# Kiểm tra môi trường offline

Giữ ảnh gốc, TADiSR, DiffTSR và UniHIR; bốn OCR giữ nguyên.
Dùng nguyên Dataset models/packages đang có trên Kaggle, không cần cập nhật asset.
RESTORATION_GPU_CHECK='off' là mặc định; chỉ chọn build_and_probe hoặc unihir_smoke khi cần kiểm tra thêm trên Kaggle.


In [ ]:
from pathlib import Path
if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle với Internet Off; hãy gắn Dataset packages/models/PDF.')

INPUT_ROOT = Path('/kaggle/input')
# Để trống để tự tìm qua manifest; nếu có nhiều Dataset cùng loại, điền đường dẫn.
PACKAGES_DIR = ''
MODELS_DIR = ''
DATA_DIR = ''
PREPARED_IMAGES = ''  # tùy chọn, thư mục có dataset.json; chỉ đọc descriptor

# Chỉ điền nếu runtime riêng ĐÃ tồn tại; GPU check tạo overlay riêng khi cần.
RUNTIME_PYTHONS = {
    # 'unihir_restoration': '/path/to/verified/environment/bin/python',
    # 'tadisr_x4': '/path/to/overlay/bin/python',
}
RUN_PROBES = True
CHECK_OPTIONS = {
    'verify_asset_hashes': False,  # mặc định chỉ stat kích thước; không đọc hàng chục GB weights
    'check_existing_runtimes': True,
    'quick_restoration_checks': False,  # True nếu chỉ cần nhóm probe restoration ngắn
    'max_runtime_checks': 8,
    'max_scan_dirs': 500,
    'max_asset_records': 12000,
    'probe_timeout_cap': 180,
    'time_budget_seconds': 900,
}
OUTPUT_DIR = Path('/kaggle/working')

# Kaggle-only; không chạy build/model trên local. Điền DATA_DIR để thử inference PDF.
RESTORATION_GPU_CHECK = 'off'  # off | build_and_probe | unihir_smoke; chủ động bật nếu cần
RESTORATION_CHECK_OPTIONS = {
    'books': [], 'pages': '1',  # [] chọn PDF đầu; giới hạn cứng 1 PDF / 1 trang / crop page
    'steps': 10, 'ref_count': 1,
    'gpu_ids': [0,1,2,3], 'placement':'sharded',
    'max_memory':'18GiB', 'activation_reserve_gib':4, 'entry_reserve_gib':2,
    'max_source_pixels': 16_000_000,
    'load_timeout_seconds': 900, 'inference_timeout_seconds': 900,
    'total_timeout_seconds': 5400,
    'runtime_dir': '/tmp/vtm_image_runtime',
}


In [ ]:
"""Read-only environment inventory; no installs, downloads, weights or OCR."""
from __future__ import annotations
from datetime import datetime, timezone
from email.parser import Parser
import importlib.metadata as metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import sysconfig
import hashlib
import time
import zipfile
from pathlib import PurePosixPath
PACKAGES = ('torch', 'torchvision', 'transformers', 'tokenizers', 'accelerate', 'safetensors', 'huggingface-hub', 'sentencepiece', 'protobuf', 'numpy', 'scipy', 'pillow', 'opencv-python', 'opencv-python-headless', 'paddlepaddle', 'paddlepaddle-gpu', 'paddleocr', 'paddlex', 'mmcv', 'mmcv-full', 'mmcv-lite', 'mmengine', 'mmdet', 'mmrotate', 'timm', 'einops', 'flash-attn', 'triton', 'pyyaml', 'pypdfium2', 'pandas', 'tqdm', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja', 'cmake', 'uv', 'diffusers', 'peft', 'omegaconf', 'pytorch-lightning', 'lightning-utilities', 'torchmetrics', 'zhconv', 'fonttools', 'decord', 'bitsandbytes', 'sentence-transformers', 'opencv-contrib-python', 'vit-pytorch', 'keras', 'h5py', 'ml-dtypes', 'optree', 'namex')
PROBES = {'torch_cuda': ('torch', 90, "import torch, json\nprint('stage: torch imported', flush=True)\ndevices = []\nfor i in range(torch.cuda.device_count()):\n    x = torch.ones((2,2), device=f'cuda:{i}')\n    value = float((x @ x).sum().item())\n    torch.cuda.synchronize(i)\n    devices.append({'index':i,'name':torch.cuda.get_device_name(i),\n        'VRAM_GiB':round(torch.cuda.get_device_properties(i).total_memory/1024**3,2),\n        'free_VRAM_GiB':round(torch.cuda.mem_get_info(i)[0]/1024**3,2),\n        'compute_capability':list(torch.cuda.get_device_capability(i)),'gpu_matmul_sum':value})\nresult = {'torch': torch.__version__, 'built_cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'devices': devices,\n          'compiled_arches': getattr(torch.cuda, 'get_arch_list', lambda: [])() if torch.cuda.is_available() else [],\n          'module_file': torch.__file__, 'native_file': torch._C.__file__,\n          'cxx11_abi': getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', None)}\nresult['gpu_count'] = len(devices)\nresult['peer_access'] = [[torch.cuda.can_device_access_peer(i,j) if i!=j else True for j in range(len(devices))] for i in range(len(devices))]\nresult['memory_is_per_device'] = True\nprint(json.dumps(result))\nassert result['cuda_available'], 'CUDA unavailable; this is not a verified GPU runtime'\n"), 'torchvision_ops': ('torchvision', 90, "import torch, torchvision, json\nfrom torchvision.ops import nms\ndevice = 'cuda:0' if torch.cuda.is_available() else 'cpu'\nboxes = torch.tensor([[0.,0.,2.,2.],[0.,0.,2.,2.]],device=device)\nkeep = nms(boxes, torch.tensor([.9,.8],device=device), .5)\nassert keep.numel() == 1\nprint(json.dumps({'torchvision': torchvision.__version__, 'nms_device': device}))\n"), 'torch_numpy_bridge': ('torch', 45, "import torch, numpy as np, json\nx = torch.from_numpy(np.zeros((2,2),dtype=np.float32))\nassert x.shape == (2,2) and x.numpy().shape == (2,2)\nprint(json.dumps({'torch':torch.__version__,'numpy':np.__version__,'numpy_bridge':'OK'}))\n"), 'numpy_cv2': ('cv2', 45, "import numpy as np, cv2, json\nfrom PIL import Image\nx = np.zeros((4,4,3),dtype=np.uint8)\nassert cv2.cvtColor(x,cv2.COLOR_RGB2GRAY).shape == (4,4)\nassert Image.fromarray(x).size == (4,4)\nprint(json.dumps({'numpy': np.__version__, 'opencv': cv2.__version__, 'array_image_bridge': 'OK'}))\n"), 'transformers_hunyuan': ('transformers', 180, "import json\nprint('stage: transformers import', flush=True)\nimport transformers\nprint('stage: AutoProcessor', flush=True)\nfrom transformers import AutoProcessor\nprint('stage: Hunyuan class', flush=True)\nfrom transformers import HunYuanVLForConditionalGeneration\nprint(json.dumps({'transformers': transformers.__version__, 'hunyuan_api': 'OK'}))\n"), 'paddle_cuda': ('paddle', 90, "import paddle, json\nresult = {'paddle': paddle.__version__, 'compiled_with_cuda': paddle.is_compiled_with_cuda()}\nresult['devices'] = paddle.device.cuda.device_count() if result['compiled_with_cuda'] else 0\nif result['devices']:\n    paddle.set_device('gpu:0')\n    x = paddle.ones((2,2))\n    result['gpu_matmul_sum'] = float(paddle.matmul(x,x).sum().item())\nprint(json.dumps(result))\n"), 'paddleocr_apis': ('paddleocr', 120, "import paddleocr, json\nfrom paddleocr import PaddleOCR, PaddleOCRVL\nprint(json.dumps({'paddleocr': paddleocr.__version__, 'PaddleOCR': 'OK', 'PaddleOCRVL': 'OK'}))\n"), 'mmcv_compiled_ops': ('mmcv', 120, "import torch, mmcv, mmcv.ops, mmdet, mmrotate, json\ndevice = 'cuda:0' if torch.cuda.is_available() else 'cpu'\nboxes = torch.tensor([[5.,5.,2.,2.,0.]],device=device)\nscores = torch.tensor([.9],device=device)\n_, keep = mmcv.ops.nms_rotated(boxes,scores,.5)\nassert keep.numel() == 1\nprint(json.dumps({'mmcv': mmcv.__version__, 'mmdet': mmdet.__version__, 'mmrotate': mmrotate.__version__, 'rotated_nms_device': device}))\n"), 'pdf_renderer': ('pypdfium2', 45, "import pypdfium2 as pdfium, json\ndoc = pdfium.PdfDocument.new()\npage = doc.new_page(32,32)\nbitmap = page.render(scale=1)\nimage = bitmap.to_pil()\nassert image.size == (32,32)\nimage.close(); bitmap.close(); page.close(); doc.close()\nprint(json.dumps({'blank_page_32x32': 'OK'}))\n"), 'torch_attention': ('torch', 90, "import torch, json\nfrom torch.nn.functional import scaled_dot_product_attention\nassert torch.cuda.is_available(), 'CUDA required for attention probe'\ntested = []\nfor dtype in [torch.float16] + ([torch.bfloat16] if torch.cuda.is_bf16_supported() else []):\n    q = torch.randn(1, 2, 16, 64, device='cuda', dtype=dtype)\n    y = scaled_dot_product_attention(q, q, q)\n    torch.cuda.synchronize()\n    assert y.shape == q.shape and bool(torch.isfinite(y).all())\n    tested.append(str(dtype))\nprint(json.dumps({'tiny_sdpa_cuda': 'OK', 'tested_dtypes': tested, 'model_inference': False}))\n"), 'diffusers_cogview4': ('diffusers', 180, "import json, diffusers\nprint('stage: CogView4 classes', flush=True)\nfrom diffusers import CogView4Transformer2DModel, CogView4Pipeline\nprint(json.dumps({'diffusers':diffusers.__version__, 'module_file':diffusers.__file__,\n 'CogView4Transformer2DModel':'OK', 'CogView4Pipeline':'OK', 'weights_loaded':False}))\n"), 'transformers_glm': ('transformers', 180, "import json, transformers\nfrom transformers import GlmModel\nprint(json.dumps({'transformers':transformers.__version__, 'GlmModel':'OK', 'weights_loaded':False}))\n"), 'peft_api': ('peft', 120, "import peft, json\nfrom peft import PeftModel\nprint(json.dumps({'peft':peft.__version__, 'module_file':peft.__file__, 'PeftModel':'OK'}))\n"), 'restoration_dependencies': ('accelerate', 90, "import importlib, json\nnames = ['accelerate','safetensors','sentencepiece','huggingface_hub','PIL','numpy']\nresult = {}\nfor name in names:\n    module = importlib.import_module(name)\n    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}\nprint(json.dumps(result))\n"), 'difftsr_dependencies': ('omegaconf', 120, "import importlib, json\nresult = {}\nfor name in ['torch','torchvision','pytorch_lightning','omegaconf','zhconv','einops','torchmetrics']:\n    module = importlib.import_module(name)\n    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}\nprint(json.dumps({'third_party_imports':result,'difftsr_model_code_tested':False}))\n"), 'flash_attention': ('flash_attn', 90, "import torch, flash_attn, json\nfrom flash_attn import flash_attn_func\nassert torch.cuda.is_available(), 'CUDA required for FlashAttention probe'\ndtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\nq = torch.randn(1,16,2,64,device='cuda',dtype=dtype)\ny = flash_attn_func(q,q,q)\ntorch.cuda.synchronize()\nassert y.shape == q.shape and bool(torch.isfinite(y).all())\nprint(json.dumps({'flash_attn':flash_attn.__version__, 'dtype':str(dtype),\n 'module_file':flash_attn.__file__, 'tiny_cuda_forward':'OK', 'model_inference':False}))\n"), 'torch_build': ('torch', 90, "import torch, json, sysconfig, pathlib\nfrom torch.utils.cpp_extension import CUDA_HOME\nprint(json.dumps({'CUDA_HOME':str(CUDA_HOME) if CUDA_HOME else None,\n 'torch_build_config':torch.__config__.show(), 'Python_h_exists':pathlib.Path(sysconfig.get_path('include'),'Python.h').is_file(),\n 'compiled_arches':torch.cuda.get_arch_list() if torch.cuda.is_available() else [], 'extension_compilation_performed':False}))\n"), 'torch_restoration_ops': ('torch', 90, "import torch, json\nfrom torch.nn import functional as F\nassert torch.cuda.is_available(), 'CUDA required for restoration tensor ops'\nx = torch.ones(1,3,16,16,device='cuda',dtype=torch.float32)\ny = F.conv2d(x,torch.ones(12,3,3,3,device='cuda'),padding=1)\ny = F.pixel_shuffle(y,2)\ny = F.interpolate(y,scale_factor=2,mode='bicubic',align_corners=False)\ntorch.cuda.synchronize()\nassert tuple(y.shape)==(1,3,64,64) and bool(torch.isfinite(y).all())\nprint(json.dumps({'conv2d_pixel_shuffle_bicubic_cuda':'OK','dtype':'float32','model_inference':False}))\n"), 'unihir_dependencies': ('accelerate', 120, "import importlib, json\nresult = {}\nfor name in ['torch','transformers','accelerate','fontTools','safetensors','einops']:\n    module = importlib.import_module(name)\n    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}\nprint(json.dumps({'third_party_imports':result,'unihir_model_code_tested':False}))\n")}
PROBE_GUARD = "import sys\nsys.dont_write_bytecode = True\ndef _vtm_diagnostic_audit(event, args):\n    if event in ('socket.connect', 'socket.getaddrinfo'):\n        raise RuntimeError('Network disabled in environment diagnostics')\nsys.addaudithook(_vtm_diagnostic_audit)\n"
DIAGNOSTIC_DEFAULTS = {'max_scan_dirs': 500, 'max_asset_records': 12000, 'verify_asset_hashes': False, 'check_existing_runtimes': True, 'quick_restoration_checks': False, 'max_runtime_checks': 4, 'probe_timeout_cap': 180, 'time_budget_seconds': 600}
RUNTIME_INVENTORY = "import sys, sysconfig, platform, json, importlib.metadata as md, importlib.util\nrecords, seen = [], set()\nfor dist in md.distributions():\n    name=dist.metadata.get('Name',''); key=name.lower().replace('_','-').replace('.','-')\n    records.append({'name':name,'version':dist.version,'active':key not in seen,\n        'location':str(dist.locate_file('')),'requires_python':dist.metadata.get('Requires-Python'),\n        'requires':dist.requires or []})\n    seen.add(key)\nmodules={}\nfor name in ['torch','torchvision','transformers','numpy','cv2','paddle','paddleocr','mmcv',\n             'diffusers','peft','accelerate','omegaconf','flash_attn','pypdfium2','fontTools',\n             'vit_pytorch','keras','h5py']:\n    try:\n        spec=importlib.util.find_spec(name)\n        modules[name]={'available':spec is not None,'origin':getattr(spec,'origin',None)}\n    except Exception as exc: modules[name]={'available':False,'error':str(exc)}\ntry:\n    from packaging.tags import sys_tags\n    tags=[str(tag) for tag in sys_tags()]\nexcept Exception: tags=[]\nprint(json.dumps({'python':sys.version,'executable':sys.executable,'prefix':sys.prefix,\n    'ABI':sysconfig.get_config_var('SOABI'),'machine':platform.machine(),\n    'include_dir':sysconfig.get_path('include'),'Python_h_exists':__import__('pathlib').Path(sysconfig.get_path('include'),'Python.h').is_file(),\n    'all_distributions':records,'module_specs':modules,'wheel_tags':tags}))\n"

def run_command(command, timeout=30):
    env = {**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'HF_HUB_DISABLE_TELEMETRY': '1', 'PIP_NO_INDEX': '1', 'USE_TF': '0', 'USE_FLAX': '0', 'PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK': 'True', 'DIFFUSERS_OFFLINE': '1', 'HF_DATASETS_OFFLINE': '1', 'PYTHONDONTWRITEBYTECODE': '1', 'PYTHONUTF8': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'MPLBACKEND': 'Agg', 'TOKENIZERS_PARALLELISM': 'false'}
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE'):
        env.pop(key, None)
    try:
        result = subprocess.run(command, capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=timeout, env=env)
        return {'status': 'ok' if result.returncode == 0 else 'failed', 'returncode': result.returncode, 'stdout': result.stdout, 'stderr': result.stderr}
    except subprocess.TimeoutExpired as exc:

        def decode(value):
            return value.decode('utf-8', errors='replace') if isinstance(value, bytes) else value or ''
        return {'status': 'timeout', 'timeout_seconds': timeout, 'stdout': decode(exc.stdout), 'stderr': decode(exc.stderr)}
    except OSError as exc:
        return {'status': 'unavailable', 'detail': str(exc)}

def read_text(path):
    try:
        return Path(path).read_text(encoding='utf-8').strip()
    except OSError:
        return None

def hardware():
    meminfo = {}
    for line in (read_text('/proc/meminfo') or '').splitlines():
        key, value = line.split(':', 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
    limit = read_text('/sys/fs/cgroup/memory.max') or read_text('/sys/fs/cgroup/memory/memory.limit_in_bytes')
    current = read_text('/sys/fs/cgroup/memory.current') or read_text('/sys/fs/cgroup/memory/memory.usage_in_bytes')
    limit = int(limit) if limit and limit.isdigit() else None
    current = int(current) if current and current.isdigit() else None
    totals = [value for value in (meminfo.get('MemTotal'), limit) if value is not None]
    total = min(totals) if totals else None
    available = meminfo.get('MemAvailable')
    if limit is not None and current is not None:
        available = min(available if available is not None else limit, max(0, limit - current))
    quota = read_text('/sys/fs/cgroup/cpu.max')
    try:
        affinity = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else None
    except OSError:
        affinity = None
    cpu_model = next((line.split(':', 1)[1].strip() for line in (read_text('/proc/cpuinfo') or '').splitlines() if line.startswith('model name')), platform.processor())
    disks = {}
    for name in ('/', '/kaggle/input', '/kaggle/working', '/tmp'):
        if Path(name).is_dir():
            disk = shutil.disk_usage(name)
            disks[name] = {'total_GiB': round(disk.total / 1024 ** 3, 2), 'free_GiB': round(disk.free / 1024 ** 3, 2)}
    return {'os': platform.platform(), 'machine': platform.machine(), 'libc': platform.libc_ver(), 'python': sys.version, 'python_executable': sys.executable, 'python_ABI': sysconfig.get_config_var('SOABI'), 'cpu_model': cpu_model, 'cpu_logical': os.cpu_count(), 'cpu_affinity': affinity, 'cpu_quota_raw': quota, 'RAM_effective_GiB': round(total / 1024 ** 3, 2) if total is not None else None, 'RAM_available_GiB': round(available / 1024 ** 3, 2) if available is not None else None, 'disks': disks}

def distributions():
    records = []
    seen = set()
    for dist in metadata.distributions():
        name = dist.metadata.get('Name', '')
        key = name.lower().replace('_', '-').replace('.', '-')
        records.append({'name': name, 'version': dist.version, 'active': key not in seen, 'location': str(dist.locate_file('')), 'requires_python': dist.metadata.get('Requires-Python'), 'requires': dist.requires or []})
        seen.add(key)
    return sorted(records, key=lambda item: item['name'].lower())

def offline_inputs(root=Path('/kaggle/input'), max_dirs=150, max_files=80):
    found, visited = ([], 0)
    if not root.is_dir():
        return {'exists': False, 'files': [], 'truncated': False}
    for parent, dirs, files in os.walk(root):
        visited += 1
        if len(Path(parent).relative_to(root).parts) >= 6:
            dirs[:] = []
        for name in sorted(files):
            if name.endswith(('.whl', '.tar.gz', '.tar.xz', '.zip', '.safetensors', '.pth')) or name in {'config.json', 'inference.yml', 'bundle_manifest.json'}:
                path = Path(parent) / name
                try:
                    size = path.stat().st_size
                except OSError:
                    continue
                found.append({'path': str(path), 'bytes': size})
                if len(found) >= max_files:
                    return {'exists': True, 'files': found, 'truncated': True, 'visited_dirs': visited}
        if visited >= max_dirs:
            return {'exists': True, 'files': found, 'truncated': True, 'visited_dirs': visited}
    return {'exists': True, 'files': found, 'truncated': False, 'visited_dirs': visited}

def diagnostic_options(options=None):
    if set(options or {}) - DIAGNOSTIC_DEFAULTS.keys():
        raise ValueError('Unknown diagnostic options')
    result = {**DIAGNOSTIC_DEFAULTS, **(options or {})}
    for key, default in DIAGNOSTIC_DEFAULTS.items():
        if isinstance(default, bool):
            if type(result[key]) is not bool:
                raise ValueError(f'{key} must be bool')
        elif type(result[key]) is not int or result[key] < 1:
            raise ValueError(f'{key} must be a positive integer')
    return result

def diagnostic_json(path):
    path = Path(path)
    if path.stat().st_size > 16 * 1024 * 1024:
        raise ValueError('Diagnostic metadata exceeds 16 MiB; not read')
    return json.loads(path.read_text(encoding='utf8'))

def diagnostic_payload(output):
    for line in reversed(output.splitlines()):
        try:
            value = json.loads(line)
            if isinstance(value, dict):
                return value
        except ValueError:
            continue
    return None

def discover_asset_roots(input_root, configured, *, max_dirs=500):
    """Bounded discovery prunes weights/objects/source trees; explicit paths take priority."""
    markers = {'models': 'models_manifest.json', 'packages': 'bundle_manifest.json', 'prepared_images': 'dataset.json'}
    candidates = {key: [] for key in markers}
    root, visited, truncated = (Path(input_root), 0, False)
    if root.is_dir():
        for parent, dirs, files in os.walk(root):
            visited += 1
            for kind, marker in markers.items():
                if marker in files:
                    candidates[kind].append(str(Path(parent).resolve()))
            dirs[:] = sorted((d for d in dirs if d not in {'sources', 'wheels', 'objects', 'snapshots', '__pycache__', '.cache'}))
            if len(Path(parent).relative_to(root).parts) >= 6:
                dirs[:] = []
            if visited >= max_dirs:
                truncated = True
                break
    found = {}
    for kind, marker in markers.items():
        explicit = configured.get(kind)
        if explicit:
            path = Path(explicit).resolve()
            if path.is_file():
                path = path.parent.parent if path.parent.name == 'snapshots' else path.parent
            for child in (path / kind, path / 'kaggle_upload' / kind):
                if not (path / marker).is_file() and (child / marker).is_file():
                    path = child
                    break
            found[kind] = {'path': str(path), 'selection': 'explicit', 'exists': path.is_dir()}
        elif len(candidates[kind]) == 1:
            found[kind] = {'path': candidates[kind][0], 'selection': 'discovered', 'exists': True}
        else:
            found[kind] = {'path': None, 'selection': 'ambiguous' if candidates[kind] else 'not_found', 'candidates': candidates[kind], 'exists': False}
    found['discovery'] = {'visited_dirs': visited, 'truncated': truncated}
    found['pdfs'] = {'path': str(configured['pdfs']) if configured.get('pdfs') else None, 'selection': 'explicit' if configured.get('pdfs') else 'not_configured'}
    return found

def diagnostic_relative(root, relative):
    value = str(relative)
    parts = PurePosixPath(value)
    if ':' in value or '\\' in value or parts.is_absolute() or ('..' in parts.parts):
        raise ValueError('Invalid portable asset path')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError('Asset reference escapes its root')
    return path

def inspect_asset_records(root, records, *, limit=12000, verify_hashes=False):
    """Default checks file presence and size only, without opening model weights."""
    expanded = []
    for record in records:
        if record.get('type') == 'directory':
            expanded.extend((dict(child, path=record['path'].rstrip('/') + '/' + child['path']) for child in record['files']))
        else:
            expanded.append(record)
    records = expanded
    missing, mismatches, checked, bytes_declared, hashes_verified = ([], [], 0, 0, 0)
    optional_bytecode = []
    for record in records[:limit]:
        relative = record.get('path', '')
        try:
            path = diagnostic_relative(root, relative)
            if '__pycache__' in PurePosixPath(relative).parts and path.suffix in ('.pyc', '.pyo'):
                optional_bytecode.append(relative)
                continue
            if not path.is_file():
                missing.append(relative)
                continue
            actual = path.stat().st_size
            if record.get('bytes') is not None and actual != record['bytes']:
                mismatches.append({'path': relative, 'kind': 'size', 'actual': actual, 'expected': record['bytes']})
            elif verify_hashes and record.get('sha256'):
                digest = hashlib.sha256()
                with path.open('rb') as stream:
                    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                        digest.update(block)
                if digest.hexdigest() != record['sha256']:
                    mismatches.append({'path': relative, 'kind': 'sha256'})
                else:
                    hashes_verified += 1
            checked += 1
            bytes_declared += record.get('bytes', actual)
        except (OSError, ValueError) as exc:
            mismatches.append({'path': relative, 'kind': 'unreadable_or_invalid', 'detail': str(exc)})
    return {'status': 'issues_found' if missing or mismatches else 'analysis_incomplete' if len(records) > limit else 'files_present_sizes_match', 'declared_files': len(records), 'checked_files': checked, 'missing': missing, 'mismatches': mismatches, 'declared_bytes_checked': bytes_declared, 'truncated': len(records) > limit, 'hashes_requested': verify_hashes, 'hashes_verified_files': hashes_verified, 'optional_bytecode_records': optional_bytecode, 'model_inference_verified': False}

def inspect_runtime_kernel_requirements(plan):
    """Compare the measured kernel to a plan without importing ML packages."""
    mismatches = []
    for name, expected in plan.get('kernel_reuse', {}).items():
        try:
            actual = metadata.version(name)
        except metadata.PackageNotFoundError:
            actual = None
        if actual != expected:
            mismatches.append({'package': name, 'expected': expected, 'actual': actual})
    abi_matches = str(sys.version_info.major) + str(sys.version_info.minor) == str(plan.get('python')) and sys.platform == 'linux' and (platform.machine() == 'x86_64')
    return {'state': 'kernel_metadata_match' if not mismatches and abi_matches else 'kernel_metadata_differs', 'mismatches': mismatches, 'python': platform.python_version(), 'python_platform_match': abi_matches, 'expected_python': plan.get('python'), 'platform': sysconfig.get_platform(), 'model_inference_verified': False, 'note': 'Metadata only; missing overlay packages in the kernel are allowed. No installation or model load.'}

def diagnostic_wheel_metadata(path):
    with zipfile.ZipFile(path) as archive:
        names = [n for n in archive.namelist() if n.count('/') == 1 and n.endswith('.dist-info/METADATA')]
        if len(names) != 1 or archive.getinfo(names[0]).file_size > 4 * 1024 ** 2:
            raise ValueError('Invalid or oversized wheel METADATA')
        body = Parser().parsestr(archive.read(names[0]).decode('utf8'))
    return {'name': body['Name'], 'version': body['Version'], 'requires_python': body.get('Requires-Python'), 'requires': body.get_all('Requires-Dist', [])}

def inspect_runtime_dependency_metadata(root, plan):
    """Resolve the planned overlay against this kernel without pip or imports."""
    try:
        from packaging.requirements import Requirement
        from packaging.utils import canonicalize_name
        from packaging.specifiers import SpecifierSet
        from packaging.markers import default_environment
        selected, problems, checked, extras = ({}, [], [], [])
        for record in plan.get('wheels', []):
            item = diagnostic_wheel_metadata(diagnostic_relative(root, record['path']))
            if record.get('name') and canonicalize_name(record['name']) != canonicalize_name(item['name']) or (record.get('version') and record['version'] != item['version']):
                problems.append({'path': record['path'], 'state': 'wheel_metadata_differs_from_plan'})
            selected[canonicalize_name(item['name'])] = dict(item, source='staged_overlay')
        pending = list(plan.get('requirements', []))
        roots_explicit = bool(pending)
        if not pending:
            pending = [name + '==' + version for name, version in plan.get('kernel_reuse', {}).items()]
            pending.extend((item['name'] + '==' + item['version'] for item in selected.values()))
        seen = set()
        marker_environment = {**default_environment(), 'extra': ''}
        while pending and len(seen) < 1500:
            raw = pending.pop()
            if raw in seen:
                continue
            seen.add(raw)
            requirement = Requirement(raw)
            if requirement.marker and (not requirement.marker.evaluate(marker_environment)):
                continue
            if requirement.extras:
                extras.append(raw)
            name = canonicalize_name(requirement.name)
            if name not in selected:
                try:
                    dist = metadata.distribution(name)
                    selected[name] = {'name': name, 'version': dist.version, 'source': 'active_kernel', 'requires': dist.requires or [], 'requires_python': dist.metadata.get('Requires-Python')}
                except metadata.PackageNotFoundError:
                    problems.append({'requirement': raw, 'state': 'missing_in_kernel_and_planned_wheels'})
                    continue
            item = selected[name]
            if not requirement.specifier.contains(item['version'], prereleases=True):
                problems.append({'requirement': raw, 'state': 'version_conflict', 'actual': item['version'], 'source': item['source']})
            if item['requires_python'] and (not SpecifierSet(item['requires_python']).contains(platform.python_version(), prereleases=True)):
                problems.append({'package': name, 'state': 'python_version_conflict', 'requires_python': item['requires_python']})
            checked.append({'requirement': raw, 'version': item['version'], 'source': item['source']})
            pending.extend(item['requires'])
        native = plan.get('native_sources', [])
        return {'state': 'issues_found' if problems else 'partial_metadata' if pending or native or extras else 'metadata_closure_satisfied', 'problems': problems, 'checked_requirements': checked, 'truncated': bool(pending), 'optional_extras_not_evaluated': extras, 'root_requirements_source': 'explicit_plan' if roots_explicit else 'planned_kernel_reuse_and_wheels', 'native_source_metadata_not_resolved': [r['path'] for r in native], 'note': 'Default dependency markers only; no optional extras, installation, native build or model execution verified.'}
    except (ImportError, OSError, ValueError, KeyError, zipfile.BadZipFile) as error:
        return {'state': 'metadata_analysis_unavailable', 'detail': str(error)}

def inspect_offline_assets(roots, settings):
    reports = {}

    def inspect(name, root, manifest, required=()):
        result = {'manifest': str(manifest), 'required_missing': [], 'state': 'manifest_missing'}
        try:
            data = diagnostic_json(manifest)
            result.update(inspect_asset_records(root, data.get('files', []), limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes']))
            result['state'] = result['status']
            result['required_missing'] = [r for r in required if not diagnostic_relative(root, r).is_file()]
            indexed = {r.get('path') for r in data.get('files', [])}
            result['required_not_in_inventory'] = [r for r in required if r not in indexed]
            if result['required_missing'] or result['required_not_in_inventory']:
                result['state'] = 'missing_required_asset'
            if name == 'tadisr':
                shards = []
                for prefix, index in (('transformer', 'diffusion_pytorch_model.safetensors.index.json'), ('text_encoder', 'model.safetensors.index.json')):
                    index_path = Path(root) / 'CogView4' / prefix / index
                    if index_path.is_file():
                        body = diagnostic_json(index_path)
                        for part in set(body['weight_map'].values()):
                            if not diagnostic_relative(Path(root) / 'CogView4' / prefix, part).is_file() or f'CogView4/{prefix}/{part}' not in indexed:
                                shards.append(prefix + '/' + part)
                result['missing_indexed_shards'] = shards
                if shards:
                    result['state'] = 'missing_required_asset'
                result['prompt_embeddings_present'] = (Path(root) / 'CogView4/saved_prompt_tokens_nocfg.pt').is_file()
                result['prompt_note'] = 'Notebook 1 can derive the fixed prompt offline from verified base assets'
        except (OSError, ValueError, KeyError, TypeError) as exc:
            result['detail'] = str(exc)
            if Path(manifest).is_file():
                result['state'] = 'invalid_manifest'
        reports[name] = result
    models = roots.get('models', {}).get('path')
    if models:
        models = Path(models)
        inspect('ocr_models', models, models / 'models_manifest.json', ['tonggu/layout/finetune_infer.pth'])
        requirements = {'tadisr': ['tadisr_cogview4_realce_17500.pkl'] + ['CogView4/' + n for n in ('model_index.json', 'vae/config.json', 'vae/diffusion_pytorch_model.safetensors', 'transformer/config.json', 'transformer/diffusion_pytorch_model.safetensors.index.json', 'text_encoder/config.json', 'text_encoder/model.safetensors.index.json', 'tokenizer/tokenizer.json', 'tokenizer/tokenizer_config.json', 'tokenizer/special_tokens_map.json', 'scheduler/scheduler_config.json')], 'unihir': ['model.safetensors', 'ae.safetensors', 'llm_config.json', 'vit_config.json', 'tokenizer_config.json', 'tokenizer.json', 'vocab.json', 'merges.txt']}
        for name, required in requirements.items():
            root = models / 'image_restoration' / name
            inspect(name, root, root / 'assets.json', required)
        reports['difftsr'] = {'state': 'excluded_by_configuration', 'required': False, 'assets_still_present': (models / 'image_restoration/difftsr').exists()}
    packages = roots.get('packages', {}).get('path')
    if packages:
        packages = Path(packages)
        inspect('ocr_packages', packages, packages / 'bundle_manifest.json', ['paddle_manifest.json', 'tonggu_manifest.json', 'system_assets_manifest.json'])
        restoration = packages / 'image_restoration'
        try:
            plan = diagnostic_json(restoration / 'manifests/tadisr_runtime.json')
            reports['tadisr_packages'] = inspect_asset_records(restoration, plan.get('wheels', []), limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
            reports['tadisr_kernel_compatibility'] = inspect_runtime_kernel_requirements(plan)
            reports['tadisr_dependency_metadata'] = inspect_runtime_dependency_metadata(restoration, plan)
        except (OSError, ValueError, TypeError) as exc:
            reports['tadisr_packages'] = {'status': 'manifest_unavailable', 'detail': str(exc)}
        for name in ('difftsr', 'unihir'):
            try:
                plan = diagnostic_json(restoration / f'manifests/{name}_runtime.json')
                reports[name + '_runtime_manifest'] = plan
                reports[name + '_kernel_compatibility'] = inspect_runtime_kernel_requirements(plan)
                reports[name + '_dependency_metadata'] = inspect_runtime_dependency_metadata(restoration, plan)
                records = plan.get('wheels', []) + plan.get('native_sources', [])
                if plan.get('bootstrap'):
                    records.append(plan['bootstrap'])
                reports[name + '_runtime_assets'] = inspect_asset_records(restoration, records, limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
            except (OSError, ValueError) as exc:
                reports[name + '_runtime_manifest'] = {'state': 'manifest_unavailable', 'detail': str(exc)}
        try:
            catalog = diagnostic_json(restoration / 'manifests/asset_catalog.json')
            for name, group in catalog['groups'].items():
                records = [dict(r, path=r['path'].removeprefix('packages/')) for r in group['files'] if r['kind'] == 'source']
                if not records:
                    continue
                reports[name + '_sources'] = inspect_asset_records(packages, records, limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
                reports[name + '_sources'].update(revision=group['revision'], license=group['license'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            reports['restoration_catalog_sources'] = {'state': 'manifest_unavailable', 'detail': str(exc)}
        try:
            records = diagnostic_json(restoration / 'manifests/embedded_sources.json')
            records = [dict(r, path=r['path'].removeprefix('packages/')) for r in records]
            reports['embedded_upstream_sources'] = inspect_asset_records(packages, records, limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            reports['embedded_upstream_sources'] = {'state': 'manifest_unavailable', 'detail': str(exc)}
    prepared = roots.get('prepared_images', {}).get('path')
    if prepared:
        try:
            data = diagnostic_json(Path(prepared) / 'dataset.json')
            reports['prepared_images'] = {'format': data.get('format'), 'version': data.get('version'), 'snapshot_id': data.get('snapshot_id'), 'pages': len(data.get('pages', [])), 'restoration': data.get('restoration'), 'state': 'descriptor_read_only', 'pixels_verified': False}
        except (OSError, ValueError, TypeError) as exc:
            reports['prepared_images'] = {'state': 'descriptor_unavailable', 'detail': str(exc)}
    return reports

def runtime_candidates(configured, *, include_existing=True):
    result = {str(name): {'python': str(path), 'source': 'explicit'} for name, path in (configured or {}).items() if path}
    if include_existing:
        for path in (Path('/tmp/vtm_kaggle_runtime/runtime_config.json'), Path('/kaggle/working/runtime_config.json')):
            try:
                data = diagnostic_json(path)
                for name in ('paddle', 'tonggu'):
                    if data.get(name + '_python') and name not in result:
                        result[name] = {'python': data[name + '_python'], 'source': str(path)}
            except (OSError, ValueError, TypeError):
                pass
        path = Path('/kaggle/working/.image_runtime/preflight.json')
        try:
            for name, item in diagnostic_json(path).items():
                if isinstance(item, dict) and item.get('python') and (name not in result):
                    result[name] = {'python': item['python'], 'source': str(path)}
        except (OSError, ValueError, TypeError):
            pass
    return result

def runtime_probe_names(name):
    common = ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'pdf_renderer', 'torch_build']
    if name in {'paddle', 'ppocr'}:
        return ['numpy_cv2', 'pdf_renderer', 'paddle_cuda', 'paddleocr_apis']
    if name == 'tonggu':
        return common + ['torchvision_ops', 'mmcv_compiled_ops']
    if name == 'difftsr':
        return common + ['torchvision_ops', 'difftsr_dependencies']
    if name in {'unihir', 'unihir_restoration'}:
        return common + ['unihir_dependencies', 'flash_attention', 'torch_attention']
    if name in {'tadisr', 'tadisr_x4'}:
        return common + ['diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies', 'torch_restoration_ops']
    if name in {'realesrgan', 'realesrgan_x4'}:
        return common + ['torch_restoration_ops']
    if name.startswith(('docentr', 'degan', 'learned_documents')):
        return common + ['vit_pytorch_api', 'keras_torch_hdf5']
    return common

def inspect_staged_wheels(roots, runtime_inventories):
    packages = roots.get('packages', {}).get('path')
    if not packages:
        return {'state': 'packages_not_configured', 'wheels': []}
    try:
        from packaging.utils import parse_wheel_filename
        result = []
        candidates = sorted(Path(packages).rglob('*.whl'))
        for path in candidates[:2000]:
            try:
                _, _, _, tags = parse_wheel_filename(path.name)
                wheel_tags = {str(tag) for tag in tags}
                result.append({'file': str(path), 'wheel_tags': sorted(wheel_tags), 'metadata': diagnostic_wheel_metadata(path), 'compatible_with_observed_interpreters': {name: bool(wheel_tags & set(inv['wheel_tags'])) if inv.get('wheel_tags') else None for name, inv in runtime_inventories.items()}})
            except (OSError, ValueError, zipfile.BadZipFile) as exc:
                result.append({'file': str(path), 'state': 'invalid_wheel_metadata_or_filename', 'detail': str(exc)})
        return {'state': 'tag_and_metadata_comparison', 'wheels': result, 'total_wheels': len(candidates), 'truncated': len(candidates) > 2000, 'note': 'A cp310 wheel failing kernel cp313 tags can still belong to a separate cp310 runtime; tags do not prove CUDA support'}
    except ImportError:
        return {'state': 'packaging_module_unavailable', 'wheels': []}

def readiness_summary(report):
    """Per-test observations only: never promote these diagnostics to model readiness."""
    groups = {'notebook_1_classical': ['numpy_cv2', 'pdf_renderer'], 'notebook_1_tadisr': ['torch_cuda', 'diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies'], 'notebook_3_hunyuan': ['torch_cuda', 'torch_numpy_bridge', 'transformers_hunyuan'], 'notebook_3_paddle_in_kernel': ['paddle_cuda', 'paddleocr_apis'], 'notebook_3_tonggu_in_kernel': ['torch_cuda', 'mmcv_compiled_ops'], 'notebook_1_unihir': ['torch_cuda', 'unihir_dependencies', 'flash_attention']}
    result = {}
    for name, probes in groups.items():
        states = {p: report['probes'].get(p, {}).get('status', 'not_run') for p in probes}
        result[name] = {'kernel_probe_states': states, 'model_inference_verified': False, 'note': 'Kernel imports are separate from staged assets and private runtimes. Passing tiny ops does not verify full model inference.'}
    return result

def collect_environment(*, run_probes=True, input_root=Path('/kaggle/input'), configured_paths=None, runtime_pythons=None, options=None, checkpoint_path=None):
    settings = diagnostic_options(options)
    started = time.monotonic()
    inventory = distributions()
    versions = {}
    for item in inventory:
        name = item['name'].lower().replace('_', '-')
        if name not in versions or item.get('active') is True:
            versions[name] = item['version']
    roots = discover_asset_roots(input_root, configured_paths or {}, max_dirs=settings['max_scan_dirs'])
    report = {'schema': 1, 'diagnostics_version': 2, 'collected_at_UTC': datetime.now(timezone.utc).isoformat(), 'hardware': hardware(), 'packages': {name: versions.get(name) for name in PACKAGES}, 'all_distributions': inventory, 'probes': {}, 'tools': {}, 'offline_inputs': offline_inputs(Path(input_root)), 'prior_vtm_runtime_exists': Path('/tmp/vtm_kaggle_runtime/runtime_config.json').is_file(), 'asset_roots': roots, 'options': settings, 'assets': inspect_offline_assets(roots, settings), 'runtimes': {}, 'complete': False, 'status': 'collecting', 'execution_limits': [], 'model_inference_performed': False, 'configuration': {'input_root': str(input_root), 'configured_paths': {k: str(v) for k, v in (configured_paths or {}).items()}, 'runtime_pythons': {k: str(v) for k, v in (runtime_pythons or {}).items()}}, 'build_inventory': {'Python_include': sysconfig.get_path('include'), 'Python_h_exists': Path(sysconfig.get_path('include'), 'Python.h').is_file(), 'sysconfig': {k: sysconfig.get_config_var(k) for k in ('CC', 'CXX', 'LDSHARED', 'LIBDIR', 'EXT_SUFFIX', 'SOABI')}, 'selected_environment': {k: os.environ.get(k) for k in ('CUDA_HOME', 'CUDA_PATH', 'CC', 'CXX', 'CUDACXX', 'MAX_JOBS', 'CUDA_VISIBLE_DEVICES')}}, 'note': 'Inventory phase only: metadata/imports/tiny ops, no installation or model loading in this phase. Optional build/inference results appear separately in restoration_gpu_check. Assets use size checks unless hashes explicitly enabled.'}

    def checkpoint():
        report['elapsed_seconds'] = round(time.monotonic() - started, 3)
        if checkpoint_path:
            path = Path(checkpoint_path)
            if path.resolve().is_relative_to(Path('/kaggle/input').resolve()):
                raise ValueError('Diagnostic report must not be written to Kaggle Input')
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_name(path.name + '.tmp')
            temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf8')
            temporary.replace(path)

    def bounded(command, timeout=30):
        remaining = settings['time_budget_seconds'] - (time.monotonic() - started)
        if remaining < 1:
            report['complete'] = False
            if 'time_budget_seconds' not in report['execution_limits']:
                report['execution_limits'].append('time_budget_seconds')
            return {'status': 'not_run_budget', 'detail': 'Diagnostic time budget exhausted; partial report retained'}
        return run_command(command, min(timeout, settings['probe_timeout_cap'], max(1, int(remaining))))
    checkpoint()
    if not run_probes:
        report['readiness'] = readiness_summary(report)
        report.update(complete=True, status='inventory_only')
        checkpoint()
        return report
    commands = {'gpu': ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free,driver_version', '--format=csv,noheader'], 'nvcc': ['nvcc', '--version'], 'g++': ['g++', '--version'], 'cmake': ['cmake', '--version'], 'ninja': ['ninja', '--version'], 'gcc': ['gcc', '--version'], 'ldd': ['ldd', '--version']}
    for name, command in commands.items():
        report['tools'][name] = bounded(command)
        checkpoint()
    report['pip_check'] = bounded([sys.executable, '-m', 'pip', 'check'], timeout=60)
    report['kernel_runtime_inventory'] = bounded([sys.executable, '-c', PROBE_GUARD + RUNTIME_INVENTORY], timeout=60)
    kernel_inventory = diagnostic_payload(report['kernel_runtime_inventory'].get('stdout', '')) or {}
    observed = {'kernel': kernel_inventory}
    report['kernel_runtime_inventory']['data'] = kernel_inventory
    checkpoint()
    for name, (module, timeout, code) in PROBES.items():
        if name == 'difftsr_dependencies':
            continue
        if settings['quick_restoration_checks'] and name not in {'torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'pdf_renderer', 'torch_attention', 'flash_attention', 'torch_restoration_ops'}:
            continue
        print(f'Checking {name} (timeout {timeout}s)...', flush=True)
        try:
            present = importlib.util.find_spec(module) is not None
        except (ImportError, ValueError):
            present = False
        result = bounded([sys.executable, '-c', PROBE_GUARD + code], timeout) if present else {'status': 'not_installed'}
        result['data'] = diagnostic_payload(result.get('stdout', ''))
        report['probes'][name] = result
        print(f"  {result['status']}", flush=True)
        checkpoint()
    candidates = runtime_candidates(runtime_pythons, include_existing=settings['check_existing_runtimes'])
    seen_python = {str(Path(sys.executable).resolve()): 'kernel'}
    checked = 0
    for name, candidate in candidates.items():
        python = str(Path(candidate['python']).expanduser().resolve())
        state = {**candidate, 'python': python, 'model_inference_verified': False, 'probes': {}}
        report['runtimes'][name] = state
        if python in seen_python:
            state.update(status='shared_interpreter', shares_with=seen_python[python])
            checkpoint()
            continue
        if checked >= settings['max_runtime_checks']:
            state['status'] = 'not_run_runtime_limit'
            report['complete'] = False
            if 'max_runtime_checks' not in report['execution_limits']:
                report['execution_limits'].append('max_runtime_checks')
            continue
        if not Path(python).is_file():
            state['status'] = 'interpreter_missing'
            continue
        checked += 1
        seen_python[python] = name
        print(f'Checking separate runtime {name}: {python}', flush=True)
        state['inventory'] = bounded([python, '-c', PROBE_GUARD + RUNTIME_INVENTORY], timeout=60)
        detail = diagnostic_payload(state['inventory'].get('stdout', ''))
        if state['inventory']['status'] != 'ok' or detail is None:
            state['status'] = 'inventory_unavailable'
            checkpoint()
            continue
        state['inventory']['data'] = detail
        observed[name] = detail
        state['pip_check'] = bounded([python, '-m', 'pip', 'check'], timeout=60)
        for probe_name in runtime_probe_names(name):
            module, timeout, code = PROBES[probe_name]
            present = detail.get('module_specs', {}).get(module, {}).get('available', False)
            print(f'  {name}/{probe_name}', flush=True)
            probe = bounded([python, '-c', PROBE_GUARD + code], timeout) if present else {'status': 'not_installed'}
            probe['data'] = diagnostic_payload(probe.get('stdout', ''))
            state['probes'][probe_name] = probe
            checkpoint()
        state['status'] = 'probed; model_inference_unverified'
        checkpoint()
    report['staged_wheel_tags'] = inspect_staged_wheels(roots, observed)
    report['readiness'] = readiness_summary(report)
    report['limitations'] = ['Missing kernel packages may be supplied by a separate staged runtime.', 'Wheel ABI tags and tiny GPU ops do not verify full model execution or restoration fidelity.', 'This inventory phase does not execute model weights.', 'Optional restoration GPU checks are reported separately from inventory; no network downloads.']
    report['complete'] = not report['execution_limits']
    report['status'] = 'completed_with_probe_results' if report['complete'] else 'partial_limits'
    checkpoint()
    return report

def save_report(report, output_dir):
    output = Path(output_dir)
    if output.resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Output must be outside Kaggle Input')
    output.mkdir(parents=True, exist_ok=True)
    json_path = output / 'kaggle_environment_report.json'
    text_path = output / 'kaggle_environment_report.txt'
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    lines = [json.dumps(report['hardware'], ensure_ascii=False, indent=2), '\nPackages:']
    lines.extend((f"{name:24s} {version or 'not installed'}" for name, version in report['packages'].items()))
    for group in ('tools', 'probes'):
        for name, result in report[group].items():
            lines.append(f'\n[{name}]\n' + json.dumps(result, ensure_ascii=False, indent=2))
    lines.append('\n[pip_check]\n' + json.dumps(report.get('pip_check'), ensure_ascii=False, indent=2))
    for name in ('asset_roots', 'assets', 'runtimes', 'build_inventory', 'readiness', 'limitations', 'restoration_gpu_check'):
        if name in report:
            lines.append(f'\n[{name}]\n' + json.dumps(report[name], ensure_ascii=False, indent=2))
    text_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return (json_path, text_path)



In [ ]:
"""Kaggle-only restoration check using notebook 1's exact offline implementation.

Inventory remains read-only. The separately enabled GPU check may create a
private runtime, compile FlashAttention and run one UniHIR page. No downloads,
OCR, training, kernel package replacement, or corpus processing.
"""
from __future__ import annotations

import ast
import base64
from contextlib import contextmanager, ExitStack
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import time
import traceback
from types import ModuleType
import uuid
import zipfile
import zlib


FLASH_GPU_PROBE = '''import json, torch, flash_attn
from flash_attn import flash_attn_func, flash_attn_varlen_func
assert torch.cuda.is_available(), 'CUDA unavailable'
torch.manual_seed(42)
devices=[]
for index in range(torch.cuda.device_count()):
    with torch.cuda.device(index):
        q,k,v=[torch.randn(1,16,2,64,device=f'cuda:{index}',dtype=torch.bfloat16) for _ in range(3)]
        y=flash_attn_func(q,k,v,dropout_p=0.,causal=False)
        reference=torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),dropout_p=0.).transpose(1,2)
        cu=torch.tensor([0,16],device=f'cuda:{index}',dtype=torch.int32)
        variable=flash_attn_varlen_func(q[0],k[0],v[0],cu,cu,16,16,dropout_p=0.,causal=False)
        torch.cuda.synchronize(index)
        assert y.shape==q.shape and variable.shape==q[0].shape
        assert bool(torch.isfinite(y).all()) and bool(torch.isfinite(variable).all())
        assert torch.allclose(y.float(),reference.float(),atol=.03,rtol=.03), 'FlashAttention differs from SDPA reference'
        assert torch.allclose(variable.float(),y[0].float(),atol=.03,rtol=.03), 'Variable-length kernel differs'
        devices.append(dict(index=index,name=torch.cuda.get_device_name(index),
            max_abs_difference=float((y.float()-reference.float()).abs().max().item())))
print(json.dumps(dict(status='passed',torch=torch.__version__,cuda=torch.version.cuda,
    devices=devices,capability=list(torch.cuda.get_device_capability(0)),
    flash_attention=flash_attn.__version__,module_file=flash_attn.__file__,
    fixed_and_variable_length_verified=True,finite=True,
    weights_loaded=False)))
'''


def check_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf8')
    temporary.replace(path)


def check_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()


def check_offline_env():
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE'):env.pop(key,None)
    env.update(PYTHONNOUSERSITE='1',PYTHONUTF8='1',PIP_NO_INDEX='1',PIP_DISABLE_PIP_VERSION_CHECK='1',
        HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_DATASETS_OFFLINE='1',DIFFUSERS_OFFLINE='1',
        HF_HUB_DISABLE_TELEMETRY='1',MPLBACKEND='Agg',TOKENIZERS_PARALLELISM='false')
    return env


def check_tail(path, maximum=2000):
    try:
        with Path(path).open('rb') as stream:
            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))
            return stream.read().decode('utf8',errors='replace')
    except OSError:return ''


def check_stop(process):
    if process.poll() is not None:
        return
    if not getattr(process, '_vtm_owns_group', False):
        process.kill()
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    process.wait()


def check_logged_process(command, log, *, timeout, env=None, report_path=None, new_session=True):
    """One process group; preserve bounded log tails and kill descendants on timeout."""
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    last_notice = started
    process = None
    with log.open('w', encoding='utf8') as output:
        try:
            process = subprocess.Popen(list(map(str, command)), stdout=output, stderr=subprocess.STDOUT, env=env or check_offline_env(), start_new_session=new_session, creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
            process._vtm_owns_group = new_session
            while process.poll() is None:
                elapsed = time.monotonic() - started
                if elapsed > timeout:
                    check_stop(process)
                    return dict(status='timeout', seconds=round(elapsed, 3), log=str(log), tail=check_tail(log))
                if time.monotonic() - last_notice >= 15:
                    phase = 'running'
                    if report_path and Path(report_path).is_file():
                        try:
                            phase = json.loads(Path(report_path).read_text(encoding='utf8')).get('phase', phase)
                        except (OSError, ValueError):
                            pass
                    tail = check_tail(log, 1000).splitlines()
                    native = next((line for line in reversed(tail) if line.startswith('FlashAttention build:')), '')
                    print(f'UniHIR check: {phase}, {elapsed:.0f}s; log: {log}' + (' | ' + native if native else ''), flush=True)
                    last_notice = time.monotonic()
                time.sleep(0.2)
            return dict(status='ok' if process.returncode == 0 else 'failed', returncode=process.returncode, seconds=round(time.monotonic() - started, 3), log=str(log), tail=check_tail(log))
        except BaseException:
            if process is not None:
                check_stop(process)
            raise


def check_payload(payload):
    raw=zlib.decompress(base64.b85decode(payload['data']))
    if hashlib.sha256(raw).hexdigest()!=payload['sha256']:raise ValueError('Notebook 1 definitions checksum mismatch')
    value=json.loads(raw)
    if value.get('format')!='vtm-notebook1-restoration-check-v1':raise ValueError('Unexpected definition bundle')
    return value


def check_native_receipt(python):
    folder=Path(python).parent.parent
    data=json.loads((folder/'install.json').read_text(encoding='utf8'))
    verified=[]
    for record in data.get('native',[]):
        wheel=folder/'native_build/wheels'/record['wheel']
        if not wheel.is_file() or check_hash(wheel)!=record['sha256']:
            raise ValueError('Compiled wheel missing or changed: '+str(wheel))
        verified.append(dict(record,path=str(wheel)))
    if not verified:raise ValueError('No compiled FlashAttention wheel receipt')
    return dict(status='built_or_reused_verified_wheel',native_wheels=verified,
                note='The separate CUDA probe executes the currently imported extension.')


def check_collect_logs(runtime, output):
    """Export logs/receipts only; never copy runtime binaries or model weights."""
    runtime,output=Path(runtime),Path(output)
    if not runtime.is_dir():return
    spec=output/'spec.json'
    cutoff=spec.stat().st_mtime if spec.is_file() else 0
    active_native=[]
    try:
        current=json.loads((output/'report.json').read_text(encoding='utf8'))
        active_native=[Path(r['path']).parent.parent for r in
            current.get('phases',{}).get('runtime_build',{}).get('receipt',{}).get('native_wheels',[])]
    except (OSError,ValueError):pass
    for pattern in ('build.log','build_config.json','build_progress.json','build.ninja','.ninja_log',
                    'install.json','preflight.json','worker.log','ready.json','response.json'):
        sources=sorted(runtime.rglob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)[:32]
        for source in sources:
            if source.is_symlink() or source.stat().st_size>10*1024**2:continue
            if source.stat().st_mtime<cutoff and not (pattern in ('build.log','build_config.json','build_progress.json','build.ninja','.ninja_log','install.json') and
                    any(source.is_relative_to(p.parent) for p in active_native)):continue
            if pattern in ('build.ninja','.ninja_log'):
                short=hashlib.sha256(str(source.relative_to(runtime)).encode()).hexdigest()[:12]
                target=output/'runtime_logs/native_debug'/(short+'_'+pattern.lstrip('.')+'.txt')
            else:
                target=output/'runtime_logs'/source.relative_to(runtime)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)


def check_inference_evidence(summary, validation):
    metrics=[m for m in summary.get('measurements',[]) if m.get('variant')=='unihir_restoration']
    if summary.get('failures') or not metrics or summary.get('variants_reused'):
        raise ValueError('Fresh, successful UniHIR inference was not established')
    for item in metrics:
        worker=item.get('worker',{})
        if worker.get('state')!='ok' or worker.get('peak_vram_bytes',0)<=0:
            raise ValueError('Missing successful GPU worker evidence')
    if not validation.get('restoration',{}).get('validated'):
        raise ValueError('Output manifest/pixels/geometry validation incomplete')
    return metrics


@contextmanager
def check_shared_definitions(sources):
    """Execute notebook definitions in a registered module, including dataclasses."""
    name = 'vtm_notebook1_check_definitions_' + uuid.uuid4().hex
    module = ModuleType(name)
    sys.modules[name] = module
    try:
        for index, source in enumerate(sources):
            exec(compile(source, f'notebook1-shared:{index}', 'exec'), module.__dict__)
        yield module.__dict__
    finally:
        if sys.modules.get(name) is module:
            del sys.modules[name]


def restoration_check_child(spec_path):
    """Executed only by the Kaggle job supervisor, never by notebook import."""
    spec=json.loads(Path(spec_path).read_text(encoding='utf8'))
    output=Path(spec['output']);runtime=Path(spec['runtime']);report_path=output/'report.json'
    report=dict(format='vtm-kaggle-restoration-gpu-check',version=1,status='running',phase='hardware',
        mode=spec['mode'],native_build_verified=False,flash_attention_cuda_verified=False,
        model_weights_loaded=False,model_inference_attempted=False,model_inference_verified=False,
        offline_requested=True,parameters=spec['options'],phases={},pid=os.getpid())
    def phase(phase_name,**details):
        report['phase']=phase_name;report['phases'][phase_name]=dict(status='running',**details)
        check_json(report_path,report);print('STAGE:',phase_name,flush=True)
    def passed(phase_name,**details):
        report['phases'][phase_name]={**details,'status':'passed'};check_json(report_path,report)
    def guard(event,args):
        if event=='socket.connect' and isinstance(args[1],str):return
        if event in ('socket.connect','socket.getaddrinfo'):raise RuntimeError('Offline restoration check blocks network access')
    sys.addaudithook(guard)
    shared_modules = ExitStack()
    try:
        if sys.platform!='linux' or not Path('/kaggle/working').is_dir():
            raise RuntimeError('This GPU/build job is Kaggle-only')
        import torch
        if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable')
        devices=[dict(index=i,name=torch.cuda.get_device_name(i),vram_bytes=torch.cuda.get_device_properties(i).total_memory,
            capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]
        gpu=dict(name=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,
            capability=list(torch.cuda.get_device_capability(0)),torch=torch.__version__,cuda=torch.version.cuda,
            python=sys.version,devices=devices,gpu_count=len(devices))
        report['hardware']=gpu
        if any(d['capability'][0]<8 for d in devices):
            raise RuntimeError('FlashAttention 2 CUDA/BF16 check needs a supported Ampere-or-newer GPU')
        sharded=spec['options'].get('placement','single')=='sharded'
        if spec['mode']=='unihir_smoke':
            if sharded:
                if len(devices)<2 or sum(d['vram_bytes'] for d in devices)<48*1024**3:
                    raise RuntimeError('Sharded UniHIR needs multiple GPUs with sufficient per-device headroom; no build/model load started')
            elif gpu['vram_bytes']<75*1024**3:
                raise RuntimeError('Single-GPU UniHIR needs >=75 GiB; select placement=sharded for 4 L4')
        if spec['options'].get('mode') not in (None,spec['mode']):raise ValueError('Conflicting check mode')
        passed('hardware',**gpu)
        phase('load_shared_definitions')
        payload=check_payload(spec['payload'])
        scope=shared_modules.enter_context(check_shared_definitions(payload['sources']))
        passed('load_shared_definitions',sha256=spec['payload']['sha256'],upstream=payload['upstream'])
        options=spec['options'];packages=Path(spec['packages']);models=Path(spec['models'])
        if not spec['packages'] or not packages.is_dir():raise FileNotFoundError('Configure a valid packages dataset root')
        if spec['mode']=='unihir_smoke' and (not spec['models'] or not models.is_dir()):
            raise FileNotFoundError('Configure a valid models dataset root')
        cfg=dict(packages_dir=str(packages),models_dir=str(models),data_dir=spec.get('data',''),
            gpu_ids=list(range(len(devices))),execution_mode='sequential',
            output_dir=str(output/'image_dataset'),books=options.get('books',[]),pages=options.get('pages','1'),
            limit=1,max_books=1,variants=['original','unihir_restoration'],crop_types=['page'],region_ids=[],
            resume=False,allow_partial=False,max_source_pixels=options.get('max_source_pixels',16_000_000),
            max_output_pixels=options.get('max_source_pixels',16_000_000),max_storage_bytes=1024**3,
            timeout_seconds=options.get('inference_timeout_seconds',900),
            load_timeout_seconds=options.get('load_timeout_seconds',900),batch_size=1,deskew=False,dewarp=False,
            layout={'dpi':200,'reading_direction':'vertical_rtl'},book_layouts={
                '1':{'dpi':225},'2':{'dpi':226},'3':{'dpi':226},'4':{'dpi':181},'5':{'dpi':181},
                '6':{'dpi':226},'bien':{'dpi':200,'reading_direction':'horizontal_ltr'},'kinh':{'dpi':45}},
            variant_options={},runtime_pythons={},reviewed_regions={},kernel_torch_source_policy='warn',
            model_options={'tadisr_x4':{'tile':768,'overlap':256},'difftsr':{'steps':20},
                'unihir_restoration':dict(device='cuda',seed=42,steps=options.get('steps',10),
                                        ref_count=options.get('ref_count',1),
                                        placement=options.get('placement','single'),gpu_ids=list(range(len(devices))),
                                        max_memory=options.get('max_memory','18GiB' if sharded else '80GiB'),
                                        activation_reserve_gib=options.get('activation_reserve_gib',4),
                                        entry_reserve_gib=options.get('entry_reserve_gib',2))})
        cfg=scope['prep_normalize'](cfg)
        original_assets=scope['prep_assets'];original_overlay=scope['prep_overlay']
        def observed_assets(*args,**kwargs):
            phase('model_asset_integrity');result=original_assets(*args,**kwargs)
            passed('model_asset_integrity',model='unihir_restoration',identity=result[1]);return result
        def observed_overlay(*args,**kwargs):
            phase('runtime_build',note='Offline compile can take a long time; build.log is preserved on failure')
            python=original_overlay(*args,**kwargs);native=check_native_receipt(python)
            report['native_build_verified']=True;passed('runtime_build',receipt=native)
            phase('flash_attention_cuda')
            result=check_logged_process([python,'-u','-c',FLASH_GPU_PROBE],output/'flash_attention_probe.log',
                                       timeout=120,env=scope['prep_worker_environment'](),new_session=False)
            if result['status']!='ok':raise RuntimeError('FlashAttention CUDA probe failed: '+result['tail'])
            data=json.loads(next(line for line in reversed(Path(result['log']).read_text(encoding='utf8').splitlines()) if line.startswith('{')))
            report['flash_attention_cuda_verified']=True;passed('flash_attention_cuda',result=data);return python
        scope['prep_assets']=observed_assets;scope['prep_overlay']=observed_overlay
        if spec['mode']=='build_and_probe':
            # Model preflight also enforces UniHIR's full-model VRAM requirement.
            # A standalone extension check needs neither model weights nor that worker.
            observed_overlay(packages,runtime,'unihir_restoration',cfg)
            report['status']='native_and_cuda_verified';return 0
        base_worker=scope['PrepWorker']
        class PreflightWorker(base_worker):
            def __init__(self,*args,**kwargs):
                phase('worker_preflight');super().__init__(*args,**kwargs)
                passed('worker_preflight',probe=self.ready.get('report'))
        scope['PrepWorker']=PreflightWorker
        preflight=scope['prep_preflight'](cfg,runtime)
        report['preflight']=preflight
        if preflight['unihir_restoration'].get('state')!='ready':raise RuntimeError('UniHIR worker preflight is not ready')
        passed('worker_preflight',state='ready')
        if not cfg['data_dir'] or not Path(cfg['data_dir']).is_dir():
            raise FileNotFoundError('Build/CUDA passed, but DATA_DIR must contain a PDF for actual inference')
        phase('pdf_bootstrap')
        scope['prep_bootstrap'](packages,runtime)
        passed('pdf_bootstrap')
        # Reuse this exact configuration's verified preflight in the same job.
        scope['prep_preflight']=lambda *args,**kwargs:preflight
        original_worker=base_worker
        class ObservedWorker(original_worker):
            def __init__(self,*args,**kwargs):
                phase('model_loading');super().__init__(*args,**kwargs)
                report['model_weights_loaded']=True;passed('model_loading',seconds=self.ready.get('load_seconds'))
            def infer(self,*args,**kwargs):
                report['model_inference_attempted']=True;phase('model_inference')
                result=super().infer(*args,**kwargs)
                passed('model_inference',note='Output pixels and manifest are validated next')
                return result
        scope['PrepWorker']=ObservedWorker
        phase('page_preparation')
        summary=scope['prep_run'](cfg,runtime)
        phase('output_validation');validation=scope['prep_validate'](cfg['output_dir'])
        metrics=check_inference_evidence(summary,validation)
        scope['prep_export_images'](cfg['output_dir'],output/'images')
        report.update(status='passed',model_inference_verified=True,inference_metrics=metrics,
            output_validation=validation,summary={k:v for k,v in summary.items() if k!='measurements'},
            fidelity='unverified; a successful run does not establish correct restored characters')
        passed('output_validation',status_detail='pixels,manifest,geometry validated')
        return 0
    except BaseException as exc:
        report.update(status='failed',failed_phase=report['phase'],error=str(exc),traceback=traceback.format_exc())
        print(report['traceback'],flush=True);return 1
    finally:
        shared_modules.close()
        report['finished_utc']=datetime.now(timezone.utc).isoformat();check_json(report_path,report)
        check_collect_logs(runtime,output)


def run_restoration_gpu_check(environment, *, mode, options, payload, output_dir):
    if mode not in ('off','build_and_probe','unihir_smoke'):raise ValueError('Unknown restoration check mode')
    if mode=='off':return dict(status='disabled',model_inference_verified=False)
    if sys.platform!='linux' or not Path('/kaggle/working').is_dir():
        return dict(status='skipped_not_kaggle',model_inference_verified=False,
                    detail='No model build, install or inference is permitted on local Windows')
    output=Path(output_dir).resolve()
    if output.is_relative_to(Path('/kaggle/input')):raise ValueError('Output must be outside Kaggle Input')
    run=output/('restoration_check_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
    run.mkdir(parents=True)
    roots=environment.get('asset_roots',{})
    spec=dict(mode=mode,options=options,payload=payload,output=str(run),
        runtime=options.get('runtime_dir','/tmp/vtm_image_runtime'),
        packages=roots.get('packages',{}).get('path') or '',models=roots.get('models',{}).get('path') or '',
        data=roots.get('pdfs',{}).get('path') or '')
    driver=run/'check_driver.py';driver.write_text(GPU_CHECK_DRIVER_SOURCE,encoding='utf8')
    check_json(run/'spec.json',spec)
    checkpoint=run/'report.json'
    try:
        env=check_offline_env()
        if options.get('gpu_ids') is not None:
            # IDs are logical under the platform mask, never physical host indices.
            ids=options['gpu_ids']; inherited=env.get('CUDA_VISIBLE_DEVICES')
            visible=[v.strip() for v in inherited.split(',') if v.strip() and v.strip()!='-1'] if inherited is not None else None
            if not ids or any(type(i) is not int or i<0 or (visible is not None and i>=len(visible)) for i in ids) or len(set(ids))!=len(ids):
                raise ValueError('Invalid restoration check GPU IDs under inherited CUDA_VISIBLE_DEVICES')
            env['CUDA_VISIBLE_DEVICES']=','.join(visible[i] if visible is not None else str(i) for i in ids)
        process=check_logged_process([sys.executable,'-u',str(driver),str(run/'spec.json')],run/'driver.log',
            timeout=options.get('total_timeout_seconds',5400),report_path=checkpoint,env=env)
    except BaseException as exc:
        process=dict(status='interrupted',error=str(exc),log=str(run/'driver.log'))
    result=json.loads(checkpoint.read_text(encoding='utf8')) if checkpoint.is_file() else dict(status='failed',error='Child report unavailable')
    if process['status'] in ('timeout','interrupted') or (process['status']!='ok' and result.get('status') in ('passed','native_and_cuda_verified')):
        result.update(status=process['status'],failed_phase=result.get('phase'),model_inference_verified=False)
    result.update(supervisor=process,output_dir=str(run))
    check_collect_logs(spec['runtime'],run);check_json(checkpoint,result)
    return result


def package_gpu_check_logs(report, output_dir):
    """Small review artifact. Private runtime, weights, archives and spec payload are excluded."""
    output=Path(output_dir).resolve();gpu=report.get('restoration_gpu_check',{})
    run=Path(gpu['output_dir']).resolve() if gpu.get('output_dir') else None
    archive=output/((run.name if run else 'kaggle_environment')+'_logs.zip')
    temporary=archive.with_name(archive.name+'.tmp')
    selected=[p for p in (output/'kaggle_environment_report.json',output/'kaggle_environment_report.txt') if p.is_file()]
    if run:
        if not run.is_relative_to(output):raise ValueError('Log directory outside diagnostic output')
        selected.extend(p for p in run.rglob('*') if p.is_file() and not p.is_symlink() and
            p.name not in ('spec.json','check_driver.py') and p.suffix in ('.json','.log','.txt','.png') and
            'image_dataset' not in p.relative_to(run).parts and p.stat().st_size<=10*1024**2)
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as zipped:
        for path in sorted(set(selected)):
            zipped.write(path,path.relative_to(output).as_posix())
    temporary.replace(archive)
    return archive


def make_restoration_check_payload(notebook_path):
    """Authoring only: select original notebook 1 definitions, without executing them."""
    notebook=json.loads(Path(notebook_path).read_text(encoding='utf8'))
    vendor=''.join(notebook['cells'][6]['source'])
    assignments={n.targets[0].id:n.value for n in ast.parse(vendor).body
        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}
    raw=zlib.decompress(base64.b85decode(ast.literal_eval(assignments['_VENDOR_SOURCES_B85'])))
    if hashlib.sha256(raw).hexdigest()!=ast.literal_eval(assignments['_VENDOR_SOURCES_SHA256']):
        raise ValueError('Original upstream payload checksum mismatch')
    vendors=json.loads(raw)
    revision=ast.literal_eval(assignments['UPSTREAM_REVISIONS'])['UniHIR']
    worker=ast.literal_eval(assignments['WORKER_SOURCE'])
    mini='\n'.join(('UPSTREAM_REVISIONS = '+repr({'UniHIR':revision}),
        'VENDOR_SOURCES = '+repr({'UniHIR':vendors['UniHIR']}),
        'WORKER_SOURCE = '+repr(worker),
        'RESTORATION_CODE_SIGNATURE = '+repr(ast.literal_eval(assignments['RESTORATION_CODE_SIGNATURE']))))
    sources=[''.join(notebook['cells'][4]['source']),mini,''.join(notebook['cells'][8]['source'])]
    for i,source in enumerate(sources):compile(source,f'notebook1-shared:{i}','exec')
    body=dict(format='vtm-notebook1-restoration-check-v1',sources=sources,
              upstream={'UniHIR':revision},worker_sha256=hashlib.sha256(worker.encode()).hexdigest())
    raw=json.dumps(body,ensure_ascii=False,separators=(',',':')).encode('utf8')
    return dict(sha256=hashlib.sha256(raw).hexdigest(),data=base64.b85encode(zlib.compress(raw,9)).decode('ascii'))



GPU_CHECK_DRIVER_SOURCE = '"""Kaggle-only restoration check using notebook 1\'s exact offline implementation.\n\nInventory remains read-only. The separately enabled GPU check may create a\nprivate runtime, compile FlashAttention and run one UniHIR page. No downloads,\nOCR, training, kernel package replacement, or corpus processing.\n"""\nfrom __future__ import annotations\n\nimport ast\nimport base64\nfrom contextlib import contextmanager, ExitStack\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom types import ModuleType\nimport uuid\nimport zipfile\nimport zlib\n\n\nFLASH_GPU_PROBE = \'\'\'import json, torch, flash_attn\nfrom flash_attn import flash_attn_func, flash_attn_varlen_func\nassert torch.cuda.is_available(), \'CUDA unavailable\'\ntorch.manual_seed(42)\ndevices=[]\nfor index in range(torch.cuda.device_count()):\n    with torch.cuda.device(index):\n        q,k,v=[torch.randn(1,16,2,64,device=f\'cuda:{index}\',dtype=torch.bfloat16) for _ in range(3)]\n        y=flash_attn_func(q,k,v,dropout_p=0.,causal=False)\n        reference=torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),dropout_p=0.).transpose(1,2)\n        cu=torch.tensor([0,16],device=f\'cuda:{index}\',dtype=torch.int32)\n        variable=flash_attn_varlen_func(q[0],k[0],v[0],cu,cu,16,16,dropout_p=0.,causal=False)\n        torch.cuda.synchronize(index)\n        assert y.shape==q.shape and variable.shape==q[0].shape\n        assert bool(torch.isfinite(y).all()) and bool(torch.isfinite(variable).all())\n        assert torch.allclose(y.float(),reference.float(),atol=.03,rtol=.03), \'FlashAttention differs from SDPA reference\'\n        assert torch.allclose(variable.float(),y[0].float(),atol=.03,rtol=.03), \'Variable-length kernel differs\'\n        devices.append(dict(index=index,name=torch.cuda.get_device_name(index),\n            max_abs_difference=float((y.float()-reference.float()).abs().max().item())))\nprint(json.dumps(dict(status=\'passed\',torch=torch.__version__,cuda=torch.version.cuda,\n    devices=devices,capability=list(torch.cuda.get_device_capability(0)),\n    flash_attention=flash_attn.__version__,module_file=flash_attn.__file__,\n    fixed_and_variable_length_verified=True,finite=True,\n    weights_loaded=False)))\n\'\'\'\n\n\ndef check_json(path, value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_name(path.name+\'.tmp\')\n    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf8\')\n    temporary.replace(path)\n\n\ndef check_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024**2),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef check_offline_env():\n    env=dict(os.environ)\n    for key in (\'PYTHONPATH\',\'PYTHONHOME\',\'PYTHONUSERBASE\'):env.pop(key,None)\n    env.update(PYTHONNOUSERSITE=\'1\',PYTHONUTF8=\'1\',PIP_NO_INDEX=\'1\',PIP_DISABLE_PIP_VERSION_CHECK=\'1\',\n        HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',HF_DATASETS_OFFLINE=\'1\',DIFFUSERS_OFFLINE=\'1\',\n        HF_HUB_DISABLE_TELEMETRY=\'1\',MPLBACKEND=\'Agg\',TOKENIZERS_PARALLELISM=\'false\')\n    return env\n\n\ndef check_tail(path, maximum=2000):\n    try:\n        with Path(path).open(\'rb\') as stream:\n            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))\n            return stream.read().decode(\'utf8\',errors=\'replace\')\n    except OSError:return \'\'\n\n\ndef check_stop(process):\n    if process.poll() is not None:return\n    if os.name==\'nt\' or not getattr(process,\'_vtm_owns_group\',False):process.kill()\n    else:\n        try:os.killpg(process.pid,signal.SIGKILL)\n        except ProcessLookupError:pass\n    process.wait()\n\n\ndef check_logged_process(command, log, *, timeout, env=None, report_path=None, new_session=True):\n    """One process group; preserve bounded log tails and kill descendants on timeout."""\n    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();last_notice=started;process=None\n    with log.open(\'w\',encoding=\'utf8\') as output:\n        try:\n            process=subprocess.Popen(list(map(str,command)),stdout=output,stderr=subprocess.STDOUT,\n                env=env or check_offline_env(),start_new_session=os.name!=\'nt\' and new_session,\n                creationflags=getattr(subprocess,\'CREATE_NO_WINDOW\',0))\n            process._vtm_owns_group=os.name!=\'nt\' and new_session\n            while process.poll() is None:\n                elapsed=time.monotonic()-started\n                if elapsed>timeout:\n                    check_stop(process)\n                    return dict(status=\'timeout\',seconds=round(elapsed,3),log=str(log),tail=check_tail(log))\n                if time.monotonic()-last_notice>=15:\n                    phase=\'running\'\n                    if report_path and Path(report_path).is_file():\n                        try:phase=json.loads(Path(report_path).read_text(encoding=\'utf8\')).get(\'phase\',phase)\n                        except (OSError,ValueError):pass\n                    tail=check_tail(log,1000).splitlines()\n                    native=next((line for line in reversed(tail) if line.startswith(\'FlashAttention build:\')),\'\')\n                    print(f\'UniHIR check: {phase}, {elapsed:.0f}s; log: {log}\'+(\' | \'+native if native else \'\'),flush=True)\n                    last_notice=time.monotonic()\n                time.sleep(.2)\n            return dict(status=\'ok\' if process.returncode==0 else \'failed\',returncode=process.returncode,\n                        seconds=round(time.monotonic()-started,3),log=str(log),tail=check_tail(log))\n        except BaseException:\n            if process is not None:check_stop(process)\n            raise\n\n\ndef check_payload(payload):\n    raw=zlib.decompress(base64.b85decode(payload[\'data\']))\n    if hashlib.sha256(raw).hexdigest()!=payload[\'sha256\']:raise ValueError(\'Notebook 1 definitions checksum mismatch\')\n    value=json.loads(raw)\n    if value.get(\'format\')!=\'vtm-notebook1-restoration-check-v1\':raise ValueError(\'Unexpected definition bundle\')\n    return value\n\n\ndef check_native_receipt(python):\n    folder=Path(python).parent.parent\n    data=json.loads((folder/\'install.json\').read_text(encoding=\'utf8\'))\n    verified=[]\n    for record in data.get(\'native\',[]):\n        wheel=folder/\'native_build/wheels\'/record[\'wheel\']\n        if not wheel.is_file() or check_hash(wheel)!=record[\'sha256\']:\n            raise ValueError(\'Compiled wheel missing or changed: \'+str(wheel))\n        verified.append(dict(record,path=str(wheel)))\n    if not verified:raise ValueError(\'No compiled FlashAttention wheel receipt\')\n    return dict(status=\'built_or_reused_verified_wheel\',native_wheels=verified,\n                note=\'The separate CUDA probe executes the currently imported extension.\')\n\n\ndef check_collect_logs(runtime, output):\n    """Export logs/receipts only; never copy runtime binaries or model weights."""\n    runtime,output=Path(runtime),Path(output)\n    if not runtime.is_dir():return\n    spec=output/\'spec.json\'\n    cutoff=spec.stat().st_mtime if spec.is_file() else 0\n    active_native=[]\n    try:\n        current=json.loads((output/\'report.json\').read_text(encoding=\'utf8\'))\n        active_native=[Path(r[\'path\']).parent.parent for r in\n            current.get(\'phases\',{}).get(\'runtime_build\',{}).get(\'receipt\',{}).get(\'native_wheels\',[])]\n    except (OSError,ValueError):pass\n    for pattern in (\'build.log\',\'build_config.json\',\'build_progress.json\',\'build.ninja\',\'.ninja_log\',\n                    \'install.json\',\'preflight.json\',\'worker.log\',\'ready.json\',\'response.json\'):\n        sources=sorted(runtime.rglob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)[:32]\n        for source in sources:\n            if source.is_symlink() or source.stat().st_size>10*1024**2:continue\n            if source.stat().st_mtime<cutoff and not (pattern in (\'build.log\',\'build_config.json\',\'build_progress.json\',\'build.ninja\',\'.ninja_log\',\'install.json\') and\n                    any(source.is_relative_to(p.parent) for p in active_native)):continue\n            if pattern in (\'build.ninja\',\'.ninja_log\'):\n                short=hashlib.sha256(str(source.relative_to(runtime)).encode()).hexdigest()[:12]\n                target=output/\'runtime_logs/native_debug\'/(short+\'_\'+pattern.lstrip(\'.\')+\'.txt\')\n            else:\n                target=output/\'runtime_logs\'/source.relative_to(runtime)\n            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)\n\n\ndef check_inference_evidence(summary, validation):\n    metrics=[m for m in summary.get(\'measurements\',[]) if m.get(\'variant\')==\'unihir_restoration\']\n    if summary.get(\'failures\') or not metrics or summary.get(\'variants_reused\'):\n        raise ValueError(\'Fresh, successful UniHIR inference was not established\')\n    for item in metrics:\n        worker=item.get(\'worker\',{})\n        if worker.get(\'state\')!=\'ok\' or worker.get(\'peak_vram_bytes\',0)<=0:\n            raise ValueError(\'Missing successful GPU worker evidence\')\n    if not validation.get(\'restoration\',{}).get(\'validated\'):\n        raise ValueError(\'Output manifest/pixels/geometry validation incomplete\')\n    return metrics\n\n\n@contextmanager\ndef check_shared_definitions(sources):\n    """Execute notebook definitions in a registered module, including dataclasses."""\n    name = \'vtm_notebook1_check_definitions_\' + uuid.uuid4().hex\n    module = ModuleType(name)\n    sys.modules[name] = module\n    try:\n        for index, source in enumerate(sources):\n            exec(compile(source, f\'notebook1-shared:{index}\', \'exec\'), module.__dict__)\n        yield module.__dict__\n    finally:\n        if sys.modules.get(name) is module:\n            del sys.modules[name]\n\n\ndef restoration_check_child(spec_path):\n    """Executed only by the Kaggle job supervisor, never by notebook import."""\n    spec=json.loads(Path(spec_path).read_text(encoding=\'utf8\'))\n    output=Path(spec[\'output\']);runtime=Path(spec[\'runtime\']);report_path=output/\'report.json\'\n    report=dict(format=\'vtm-kaggle-restoration-gpu-check\',version=1,status=\'running\',phase=\'hardware\',\n        mode=spec[\'mode\'],native_build_verified=False,flash_attention_cuda_verified=False,\n        model_weights_loaded=False,model_inference_attempted=False,model_inference_verified=False,\n        offline_requested=True,parameters=spec[\'options\'],phases={},pid=os.getpid())\n    def phase(phase_name,**details):\n        report[\'phase\']=phase_name;report[\'phases\'][phase_name]=dict(status=\'running\',**details)\n        check_json(report_path,report);print(\'STAGE:\',phase_name,flush=True)\n    def passed(phase_name,**details):\n        report[\'phases\'][phase_name]={**details,\'status\':\'passed\'};check_json(report_path,report)\n    def guard(event,args):\n        if event==\'socket.connect\' and isinstance(args[1],str):return\n        if event in (\'socket.connect\',\'socket.getaddrinfo\'):raise RuntimeError(\'Offline restoration check blocks network access\')\n    sys.addaudithook(guard)\n    shared_modules = ExitStack()\n    try:\n        if sys.platform!=\'linux\' or not Path(\'/kaggle/working\').is_dir():\n            raise RuntimeError(\'This GPU/build job is Kaggle-only\')\n        import torch\n        if not torch.cuda.is_available():raise RuntimeError(\'CUDA unavailable\')\n        devices=[dict(index=i,name=torch.cuda.get_device_name(i),vram_bytes=torch.cuda.get_device_properties(i).total_memory,\n            capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]\n        gpu=dict(name=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,\n            capability=list(torch.cuda.get_device_capability(0)),torch=torch.__version__,cuda=torch.version.cuda,\n            python=sys.version,devices=devices,gpu_count=len(devices))\n        report[\'hardware\']=gpu\n        if any(d[\'capability\'][0]<8 for d in devices):\n            raise RuntimeError(\'FlashAttention 2 CUDA/BF16 check needs a supported Ampere-or-newer GPU\')\n        sharded=spec[\'options\'].get(\'placement\',\'single\')==\'sharded\'\n        if spec[\'mode\']==\'unihir_smoke\':\n            if sharded:\n                if len(devices)<2 or sum(d[\'vram_bytes\'] for d in devices)<48*1024**3:\n                    raise RuntimeError(\'Sharded UniHIR needs multiple GPUs with sufficient per-device headroom; no build/model load started\')\n            elif gpu[\'vram_bytes\']<75*1024**3:\n                raise RuntimeError(\'Single-GPU UniHIR needs >=75 GiB; select placement=sharded for 4 L4\')\n        if spec[\'options\'].get(\'mode\') not in (None,spec[\'mode\']):raise ValueError(\'Conflicting check mode\')\n        passed(\'hardware\',**gpu)\n        phase(\'load_shared_definitions\')\n        payload=check_payload(spec[\'payload\'])\n        scope=shared_modules.enter_context(check_shared_definitions(payload[\'sources\']))\n        passed(\'load_shared_definitions\',sha256=spec[\'payload\'][\'sha256\'],upstream=payload[\'upstream\'])\n        options=spec[\'options\'];packages=Path(spec[\'packages\']);models=Path(spec[\'models\'])\n        if not spec[\'packages\'] or not packages.is_dir():raise FileNotFoundError(\'Configure a valid packages dataset root\')\n        if spec[\'mode\']==\'unihir_smoke\' and (not spec[\'models\'] or not models.is_dir()):\n            raise FileNotFoundError(\'Configure a valid models dataset root\')\n        cfg=dict(packages_dir=str(packages),models_dir=str(models),data_dir=spec.get(\'data\',\'\'),\n            gpu_ids=list(range(len(devices))),execution_mode=\'sequential\',\n            output_dir=str(output/\'image_dataset\'),books=options.get(\'books\',[]),pages=options.get(\'pages\',\'1\'),\n            limit=1,max_books=1,variants=[\'original\',\'unihir_restoration\'],crop_types=[\'page\'],region_ids=[],\n            resume=False,allow_partial=False,max_source_pixels=options.get(\'max_source_pixels\',16_000_000),\n            max_output_pixels=options.get(\'max_source_pixels\',16_000_000),max_storage_bytes=1024**3,\n            timeout_seconds=options.get(\'inference_timeout_seconds\',900),\n            load_timeout_seconds=options.get(\'load_timeout_seconds\',900),batch_size=1,deskew=False,dewarp=False,\n            layout={\'dpi\':200,\'reading_direction\':\'vertical_rtl\'},book_layouts={\n                \'1\':{\'dpi\':225},\'2\':{\'dpi\':226},\'3\':{\'dpi\':226},\'4\':{\'dpi\':181},\'5\':{\'dpi\':181},\n                \'6\':{\'dpi\':226},\'bien\':{\'dpi\':200,\'reading_direction\':\'horizontal_ltr\'},\'kinh\':{\'dpi\':45}},\n            variant_options={},runtime_pythons={},reviewed_regions={},kernel_torch_source_policy=\'warn\',\n            model_options={\'tadisr_x4\':{\'tile\':768,\'overlap\':256},\'difftsr\':{\'steps\':20},\n                \'unihir_restoration\':dict(device=\'cuda\',seed=42,steps=options.get(\'steps\',10),\n                                        ref_count=options.get(\'ref_count\',1),\n                                        placement=options.get(\'placement\',\'single\'),gpu_ids=list(range(len(devices))),\n                                        max_memory=options.get(\'max_memory\',\'18GiB\' if sharded else \'80GiB\'),\n                                        activation_reserve_gib=options.get(\'activation_reserve_gib\',4),\n                                        entry_reserve_gib=options.get(\'entry_reserve_gib\',2))})\n        cfg=scope[\'prep_normalize\'](cfg)\n        original_assets=scope[\'prep_assets\'];original_overlay=scope[\'prep_overlay\']\n        def observed_assets(*args,**kwargs):\n            phase(\'model_asset_integrity\');result=original_assets(*args,**kwargs)\n            passed(\'model_asset_integrity\',model=\'unihir_restoration\',identity=result[1]);return result\n        def observed_overlay(*args,**kwargs):\n            phase(\'runtime_build\',note=\'Offline compile can take a long time; build.log is preserved on failure\')\n            python=original_overlay(*args,**kwargs);native=check_native_receipt(python)\n            report[\'native_build_verified\']=True;passed(\'runtime_build\',receipt=native)\n            phase(\'flash_attention_cuda\')\n            result=check_logged_process([python,\'-u\',\'-c\',FLASH_GPU_PROBE],output/\'flash_attention_probe.log\',\n                                       timeout=120,env=scope[\'prep_worker_environment\'](),new_session=False)\n            if result[\'status\']!=\'ok\':raise RuntimeError(\'FlashAttention CUDA probe failed: \'+result[\'tail\'])\n            data=json.loads(next(line for line in reversed(Path(result[\'log\']).read_text(encoding=\'utf8\').splitlines()) if line.startswith(\'{\')))\n            report[\'flash_attention_cuda_verified\']=True;passed(\'flash_attention_cuda\',result=data);return python\n        scope[\'prep_assets\']=observed_assets;scope[\'prep_overlay\']=observed_overlay\n        if spec[\'mode\']==\'build_and_probe\':\n            # Model preflight also enforces UniHIR\'s full-model VRAM requirement.\n            # A standalone extension check needs neither model weights nor that worker.\n            observed_overlay(packages,runtime,\'unihir_restoration\',cfg)\n            report[\'status\']=\'native_and_cuda_verified\';return 0\n        base_worker=scope[\'PrepWorker\']\n        class PreflightWorker(base_worker):\n            def __init__(self,*args,**kwargs):\n                phase(\'worker_preflight\');super().__init__(*args,**kwargs)\n                passed(\'worker_preflight\',probe=self.ready.get(\'report\'))\n        scope[\'PrepWorker\']=PreflightWorker\n        preflight=scope[\'prep_preflight\'](cfg,runtime)\n        report[\'preflight\']=preflight\n        if preflight[\'unihir_restoration\'].get(\'state\')!=\'ready\':raise RuntimeError(\'UniHIR worker preflight is not ready\')\n        passed(\'worker_preflight\',state=\'ready\')\n        if not cfg[\'data_dir\'] or not Path(cfg[\'data_dir\']).is_dir():\n            raise FileNotFoundError(\'Build/CUDA passed, but DATA_DIR must contain a PDF for actual inference\')\n        phase(\'pdf_bootstrap\')\n        scope[\'prep_bootstrap\'](packages,runtime)\n        passed(\'pdf_bootstrap\')\n        # Reuse this exact configuration\'s verified preflight in the same job.\n        scope[\'prep_preflight\']=lambda *args,**kwargs:preflight\n        original_worker=base_worker\n        class ObservedWorker(original_worker):\n            def __init__(self,*args,**kwargs):\n                phase(\'model_loading\');super().__init__(*args,**kwargs)\n                report[\'model_weights_loaded\']=True;passed(\'model_loading\',seconds=self.ready.get(\'load_seconds\'))\n            def infer(self,*args,**kwargs):\n                report[\'model_inference_attempted\']=True;phase(\'model_inference\')\n                result=super().infer(*args,**kwargs)\n                passed(\'model_inference\',note=\'Output pixels and manifest are validated next\')\n                return result\n        scope[\'PrepWorker\']=ObservedWorker\n        phase(\'page_preparation\')\n        summary=scope[\'prep_run\'](cfg,runtime)\n        phase(\'output_validation\');validation=scope[\'prep_validate\'](cfg[\'output_dir\'])\n        metrics=check_inference_evidence(summary,validation)\n        scope[\'prep_export_images\'](cfg[\'output_dir\'],output/\'images\')\n        report.update(status=\'passed\',model_inference_verified=True,inference_metrics=metrics,\n            output_validation=validation,summary={k:v for k,v in summary.items() if k!=\'measurements\'},\n            fidelity=\'unverified; a successful run does not establish correct restored characters\')\n        passed(\'output_validation\',status_detail=\'pixels,manifest,geometry validated\')\n        return 0\n    except BaseException as exc:\n        report.update(status=\'failed\',failed_phase=report[\'phase\'],error=str(exc),traceback=traceback.format_exc())\n        print(report[\'traceback\'],flush=True);return 1\n    finally:\n        shared_modules.close()\n        report[\'finished_utc\']=datetime.now(timezone.utc).isoformat();check_json(report_path,report)\n        check_collect_logs(runtime,output)\n\n\ndef run_restoration_gpu_check(environment, *, mode, options, payload, output_dir):\n    if mode not in (\'off\',\'build_and_probe\',\'unihir_smoke\'):raise ValueError(\'Unknown restoration check mode\')\n    if mode==\'off\':return dict(status=\'disabled\',model_inference_verified=False)\n    if sys.platform!=\'linux\' or not Path(\'/kaggle/working\').is_dir():\n        return dict(status=\'skipped_not_kaggle\',model_inference_verified=False,\n                    detail=\'No model build, install or inference is permitted on local Windows\')\n    output=Path(output_dir).resolve()\n    if output.is_relative_to(Path(\'/kaggle/input\')):raise ValueError(\'Output must be outside Kaggle Input\')\n    run=output/(\'restoration_check_\'+datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S\')+\'_\'+uuid.uuid4().hex[:8])\n    run.mkdir(parents=True)\n    roots=environment.get(\'asset_roots\',{})\n    spec=dict(mode=mode,options=options,payload=payload,output=str(run),\n        runtime=options.get(\'runtime_dir\',\'/tmp/vtm_image_runtime\'),\n        packages=roots.get(\'packages\',{}).get(\'path\') or \'\',models=roots.get(\'models\',{}).get(\'path\') or \'\',\n        data=roots.get(\'pdfs\',{}).get(\'path\') or \'\')\n    driver=run/\'check_driver.py\';driver.write_text(GPU_CHECK_DRIVER_SOURCE,encoding=\'utf8\')\n    check_json(run/\'spec.json\',spec)\n    checkpoint=run/\'report.json\'\n    try:\n        env=check_offline_env()\n        if options.get(\'gpu_ids\') is not None:\n            # IDs are logical under the platform mask, never physical host indices.\n            ids=options[\'gpu_ids\']; inherited=env.get(\'CUDA_VISIBLE_DEVICES\')\n            visible=[v.strip() for v in inherited.split(\',\') if v.strip() and v.strip()!=\'-1\'] if inherited is not None else None\n            if not ids or any(type(i) is not int or i<0 or (visible is not None and i>=len(visible)) for i in ids) or len(set(ids))!=len(ids):\n                raise ValueError(\'Invalid restoration check GPU IDs under inherited CUDA_VISIBLE_DEVICES\')\n            env[\'CUDA_VISIBLE_DEVICES\']=\',\'.join(visible[i] if visible is not None else str(i) for i in ids)\n        process=check_logged_process([sys.executable,\'-u\',str(driver),str(run/\'spec.json\')],run/\'driver.log\',\n            timeout=options.get(\'total_timeout_seconds\',5400),report_path=checkpoint,env=env)\n    except BaseException as exc:\n        process=dict(status=\'interrupted\',error=str(exc),log=str(run/\'driver.log\'))\n    result=json.loads(checkpoint.read_text(encoding=\'utf8\')) if checkpoint.is_file() else dict(status=\'failed\',error=\'Child report unavailable\')\n    if process[\'status\'] in (\'timeout\',\'interrupted\') or (process[\'status\']!=\'ok\' and result.get(\'status\') in (\'passed\',\'native_and_cuda_verified\')):\n        result.update(status=process[\'status\'],failed_phase=result.get(\'phase\'),model_inference_verified=False)\n    result.update(supervisor=process,output_dir=str(run))\n    check_collect_logs(spec[\'runtime\'],run);check_json(checkpoint,result)\n    return result\n\n\ndef package_gpu_check_logs(report, output_dir):\n    """Small review artifact. Private runtime, weights, archives and spec payload are excluded."""\n    output=Path(output_dir).resolve();gpu=report.get(\'restoration_gpu_check\',{})\n    run=Path(gpu[\'output_dir\']).resolve() if gpu.get(\'output_dir\') else None\n    archive=output/((run.name if run else \'kaggle_environment\')+\'_logs.zip\')\n    temporary=archive.with_name(archive.name+\'.tmp\')\n    selected=[p for p in (output/\'kaggle_environment_report.json\',output/\'kaggle_environment_report.txt\') if p.is_file()]\n    if run:\n        if not run.is_relative_to(output):raise ValueError(\'Log directory outside diagnostic output\')\n        selected.extend(p for p in run.rglob(\'*\') if p.is_file() and not p.is_symlink() and\n            p.name not in (\'spec.json\',\'check_driver.py\') and p.suffix in (\'.json\',\'.log\',\'.txt\',\'.png\') and\n            \'image_dataset\' not in p.relative_to(run).parts and p.stat().st_size<=10*1024**2)\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as zipped:\n        for path in sorted(set(selected)):\n            zipped.write(path,path.relative_to(output).as_posix())\n    temporary.replace(archive)\n    return archive\n\n\ndef make_restoration_check_payload(notebook_path):\n    """Authoring only: select original notebook 1 definitions, without executing them."""\n    notebook=json.loads(Path(notebook_path).read_text(encoding=\'utf8\'))\n    vendor=\'\'.join(notebook[\'cells\'][6][\'source\'])\n    assignments={n.targets[0].id:n.value for n in ast.parse(vendor).body\n        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}\n    raw=zlib.decompress(base64.b85decode(ast.literal_eval(assignments[\'_VENDOR_SOURCES_B85\'])))\n    if hashlib.sha256(raw).hexdigest()!=ast.literal_eval(assignments[\'_VENDOR_SOURCES_SHA256\']):\n        raise ValueError(\'Original upstream payload checksum mismatch\')\n    vendors=json.loads(raw)\n    revision=ast.literal_eval(assignments[\'UPSTREAM_REVISIONS\'])[\'UniHIR\']\n    worker=ast.literal_eval(assignments[\'WORKER_SOURCE\'])\n    mini=\'\\n\'.join((\'UPSTREAM_REVISIONS = \'+repr({\'UniHIR\':revision}),\n        \'VENDOR_SOURCES = \'+repr({\'UniHIR\':vendors[\'UniHIR\']}),\n        \'WORKER_SOURCE = \'+repr(worker),\n        \'RESTORATION_CODE_SIGNATURE = \'+repr(ast.literal_eval(assignments[\'RESTORATION_CODE_SIGNATURE\']))))\n    sources=[\'\'.join(notebook[\'cells\'][4][\'source\']),mini,\'\'.join(notebook[\'cells\'][8][\'source\'])]\n    for i,source in enumerate(sources):compile(source,f\'notebook1-shared:{i}\',\'exec\')\n    body=dict(format=\'vtm-notebook1-restoration-check-v1\',sources=sources,\n              upstream={\'UniHIR\':revision},worker_sha256=hashlib.sha256(worker.encode()).hexdigest())\n    raw=json.dumps(body,ensure_ascii=False,separators=(\',\',\':\')).encode(\'utf8\')\n    return dict(sha256=hashlib.sha256(raw).hexdigest(),data=base64.b85encode(zlib.compress(raw,9)).decode(\'ascii\'))\n\n\n# CLI child\nif __name__==\'__main__\':\n    raise SystemExit(restoration_check_child(sys.argv[1]))\n'
GPU_CHECK_PAYLOAD = {'sha256': 'a9dd5d604fc36a5719ef1a0256f53390f6bdf722bf395dba60da803029e67a15', 'data': 'c-qXpX^$gGmLT?5tg5k1GKyr7ab+g6SUZssky$Yvc}zrR^}&qwbdt`XS2*2SbY~nD6+(Xz7Arv#cBY}lN(1dI7Nb1`>B~ycpa;SN+xjEj7_Ii3_9y5acDg%{$jSz`x---1ZpSmTXYaCS@7;YqN@f#3`@Hk{RW@nFNfz{z<g(qE1!<Pd{47l3#$X%_E*n?v&+DJ3$$T~l($71+&quRlB0O(2&*rnh^F%nACbLZVF*HK;()Y1^2OW&x$8Qdwd&e*LUmSVg9G#vWzkVgUqB0$h!uh0e6{cZ73ZRq9`}lo43`PRJc+>DEh`cQErv61xO~dbly6E?lo4T0EPHLU^v4H>He~$k<3vYx!8ijEn21x)9gD{A*v?FdRU4P@|i^0b27tq9QrQ3&JL-;jnVt?4eaXg)8f_hBF*uM&dFS2CXh=Nfjl392W#!p2u&#+XOig`>R1VaJi5(7WJ^6_PurA=-RUc<=5FbPrtQzC+!X_5wF^IaOu-a)wze4-DL-wz^@jRW{P9p9$mz>mb?n`3e1NAo~6#{=Z&>$(`j<l%hqiwLF#=A<q%c#RXpufvg^C(-SSCoMTy_}?smb&Ex>RTtGvG(^89HnHLA41PrTV_X;T5t}|QZLa;eti6V5;F=n6Iv5j_3mV*dv;e?{17RZ%B92?Os4A5J(Qu%MxQ;<JiGkiMfLF6BB3xahuoUpm#J^!wOclDx6~I!>iy-rGDXKO8hR^V=hgjbn1YuOY78~4DtuD;lv3_f&V-m)M<~8x9Xam+!W4+cn7R9Ibx%kQg$DS@f1pN61QRZkiOJ>zk<qbe5cs$|`tKS0>4uW8qQWLzFWlP;V)9U?GsC5+v*TNrsH-~wl1xkY~3*(DaRJkH<f;hMe5XK00S#%4vhVwx%tX1Sp^uui8Pt`ohnWZ6j<;)Ie{<TLWBkjJO%>!96o81;Mfm>@LHGtWxH3vxyy=T?R>5FHoN7(v)6xo$!tw98v4_Am+oCXGD3s*jzAaUUZ-vE%NVRx(5s+Y`ByCwev%*cQ^1{b&8ia*biic5x08Q|nP2J!&VDXQYfLje;z3}To*AiS{YVG^<DO6tt@Fc`pN5^Li00igPTGY(`zE-9>q7{Ors{@_xCvFc+A3-V~INR-5L=+BqC#80ze)<|Mlv(Y??8VCa7Vl3u>L{MDBuq%NC0;mAeZ5=uoMDro`c|8tdXyF=xOooyGjr<V$27(n1X*)N?O8{rMc@+$RjnEKNYTE~T7eM!NAm0IjLmCR~CW+=54>?cnBo~?9GPBqcFerYMog*iDb9|x-9^--@jFV}q_mx^MBDgSFcf4$MQQ^s~+~a10NfuvrMGKz`t508+ZMe#DOcOwJZ?9%DC3dKA^g0}7K!L_Vcrnfv5fPWmL8)pKrdd^&#qZS`&=dVem4&z5MAH8jW<Q0xZdLhSm$NuFLPHbVgavLshk+ms@m>VkJ{CGi<}o)^Q!9fDTUTg;VD`>|+BREiKf+8xe>ml`TWbbkUEpOiyPXE&vqI0_8xR#<Kl@Xb8LVpfAMAY~M)SbDm5bD+CUKb48)riCKcKxckPu86eqk5EhJBD9X0nw2ZHiovf!<jKL+0<Vk~qjgsr%!H=42ja;WWCf>dFojwc;2RD5gRQK>kgTYKAkuohDgTcIekt*L}r$obLnueW;ia_dJ6RRGauYiM?qO!r}u`-qvVY@d?+pT1EZ}%~#vz-!}f+!hh{{`+z?~ZWj1lb%2Zwn;Kv{vQxxq`L(^^tADesE`KepYk`7fj~3<z!Nd-6NA<P<dl^c2tbx3M1?Wr5d}*E6YQT{5b+=84ZMSaQd}(9duYi&?A43zWQxr_NlbU|Jt=~c&1Gps23_2q{PQ@l+0YNEWw)IP>uVf4S4gCX&gCc=wnRd+<#)dWB=kBcjdCNEb=@bZO^{&DNXscB^lG3oK1jLmGja(MK@eg>*I3?fqim0leEcDb*fqojnSSt}oAQ3c!dzGHy9j+~D3?8yH_=qpyJ;UVzhx8t8dCVO>;*L-Tc;<#2?R`xNlYYuXv3IUSyU8@d$p?X_DhXPRWMSg%@(rv46SeLNqNeFI6B}8V_!W0Vi-d$lGcu$!{LvAR2vD^s6)5CEB~YBE9!(0%I8VhC_-rtP<r*ZwDw#bgwqyz8OPZ!>RpqRzHWT?Q$0*C1u*!SM3|8SX264%1urSEUQo6lB1G3l*O$2gwzn@koKoDUYw1Y-#&%7Hr@8E?`Our_74b3k<zO7cVk`J5oigr}9)oK&X0ZhI^vh#g7ttx;S_p3ElUu(4oHPV|9);CA^7w(hh5KzA-hm4924jYIC(3V<`1r;Vg@&~~L#C2~l1ME;MiEA(mMiHQ(wXVbqqHtO+7QF6%i;}?s6nF!i1;<)*W}T|yzQ|QpH9V3h@F}fgS(<?r@Dn0?SQ>VCHm=x$2+T~7jT4|1Mi51j1UFEfObt@I1Gx|1)r#UF>{ylz3GOQHHVJ_}u`#HB@p!qcfqND8J}CE9QJBn+vLH*oVlq$TTCBC%3q=|g%NpZm9Nc972!#A9QhfkqQ;D4b&!&_;Ysjx1h!Gi0VL&Onw8T$9Q09G$T3xY%ejKJr2FMo<JjpyRf&^A>c54p@NnfrdtxoD)MK7wnsNLs2RLRLut|F;%8WtDv1E*^ptx^jO1?|$ByRo18SvIS3gMe?>{tWh2?2+nOsg=~S^i!uWfJK8T*2rtK_BUz4;LW<yp5fq8t|d^*e3Xv#JT=*-(N%r$Z`>kA*J=*K(I}XqZd=tA>#9Z#7mSq_6qpT9)6m;n_O8M^8Thi~=41#YxIipHi=aC`)a2+Z)S!R0X6HyIHuxx?(0ss0`uE-i2$qyuDqC%v+8~9o#;XJ(Ii6scN1_tMF6H6ejFee;Zavg=^~lL?U}zYV3OToqo{Gx8N6J2+dy+&|mf>o&fPm1|YQpjgD#l9%mij_@=K1T>m-nphS;}6G`=~l9X4j!Qifk$Q6U7h<Vyvz{ku*foDd_2&AhBu1GcTY1h&_;79tEC3&z{5@DG~w__=K@QHPB=O*i<ldqs*)DS+S^!jh6(p#!41l;fE5D9gD%aDuKI6iPmH-fKG;?@0r)G-cMhZ*`Pw~e^F8Vs7q>TucHM-c*`s7A>N?)+><Dj#{h)ZT74*K{lZN<=?BAMFk}f8zbl(UMO%kFSa0|Me_OlS-noTc$IzM|s1!(3nI|_Ij)tFbEf^7oAcCeo%AL4UM1svn<PbDf#e)g6e1+o)f>_zMNZCM|=4v9?pskHO)yDvyIxK(>{*@m_KA>|&@)k74;>=f~<XUl3)7Mj21lUE>Bw_(?K=Td-1_@xQHw3f|qsqCa9BiE=?HT_HdyZyz&N_dHD=Vo-4W;2nyHb<vo!>ha_^P3iKI|+q(V=9bY;>b2NkCc;Q{br~(#`a{StiS&U>AjbRrC#=9!X36EcupYRL}v^OVx|Cu(?*)ePGrwjVaqX!}wBQm8a5ZjmAiM=E9aEG8g!9663^gzCJh=xC_#zZHX+kG}bL9D4(B)5zM{(@mkp-FJ?ZHV8#|LXPhV+j-Q*^cs}XJur$?bdk+oG_;*dV)I1FcJz?dWC;P7se)alHsjB`koTpNG-(sT#wa14Eo8qyexi|+<&f?qYybt2<Ama)n0=;Stqxr(0r)lWN&!YLP$~_@rz~VJ$(baI9C_<y53MqLNd#p9td=&YU{?Hd!9kI2$i@0|MQl(ryq&y(t$Cq9<20R`o(GXC52NW*0Rn$t5;KHA#3JLt1FkOHO&qKg%+7Eye0`^M-(cxmA!5i3a>3oWsaSC7J3o)BVup0o;eThH<5y-&<If*N<(K#x@0!?#b2Xx6-Z?mL-n+2(o=kUN0D2TKU({5~|0Y0O1BoM*3m5uBgBy{J~lSegWMFia+jZRsXOmeM^bFGX`E8~1CutIeH`Bhf}gohT%6n}HuB?oruC@#4s(@8p?RI68HbHXh~S}l&gcMF|ei!Z2yj`~5ppp6X6l6`UJ$3SAO;V3g6k#|pZbIUdAVt`7eQk{#H>jW4X#JJ>_FycpOb>jgx2p;Q54s}4&)#C9ElE{qL8Q{&i9zBi7v?j-SW5#JMzB^9U{EafcmW<P`Tql4^#)+mY3xcwNV$H<^MLiL=8tKY66{{SdSAWDDZ03Q=&&suJx&_rXh+&*i=OV+cCHb$ElCH}Ixix{iF~HIMxxn$1=1_CI4sa~H0V$~IMU~*dRwP1*FW55thzTCQp*=%S3>egs-DxU`8nI16fhc-NMn_21Q52#Rnbw}-0Kjo4^NC>=Gc-<BCBz$UTTmE@Y_TmxA%fwBvPR2>3k^brJyjeDHlCjPvrL^9S*92*WAMyk+j=S++|?JNZL8A#$d8c}hk)O1K@?|rzIx#~$|`!k9Fqq(fQ1>Tq5?%~fCM;n4Lq=8O6o5-4MIr^fa7AY&~m!U>=SeBi)XRQ<-ZUiq9Pip8O0bcVZ60s{dDTb?wUZg?Xq>*-mR6??+36DQtQF_!-jY*7?=k2Q0vgI^J0CnJelfY2ic6XI>2sVNYqM6edNap%{TU33b?_j+!SkeW46ZzOq*ssJT^XrQT@>Bt*A$1a;hyb=pcqGEglxKR*p!<03xX?R#O5FXce=JjPt0WRY{|Orcx#248s|V9cm}s0RO&)KR2+OZ?{Ozmj>`a9=L%dxxr1`Y{44bv~y&fNzN^PyKQS{nsgo^u)Qjj{{n5&)CoTKv7?$YRreJi3h0C<pp&l7dF{Nk7i<oj;wA}ZYI1dR&Wtg5y-ugmF7!iY12s$Z5`}G2sb*Ou;ot_em)h7*QSH#bzXjz`|458C+bqzJ=@@ja$o!o5K_nc2BxzpVn&&K>1X9HT;dn5gfi9ah*?}(A+737HCP|{Bn*f$9naMiHvIdc#rlzodRTrZ$K*~lXM=_{c1EXv4Pt#0*kXUWd?>fQvJ>ex4%)T2=NRn42@r6dw)Cau{ZQijz2|8#&uZhN2_-`k-N~mB|E4_d49{#(=zZ&OXpbqO*@OW_&)S6HU78_pBu}Eb$kF^(|2R}Sm$?A17yM%wKC-DO*g>`WWay`A_mgM7LbV0CS841iV1eFz08NM|K{$Lz<m%(kd*6TcWk%~?-WM4>d5ty_B52MN*_IB^xxkInP8*csHxGws$l<jkAHJt_n1$2m#LEOQA2|zTu^x((2qpVcgjdrWv-mPE}5D%J9KnvVC7uEm!AOHTh{|?sW_ka7p026#FDpX?f`@j8fLXjcIbaWK#7@Y*O8NpC<j}YOFoc3B#(j#LKv*1{JlgzV@0oXbd9g%;}j6_jx78F#F9O{nA8+Nsa8@o$oRo6{T=&F^5Z77A_#1~yD2B2o**H}wIPJ+Mtbbtm@t_D!To&u%@!)gk`OLfY?kvR&zHb)@jfgmsiSs^9?Mv+^z=+J7R>7heNKnnytb=LDi8YTmCO7><rI!p$0JZ7un{9~)NWv*=~VL}TFo!H=8-h<79#%H%4?Q8&?esuuP2O0VVp<)}GU5Qp-cUwxD!1d79GYKl^x+d+zT#+;jCX0&Du94;FSnj;m#3oH?1$G+X86yE7O0eNyiCGA}=|R1PL7bcILS>x&axjJ+01N-Se-0!s8wZKVe)n(U&5J1MHvvmwS+d{%_OCNB`2BDHMHAN+8yA=3-~I2P>|a9V|8Yul^t*qai0Jo!`%gk=8_5l$VJfVh9D-B=$FNATJsjJ6Aiz(6nKUHan}DQRC{NQczOaT|*kZK9O5hlW<mA<3U4irj2@?XJ#}k@fTj-Jl#8&d|OAM0|X0V>+K%@F9sfb4>3K5jY9Sy!j)TIeiT?A#qY=GZ~@KNKI<u1#!X1m8sNUcH!)BH9;N1{NXcVT{P5wgU87Cqrp7g0LmPWE}<6bHZiw<7)hZ~qTi)H~|;zT(m%9-KmbTv;SFDG;#fJo9qgY#EB5gunmWf0&8O-~AhMF3e_Vr?OhX-jP52?q7)LcYifOs9-bHP+kuvwBlp-6m-p*vm`;DNV`|J<}wN_(hNYdLWjYE_v7<03SK4Ia}>|jtjYPx#<1ybZ6NQkt$X9C8?pXt5ot-gsm~ei-)d!}iAMJt*)b}f_j3Q>_1P~f8mW-D=1dsW@JB$36C3~TuM#=CF0>_d%G#gvG#D=0*m`hzxM<C~mLN54;%e3Mj)qh&jJXqcd(bCflOQ>n4_J=&3rQ!IWrBxbdu~ne1cm7|llP8bAn1q*wpvHd@Vzw{q9=?~Mog!s>pVsqO%!`!_FBQY$#GkE$7o`@=pdjj-hj0HbCAT6>(o9O067iiSMmy}lHys|T$Vg&OM6tfwo)*E3wohHwB>S8$5Qg7iUL5}y14d3V7||fPtV?Z2d`hgIXQZJbf|=Bs*V2Bvbm!xvJ;&ogG=3YB>iuUe2+(9YJ?ZlxrdTzFaw%?6#|x$fB{nx4gyk>^vf>URNSjE#CS!NT+scJVQ|H@V1!fb6(IB3t{ukXU>2fRwuC$o)5y>8qL4#9pg%mfg}qQs(x&X4XuknOnNwKdvOXv(!@w&aW%<)3L8^@Mn8I_$YFJa27Ld^dnT@`bCOkf*k}Z*#g5ni!N%QhEcHvVmmb!Jnc=JviAEsh52MwYhz=%_rE||+WiR0jc9d*>a$`<vxyq?Q0j-G952k#E|y>E`sj-Q<zd51^e93LE=sS3u4Lszu~YO*2eCM{D_iKn1ysoWrC`srOMY1|k)bX{mhe<g#O{sl8wX|yY@AVKRBqK$-XrqcDJWK~xAKaKRP-^n`a?us@3vU%*@>F=M-Eb@KTE1&jrk`<JwAR$JBVfhU5JR?yI%Z8PsWmjtJDrXQc8O%J~$LO{c#1~<VTM&LsVgIQ~K_gax$7cK%XBvOY;wy0#c;p+#RUZVE3xMOsAK>jA1RrM=6nZ=Xy%l)I$*pR19qO$pyKk}$uUfNRJ=flZ!~6yu+M92<VlZ#~8R$VlM1Z0<K#1fRhf%r(8Dr=Ij5{j19z?0stXgM@{eYGvjFBQIv)igmINx9ivH*fS7e+&nQY3jsn)x=Ff!4u?>p(Aq;QN3nrjI<Nk`j{gF`uKH^A)Nt^kG?>>aaG@+<OF#$a5DY4B!r~Rh-H+2lJud<YFEj@KkYENpoPA%fM8n)vY?0>W(T{r*jv=ws^|6+3qMxd{C5cHg0N7w1C!{uyZE3+ET4$IF!&pQwZ1|+;qb_3JzTh7<3$w0O2RVj{1AEyjha@kw*ue3o5}<OwmXOFo(4!m!AX^SP{D&oa16MP|m}qLYOS5jehV0f9m%`bQkLqLYH@HN<lRj;2;Kh)Pq^+vV?$hGM%CE0oSTIoKL1{RZb9n)MB-=n;Lsy6U38hFTDjmNx}pAae$E3n4_N37{K3*5up77M0BH;ubR@WIKYeGMm+}cAVEQ*TbXC0#}zi)LE-L`7F5Qoy7o;viTeaGto1OhGIT)4hfs-mON4}_TI)6Frh%5HXj^1}y56;a1qJXCg}iw@V5b_|HpARm`{_KUJ&L7iGx8MCBewHY&^4F_ks@qSNT+n#H?V17mm|T`ci*@W5E>s|gFJ+n=aP}AvfwH!4?P7;6C||&W)UdI46op&Vmj|fz?`kyWmqD>?GA(plxMj}XxN9gLRfZq#$#@PD4g-OhwTi*66%IDdlSs&)2!se8uXo96VFGYrvQ`g05Ao064L7uQBrjfMp{E%+$m`rsLCKfKTMJp+4RGka)cKrfnn7;1w9_yFL>Z04p@3c2%>vYoJAi2DPX2uB+%Ftdho#n8T*mBK*Q2;SKQ%OH1bYBN-@a6%fn-+Nfi|-8Nc_Co?!>~>p1LUJ?@YQu_bj^B^-;~YvmO8WHI^gIkJ6zY;uXbGc5`m(@Z?`)8L36-D?q+Cov4KY3X~5!ImH_08~Q54VE=V{LxxOn7ig63jCQ}n3O-~t(kIU?f?dDC&!G%8%ScS!BpD{pvrvN)yOT`y85(;tuGMo$Y64hrf|M9iPV6bVjti;Xy(4+LV|-6KQ+MMt&D5BsBe7rB`|pi<aSv&u)>aEtU(|dRT!(Q4pOhWl1NJ=IcHU`K`p;r^C^mkW+LS#w;^^;y!0{QV4Nf=3oqX${VMhYd;zr50m?(f7|=(MZZ%J5!3gM8FzoVe2)wWhKdQBymPp5wAWbB5U?oE_NrvPb7tT^pRx`Z%I}RcujN~kZhbr(VoKPeL^sfX(NK@{8D$G|Tfv{v_wrG$)9$h0w%VjzSQ3(n9ESSKMNkxvOBe}Zypd20oF$w$>_KAc{AWMUpPqU`@IZ!pU^2-vaWTr}L6xnqmfOX8FqBMh1i@>X>L%yGkaIkaWBtiV@2ibK1P)eFuLg=;MH(9b`-fWg@j$9HFs?w_EjDr{(D=QYSg|hYH>9f@uU0^ibNWA4GCUr8#TO7s&M)iwUYQ6qg#`+P<4z+JJ{j7`20K_JC){8r89KHD5VM1n2+1O-t%eaTOcSb@6mG*$<Z`MBKQmh`c`*Q&MX_!`OrM%1lL>m<J&8^~yogy2ui=Nv&-jS@@lSvS^YUE~u&Hxr%h-<o{xjIvd1I1Od=PGxQjni%lFP?inh*T{+Ol8TO;kDcRxb&dVXMW@Z8R<&D8&}+;hJc!+Rs5?&*ej(bOb|U1spKmymti|4_NX`LE}xPo3-2!y5P=PB($0Pgq!Mpg`?Dd+Vu2q4RT@M-$hu|=%N^qqXBq))xS3BTIt_LjC7CAOEEuAgYUqgE-v}?N<j^flbrg`GD(N7o&fK><JXstF+0f9;EaCRZs~-B71`SZX5(-Z<y%p$Xk`fWLSsxS06g5aW{$U~#Ab#@7H%?bmyh>o_!hrD?(pOd^gJB7kBEm<K8XuCARguA}S@HoBK|ov_k|J5!tJ@xKH}@NwxRV7XwtC~hzq(bn1gCUCE>RPNypgnAyR?EMC+JF>xEvWag`{*yk0c2wQ3fRzTL|a7oD1y{A0fFA{~9(JkF0gHg%^fXixc(&NR1)SvMtr%?jAR~MK_~la9KCnK^1Mts5scWkXViLJ8#62N`)rq<VPa#V&_=~ssZ|?*|)<-!t>Bo-}7MRqLFmn$h*nGm!>?}z<)}bz?(hXM7}Z!8Bh_mSNDrN2m$b=yd9<<iQFZ!G$}VYLkr|=$#CR~Xn@Zb6fe`C?Ej{G+o;f7bDB)i9k)!9Cm%2pxKVX)P7(kJuN{<?y+yYk>gC}>W!F%}GGM|a<*#ZmbIjOXN@6A@Kq?=`V7PFGX(^Y?sA-&nDB!K<x_;R}z!$LG+QN|*7<*SWxIo{M*d{Mpx_C;P*~8w+l?2gUHkL=ZWoxi<=6mh{Etru?wX{Hp>{Lpsu-$EPW-Z{lTD_r!MzMECANjyh$Ck#D8ygcVP0bdFPr1zvqp)s>dQ$0ewM{~eh|Z)d4QZ1{OF4OU<y0rym^>-HWd;aC&{#&T%abOWJVz$Lv6w;gN2|l4W#UtkCB+Uct$}%Hud|FavBBQqn7l5%_W6=pBZoiY+AQgVSWX@#mKS$M&;(@+^r`%Lbbj=JmzECz!c<!Vlw>kF75ENaZve*?zm?b4VkOnwmng?ZmaQ7Sh$I@8t)cB&ZRuR{x-InkUOJaOX#Tw1lbP%jjNyy!jssd>mYAJ@Pp4xvFnTaWcq(0$?bIbN^HBTl=1!_t8?a=%3!uqu9mGrS6?vhw)76$+>9`53z0yEeN{kxaJl%2VPbtw^!_v3d%4Zqn@-~`p_D_%ZU%mCt-W(mA(GB5YGC=<ZuOG(#Ec{+wQ^k8(l{lEs&^g(J-K;EpdzcK4;<tDthN0BR?ve(ZkHy!G!{cWMueVz5$Ksptt-!mUG0@Jry17w7!(a?_Vmb@)XoD_KhgV^CTfvK}!{EY?e>`+YjTif`3~=unuYydpTU$F%#m_)1OXZmv8jFBTJhdF|n@@Jvg>o45qxlSj;C)&c1vgv@>bkI91Cq@q{tWLL1LdQj1fK}R-S(Ey%?sK5Q;aPdg@X|7nHQ)+a`?v8@N!sF1E9R>43tu9G5Re&>HD!(R6`D;C;2+H3Yx??P)!a4gcd1B77jdgQ%*VLtZdx<K1d<}Q(Ya!?*}LQUmuARzb{S*0>Eg7?TSte``l$e9L(X5ce924ZmrR;j5z||=nA>#+?K`yZ$dP53)!-Ome=Th2fN?-V|2f>r2AnICqRdW>(4osM+S|`D==%^M)3aC$xAcS7t!r>T)-H-^ve)O^NIhd78$V#+5=QXqp!JEyAB+)hm3nQ7^k`J*<l&y8_$+iV=f36z+B=_3B-$!00PKX@F`(<wl)lr-}j<Vw%?DB2nE-|0_@rU#66>%$1IunVi@`tae{?c55ND3f#^RJh#7{^h%Y}S0-zi`Ow^`C7nbE9AeX$Ln|fZ1AENO4Pq#Hy3<6<#fw{S&D7mX>R_f2MfU>R&iX#BQ?=!A0WPCK370}~=1x^w=V_MvA9QGrC_G9~f#ntgZD{er&dD^&aq!~y|Dh?7V9cDfj1JVy8lo|Z!$1(`r(5fk)$5iph_NPTdm~`3Xs-8g%598p*o20W(g?k7^(BMcwQF0+pxeIauq3}ab@$3R+d$b2g53{u(98CWhFupuHJ$Um}43fzNHV1Cy3m?5L&;*hCpr!CRLw`BaTqn`!V^R0#P`B}?Nrc^?AHN?p3`vmN#}4wba0C5_-E(~KGuk>|Ty9`&nd(vR_#bSYe{lt+kf7BTcgwRo9eh&p20h0H;S^?qVsFN1lQ%5xsU`lP)8=wY%RO97CZ7sG<T6PtxAlvcFU8Tzr=m}RB=CgaKuqDvryOr>F?=}fu#>}4_G1aaYj`~P1`orskftue(A?0xpyul$PQ>(f=1)S<f^XH)=HjKbgAbxLa-LdR?#BZ8CYZtGHC_UGU!p38t{_8Hy03^f?1&hHKlBp}H4`U|Q8d3<Ma3*$I!R_z*e2{>{;3vRi;V)W6{2A|ozF)8Ab3h#br=Yx8a@Ehi#b&MWE?*S7@aLOxC_BSihc~NNtVt(mb6d^?QxKXX(vB@c+EtMCGh+jyIH~sU<J|{+6;{5_z)w(DKyYHIy*&6!RhJYGc*z0>`bG1x?1<o{9%~RKC<Jt`-kD#sW?b3z6pct9f5!x90|18<Js#4N=Cj)Wst}KjVo*?upKjI+A?74gUlsR`z=6peE4$nE&O?zyd)=^x44?<KaL)>f_h#J$~+Fo;mqkmqRK@OqhkSwTFYD4mVSH}hhHC`if8*Tj!wkO6VSAmHc*(IWB}lJ=uPI&kWGA~;qWv_VX^S-B$$cAw`Xra&6sAwYIyZZF)TmQ1h*I#tKs!w0g^mSuKyUon3mUt<`Wor7lD(fMJwUKZS~`q3bz-qHH>S32PSwAfUaYKFbA@%KWnsGEi`?HLyR12IO|FXXKIYV=P2OGj|0_A*~`PX`<pKgpT82^FuTT66uO5%1rkLvn5TaPu%vep$19cHd?b*_M?5W^d@2CXjt>t%5}*sIt$%u;`Vofm)7F5s!MlGSpS?T%=IEE++3R<w2S?tE)7S6b@VU(PlVQ8RxBVyxhEJaOdqMlj<Mv>CYcF{GXm5A>(O@uove)<h$4?#w?Y&WZIPiBydwwuzwS(=g$E_Ad^nB4q0JeiC{#HMD?6+Gx{;;*PGZ?mZcZNamXwdSXJl@&b^8MXM!>z#+e`gT*0leGoKOR2uclZ3(ur)F*_5H`~EqLD^?rg)OzvcIL_x9Sm+q+Npf_|{`cx&)z3)*V=t-+Jgb{pV+GJN!8@ML#53?B7&O-p-^`+NQX02#ElA3fgP33l4S-fl45*&PM#pg$OHJ$n533H;|j+Vz9&;b^Nr8tguKwDoAW-Fg%}vRc|34Y!}PA8&2#?6w~J{?2eT@^=TpV6f}=gGW2<){~vV?%ww99zfHE!39qq?TsEk*?!z^_qQJh)DoG|cluj<I8p6ke;Ys^^@lq!&c}~N!$(_#o%W;Mtz8)O_6We-8tnOdj~+h?T3deWaeHsOz18-qrR@qV>bqC_-|Qcs?Bkuj)1$MuuTS^i9>0F&y*zsR_3J}k-{_hyuLdYv+*MM{+J+5aI8;>OFJ=^`ndPyRD0byF<&N{wp^Ne)B(qzL4<!8AU>s(k_RMF&rqqo<0PWB5Dw?<oX5lELbVl;j254o5j@<C*qUwB_l2Zb9HqNqX+S%N^00n*CZ-Tb9dHm|<C%@c0`T1K3_&v#x{%{@#+4`e+l?Yh*AmwX}i5in}ucjOeMpmO({bdmAku3uK`ue&V`+S1|z{jXA0}yRD)1B7NlbuFuYq!yAKiS)DJm#VJS(17v?OfbGXgJd_j(L7Vyh0_^JrJRQ$_?qM$jSWb9?{gt4oqoi)QI&2BhzP&OYA{kKvyWgnPlqg!>qAlh<uYA<xA!r&<D^721Q5gf;wzuV}Sf(%qar-_Qp6sUE?aa#RUS@5Jdg^dvnhejT@PVSGpaSY~`hZk+7uTj{-4;Ev*#2#rdivMXL5jpj=0{-Etc3M;D-cW#dU#sRU9w0O%YwK$?&7g5P}Bmm!JK<FH#XCT<G(jiqMERY1m&1>JHExgC{ok7GcaS{R!)nNKb7>M|NKj8Dawch2Y8f1u>Ls51xoR=ZaFB_ngYbcSWaY9++8uB9@DVq+kh<MlOpOOah9@Cq7x?AGO(FWx$cN8yF*WqqU$hB=mPMy837V+`PAt7yo4`4x7?5V3fuRJV-Mx4AJx=5%YEJ7X_B2f&K)O3!&1VqggwkP5-CNxz_U4s_|JTH!z+CKnC#!_89|mP*a>VAVH%?sRh;?Bu>lv!1e9t%|MmWDDoDl2BKiEJuxigKN|MzGR-IXj{-n>oVHEJJ>RKDY81pMK1!)i<K(~<>XZmhSzH0d_9n{hM)lK4}x2!dF(p)#vpq9ty;viUL_o2ASj7%B{Lvz-N+^?96EsLj~XU@(Aa~UVdzQXg{F(%`I<|mZ-wVQAh=cZjj%C#9&1MC-mF5H{o$=i3{%{OtER;Zhxt9oWz*bt;kU$2E+{6xwMVoQ5mlt}qskq2cHg-pU*j)C8y<C+_m#R^`u*p3%XiqKhTeZpaMTJh3tal+j?j4V6J4NWYs(44Vt;;#8?N7!a_t9h4BQcOi(p4}qR!;Yk4{<m3f+?L<W5)V4R{L~G*Jk7zcT&KJI+TXH_xUs5E&r1k_(1)8E`thQW$E*2wAc`&6_4?xpDTO&^ddonv7Aoa#wW@^QMD$=!(T`Q00T&cSa}8`p6+v#1rg4#VdLk`~$NNxEHw85){ZEZDd{;N>)OKXJDc1$`Z{d@fu5^vTC|(V9%`sOn$AMIi{!;Ok;D!&NZp-7w<Y3fpTl>iNd;pPpEP9i?^ry-oeTK+1c^I{t5aPTQP=E__87!$u|?=(ig!4;(cNSBHRf&#AukT3Z3)@)R`L&oWwaw4)@>gpB=rW3@-a`(e)ZLfs@$V!2V!$EBUe}fp<Iq>EQL@k#~0d;?@4!cc(|#+_t~7yVD=;`u>yM$9wJ8-e|DB(*pH+G~C+pgMMqYJp|QyYXEBJaLeD>+37!N`~AV_@ncYQpKR%{ni3ihgBfrL2pPM{4eDfvXQ1yGGXbxJdVV?x!>-KhfsV!)1T>X#6{Jp(%+fB#+5r^ppjCseJ9{9i$^244H!JVo!xA?s5JOPKFbts9)bdbbIN4J8WSrS%I`+4A_o@p=Rcnre8xAd6Ewql2wzvxlqLmMGn(M>9B0w>(s8TbT1~K4rze0BbQg!XerfPlkS*KVom=%I=C3a-dCi2WGc};GR^CYIXs}yJ*1ErjAslj|;7Aml0v^dcoo{N%-S+%s9Wyn_)A(oXchQ#xDmOTnn3gWQ~w7tygVqIZzsJTtPE9Ln3t4wyXN^^?*QNFXJ4*6cEsZQk!OC4fm0}lsFz9D;u#k$2$6vfS*S38ba&5d;W=J;g2$fr8m2@rYmS;P!T$0tqClNpgb4@aP?tC(^!QtB_?96mpO_tHCl_55{>M+)zAg>pr1fhVJKjp0;eA2LQ327b9OamUV&n}WpQ+Mw_haf(yDNh6hE6+zISHO5sigfX)fRTP63cbr$t6SE&O7$&9>&M?_Cbdv~SiZK)A3mppqBj%z+>Co~LFsU5gNb&oW5(st;!P#t9IF2ZO=El^WN$Xv0`2ZA8mh&k%Y;F-)dpF@TN~8eImCjv9Q3SgjWfBh$X<;^OK=M&o$EFD!o$Aud67FHWnWugO(78c?px#TNmlw&cD1!=tt}((`4uk6SRRJJJwhZG;K}lem6AU|D<sXVKQSePKaiaQw3=Jsb&u%H6GSo-P&QX`>CkKXv#m*+vO1@5$fS0U>K+YEc`h#ggOX4%$*>NHDK+sx7j0QfGa5smM-RGy0n24GgX_(afjI-pFPNFPo0uEzCo|8$3vobCP2QU9xZOLJrFU6wwpDQ#X3u!u^U?5tGuuEu^gA6pQT7h-sX8}pJG&af+(z7ATe9ttOJIcyY$>E}~NYZ?UlW_dHY9~i+ratT$E>#!o-_WhRdG(@FCo3xEKjLJ;U6m+A&OX;c;hy9Y)hbL?QP3#NZ-J<eWahYZ^A76tI$P)W^mN~x#ut?b?j4&fs35gS&K5{X6iRRG+kv%L@8@#-H-!x9!^85tev~wB(N5vAIv3RPO+)uaZEPuQozhfgmKewh#Yh6QFlOnd<SV0mE4EY2xX2$$kuw|><|DuHD9aqK-2<j2*M>8{>uzCr{u)m3v=EfuO_>gEFvehe!?iYa$aDKX{*$~S=dRh{(&15TR(*X&_d?Z~Q7W^i==frEiV-Jx=NbfrH^+yYSR651V4$aXJ{55g;LJN=%T3w}{AqFEb$JY7N!3)uWC$#o22F&ZqVxy<i$w|qXI4f{Aq%3YN}1AIh0HosfHz>j(yG)-zmK9h>|#4_D8EmG;IisuF_t+m25}Zysto>wMsNwZW?jBJd42FR@8}oe2mAM{XG<^s-g2&u<!ONz0k{S@?_L!ob#0BqBrcd&Cbg6x{8?@ux9mim{lzsUxn?A^Uern~Zt$r}sVgX2qzs{fWhm9OO5s?&vc9y$m*NWkv@N?70Zl~+UZlk)R|9`s9uJgFMl=K3;xdoJ?<hCc0MFde6HjHTqT{k^tKO~^b#s$n=et2wnqljL6L=Pq>!_0yXjbTJTS7O%LI55x4FkOQjOQrH2xFFRlFyksqKq*Nq`gIDVnCA<Fu0lW{Q}q}xU6mhn{hG=zX$T-M_!c8WD1*WD{mHt<y6LK4st)ApgUQhMrhA&A!CrT?aM2G{p6-nOcFrJf?dp@Tqb&rt8Tk8(Y?foE~XCp5)%-?<2B(57R5Y5fDEuyV$2w%V38rSy<~<Sgy&d_Mp;|iXlO1h(g_K<yr-Moo<mGvH7U^$+Uafx{@R*LP{pC%v~wm=ilYYbsx@5+@qvcaU>12CU}mVgz@)8_KcyhMG5QV4*hwm?cwtBuGQqIinEs3o*p@DZ3vj*Wh+E6IY1v4U2}U`ha|dFPaaOV^BR}>c{}$v;l}VEi6FB4)yy&(Hp?RhMJY9iCS2Xenr|=!j$VGoX1j+R&qAoNC(*(pi<2Xk$2ro1Iwe!3l&V0G+bbe4HmrrjMk5V7)jIQ!*M1Aro2X#2q11)}8o%vN}>qGm?WpLX83YAeBdNp<MvgV#o<TI?6<qIt4{Zo;k7NkB^kT1x7Y6L$yJMiAe4|<x+HxQm=`gxuditA$0emZ90v^p{$pq@Xgl$9ie+ba*yK+L2odYobI3KK$F<czCYbWo>!uka@cS6WdyCn>@o!J>%Qk~CyRIPC027Dym_`8Hhn47g_YD>oIh1T{Nwg#rB<U6|6T&oP(PCCk#RnJhGzME4k`7V0N5{k&)BgDvXI=vr3Nx$AQiXNy(`)@^G!6`_gcdBMyGFwDT(v;=-d$JFI)*tn3DaWJ2;CqL7Yr-o^Jl>3_E!=)Y`CX)}0dRaMlMxelKi?7s@;;xhn?n=2!QLKD?FS!cCsN!4-s|TBys{yKLqLnO3VBf{Jp3Y&6ZYJop<(bv7TTpoJ=1Rf0oh9AGi4m2a3Txd}Z`!z|=t&NVdou5^oIr>4h=zDvEC0lKzP5IE7S#L#YM$40B)K{9?cR!AaksG{HwIeT9U|Bf**$NsjGe61Xw-C3j?uDY&ABwo7#$qbYQm=VG5?O0b{X?3^J@jTfmf%cg^nmItzVWix0uYB($iLD_YDN5kUH1bxpM=&D-%=Hc{{NbCL5K~xh)?GN>B>bW!^Q6)1?w|VikgM6R4$4W1<AeeMmAC$V7!yHc>(YVmCuI_8z>2E+|E-avsPrq}*o03QnLGZhef|2N>Rb7h}ve`j9TLg40C?s881;@@aGZ>#~HT=*h<i)m_dkO6r#12h|O;Z&KJL(<CheErQAjJs<&vFk7b}=3h3=baTGD@VJ>>b2-JWsgVnN17OM2?wy<Sx;=R}ICmXJ1}yl6>KR$D9lHp~?xjs+)JcTZ6k&9;wUsB2SW{CSKrau(bv*d?v_u4p^^{Yyd+6Jp&5}!ajM*)d-Hh@QxLkoR#TLxq_R=orG5KBwo=+Wn4tdDC7$D`Otmsm;(V5XF6tC=t@^H(|&&jxJOE3@9%r1rPI`A)<Q^~}$9uPQSF(e>5BEQ_=CdHFf=5Bz%4bbdc(bI{qZcz3k{^3#XY(3d%oWxYcu+hH2&0le(!;ZV6_BCx!&pI6u$Y&H8Bxga(DGp92l$u8r!V)kL{(o)qV@oDT`J$X;xL8AF!zHr5NZe8hr_tfW2$Qm1Z;H1R@RL<4%~3{~&{<K^R-NSnZlNZfOl@6gFrE{mkVSgbKu0BTrsCGt;68PLa73PJzt`e4b(8{^+N;%QZV;K^^H$mCvB^AhA(a7p9GJHhS;h^kJ=Lo6#_8c?o<jpw?U%zq(JSepQ&+=rtTUFy0UCDA4b)RHa5I5sa5F@<YqF|RW1ZGEd9+!sksS$s&KS^9z1s#0X?2H33U0(IGrT3w4Li_fsU4^{qfyQH9G52&-5M=_ObpT3C4kQBmQJ%038&i8QtLamt~dmh6258mvWb6HWhE<KuCeW`4Xav3+Z>Jyw#wkcHW1vJ%vRttqNe8hirwJ*E$h8K$v6_62Ok~jt7KM@h%8%gknZM><pT)9sKR$Km<b>O(WGuW-y%}NDkr1{rngOgzq5rVBUTSbVH9z*cA|OoM@GbzWYZPt@oUe}t@C0+f$Jth3o;v(v?%L2jkD~v9x4Nr5Mj|n-LlX@4s%=8LEoCn>Dv?WWuqb<q-3(qrgHN3R5ZS-l+1-xLC}GN6+PH7{9tyMJk;tTg+90Wi2A?^z4IwcJQ+C*E0PLJ6N7d>{59rZ=hoiV!$5|$V1!hh9cMd2+ip>4z!z#U;SK0i(R^t6*#a2nbh~!|^8@OO{b9yl%@5sGP9!g_NOnZK*+RbPsFxk=+P+9*F-7%gGgFwsAkFAPy?IBYN3wvdHk+q({{&S>?<!2^bWws`TwRwqUgJ8eFzyLmT^K^M;LKrapEC|%Ymr5c?A{(*&-mb|lS^*3<g^l7QM8AgFGo&$`FwW%;JbO4hEi9uvKf^gSaRc5or+c#HL|bQ1{XKg5gT%_>$6KVZ&2}`vuo&3(}HDX`K4x&Gb&kkCWNlx^0=@C?Tk>d&hc~OQyp6d$MjNX)?HFzs{LhM`GQT0i@bbYZ+<PKSWHZ{ueRj*64twdk87GxI=gkNt31)MG7Fax1YUOyA+2DsTwx^37xb9VpXF&Z+sVzTg=(ZA1k{yP>sBRUEx=bVf|7H)C#6(!&Py7OY7~Fc((<}mGJ+$V6~waI>InLkQ<h>h{2UcJWuv<py4-p>?`|Ej(p)8xcTy&tVRqf_b<7ZX?F5Z?GPSea=L65l%zX&Td|L)!$qk6yc}=fMve8Yu269z(%elN~1-fJ!iPA*%7R%qRS^m73tXpPy*K|n(qw@Vq@GYLs*l!@+)Ri##p5*feFza*sye_Bpe68yV-^c}02(zgpv=k_vH71*zlysw<Ru-}-8uo&~%xZECG9n4id8+UMH_lZPH%zq*acZXcqWoor>3FRWmrgXL$(hF4K+%T;KTK32*U9x%;Z2oi^yJ6EWH5n2yB>^AVr5fEWdTYDvv7*RAlMTTy+o8xGqTtU`=XMc08(0*7eYgJs<eDWO-srdlK-JI$-+0i1ZnI~K{5i_kJ~L2bzIqOrf&+8;~>e~PHshX>NgAlTi^>W>^oPX1unfpAsx`Tv~*A|D=u$QEM*VxVrldVhRRh*vI+)cQ@78p^poK&odtHp6;tp!9~luuRvxNL%HXS}&jTKI?~GOZwquI~JihHYHI{QW<zc022u9EBwy<rLl*&6#dfnwcHuTOkE2pw%dgOUGwfrSz9`Rg$c~g_KMMyDYYD5I-JBFxbUa>WqtME5`a!*oJg80If10%ySgC>}k3!ij53Cp6~$vM~Az(QTPr&M<+f5g){U1ceXE9pHgU3=+5?N9pQ#XOm(^1frB#L1M8MDWTv?l+#1P5Wqxdm*++-2v@iASj!;{mhwLl4!V~oZruQe%Hb+Is#sWSKPjPr&C?*m(yS6IRpY#m;SE&yrE1xGOHEc?UT=hLETEbfgvrT5G1$m7MV9ZF4ZP0I=P@iTdLBWZumR;l5%DsxP;GMDs+K9Qqck7Ti|DADKNGpU1&?_hQlPVa_;Hem}<dzF*hwi$MRK*ZlQ&FipKAzc#WC)*#Gd)4x}<SAMnv61x|>AYhjajHJ)JD6h}9dmH9$%_*AC!6SDb|{CPCD+B%EKz(z3c;(*>(Yj)wL=S`C|Q@Mz22UA#qf)^aW{Zh2au}CjYp`7`GPHRhXkDA;x6_hI)1qzYZL9ktud|c5YUQkcIsG_mbn+`{*!M>VAF>IrEZu>Bl0B%8P`jAh(j;<niBzMkfXH}v_d7@7vvQ!K$yB4Y1wI8#PI>lH}M3z&Pz)MPsu_tM(E4o|h9Ty&x>(=?9*U^>E^VVe5Sc9DkxPsdwzbmGwqG`}}cY?Yu3)fu7a#eUKKD?4i;dJS;+b@Qw6z;&FAt&YgQ#5ZKl$;gS#Jx!-pcX}%#*OU0Q@7@=xA(f;8LhW<%d5`uYGB>-S)J<Tm%{1`@8Id{D?mjho*?y$!%+ZeJxi`rMLT7{S)Jz?|F~(37P?5YKM<t{<Xb`o`7-B+fkpLIfV|Dt7I&!^S8;PNfqYGh%fHTzIDH&6y3kgAhoEh2@U>lanZBsKSd61YF_2$fftFpht%~Fupj0~${Ky(!;aRKI7E8SiSma7=K{goeMK+v<l~T)XZE@l?Sy{rkG(VVhVjHmso93i^ue~tM<xIiK%xhs`Pmw9$qZLX7trlZ+k@r0}VoJB<PKj(t-apFWS~5T)DK9V%M~BH^PCIH@rSjz0&{UHKh7y0eD9<!YLUB3K%MA$Ep_{wGniJDHj%dxISx3gZa4f4Gnt$sJh8+al;P>!^&V=aA$j!-|A^f%Qo-BsLfUN&Rvf&$AgvNdIYhl_tMK%Wk2>Nv2h+pA-BR%^2JndM$3MLm2B}OhaAX^ueTsYjzY0I-oaR<E$%kxH9M(@t$iKP}<i@8bKeTxhx4>FA?2(|Toaa#|u+pA_v?e?W>jf}IG<CNz)$DNX-?$y_4G(UA~uIl1v!7V*4%Q}Yluw}+a*p)p?4SYBv>8Y;gW{G}*f}$^dS5>#bjHh8&@2kT0=`Pq*X@=+W7q{KqR$Vf;<#$>Z%|{c%AP3cDYsC;@G>go_twq-!*TN`R@{!e+eTQ$suD2>P1*n$=m9S^1wgx4*qrKX(HmXUQ)Nw{$pq;tV+%N8}Y?LmMDIE&~xo8+tptOC+^L8VPIlAT}X@9>{l7Kz1QkriGtJbsXoTAvRhEb;Zv>d%DZRVZh56;|WEx+-R>omR`h9!IxSyqlLn%l(aj3P(b$&JV5c=yZK$+<==O*ET#2Dl%=)?_1#FI?w9D7$XGxLud_TYW8Oh3foDvqCe}2eZNx24r0(_~0|s2R4#FM_t0-T&?eq%HM1^fx6|jwCsuwis*DXXS$qlb2SwPhI8Dj)vOC=QoD=U1AaCx3rh+rr-s@?ao!@HT%h(aqMX|(b+H=Vb(N7c1JD)oLPHCKEZ9Bdu8o&x+su0jmgZx%&X)@Z$;$*}JeJZoqVVKPA1b?NUyGYiYyxLv2}`u&=^!Cj)mG33!|kj}iXN8*u3Q18Z7@MRLUw7rTG5D=&HheiPP8-n)x}0GlGy!<;aL`n!PVC4vL&PFRnSB-xFXgHjo7h#mN)n#I3zo8mg9;M7oZ5Qj>S<+S!qFeEE(sj1-7+Fg<P9=s~GM2;=eWhD;3IzYLXUy$u2BDx|)HmK8}BwQkkci1*K8~xkeL`E#?iM%NXM)W01xI#|LG2*Djc;kiy1sjzFs-^;3Ds-D{D{{wuhgl<$d$pw`{S#!^m6V?hs7R%BR%(IcZRCZBbXGRM2qb^}q(HOZ;9%o9YH7-L?fOnZaeua7=%yq3l4kHV_;D0!-L+RGQqYdOYl<JgXeE@Z5-%#6mMk1<#~d?w(U0-JUyEBbCug=$SdMGI$mQ>|G}xvNZd^7aNPLgX%AYQpoh*RSKHDBqqJoPbNHxb&=31qnk+N|rbd7ldlvR;eFxp}6IcWcr7fJk74#rStJz`w@F|6TxgUj&AH+K`<XJf5iOVgN3Xse{;bU%z3LX@`N6hyFuq!;>L!21K++Ic$RyyGY`h69q>a<sPnRSZLXxL0#en+_tzwF*>%qK;gl`6j*6z@G83QV+XkH_`<~7c%3T1XR!%ORau1qid^~^eWMe~3zanL+5f0M(g$e4VpTnKyZP8gM=s1?3{KW8EZ3^i)va2K5l*p&|583~=8LDhAuC+Gh>soNZUqHN9`;sr6aVF#C+SxkQSsu-)OLo&G!O9jiwanN^2e+Gdrqg?6UUd2@Z{I#a{m+q$&}x%AQpcK$g$FP!ohkRKqD;2fvWy2^1w*oCvQka&S1J@KvRW267Z@=+Ou>VU-(Wa&t*q73D0WVt3oyhqy#y~*A;1m0N5+n*Qg=sZ-*mG5D({rbVW-sJur4Yq=%ZZER4%AFuCFMkTu*)P^<=&T#I~T|yhZQWPwU)xrrC|2lkJyBg%^peY2pO-)={j!<)dCKJMmk@CKNfxn=lzLXS{stTArW6`bHx<_V?&+B5K&lVCF5mpL1Wf7&-UHysB#o=I$Dn6R^_a+FKHo&i9{dSE|jsg-|dyDVM5+X9co~6V*_88?Y1C0(S|!71#H|X^S?#GVfUcb{7J!CH5}e;A8I&RRZtzVS>bA+U1*ud50BHC$xvlb}i?6(nX)3(%mn80twnessq<2u;3A$_dm6kHRnPtr^KwK9NkAPsd7e=CD4-^oe<nmmjr;Gyk491eAB^-J1CPm#~&W%eRk!Q2)^CUYQAxBDzR_dL(*hG+4}rJ#*uwKiprFD?;2-Q*|am}UY@+sRV)>qX5vq)*?bz&`72hgnGiMd#ab??s*i?@ZpGB3U=*=q)SQ>N+LesXaVD1X=L&yl<?d6MPbDN=xaalyc;-QJnvN3#^pkw(Fzblyf>-r^b9(3W$>~PK-Sz$Z&y5mM)IMsTpqtH`yFt&yJ~(&!U0<oOo&U{c0hz+ZWTUdNM9-cbgI$r6{Av{=%Q6eL5U}og=y${tTDssmDP4`D#L*buvAS#|CUcXnh0z6;-Eduli{?n|S~KG~Mjzl!9)W49ZUtciC0CZ`>0ykQe33#Sk$0RnX<1(%oVsyQRg^?BkfYza0f)XVS|;1a>eK91d|k3t`E_BZy7&JJm;G|Qn<9lNfA3fhw^vb9V{$E(UcpN6%#UWHw+4{;yil_3^U}Lh&~{H|&La+0Nu2z1Ik&wmjORgVJKDCnBB9+{wr2v0jaHyRE%ar{#YQVTM3ELHqDdu7u5M4OJDl2HLzCTgkh9DGs0<`BH*_$xWAx^i9A;qV(}E{5yB58XDW;kfzkBgrv$CrwSUq=3$AQwM%JK)7WM?cFq3WDE^D1}g;aA6RJWSeobSU1udI6F;@8^J$(}`>xrkZjLwNO$?P1I+~1s|kkv5&tGr}*vTM7Nb9p_FKNP|C|&jWO@yjRFdPD)UWd+2T3CN>?aSm8@W2Htt81oQ$J1V^Rp&mX)YL&IX;BmWyB}c=f|&57m0p9}bnhLETU)ab?v6aK$wzx1n0Cv`eW_sO!y_y2IlA<Pt9=b&0rECi8q;2!I9fNs*!)KG$W0Y@0G>)rwZHoYu<kp<9{1tXsxvuYZITljo5iBh@SzkBw<|_viQ%yE6z?*IcWJ#I-zQWOEMzzKF^(^Wl**tuM`v<|4bz85THgm*VQkT{obQD~WQ=B`FV02}1WMB*|<3Dx4xYQS?Yh%HlK+FyUQYQWIdwXrSue4mXVV8u;Eyr+CAiq7dcH0S{Df>taL3iZqj0%S$9-JW2owt)U&m8|kA#Xo;4AZngd7F-W9s`ahpJwh9+3pdu*~GuV$F-Y|fMx3}AmAB!))6z#pD{bQx!#<>sfUVN4l)g&uLitAG~*OfOb@*;G}Q9osiTH;?P6LOGU1}!Hztny?Pa{Xn#7R8+a9f)a4UBjAtj;U0kUh&vSq9V#`)mjscmlW-4U8Uq+CZoeR@ro;lDaltL^2oUH?_-eBJ+@oQ+>eyDn^S64F_kez6rhLpdCiKkfUgQEBbApbbZ#dEf4+Zw^6vBqkbVE{5$6RT`7jGZj&{uFOr%tB#z=YV9fNI8U~NK~a8&2c_DpRoh%AD0Z?UM%0bP{_L@Bz2^qhDX`&UpG?JZSfP(Wg&?$nQjr>?Sq#6GH%waN{thVPsFnle@4gY|Vj4Q8-nx|zxwb+cCq&T|6=sBEzCR@OXjt}JbyNe{Y);^atr-$*86sc`z<E=5nkHb{wkDkE|WybY_9XJ3y4N-m0*UENd{7-YY@oVmGIwQsVL<g$17I`^xJigA+_FZ^4{5$zs0mU>ol<-fi5^?s{8Ym&;Z`D@<1Ld3es3}ba-7~!kp=%~DWan4t>O~D5Y>(m;G1a;vhZ4>dOG;D!c)^hK2*MQ*$G*}SPK=&vU$e1DOFP_#lYHJVF48*pcx);*rt;ji@^umsygfCsjKK`Sm0OZA>sGBl()v2+WPRn$1Z8caL@xW35Fn*ni4N;JO^RicsbL?TAoccoM0^7ekjBN##&tO@UDgaGyEe93#%D85zH_F=~f2~*vRUDRpRy5xxMe(M@68J^U;hF&H+}yb(Ss7^Pa83zpvtTon*(hetwz#lqV5KlB1t@n-UD80weE=ozH9{?E5ZtblYd5e1B|N7-a$u+&#b{bXBFzD#<cwdc$xi83rg~?qWn|}3eijruN^daz%JCU;eQK7+nsQv!*wP7u!D#5LVI@~VjNz5n3QC1Hb786|eu2A8m_L#OVVLkbGQyo)mZ#F>C@7c!z7qhKD)i>?xjn0DDPaYHjr!2Iy`o0x;Ki>w+pJg%54@eI(^)`=O>$500u%xxXMv%g3F)K7%<Ac7IGqM4fqKpvZRKA4AQlT&YiDM$Ct1lP1#p>C_UJwNBGYeqkq6$PjE|eUkUxiJ^CJwme{`$uAIW4ZY3#>aYs<`?aykhoF^x#6Ar^a9q1XUJ*A8Q4b56x%L<%A`g(TrPc-oRLf!t)!%lwR)329<f3Mx;7q^g>z({`jvf-MPh>sVxo#0JS*f<c-8`ogLuwG=1HiyTgQH)dSVgc>_H*}HOtSjX}EG=clpTRItHhGUT}67kBg$}<ESbCSdX(3i+_Wa?FBepaB1+9OShHmz4V@D9oLcrL!^SVBq>7Td5R;o0C?d6VqdVOkc_WW8y$nprk^xfyZe&pAgcT|QcIxVE-r6|tri)AG2)xSmgBsZw5kEC%U7jJSmt#DbJJNR0x+mdER~mXB7*9Ky);az@e>vC2xR5r}Po(8;Wb&?<JO&5wu|QP_B#uEiV@vuLXHe75dkj$D^Q`kc%T4*MLTW)4IXl=&dSk17I`1uh-_Y7N!&g)>FpJ_3}}QiWAk795{=Nvt<ig>r{TKlT`HL!G4Yo|gw{{3Cyi#`X-8L{V)l$ry@cWqD?4Ta$;ns=1E1lTY_g0lWQ+IKkXRbdQlO9M~ULkUOwe0EjPH!^X%`5_;!xI1XnHsk}ERl^f>r6?~zJHDp#cLLKNgCDB)ACx`t7f?=599hOEImxH(@4lAECmC%~vN%4cjaw>F^v3u?$0qh7V*jn_wQ?jNS1YivJW?XEHDiZXM_8zc~)M<-V7X-(v=YrualGZF!4jz3J098v}FgW(^qyVZtxAlbwbDh9kBha&{ETpQ?{ka3evNQc9uv~V8CB#^~#;E@mU>Y8lUMP-{4TsGdhrxv(d)H8UhC!iVw@i{N#VI-4Jy}G38G7M@|G`7K5-Y(<1BK)@uBLP@ooNbCpsIy<Ib20y504rz_FoCzKtnd5bP8^VGmlwbHw&gjk`PDZkr%ud3_zJuv6K}^-#7}PN0v2R_nkXEU*?xJRrh%4<6Nv=Cm98SgZhokHqj8AjzVU?y6J7JckT0Hs-7yt&tsWSDI<@SxXweojsg#@Ru$&Wb_J%VX^6=eadKJygjSN7EM0A5b|1UWVC-YEoM7gWfKVaP0LpCb$QqakiU+kRb|@tTer`2eyXzQrU>)C{oeHI|bC#n?0P-<V%}w(6C+8W2p$sin(OAbFMunAKzDnu9r+8$`D6AD4l#3T*H5OBZ_lj%2!PV=<rB|QUm-VRlrpf+YCh3*!AuHfBh1|#Vsk`m1x@d1bUa_U+>dD(pD6;zTm2bjGLNhneiUME)%gmt~du8a)Ft}GO3)7;Csr!~(y)M7YW${NNGzWJ@K~pH)_^dmzDal`vD$|PXY5g>pM%ph)Bh5u*0%_mmkCvx|Sf>x7TOTj_McsDLXg!f3aq)`^eb2qu)m+scAh(*7uzIHRD6sYkey8Yx?^x7FIT847W5b^7+?8A1mu=*{yEnIVsV-gV&dWJWgUuVdlYu#LJ#B8b+RuHd(M1b2E1sMcPbF5dYc4BR%Vqsib@V$zqI+e+)Q5`bHl?JDMK(Ak|5NOd#WKHI5LPCW>8(109tOb_f9GNttJm($GRkhzl{ca&K-=h|$vZ;ZQ;0Uibni-9B}xjJYZ5`;3w%AtPCB?_3T&sv_>9wHbP7QabKaB1cmz^t_RNLnhiYDCm>=cJNb$(rciPTbE6X2dDG!CN&t>%6TRRCh-S;obp6J>FAM>$cX>F9dIeeEV?3Ueuey=0kIyFlJQKRoPU(O@91HA)TmvK+ee+H;Y=QV)}vP>qCO&K>?Ti$X>=rLTgR95^L=qlf`a!$|rfTQ&6mEdrXT}c;RVF;4Mrcx3rz$H{gs+2#{Sw>*2(_O0)?;#wLBj)0Zo>lC;bigI$7Mwcs<#=!8Pr``6_M;1sD6{c|#u&_Cz5A?Nkncxv<Rp36E+bUNWKzt+zze2n7@>N9pF<-AQ=w5G*4**T)6pl8I<5|xpAre>U1&Llxb_jon8SpVO@Mrv05Z-B26{AoWr_0JWvwlhkwVlWe;Rju<)Wf7Uml`s!6Xd=8HoC0kAw7y!O{TsE6J{IM&3}NxWWpym9<S#I)+(r%(U*6gZ$cmvKv8Gu(p!4N;qSsK@3*wI`WbgirZ);lPw6{LMlzhwza&-=%if1i3%PrVk~B9IVSaqWd(g^Om1l?GuYM}ZMfm^6SG3x2^DW*Ru*VN#pswVWvTbJ-!aeOLvwUlnmCVRt)xHNUyV+dXx?yh_@GTdQ~A7D_8BUN7!Bjaz0n6PcEhwT<iu{tyf>*LX!_2+b!pxh@JtAXeC9Z08|-k{k-+9J)iagr*$oosa;%*9KT_UV*1W|KDo)e~BOKkk=>@R<w4+2R6#1u)SymYmsk?j2Rf&k)z2m7@U`pmVD=k<Y&Do0%@o!wK*OJ&|HRYFPb|u@!%fo--&Z1prd!jV}qZ(K3N<LtN6Jo*GTT7&94l{x?t<dZ~cY#?#tMA`nXvPvTAeF?e@25IXEpvR^eB|g$u3@R0`;F^mnNFRLIgye=;ibU*Cw`Y4C+#UhfO(Fzi-Sb3ZYzJA8(v`X7|O^*3QV&~KFCPO@S%UJ=Aes`fp*PR8fX{u5D+{Vy#S=@3354@r!ehS-n@Dt=2LXzQD0b%#7w=XFP=fK^AuIrO_=)#zvBJJG2p^e(GLI<F%#8LdS9_Ht2~NTdtKOElfXw0Tc~vK=A9&U;e@UUtD_-I(M(s1H(x*zRasL$2|rABt0>>#nlHVfxZHb`=|b84oQGXTv`9ml{u;&-Ev?tcT(qTLB~E8sw6rC|3<wQ{%)w~2WR)vk*kCC?k9Hib1O(VTCD8YgF>&pRBXb!GtH_pgph^kgQg<G5Y&S|d{-g^JWiA1j#ki{nWsn|p7t32rUYEcPsWyzYh4h|r{lM@I2t{%>Xj_ur(?x&~Ka|5=my{gxVlgjepZ4nIV?LXM1RZj-W=F)#g}#*HlGPyVkBgdnXxC+7;?%@C1ZiQg!NqxF@@v+sn3-J~yKwO}Sm^@CM`3w7T~+=vH#-&QlF9-%8lDB)MXKCyOwDS!cM3ZfB1fhxi<PAlkaIfx$VreV4|F|5YSkq^J_&5z1b1)s+8iK^)nm;ZO3-8&TYgFM6~3GRQX{sMjLWwqOq;XK7Gn$-W(OB4#LtT{v6K@oY=1$mbvjg7^O@~xv?#NP<U`P0bsOj9nj);z48dRRW(yOxw_95~#rl+fisJEVDx#USI@XfA6sLT=nnaXE4Mshjm+mV~gfU2cenKb~1oQ&M7%)j`*y4jVbUb@H$f)9-pvq$BCprHG@dN0yR2+yZ<DTmlw_n$>9Z)<+-iO^%$AGT97)8D}-ID7_n%Jw{5QTAdmyf!p2u`=6{X*yy&(>4a&ir|nfRqvr#4x#zc~6@7J3Jr$a7edw@c_0UgVFu7b0N7gf6mo+#VagZk#c$in#OC6AaI}*C#yfT+_!njDbKP1SsWy-r2j2YVm%$~@eZ?y*@@BKI>;&uOmO(WKS2y7<W*O_ib<a&dJLyw2S)|)Znav4<mPly566O2SWzc{xl9VGfp$y&Cz~;sjG+RIR+rN~6%8<|KTHVEJ1>hWD&HRFMMymmf;~#;t=ii*aLF?2%ifM%l~O8n=vcQ$TjgB*z^A=kD=lU|O+$yI6byeA1`%uy7GS)QPxtP-_)*#8<4aE)Oce+_a;O*n)WZTJJO!tb5(t8h*<13eCy3e}7F=3LhZX0Mz=WMIgf8+Y{h{9xj>E3gj1oZB8>W3D(9Up$*Z*?Kyy-?Dpci=(AQht*1?17W5K&Iq-onyUuz)@IrM=f`@u4i0DPRO8OEljtjL4@^Tz(nAQYeKvjNjxwm>%VCF%ZmF-W+Y3r=3H#iWn91`;|c0t*AqIY=qSdYNVv^!d5a6IC}{ov3yOUY#oTq9f?A6wSsp`3uc-Ca&`l}+}hn$C+sb~%K2U4V1sTdzb)W>tGWA-<xIYnlcFI?OJKL01E^mU1QmB0AyzTryHnB*=;eGcvAo>ZyfN~HMFVfbsGe3YknUdkNE#<*R9cNAs1|5QTq(ujJXu)PwM^-Ssy$ub8~bDs)j83nQ?=A`YDJd>ZO=zyYwv5$fa5Cp9RCwP=&3KH)hW+suQVZ<Z>QD_(;>`J#!>5#uhVNpz_kp*`EJD=uSK>SI1xj(cw7dz1&eU`sO3oA6w~V5OeQvQ!?qhg;iV#RiDU37y6oxf-R?Sd_?r9uQyh&(qP(lL1W27aqOH+BF}3c!b?6eQ?RU{5x7N?=pTB!^_V)B>|D|_&^vy8_1?4;N@8a<5<5O&MwD)M_Z;!V3T6^35(W58*C%cb=z4oKgllE?Z_wmlq-`&}Mvb&A;=5LN(9lk#G&R)MeJvhoWbXS3H4qqNMCj{wdJ^lh13;GAFd=$+o9~m9v%=}R%q6BtG_`QrTubt;WikRtO*L+2s&u8SlJY`8duY!2B$zIV7?xk_H)tttlqrw`$kG~v-S9sCqBJRHbd;k*{%-(<g6@UAQVDRLxpZr7|B20Dh4SLPp)&)BYfh=_f$lo{(Mx1u`>kwbl);_ktvR#WPB8QLd(*^dICnqntm)^$4QG9_BH#auUt9rI-ARWP{|M!3U_22zpfAM#J^S}Ow|NbAE@7}>HEFv5Dthaw~0s`dLo_OhpFazT(o2H%3&2VzjOvfluz`jg2`~L7E*o3kkmTh3!#^e31XGdE#QwJs27!WH;E^eF2>|ztk0&k5PQ4(J?+grOkn>FtBjSsAUMQ>E5fqm|)RvXW@+t1q1KCaJ~XyRGXCzWZ?-)Dnl8Z`E{cJ`k?w>rGOzHXASD8=v1LA=SoHoyP=X7v5-?Ixpv8s{y*wrX0rP-DB<ZZ)<aJ=)uU_V-~pNq0)`e~QlsFu+;&dJH5o2MZUXq02r`LO{(lSkb90)i^ltp8c}Xcy$1O4!xabt8p<4g4pV10MY{<GGeN5bo#)qBCefo_HP^Upb-x@p{4@hMVNg(?^96KbRiT#!N>FdCKcNJ)vtc>yxo>l_xIE6HVV3FG6?<1f`e-SkD0jTB`$Al{BnL9ieHWW_~J|8oL~L)Cp%kDzud$}@$dgn@u%}R3dBh`FZ=i=!05miOKJl}kKoS*wtz2wH4ftO+!w#ZckufJK3)eg6pX3<AAbE;Tzg}K;RAjP+xTD%gc-Lo1wp`5kdb&B48}2GFgA3E9y%uhQ(huh--L7@<{+7XvWYKG{Jtbec}S1G!{DXlvm5s3L_K~Q_tWW9{#&@9L6blj(`gdZSvbh5AYsauH-Xkk()N|u*pRe0Ov!m@6wXrg2xfvb0evHcT>w2t{tS2%Y{`o`#V*CIAwz@>8(GqTf5g~Oyaj}~xv?RRGbZXOuvKDsNZk3Lo=A8U1s51gMM!16t|EM-((@WUIYHP!(e4`W#RaGdacpQVK;&4CN1ctJ=@X`&ui?dX4_l3Lgv{|enO!y&i5LIrpZ(^q#jD^t<u?tOCnArnM~&7ljNuQz{*8DG%}<e%;oRvY_X2dojSYwPZfrC~FHacPB5&td@9_D9YBi_BQSE#Y(8nq`4QKhr`GN@qW(#vVl#EVXlM^r>FwKG}K>5kc%^&{qUy0YSB<Rd8QEFPsJB9cXNP|dc1NdMVFld=m5=@c*zW*GmX7F8ht=42UDhbsLG(pbI&`7)0`g`2>mzz_G<Fa8;A0r>b-+?4|*+^NL%-xbxEvAYwv){i&_G}DG3u{O@mPu<N6pi`d3dkKf>rkKVCd@ra35^$X5X6S!#1dfSQecRByvQtNn?UV{q0YW0R|nBMwfBV*e^OkOE$r*-Fqly&CY&%Mh6ETDU3Lc(5a5WRM=3R>Ds(g2>xT1qNXN?Xjuw+L7ojFJ@!2L8q3<Au*MX%$=Rs@ZA=38~n6imKm?Z@FB*dpExB?D}U~VIr`(_VXVvq(96=~kKyvc`(1b}2XxKxf~0cHW9OxXh1Cd>y$EJJiemq(1cL!F{SNOCdrr(+HtM69Q2WVoLNw+=PcJAGf{Ac|uYTxTB1Ta+%jTSLIBA{T2?y9?2Qz+1L}99beGhr_Et+Ex_{L4~Vf#Nejf`0t?O=WHn2z>|C!e=u!ahj_w20d-?`+XzOZa1aJ`HxI<^In{4ub81@U1E#*mKn@z@+{28A9S`_I2J+SvR9p^kCwqs^pTUbUc1>Y_Ns>3z-iY#Jd)zZUWTxa6$+me2b#j&wTer8bKN!qmWAJ)kg()&j<eJEgaYLiiS_%jn;<%x%;SCzA2e&p4_>2D;7n5Bt`SlO_1xVs2VSEYyobfHzAMkUd(a?XWI8l9++L0q@Y4?F_0ELH-;{(w1zV?if5EaNhKxgFQhs(nyedM?dbnt<qqJd2N;k?@1R6ok<D%^oj|LLFo<9`G*H=P3M`5GEz*Z{QlR=c?a|9{flYHxL(v|1pcI!(=|eDzIP|EX=YtpCHW#h=YXASh=TcF&LmpxOr!3I<GpFbxXsA1?6W!-u|~j^PQURWYD)tK`=ve1Hmrq0u0r8Pg$<?gJFTxkoFJMZQZ+a^Q;rmu<v?8L40mp^NH;Xx!>@Xpy_Zl3Wbl17o924n$luX38XmJ*Ca;Cd<PGwE7T4EA;vx$SetgP=HH1MU==u9xy)+orvopucn5X_v)a!$Z0+_W!VpPb~Q@S;hO|PB}yWDPUaV5@uAUR2LKG7_`#z8J56yqM^Yv+H9zGC?+*7pG=n@lIr0vVzBxWPI_tLNlnaHx;ywKbqE4d$vJUkkf*T(%*QcA$&%E}Q*WP{NZ8g80Uf6|!)bY(fv1h+d{i}c*X>O{xLe^(96rtw8UOSXXQI{jW04)J8y-qm#G0MCjbekVKirX=&lfXPwx#s7T&H=w@n<d{E#Fo$)^^~gm0v!bCrE<Jf^+FT^Cm<dm7Lqg&%r5f6mJuJCN*n2Y_UiT9qjP%xc|cx6lSx1hed5(O$A`!JVjpC@2A`mzw-X6!8yk;X;>9!Z&FTJ25GIISs&#FOm&EQIqz5_W7EfE4FG#za#o!QR97jB#^nrXQBa1aDPHf;tbUNT^Z;-(iR3(y6G}G9ZFMviJ@+l5I8JWRm9v#CyJ{YX}05v`=!YurFOiQFh$OZB<k5o6Vv5;Ed!@pnYAunLf*7!|B8gHo~gJ(EQy>;0o%*HjAAqb6ZX;L?9ZC!3x?t5gu2}S+Dtstesq8`Jv!zTUz@?UHI@-Wo}=x85nIXa7GRj5;-8x%COLL1RlWE69yNTl;Vxgu-u8iy}VXvRoaZd|b3=n7A|?AK?fQc$y&j3mRnNTsXnbB)C!ma=t9_xk+--bJN0PRX#4k}mQt4Gitk<bvYj!rql0k)#|Cq-(3x((x7@-Lr;%C*-qDThXC9lCp!IL&`Qvcc^NyhcDe#3e1RfVj9rKjW=*`1Jj&>*c>tb#USDmQJJ9)3dbDagt6rT_y=3RhKN&hWy&#@2U!?dX_(Dz8UO<6ygvcS0F_2mo*d~zaU&cZ!1ae;|I-x1*CVX}PM4vM(G>e!fyk3QV<ZU`T|z|w`#!l1020`lzJcwQk05h^0oB`&p(F~rOaT<UgclqQKvP#9?#LgYR~SbEMBQG>;DFZ{uLV)astmGrFr|b*&4-C;G*>4;o#+m5miYiS%2F6?5SSGiRBUq!VnrD92KOoc*NwP^(Wb1Ar{m;W+yY`@1N3he_!F97wCF(7cqycA(?hL2s`G5iY9u$7F2eNEq?sgv!xxl657!tZGFC_SM|d1T0N4wUjmtm|b&!!@G?7MYR%Ip94wWzkNO38fgrf5lXM@mTl%T7-Q)!+`<%RuZV4Lidp-(B6)LWm{Gr2^XJHqOzU4<j)h6#<QsSZN|VCGT!726?!vCVG3`Z6Vj>nlYMSYw>en7^mQmt-W`FE?dbv_*1lLnBIesgJQKFg*mNWQoWO02BSI<2Ne6G>q+FoFo8y5YoLe?ut6UM7t`Awk+w)(TrdsSAI|gV)FxCb!&>_k)CeSqGb<tI$>iYD7MoNaK5nvy*MA#K_G<*N7v{NLYa~xr8@qAqyUEqd|%(F)M+4oV8<Wb>jJ$Iss9fKcH{=l@53pvL1vxSE7qINPMsR?J6{7{H8Qm0g4j==XoJJV;ike(PSnLN&6++IZnj$O4}bXeUwlBX7uDy!+vUFQ5pkCpTcKGlWH-0{LGkE~jh7%OlKlY#$?8c`mUgXjAkezy0BMMtUu`6!#n*a6Q~Vs+%cI7x!f74%DS38bUJok#f#6|T7AJ)+?+-FrpDbU{tc6O1Mc7OnARSPaLc~~Xgnap_Whsv<bRvF8dsvG_IoX+p#mb4yaAJgd9g>w14EtBOsdw*57|(Ag8Jj<NeI}~UK%b$_l#Xkif@0}Q8dq5EJD8AqtF;Axw&BkX{Mm&+d-vk2@)~Yya+U2?teX7QR$YD`#zS<dt4cvyyn%_DW){SQ$&fT$P`WnzVj+%xvFOXz!Y|tkzw9jhvb*rh-s0(TVK*D$i~O6d!kg{IQ+FhBRjP7$WP|5jfQi={<#XF2PZN;<ljKFD)sS4Z8+Dv7<C2M9kC(7bQBPNB>Ga-ITUx>g+r;2J-vm|L8is;8fF&Mp!t)<~^Ot}4&9CAAqCj<>uQrXu1iOAzPjBP<P3_?E6PphI>DlX7OyJ8KfmPcq@zH%y8~djQj=H6OdGH$<aG2in0DmKE^4e~`XQvL+Vs1otj81h)!|#g~5c<`nZ1_y8d?;5^?Kh@c>0mbRN~&t~d^V$F7ueq%W?Jz}jSZO7SR5e@fA{uz<FT1tlo3!gWZ|MIJ^-aesh5?{59YS#1cWovCJEs`AdUSXsvpL4pd=qOc3M_$qVXRtrr8IJq)zb+A!REC>Nf$KqCU_SL1>SxQQQ-$thJ`%X&=ZLEmhJY=%lIG*idPfm42$nMbfF>3mY49ry$EG6%80U22(PT4b)XHka+N*h{W|4Fi-UEW%G(ZL(vI#@rN&efVv(f6#3z+n|A%SUBB6?-)`0QsS%d|x&Mc+J~YiaFz|9xpK<#FrU!Uxi2|E`Nv2%738s{HNmNLGjS}Qq&+rh4jYafEi&UJbRnCx<=V?NmDdb5u?mdK8H3<i(;6rMa{kk&!jvMt;%a|qAF2YBy&Ij90|I{O%-LgDBRAGFi=^D9jQ=gd|kyxJ&{8y+6qnX2K%YQb<!?KLm-2_T}n5`iIj4L}E{myP23(aYmD=ZEYtmwx=1$(2t04m+~%n6&M`&k7bOy)oN?f>^T|D8Am?3(yfrBe59&S8q_{Mi`v2oDRN7Z`sE<y`9zb^Yi6RO}Bf<K#LDh8KZlw}YC`_Fo*Glsc;51%x5mO&TdkfSXj1mD?UDVaeBrr=``-4&J?O9G-l0aN_qjWkFdD!+tHQb{ov&i@`XEul-F^<LfXzOa|>|t9w!<WyP1lh5xl5KY8@C)m2T|nvMK_DWsnf5ugKQ&9M#9(!W4yd`QFeR%0ctcW@w{{SsZ2ULA^^X6vI|la|?an&LS{{M8@kcmO(OTp<1df)j|Eqo(cVLx&7O5M6&8#($Cy=2<fs&YOc+kxM;V@$dgNyLt51^X<`=5j+eTMpuId=)cW4z*N@jcoEr77=Ml%<{)H|gx`ZoV!U}^q@y|7e%LCZX2gs>l;=*`SA}dZmWmMefz845p8!Ee3rK+Ka(eeYdQvrj9!3pvFCijbCz1iw3D`FN*A;hvOfSiM?vMSNWPjx0RTtzdh6`{K&g;@Q41dbc${);>(EeZe>X%eVxq#stz=yg*4%sc<zT%-mw{OhU;9}v7?(G1;ATQzp^vfWi)t8_4Q}TY(?gWe;zLcp!ce|M5ZK45aLf@b#b0FFsae@HTNpf0~Ai$=8K6vEc&Le(_b}^x*AJ;}r6W?~?_h+E@9)`h1QWpn6xdDhc^y@ggD59Tnzy_|TtZ;b!=WcDtW##!S5*nC?9Cyq5!z48I+pT8%@!szC=2g3-iUOB}5;gosTkX9@1OC}=-^1s7Et6Jcj`~fN`l>m_V0RV$9{=)S&qk^I@)JYfaHs5wXNvfXlWf>TG}$B<jg&$pMj-{ZAlKouH;2DyD38oza>)iHo^{0j6hj_0wwf*8aPWT_dJa>enO`$#Jej<gY0};tpQs3RWJYo-!iFKn#x@50(TwA!pP>g08udp}aO3Hps-0v;cYN6_&yN}+^nK@3XAd;JSsKvysxpsyQGiZ&WUG<S(0uG6>rL@q_!SJCRCskoY2eROKjK#Ubph1qJ1-pG)Wzi${;SJczU@>%eASgj7%<fy8pu6_J5E9c-J0!Ib9K`ItMsTI1*s|j&JCW|#7|eW01&roF0>SyH~1J>{|gM2S%iW+K~Tyd_^c~RL9gY&V*Tnt0BH<xX3TyoJ2O4yCM%mvIl>`-ExdehVpW87Q3dR}!1$#k)zvuk9Q1{XWEJCkmg8W}ft?gNyfKP~a6TKv43+QSR}eP%r!L?xf&~6O=X=79w$(y$3@Q>}j~YK08-gLN<@@C-ZtpN<R);11E$D{Zayu)ZWnOvcmC3<dm>;}xQW!?9nI%=ZU!4^xZ(vb9g;WL)bWPCxt37_i?yJ?P4nj{@Q!5;~Ynmr9sLyuWwe#{0C01$zR#il;Y7v}z8MUCh%DG)xw|7L%A%;_o`U>KVx^P3kDdnZcG{j)@TD~06wxR&<91(a7++2RcxK2Qa$Kvl{UA&ozV#l@q`QDAm%njg`IcTDa4U5f9F|FhOuhnBMSBWP+2r)Ne6%@Vk^!mHTSL5sHcLNN!O~q<UJ8HC{*^P}YK)ONAo%YahA~{W<b$b}|25=S=pd@^C5<{(cWLclmc~cdA8^+lCg?DYwWt}b#11@;<F5v5=c;i&hvi%X7uz&I_L(4Q7R-@F9`%*0rnp&VCJyUojei2k#Zp(UC%ktMN%3rJUFatPvj4h$M>UB8b4DRAZqZME1o@z}wpVi#t%cj?ze)7!u&u#kq2|6F6AuvCoZLzr!m^y}xfWMX$QFEnksRAWfvea!1vmP||xXsdqFE(`jpoTO?^yh<1Sfv6q{2r#JS^{(3nkk@LD38kg&Si~Wuhr=Fszxo`%>lh83g@7D5GM1j@4!f{M3cF7qS4jm!Y3NFrt|loYs8h2MI{!4k_j8cGy!5cTLnQ|dnJv!35NB~l!-bdcWE|dv%!&BFuP^&sikyTu=uzyIZ7RK)XoAvv6N@##)^$+C-{b1k;ZHeT}j>wFgTJMNVBlOH>2)g^K^uR?kg$#k8nj~uGA^E-7K37$@5@17%B(Ygqaq#Q0^*d^!h%SC26&48mL>10fQZ4xUQ~5#UzRJ;uQG!0Qwb%<x#jkYgtt=_XWb9E$d09>QqHZkL3X+>y^rV@0}airUjJNK9}`BxBDtiO%Dgsq0xMSBXEA6>xoM+Pb#K`N^AK!X=bx9s@B9$#NYkRfBkoV^N;@Hzx?wD^+=D{lPyl7fcdEbM-cy?|MFk`$A9-f{fGbdud#_A4<>&GB>ZC_MsZ1b;a9C_hbKdHL~%OGHbAYmdLpdG{(tt~{X3GQNEH58>T1q+Ts1DWq#oO_`e>g!9*<!T<H6Xou={CrO0DjaR93&HA9|SeJ%$Brzy|N~Hm`+UjF;WD4cC`7e&9dmjx^(+_!qv2$VcU~s=C!0k3oZ(>8i?%jEszojEsnkEP$>OyO>Hv^E2sWyJ@JG(5EK5bD@SGK&otIpeu(;d;iO<%TgQNSXj(@TOeV1QHNHp)zic~8!yyJwn3t&2ZN-Ywwj0GQ=n38a|zF0HDIlm<A_J>_0KqxP(mHUYnbwmj3+hx2h$0F3j376)0_*KBmmbqT)Pe6bcYPSN6SoY(BBV@0ti+e8rsAX!1D!l#NLPUBa-UyRJ88yKx?NDLIJN$9*sk;89}3zKPw{5e-}1Kczp_=E|V^sFz^bQiBe6392j3Xb1(qm8r?917p1N}wSL3M(YeOJt!4MG;FTfbY28F0Q)#e|5rJF3;Yf%S^ZNQSpW)q>+x6l0Cfix5L(TRf`Fwry(~n%du|O#z=nF*!U@EE`3zRwm6^35x5cf5dvQ)i9Mw0~EK8N|15eS2?jwhYYhLU7*tM$_T5V2`cQEUcP*+zcV3!ZtxL)~Hh!3X0&{1sUcYZ%a&imfD04gmzGMHsDJSdSd|Qe6)h7xamBsch%gY_}<wWCj}IxW5aMG^Br^TSGZ0O%M%wA=Q+lrpuRgYkGOi)t_et(WJAI@nsJ;{=>+U%vSRvP1;_ht#%eP${AA*4#nzTPAnL3qnkN%g2M2rf5^JR;f`jU{4Q_<tlDr6Q{ZkxikvNx<9-h92sn;b_Q-WLuRdAO{4LI<Zw4%-HQb<RgPF4@@;&GGyoB6W^+)WDFldQde1Zw49`|tr;~7a}$=Lv@UEx_hp=^IE$;~?z06E8j-SN_%6_|_2f_!|}NU00eg;t?KLw$(V9g|9_M(Qi26QZF%>hP34Hq-}M%9f6j5V78=*kB}0fWjhJXNTd!0}wd<(I~;g)wJtw0HsDZC$K&t$&@ATfDC;OJNrRiMhC@h$Akd<>!`X?QkAIzp1Pp%lr=3o;;A1V1n19FQZPYPN_=#!KfZzuqj5KFvkXW&{7CiD$z(`((-$|r>#em{&mVw028S4r)Eg8z=hzeyF!PQ&3f1zjqWspR$1OS7(qq*sU6Iy5`2IMHqhbHC6LpU_575_qxxJk}p7oyEfjap6$rRNZO#6rbhSIeN5GI|}u&uDdJmL-m!I)eb_1S?9c1jb}MLNsTrXWA)0Nv$=KL)@Hbh2pNXK5`_p^}1XG#RouX_0UML9p<+c81-?nwy7Yih*T=;srp3fI_u=dS4uFYwL(;hVGt=R)gylHbz>V3=b42V~CLO<ufHq1(fwPKtRK+3wpQ)X@pcQ1;vmSfGArS_6u~s7CT17F6v{h0t~LwAjtznk|;H!zdX~UK(u*4i1N1rRWcu0sAVb>1);@L0Pj)Y0XpQT6-Dn3IvgoVYxB@h!&Zp}S2x@oT(g?lsL8XGz(N}vR)N|hSFT>UcKPCy8yhvFtt(rCH%9?;X~itq-US9hTz=~M#ztK9Il26I7cXHUqp4!V*cmH#nM;2Q|KHdk=9c*a&7QDGD>c{86B(To;+twpZ7^mT6Vuvu4f_C+tj4x7IU>|bvjek~o$S(E!nngGh|&?>9Ek=zmh-OJfwBH#ZgOnW5inU8u@PBIiPC6Qwrr0#)C8tHcd0rN($1E)l8f0&*<AH6eEQj6*lcptkTZdO^Kp5@J3bmX*>f_W3%x=WH56qWMm5>f$mnWK3ZsVXsZ~UXnTG7LS%iy^2JgA65t*bse=v*b)3>41#4MciNFyhNF}~gAZkkA$6tb};mv+oC)u_dI<T6TqwxBQ%em2sDJxNtA<W-BdqL4rzLslfF#F9gFd4vqyl4u%Jy|Z6NfCQzWKTIe~HAk9Xrz);XWhvs#%?T6CcwH*C#-2nvd7~&7=b|is^b{6wiS8y*Wr0{_1#f#^7_w__Ae^vDYk4IjMyEgAZw}j>a9MYoxdLG^XskGfTX|tF)Xc&GLIriPE6h&BkJEwlp@WV%87v{mn-i;De#D){p3}iU3aW2mj9^MLf^RGpNKaCaErcX4Xpv%z$ts2~;=qa6bneO#Je%h)t`&7(H(wCp)F=$gkRe7)um{V_*Bqp4i}ls@5<h^((4sPvEK139!H2K@;OJ{F9KZOD+h6<5hu{C;&Y%9+5OV7HXTQGv-oGDz?@vkv)SD90bV$&IS~HsPL$yej%zFNk8aj7g|K;u1e|YEhe`mPd`O}|{UU*X=bo-UBGK7xb_};BQeM@akTnV9x+@%~{UwcRIeDC)6USm*?{`iMmAN=gj8()?q>zb#rYrd%h#VJaB?H~W^bH~5<_3dB$7j*XcM{gXx{Oa+$zr6E@|1B1RJFkBE_;+8?nFy)rL?=?SIn&B%FojMgvFt1rKr{eUr-HL&HEJ+A-1)=T?tJikzL<UZ<Cl+K_=8S~TzH;W@p;%<6_<p}@KGf6*1)Z=yPYWFJL0=cq@O-_^zyflzW&nfU%n3f$|lOKw_nN$hNJrD_g*;q$-mrs`yI*5N8fql&Og3?^u~L4o`381YcC!B>2;`a^!z)2e(xIwf5E)=@Yloo3^S<-^XW~PK;J)l`JZmR{jH<#{NVWeZ^8TpjE}$l>pLI(6HMS7o#)=WVdsyQgnn3Q*KDTJx~U=RUG{R!ft1HDe);J2_dooPAAx+i;lFn#Z+eM|Sr2-}YqC=@D8$nx=Bu@Jy0qYJF9lLtn0l~W8kL2ot*ty(V;TZmOf@w}HYwm}%5JRa_PVw!2vW_4GH2TfrCt#B1omc_iue=N20{+HyKF`gMujsw%9$PI%#LzqM>(^joY_&%>?mh;lruX@t`6fy+O$E~W)g~Yk6hJpI<ku@U0ZY=N;FsNgto<0L`$ABlMBXnKV|V(?1DkJn4Ih0e3q^Gscp$!bkdYy7k!MS=tO&LpQ+0a@ov!KbT?$GPn%@Qb)f*S&Y~u<i`Us+*wB+apZIIOQo1%tjo>j*6VuqKMKx9)IuHN-&H4KB1D}j7xZSkbYpgBT9*hI=*SfR$wpq%lOKFOZin+ytZU$p_-BE~u+-Lx#YQY4K-M?t4Q9yfBS8WswhOWIfw6XCt&xaxZX%>u1mVudw@|Qpi>iD>yjA?Dv`mXFkulM;$Qmd-Wa^$OeO%w=e8sswqRo}MkP#YCgN+T{$Gig&FFSOvdWELt@s3$et!x?JT_`i|6u7e_2-Ba2@1hce^;N8~3&)O-)9`g*KMlkjEuz8qxLWFPJcU1`q<j~pfmXddec^GyatmTmK@YjXNh||t`0Yb#6V}81ZfU^_Md0icIaSyNvh^7aA+i8CAP*L0S4G3#m+Zc$7ky0^N%8Z)&Ho;X@w27<cin=(tz$=CHB~=Z(7Av1;5yo@LEJUv%Yz(wfj*R3C{ehqt{^qFXM_yP8EIc~Mf8nV>wWK`sONT(&!A*HeS<_oFdn|`WnH;es8n^^p_HJJb2Q=?Gys>Zz+Gd74rtgTXX}K=11*#v|r#P0yQPM@(wxI8!xAab4ddCw7Un62-)aGFf?cOh;!;AcB2M=7VtzjuxbJzUk*4MR>>h|df%w}gYOfc<eYfp^ZZaVCx9VWO}9z?}v8wNc;?Q$--{muu+&;923U*6_#SPitM55(Cc1&9Vw9>0U4bYSN~J<_18DTVQHZaofG*BNL!c%%V#fs$Ar|L|LP-v7@WOO{OxePvn`K^=yI6cXoaf!*WCr+alg*zqLKyZCq$mZ^jKqalOLDFvG&IlWj)@I=xF0l=PlaB&Nbr5ukqcsvgL1k?I-^!`8Jdi%Mf7k`j<%V=b&VZ5o~*+~+J&iDTV5BQTDTv4VLiGHAKa|psQ)Q{tD(H;`Gxr>pvb?wj+a1+{xSx>jJ4v+=tL+cVA*w-<@lfJ`tI=qg9h!r0B+>?)9elmIT@?%#oU%Ku)83j~$<BzxA`S#KK|9<Bizd!oPpN{_VJAZ!f#iN(M`r&i0eE8+>-u~8q-u{no9liA5w}1176F0%7><F5%j-idbm7gKX_uBP!Yh%tquGMNa5O?d2Zi`PdcHg!7VOT927T$cVIl^fWLboO$VLln@B6+kk+1l!)>?E|jDR$Cj$BaiYT{}*by>57aj^6w_qwdiQUqAZB?{B^R&e6|)h<@3xefiGoZy&w+&e0oh-+KF{<L|%q;gA3A&+mN$2I0e3zH|K3caFdD-CKYB-O<b6KK|Bw$KU=n4$jYh_~8%Vy7j@&Zh!yB@Z-Z*eymwVb|f<U{1*<B#IgFCfQQ^zPk9l3?te#z#7poyV))@WQ4ckUP*k9UW3F1_h=&m7IGeNr>cmyRi*y8(d0^j#7OlM|R{Mn2z85Q2NkwX3LyCP@apEWA)ss>!<fq;^IW&&8Drf2hU9`C%>K!Qc6oSbFB%Cs9*}5^|p3WQCcQv5&Qvzc>T6=okRW8>jX$_SK#(5uO1?~F<Oy#?Lxnmuj<)jnTrbn}k0$gJ3AlYI5c}JP>{A47Xx*=~*8N6{HuV$C_Vsn@Qcb!*&dBil=Amqjqi+vHsPB9Er4~6=kpG8y^^H4!F*4m5;P9?h7k*$8;J^IE=cYgWe?N`2f=l}fo(Qm#L{C#lrvzL$m<#}x~V1h2K%F(}l_4dnOhgu+qZol=bqxZf)D|ag}U-)5;#^32^6f?ip6~yS3uYUL+KRSBn)fr|5K>E%Pj-Jm&F6*RcPH;wAlVn=2I?1$tGn|iJe_zL$D~G@T(Ve&6;w!~}zk2KKZ{GRMKQrpU3KBTyVx-6LXuQQwr4l}|r+Q>zar~3Fj(_l`92dZa@)kUI>+Rn%?AgHGdjF-{@4pV+<$e3v4_U8(`PnjMBj%deP~4arIzf>k@7I$GNEH9xL{a5$13&o0IK_jXORDI&=^BM3vZgU)y+IQciLAHnQWrKm&E9T8sW2PMwF}~fdonxIoTy3#a4gdqZvXo`w}1TphyVB*yeG^-Yq<Tr*Few9(Yf{RyGP&rWgg%{z`pa9<L|%u=l5PQ&5f&p^oIv`{`B3Wul-u6iZ49*=|_%V`{mJl-_fZjW<U9i1Du<LB7uM5=;vR#jRW?_qnH1QqyDoW(kB(fz6JuPiUQ*J&9{#J_`l0QDD+)rISm3v8=?wu1`8eqc+3B5#Z{F?Vl@~~Fn0^9di3HuN5A+5%9&sN6m(`r0#wUh{R(`AH#Bbi|H@?K|J?cDdq+R{xlZGpxJ<Z8Z4V^_5!yBMXNumLqIag~nR;A4LXsQLn=FHqgyuIl7wR?7t70Q^R6!Hqa?Apy#p_saUiO}WqO4Mclq*k%UNg?yQW=k3Q-vwtY!)PT!=#>U7`0@J0<4*d0aj5cr=e$W$?>CLja(^}xiRRImKH;in%4N6JrJCrG*jkz@xqX`bL-9vKRJ5$o!jqz_xR^Om5Y_wXMg+X+rNO9s31K5RxYAh788dWE_)oR;#3Z@O9W`z^Sve(gr+N(8(3%vtWVc9={|$j>r-xDcoPPL>1IX|bvS0xTuY=$p-Ne@r&P1vNUaYKwu{SS@i@Ck!aP!rou|sF@vN&n@vCo)6XoV<bqVI?Bpr<vz|G+{C2{thRQDZoX9vq~w!8AII{X&z@wSsrwkbY6!ID?ADu)9MUF2UL&06D_ZvGSBDaE$<%o5kji=&4zT{S{!x;p5}qeK+KGlaK|4%NFi@D2uGrw8y9z7U`AD#X8V$a{mD>op4l>NYV`d!l_oX9vq$J83s-*(i~h)#V9mlmadg{vt`U(=l+s3+_Qkjj(L5S8Jzufo|5LjOx%a9tY_xOZSrE4KO06K=(U<E+4|kkHl44<ke+VRT5n6HB&)r{3+dz#$&RL$<6k%v0B2+_>m7BHC)SYaaA+F$dQH`=HRQ2aH`$lxM^9HImQhIcjde6Ttn%mQD1hVVRTw8r*^iXcijgL$<(k3=gM7!G;3&2c0LkGt7PYJKm;?Bs!*ieP6lfv1Ji5Z-9XD~i|szoKi04UPg+~s3EpVL!gw&%n)wyiGXF>P#_Tma7;Lgkv1;pT5tLydmtjl1lWmO~7jyuq$cPf7#)B4c=r@uN=0an+=KD%+sT&L1o&F{!XN+wUiaTYEg&RF_j7h~k%hK(FaqYRYT#-Ma4l6QS;Nd-bDPub|pxf;NN~Ch<jsHG+<Bz0j{MYS&dF%KkzS7<K;3v2ze*G1A#J%Gm|A5=-=igyv*n6n;f1mpb01-$`#)vku2VpHI6ia-gbjleNqW(dKXfQnElA;sLr-Nbtj-V#jAD9bMl^>>Cos_bj@H^8R0gaiwzPbzmO%2ZX!TIyXd{Wp%U0Fhfm@#WmUUMDm!_7aNJe1$;fDtEnb9$2S>^Vt-f;mZe{v4?g!ty+1-J{<%adY?+^d)TPcZOp!A%5G76r90R^i>JuU=S(2mAI8W)D8;XmtQfCu^Q^U#Yr_94>cjeP9b31zPrc`C_8CqfW@ESH+5&eYs&RIcXO7WrYDjiTwO5^CWx*)<ihq^@x|}*IL2!5P&6B=l9i^Ty1ZK5L+1Rpb)Yr2p<JAm*PH?0d=Z}(VaB8-Y*BQ9ySysT#vG>|Z6l}y?1~b6C*#U;^#?U?f(JUOdG*WPNGe4tZ%wGXo516suiV^Edv#18x!u8gX`7RAABGCB9BSw)?;RM2tq8GxAb6w$njzkJg~7#Fo0}X(tK|IqcaF+lhw032Zro|&g|{gstY*gtHxH@NN6Q+;8@mz3#pMSV>&rC>`*Y$_c!Iy*#QV$XA}ev1SdLGeUnNf`@A$o80l#DPoO<!}IB?zVhICxqB0Ay?U8AL~&g1}I_cxc2p_Z;*xpev3Q<rOc&t+f5<&_48^7>VdP)%VBOm;@`%BkED$yfeMe$uShAIJ*B6yp?n%AT!Hm%_8J;(T6DwmE8Ur7S3MRMU!*0?+P39<hhf)>tlP|5HVb#aQD-c9$7{!a~*TN;45qu-~DJ=gC4Zb*ltVi1fsHF?nssevhfB>f6?i`UGooqQ8M%cb6nFCih%}_v?x|4%$y(*jl?<L(e#yS7ZOUCqR5b#nt+vgnaC0qn+?zmD4y179SQTSaf5+L2#aL31P-}`&m0YFvHa^j>o-65aZB3B&-Fk^T<_$lvAFPQ>v4`D4&euDk3Nqy$8M8r-%K?0ON4Uwz-W@qsE0gvq}mG4Fy^ppmNWyL}*ehmKl^QTe-=axYY0M)!U&4LwnGo=B$40&IQWeT%dps{d59+J1a0{@MS7(KKeUJ0nai2j~Y@O?j$O9H^sM`#)n<;Vb}PuCqC@)52`aPR&KeusoiiR8j}Jlyb0Tnh0UN9>_BnAd^1=HLiim8ts4swMkSB4?MZ(!3QfFn7`rpU*wbL_%>;v}3S*h)V63rYYDv34PGGz-lVyHq1@;ybBD;^k<jCwK3>%38!-$YW!}3l8soOwbN&`r}+1QMVP6!^9@P?F0_v?Jh4y5JSA4H~%dNLjL((wtTRLtf0=?E@SP;Emmgx(G9D=gNS9-IYcs$a}@4VXqzm&jb1zs%IyzzF^ozid%4)P>6haaOp2S3BX&?!9e?_`*Kar|E;<V)ahcwNB_zPSOh)Yl~(6N5dXY=MkPB9;v}oI;|oB?niIMh2C=Uyw#=xx>gR1=bX1f%^gPHNEfkROI8VY3^}AhtWOWIZb6)t;s+8Fh3Zm8a>%V6LohumHOSM}p3VimK4`~>-7f4|vz8cImh&D}X{1cd@`tqhu*m@)fz4@WFCDpfe+JzkXZ(6)&XOy_s^&FB*9qNQ6$i^oIK3DWeWioqA-R#;>^4TcVu|O*z(tC30W_9P7_z5qI2~!O)p7jW!QC0f`Ox1KJX6x&Cb4=++kt)%szLi(PWK+D4zNYlFRPfI+I;6$Mr`wxtOi)y=IQxAjHO?-;O{tInY%gDSzv7G)X&ejo0^s%P`YVuZYGmh+q|FT=fe9=7`~CGG{=+&UF9##;yGYaPsPJ5TF<$B9}OJoVs_}cpWitNrsQ?u4#zI~dO*z(wdkIm7$cr3XwFMStw#P}7Q_Cs4l(vcAG0>BM03__-aR88B1<~y9=i^fsp75?cN|9nu!RltOPPA5g%k{1c!V+U>>9^y-C2WeE2&CJdF*E5?jYyM$dxOOK8Xk#Rqt1rrXjG`6`_ViF^YVUS(jx&qnJ6c7TK^sO$-D5>ngxU0ROQw2#TQ>`iVKs7lRcST?lY)umbRk?jgG7>b5dV<WD?y=Tbfy_Kf!U+sY@=&D!mbBd<~0OEpn|Q7yrZ!%U;9cTPg3tJM%_ZpJDs&py^qhJ{8Fpp|O}lN(^FZDR6ft8ZW|BTo(lug!htG!zN<*511Yu<%!PB>)X!eJ;|F)nt=#YB3`>6o|JJVT?W2IMnrUXSBAsvhJEy8C|>*IzS*tVvdyMRgaI@(=^soT6Q&>hXq@oXBR9vPzO@Ez!9`47YW!uR@k+HQ$&Ghi<eL9f&zRV>m|&|sBJXK(lqs6)2_vJT5Bhmo>ZrGD5r|^VQxBWk$zmDR-dW5qTm+W7(t;rrdJMqlg*kbt$uef8K;V_73WF`rjwNGa}!-+`{-01YTI{HYZEG%4~6wIZJ9KI(q3rYe<mSIJ5#B(HAb|M|FTH%4CQ;{y{NKPP~qpR+tgcyGtI@Jxl9JWP1|HpxPdU&Z1^!8vHL!Ki}&g8W#+jvp?gm6x!s9ADi#?ggTkE&e6G4LajGs*pEC^3eQ7Y8cI$E8z4HL*Sr2c}Mzi2#5Fvl!UF<+-6d8Cu78?Cv_%Sew(kgsB6miA^Csx3pDOJFIqE%OCK>vQGnhJMn+FH2`numO;x;yYrb=iT*z`s8z+Zw2Kt8Wa<S_KVOn7B@`Vk$d^<RjFfv*tG1qd(o8j7C|rSIrtPB6u*tH9xrk8pzJ1x0|%HE<32(Fb26?Q<aiVTw7@>vK&6GIsErYCNN7V*XeKXHHY#db%Go|6JwGjfZx!YcKIARF-#FVZcCWy!600&JqUuBH5OS1oLJ&zzy}@mP*a2u;enPO%c+B3%g)6qfXtAiJbJ6z5D!?4mmT&s<)HGAMxcdg-ag&MMM{IUsq7xim~VB-=Y>_nwt4PrM-+n9!1_k|N?w3%4$OoJn%okF^|xl7XDk$-abjpus+|f-iH+?&K`FMHitl`Fz4xg(OflV-CEi@&XsYjx1sY>J>RYRZ5-h0HP~PftiA{ovhP`L<T$J0PI)`|aj#W;>ANG~c;Nf)>O~+dBjE*GZzlyRG8-E7V!CMI2Z?|nXtunl(9nBo#q(D@OrbRX(@WBukr-TAtp^+XY<mhH2ynlh)c0f&B2Yt8)LOX4Z)3)-OFu)ih<yvRdJ2(K+G{~xeFLMZ%20}Q5TQ?Ry-5&>EQXxDW@L!<39%AeSA%5e=_{IVd7E?atFKSQsv%F?x!55`j%|-b4eDI0CQNS!jg*Yi4cvhiDWz3ZNdD=M##qJhmWaCqf%$%v9Tq2Gv*i1tZd+gNz&MIzEh{u#gS76t~lO!l+(0pO8$ptG#{lSNd)`knQ!<)v>5Ps6JOsv#D=~*By_!fn=6;h@ifdAG@mV|nN_+PR5p~Xxr*DgG?vSJlcq!1KYS$Xhp90v!_1y^A(*lkIvdO$<cUMG4e2P4u_r{+V6!G@UcT@@CDPg|Uw9c_)3p^vnb<HWbB3_=S+T4VJv-65S<3rB6XscntLu{Y54vhAL=Y4*!JwGktMZ4=>%e)3c8@~{dWZ81*R+*r69M{MLN8$v;xW_Sc2F7<m`8FYyrWH~y1lk}Q<*;thRJiZd8>EUs9w3vQB)*oI1b)nh0`nUwb4)5qto}zbOpxc)OF!de5lt?EDC8I{y9H;Yr3exi26nmLJDySMS3H*KXi06GK?s;(=QTI0v5k65`+XZ{T>2E!&K?_FQb8<yHZ;~HL@nsosWF_E_-R5}E>5n_vCbW&e=n{Df5>eZ&X%SXf<}r7g0yJsH!F7yFiG$Da>zQIakk-$!6sNk=AwS&?01_>o!4pHga{$k>iJ`V@wus&ncBas(&X2;eLD9&DeUahWwXIHikjI(}0Igw)vVd87y3Nt9deEqbf9Va1Rf5+Wv7>ZYQF0MxHQwq_OPJ)NKgtO2tUXH71HhIAtW_zl#J9z2DwJ&Nkg7*T6ZBd$+e<Iu#FwSqL-6?3CxF5|Iywj=%Fmcz?rx^-Hs%6&f}_OVkifHEC3Jey*e@${Gc*TkYB%FvHo~+Jm@t4$CHALDuWfzmbh-&+p9U@?v5~L9_X%XS=M#<vtOG2I3Cm7vUQNM^Fq3KBY!XZ0ip?Ov&OgCB1sk8xkL-)_NMO~)1N&)bk_--stCMlFx09dU*Y6HXk%8>Q1x9KCBR9eM^Sk?a^s|z0R1~%k0m%UZ`J6v^iosyz#U{21JRdk*?B#<EZJZa&AFN_>K(Uy7uo^W77uz=9x$ZVohMZq;ro3_C?B)XA2lq#Z0YB#vH0P<${+rc^L$tfq=#6zVg~l;xh0|WRW_oDIa>+-G*a<7zP{o3posJ)z#&hHcEaAf@(?Ra@T-wLBww-2gdqP)R@T@FIK=zZhyQ(%=%-gI&WZVqQSwn4x_(FYQ7T?K0cFJ4x7+i&?)$f)QlW2j}ZY(_Bhdoi&a{%Zx4}mYbWP)@B8tkM^$}wZt#IJH0WaV^&EH9T&VdsGl*Ibdf`KOGZ!FXkh&wMRwL4^WmIknZ8!}C)iX3u9{+dr*M)cAhOiPtl(0FC?%B%vmHNM?Euwa=fYU^z##K58c^U((UrV65_?$+eLS3Us3moWxv)S2GN&51nYoK_z$qtG-7YLj#^tF?JIsD&oohi1if!TbfGf8suVptvLbfm2*x=*&_tGAsV!Tu9=gsV6|c<EnGtsV5Lim2GJ?C6r9{r%U+T2{w77K3|$S_Bo_i(U)drr{mMt#Pl2ifZmg2dQTHqGfLS%|zRD$8vo@pbIUFhkQ;DQHq^`5HDDJb=h(u2_@1+8xV6}j$jgV^M@Dp3O#oVv$^K0yuqv`5xX3Y_=<uH2}Y$QgQ?+<etb<*B88VqU6x<;pM)$1-bJKF$L$+cPXFCgkFFNs%_@q7`!!o7BSkd{<n>l7jc1+Th-BRNQQV2eTaP`tO5g&w!40=m{s7=Ck;Lzym3bR6X<fC7b_D2E1qN4awp*1%YV<@mr6ZR^n!I3oSgD5sb(DsXfk>GwN+lsSPq%}r1ui8@C0oJ<APDQg{F#A?;AUf0`$$|&f=JQb823*hsPt0Px>V}EsRq+l}hL{9oC$Tkrg8EYykk>Mi4E`l~B7GI3XRB<FWBC8Uck;r-Lg##{@{8=d`=#E(kPd<DyMc<UwyQ7)epS&F^`KP|)*b<5CUrg&IKGm#buZBQeS$vd2R>nMi4l6*pgjkBn0E!ht5go4pi5112kq<vjq+TTVBX;K+LWXrC?_k<&?eNkFk7MKm>l3SL3w6Ms<;NF22=;f<VHzygmMN@cl;<ZQPp2dJX))_<@v_-=uRrwBU*~&HDI}oyFv8ov_|<WkY>LteVg?7615%a?U?vXJwkyK(Flb_OD4J+NeoeI-AytBMbn=<c@nGM%q#3j1Wi>xIy^kh$P(n1(2{E3GO{z9vOnVRiA*9Oe=Mcj##0@r_nmo#K=fRpv_$Q#X=pq=3NE2`v>*8JI{1`wB?`0mjSwLGF*rQ+^vOs^~;^jc(XLDtb_O1{4{0cF*OT-Suj(FO1;w|0|tD%$BOFhThF(rR77(V8**A#o0h)BxFS79%Ev01T%4m|BktKFy;EC-aD9^Hj_|9sHuq(U;7;O)hzoGqY^<O}cOqsseK?A3A`hiXw{&Wak9;u0k^7M)1Ty=XQfXh*R{j%G0#eZp>qBvXB8tb&d$V&rdS&9RiN`<k}0fVh_xk!{CZuFa(`_d(H}rKV#lRMx^+(d}bJe~&#hYNvZy%VE4hnZ|GDQlPs}IOvGw?_9SyYyPxHFwXol4c(?02cJ)_UcS~~IK;tczo0$3jv!MX8buw1wyu8e(r1&WuKa(ODLf-ckgu~23!X2y#+h$}lwp%6A-FfOXk%~HbTfjR&r3Js+)QO<@)v88jIFg3mxz4qT^wO+F=J7^%TY{csg@m=1t?AA54CZh0|p70-><6O;4L^yWSl>+TaRklIPHc}*@7;NL%M0s#dSiN!b`goO3X59So(~Uw|FVH>80UOVq)i+6Vbg!{?m8SIQ!hlf5s&oT1=bVQ0Fced6gS(gHwqt&2piq_my$;e7N6k-wn|e&6cXS?rp?9%g$-{b>V$dbmfOTr&Fn2Y~Z!+7ilZ%RW<6F5%x6}>JjJ;&t!v8)5zjN9m{!`M7GI(tgXhuDxOpuz`y(W?>dH0WBt(Kh@dQoP;?*uj*6i32>+~y(&olOYiF>3bMfJw{lU!^M$oWwk!%XDtQ}zMZvRK*pQl_~;|Jk5+(PA$X-V>6LvtKz--RSJw~2kc&%0qCKyUz$N_0C3Mx;VD3seB$^0=M13od~?$Llp+#raK}6&sgB3ji0bO2|j3*sLoMHG1FguL3oQy2YJl(^%`ebbpl+kfY)hai)tY^zS9gNTa&cbTE-k0MViQo!C_rfaZ2Y5r7P;F3QMrkWeCeq~w>Lb%;6ctzLCwfEONUEgBjze`_BO^NDO<OlVoVjeP#PH1*iq;JRmqtF)<225n41HVr<$53wHz-Ui_WVvVay9dIvco#rU_%3)NjdovamGg!lTj{U~ChPoLk)P@`SV_a?kYdGGS)y|(+N6tL`q8ntnopq+gone9->A_GDMBIB7T6AAC_uN+cmT_~4$B7$!aanW<&U3-F&t1Q~8tf(GM*AQcFSW;wgJfm#AXx?;@3ZSl0~z3m8+eyA-D!>zMgVJ#cv!nhEvHmyb$OAayvCuSv}ebt&7KM1NqR7VwNlmo@&!OZkt=~YBiq^7Ubh`Yl8_KXyXo`R;MuYI@^%<Gu*6j@{+6m`w4WM<4!mMP;1ZS0$Q$QoNW+9zBq4dnS|(HKhL;Z8X*+ndRl=f#;bVB^B@}{w4Euw_&}cNaY?pBW?<XyGSIa^VJGkIZWl9nIK(+>v4&)B*+_m9oXio|h^&pvdsKlvFefX=6mP&1kb)~pi!6lm3j&P0p49y+A@o9gD?n($gMW=5#LYB@B?7%;*W$Ui>5yo*yNdg_;+)a>&q8KQ{*SPdgPD={(bhYO~FWtsVRk#HX>S66ghIKUvbbh|OozNAY`DVzt!*iNFkf1AGeTtV7={T+y+^9tN1@VbyGS!-3fmO*1_Zy(s3qTwEC3ZAlD!Je3#*O<paqvDRNvQH#JWD;)VKl8!MVVN#G3vV9$L-M8aB(Fn0QPLr$DDNLuoP6<9Hvb($%xA~=j3WzCK)54hofLBkmeAF9Ipox0)~=`?Cvp38p|-!AE0z$rL_A7wHxn@bi|r595?A^mKn8(5in$xr9!MFZn!H1m+SMsV!P=fv!%Jp8VQ~ZVF#UN#KGPpZDiHhZVqQWzkW@jEdQ<aTG~VemEFS-c@|BpJGkWIoMS0Eus(wV^Eij>yhjz8Dlg*Dhw>qnoFI<kGVCLNjWvy9)-a43_K4yIJg%=}K3P%Wen*AWtl(-bXBBBqfmK7gW}@PF0YN;ME2oHaZKA869fN*<praL)M<5goY%wg(wt^n{5L-3vio|<b&7xj-m`W?8^Hm<p!?r%SpEun~3&z0SSP(I0%<C5#-(hP^mzqhNzO^jeSTM%j0%?5LL3aAR?Gl(Oa})yCHigr+BJ31}*b7rab=a0n@ikx-tkjCEHF6K9CIE|u@>!=gZn9Kt%k8}0|Ky&gezHAmw(W?`q=hHd5@tyuf)>Sa;sw?@QAVrEVYdoVn=%3}uCP`XhgljkV$N})zR@qPWemHw*IsR6KA!pc7!u8@Hx{sUWaS$Rcx-IJ;7zvw^B^vGO<wirv1{F&q{BnYx~0>`oNlZRnYc_bS=5Njer3z6A_#^tP?JSgJ#E@FjOVzb=F{*>B?ExLi4Eg9wo?7k<TtT2&vX*kj6h)E8Exm+Vk4i28tj%Ls8BOLIEA0LjH>pdHJNP-N;H2~D9(>Pk1A;XwsG8{j5~by4$rp7Rcoo;aml8jZ3L2-i23WA*y+nPf9V4kpLi3J@Lk9}HY5Dn=qh^=G4A*5R>V}>xq@nK8OTDD^tTdmDe-z0MVmt-;(N*)MO`IDKTPXWqJA*t2ig9Z9)#O0IqyR4u5xu2855)AOpePLq&7<ToQEq>?08#`Vz+V3IyvSnr=55LQ7g}wmz;?zi6P151(~y08iG*8K6as47VaU3Hh&X265OEmNVcw2CQH)V@_M!KVYRIySFI&ACF5(TL=|I-_4%2w3Ch{i=v`DY5BxK|1df;tdNJRWu}aM-7gD_O{$nE*9BYpt_?jugF%ZFE2N;C9Z~O}Rh$>EDOsG`5%%ZR|i-q8{xAXJfXj>E6g6s-ztZLQB<xmb*2@z5?EFLgf1j%j6FyI+xjB==&Ao9`GDmcx+RWZTLYJKsInyEEb9<mZboEdTEXT$-fIz1cCd@U^MsU`I!UA1GL6&t8-RdSpjDz{u!2d___DzN|ZtyQxh9tkmMRVQh5*UpZdaxnk&f&3>3;-4{qf41QLxdiT43feCX*so55X#(|68k#?SSiT*L>I&qf(luJBF!jHA-j+VBp6rMKvZoVng*iCgA%VX*8@DflQ?YIvs@Jr}?OE;Ge?=y3D}CQRWz%k^bFyj2GWrr$pVf+~B4YJyZK18YLW+mdAeX;Hd$t@vZnbk#;mqj$mtgchiN)LG5`m2R(p3ouJR9F7Gr7-7ZlwsE!BAeD?l1D6_;(td4a%6+KNAOE#4uT{lVsXnSv76j7#I5)yaCbORipGil>TDn(nX?I^o2!$M)th1?DTj<aHnEwgfkoYnT`COw~@2?iQ<4lEpZFWrYTzk;Sm=(OLnds8{`}eLc#qpR*f_uw94Xgd0uHW((a(<r+eiQ1Ge2(Q(9m^poI+x)AKqOk+quZ@>Y+aw*juY6TvXV)&Llu8+OUIb@%%^W&Bb<%;c(2kigB9_QGY(^@~1My+AAVTHuZKz-`wz3YFUR<*6mh8oVYxtw)78Rr3m1=L)BG=X?xYL}y-%GcQJw7o(XLUQ5svna5{wZ72e%$nj8=8H?`WaPM`e_6R8Xq{_S?c4n)}ewov9;(X)|kgBeXD$bHK=SR8o1DTK+k}+}8!8nwYu|_v8rQ>us3ZsY_f6a=N_^Grel`7D=v}ZIt^K4Uj6GEOzY$IPbRem=GA2WX5M}>c~nrh`Lxo6CL6;8X){PWJ7Ue1}jly9bDt{IcO)iccG@=kXT`Erw(YeD3f{4?iGnNxBTt83?iqwoCa_=T4N3`@Rr`<1WWe)p}TcV2b+nK;pBaxY2Tbyz2HCf&V+E0zz*aavk&E8)dF3$e3M(JrBC!jVF)y*vTTv*z92M?Zb-&IdpK@c9>y-u&I24}Nm|+AoiO{;lJ0zjpiOuitv->u0X9dAP==^lY8jllGVC8e`MCiffEnbf!4rJ*z9qHZS=1_mZQsaYBh<rg2U=8!;VWr|0uxbHnfWLUkoXLwhuZpGu6<U$?U=C++-=&C0O3Eza}tD&fF?y!}O{8z?5<PJt&Vp2G3N%;jWo72X@lc;XvwDIWyJ4tgos4?`1v?7T4Re9>)JQFW3Qc$RW@i#67>C~#6QQvoF1gOnF6oEeV7ci-XkkqUwtT~7*APwRX#s{R!mn6M|f(`BK2FF%EFDn5|A<HeY}<^ELM40E%te_Wjm)w}{_T6@yk+ODp-yHxd}D6hC&pLOWMXZr2wuPYy|CUuW%6CaNSecELiW_og(5*gvoSt=_iYgV9n$Y%xiJtL#NZdzW22cZ-nF}n+HM@1E?<Tjn!O*xyd^)ye8Rm7u`6MUXdQ_oeHAT$)uc1AY-`U6@TCoR|gD!)kyWL9liEr!!%nD+X^Zqm(qd9h`0F>*Hqqw$cg^JaIh>4tYKOgF(okCaRQ*cFJ>K+M{qn5eboI9OTNZ)`*e$h8JgV&qY)*+~og!;%d-<jRy67@Dw7a3iAv%3vW0Cg-I>02Xsia~G!BBL~|})h#wsS$EqU{qA;!aia98^BE=u)Cot4L7IJ#u>Oh1Na@Z=hlqw;o3a=V(CMVgq}6odgG6x{s#f!J!O@$)JO0rdx88p4*4w{5`tfg%{`F0Vz{$flJ6Uf8a8f#i8S6qT7I#=dbB-&yG+^c>ANK7Y?u$K~Ufni169eTIj>Q!%ZMqSjehxNBgZ^+F4BDIjFaA{<rQ-zCZKq)w#tgWwB=A_&F>YDLH{0DXk3!Ay#eBKrW~V{fFXh{)<kmD%QVxKOphQg*2`y5~lu(~LXUD<Ank<KklsbDHe5#>;SaBxN)<t#_G>kW%DpGie8uCTO6vuYEAbhGcW*6H_q~|pezE-@2&}AD2S|UAN7I}FpO#CRidCO9c8q@fwJ!)CQXO!}T5ISbVH0YI8sh+*>1h-z;x?73w*j6M<oZB;akZ15w&&Ba<YjO?!v7`cRa*StJ6{lwAo^}mm7RE(2Ts%<pu?jFfuAuO1Avlu-d>=}1(wV~dp-RVWHYqLPA4^?#b4tdBE+^xH^1gBOe5(M;9!1~BB4DmDbi6Od#~wk)`wCrhk6%#%FIR|Z>zc0)|0+}M@b5)j*)R@FQ2;<k0Oe>cTL;$IOFR8mHa;x5U`K&J>6tNSq;TNXtfApsfY$2~G|nu-k!?XSU{x@nco%2Wega7OO&Vf#K1CbGY(ea&Z0-4o1Jb9nmrRplX-c!oX&6K1Y0Q|90b+i3k+Oa6-LQw;YnF_;m>=e56FJ!k&BGS*QJFE$j+F2}&Cw_wj^)}`oAfe(8Aj3=fHy#`+yk-rqEKWRqqN9-cr{hLy&iB*`p#6L^ipY>d)PceK8u#G5G{We0{?Ljfp-SE&m{(4vCxT|NM#VbW8Cek@V9@KS{I?~np^NNY6Ip}lx0rFw=?tZRQy}a!_yjJ=3{#~iM8Y;=9>8$Sf-f@6{bV+lyF54ORODp#Ng5L$>IR0K2AA>q+0EtL0IH3V**WK{hEu_>jb8%6Bx?oYTddg<_;q5nL0UhR-ZXJIOS>1OAjxmSSL`s=hxuzBMG2_ImiRJFzCj_#&Acy;qyVbM<ztInCnhc!h!Wz*zJa0tji*y@eJ~Anhx+c9+4{oAyLx-vDekXu?{qXP@=Y!|K@GnAFQN{pNyhftJCZbLWeH}Pt7!d_)!5FnPR~~o)3Kf&T8UNS6EbDy5FeI#)e_AsFrl$>hqs!x{=G<<EwZIJTwoQCDn<RoS15fI`Xy&p4po4MtkCIyxP3g?}8kGeRh8@9R|%#2iX~fEM7Al2jiW|h{91}@7*~J`aSq-G#Ow>!YJ6xj}yTjKG@I3J3+6xmyP{ovqXb1TKf9royAc}1;=!C)6<7pp}d-tfGI(^I-nx97&fy}8eHp-uV8RB-szmSFAs<PVYqc;;Y&u}pB2~L(<l#yP?lXDKb<f;xg6t+?2ulrz5-MCgG-Nn+TO0Zj~@eQ_u&P0s*{`HijOXoo-<ZqO>tPk24rh?)vtcWh@UEl7e12n3JyYf580WwE=N^OA-;5k(;l9f=6IUgBkx%!jXkO#`muZHYr!u05uSef7{$TQ_qN<qy1&CSe=7M!ZN@0BMn|#J-u8Hh!oAf`?S8^e?W_iy{eA}~g|!D7r`-V(b~5z)vG543nB5|zT=l|ef{2G1F58n+_l&Hq8?sToZ_l2mdD;D)3>G(P=2L3t4c@jJ)nrZKop!Ujh6*6(Mgyrz00n`KvfZ=-M%vCk&!5C@IWf23{4^<ucUG*)1ixb~UykV-bpwF=Mn)bNYLu;sykD*wBpIv<Mt(9+L<*=l)OE3%dKWhqNMg1kVn_rrOuACAEv~POHb@20q8&}E;)78zun=Vx*Iq5xm4Kn7xhnD1JZtabi(EZUcTdR$8z*<8lv>k@FOxQGYWdIU>}mdWwW>93Rx8>J*0YmX&E{Y&bLAb<jb*+P)7i?VNJCW0OloDvXA{H_gT(sV*?8n(>rOY>Nt^A^Yo)WNUcR2X#%tK$#J2P+sYd;+G3LBh@mW$=U9pdJZMF1ekzX)1>5Q{pzne8XAvMX4PcXbjjg_c2f-a<VkwDd`ct<nM;kJRcis;8kR;%*Yvk6~Q%E!h53FXro9tXJ}Po#;pQ28iACD(Z;98E=YS~3dc3RE@U+;5#!F50;Izu9m|1@qiEy`?FQ!ft^|Bb^gbIF5&z=sM;(&6>rGS&bD3O*e;WbJrCTkf%Jg6+vjy&UMPSCXDi+dbW1MwG0D0G<Z?Yj5geLdtE<A<x?CG+V42GInu0v8=clpvv7YyM9Q8?^;4hPRYO(|e|i6j^+_uX;-otm-@UzsHh1+DwIr}T*=cp(M>fiOFkGmX7#A+Jv)0%W4>%mVaj=IYsFkNH`k<jzw}F&zqDP6iNjLJE9yR00E=r)iJXda{8Ot1i{gx{WI1x+jg4T)_!U%GFE9-O$GX&-v<xpI}x0P+9b1UILaiQU1X`SXFjEmzIB|5jeZty0*REd4i5No#EoD*GlEa52ejN$I>T~QKrvF^*<C)rqB=H*)inc<`%K>3Ox^oD50x%FY~G}h?fb!U<MQldt|EOici!#m$y>x|0sCfVl3^(Np##;0vOdL+26(H+#9lX1TVEQuM4C+#L?CQf_J%}&~8s;FjW+9YAJ@{ki7o3V)tytnq6+B>MYW@jQ7*Baih85okuTZ<bjaAE~6<iHzi?8mwZLarl?d_@V0q>Z2H%rx24GFxRvn~KR!g~6s_F0xe4!d>kYbD9@hw>3u693vLnnck4Up3$;T9`Rh79$WGWHu$-w)X0Cj3QE1HA%8i~1?Rd`L)|U93_gnL+_3W>8m<}lrrwfX1#5~l7D>TTV;mbA^j=$ym)D&&`9_{?2LY>3{X^DGpdbJ6Fg!%4pX;B9!TCq?*gRL<8Ay14UmV;=JN&G-y|f7nYiCJ3CX;d28Px`d_pJu^oeM7Y2ZzIKduJSktthB3*Xy9ewo*`MVNP7>wQ96i!5w>Z3$&*UIyI_63yzu(29F%VRshw|;Oe+dHJ9juE4%VjPdxf}i&ugFd!uym3Mf-!ye(k38eF;*Jo5L8*DeJQ&}s8?RQmc3Y@W8z<h|1z1)FKw3!po6zd*2gNX<T4)-3F%2+76e2N&zhH3|E3(1)zo>@fKI0DqWv(&h+2Phc-P)G&(q9NkHStx2Z?JALYbSO;NI+-qhXoTC9K`a9$CV6?ilv<-D8o1jZ}m$o=<hv_J7!n|F2G`qgE+39aCb(>jl>FSkBm#;l_xi&r+*Yw`Y{t}v*g(ac{Gbt4))fj{D><r!Jct?8yTI}F3fX?;?`Ad={@JNRl=|N<7==J1lp`fLBLu6~5?a_=GiINvDU9UfOjkR?ySe)?>Rjxl8z`iD3Z1wS<K<vZ=^(4o+;K`I*WJx_f7XU$5XP`Vy51bY!#hW?m#z`Ia#90q;SB``*M+xkY<6wK3wJ9_u4q6>9op^K}w_dZQOt~qSs^DFay@7CNC{Wm90H0_f$nR*UBKxz{Vs*OdXa}EIt$mL9PK1l}4p2VW54QU$PS4?JY}IB)1bmKz=0P^n3<JCv{v)s6SgXguibfv!Q;Jdb?2U)i2{LO!cpF3)KrpOE5T0b5O7-C3Kns~tTZCT0R<MQA9n+Gm!81F@w5CFc1KI9+6z2feRXQl5TRo7&{wrXlg5C+N`MD<fIrx8qI@*|)RlCe&gc7y074)~1s46_d%&wH-W6jG0EMh-IYNZ-dm_xD7FoOk+cqaVJ`7g%7qf3(25bXZ!U{@SqbI6YIqHe^jSuq+bQ4Ycw7<R$EJ|rtuhXg*$5xKEI|C6Y(GOI!%ko0HPGEkzW#LL-!HE5D`83#;!rSaNK`H;0&0}6NxACE;xpn+9<&by|eQp71#y_;?~HThwVgDylKdLYtjE;SU`P`+lsU8%v_m1X#I{yck!V*w)>D?<sLMayh|q}<WN#$D?tLsQ)IbK|~l++KXRi#i{TMa&u%6^UJ9tOpAyY!Ta&)>z{n9Az%YvK|^iBc-BJtVP%nqm3NJO4)IQ;bt6=!jb|?0Y{#2iM`Ix2VUl<4D}WD-dIpu9pF$4)3Dd8Jq{z*N!=6dGCS~yv3bp*l71@d!ED(}TcF+mJqL(<Z_HGfnhOr4Phg3@5*}}-97>IYtsygfALhPY`qHRBsPDx_6WG6mQB4Ai3<Gzq+Ziy|*;52^jNUEN7}2X;1zo5)41Mhy=xt_A=IONJp0J(l>feU054;TxQU$P9k9<f%H+`R!FsKoZORJlXclv4+lzyfaD~2qhl_ECrj)tt?ZjeCH20$BD{BvoT!VE+mb292@ZP)_H6k!syWD5MVO2+tLj8b@I@zH?AA)ze327J{9#Tz9Is;VGQ$X^qrETFaKu^6&H*LQ#m0r^7|<6Fa~@SECA(}IDje!f~j>-d82ChaOYO;!UCJmX=%Gol(@;KS~un~#J5<>@?=KG64G5MD#6r1W2`ML<J$ZHxG%b~FO9h^Y1$z$f?rr3V9Z5-j3wrq=0ihk4&9p`9A&L%Ze1GMY@<4Qxy|mk4zimLB}5GO==}pPKzmJu^PsCIs=OI&|l)GjJgyLD;X+Th+^<#8KsbO@6c=7Yzg%3BM~YHuPfM<rvRMSHo4>I^^tbB@a4LFN|hUSMnR(x(B+Hv&nqnF^|wIx2SUIsJolS2@*s-(QZ%OLQ!G#9QOas@pu>tfl1l=V0qt9(Gr2S5tAsu2dJ`%Ya^A4HCD+PjD-e1LOnkYuL5$J`eavmMPMVfik1yAVY+}oCS+9{B&U*cwP#D=<5!=6iQJ<#%&`nQR;;YYR84eWqNGrsxL607L%AcST(WWlK(d9J65IW+<v`c1C84##7CFAgFw`Vg_7c*nYvp-4t%8>q#hNun3y_}JC<+!I)?TO`*Xl|8c55ZnK15|TH0n%Q7y9k5!I?w*ZZOFcTq0kVGp!P*Xg40o`;N5Le_E}^2@$&K{$N&D8pgc(`IA{t+oJbsPYAzxX!(4k2n;>A;oU=emo2XlN~;W%M~YowcAFwah*5{hn^9=+to{+pYa~7wT!HVP4AQm^EfpXuQb0<vN^BmY8iSUyTuA%u7e3>ZN$$94{30*;YnTNM*0jN467sbccgAFJ3}vZEESZJ#SIe8SYEJ;Y<u$dq5Ip_~!D2bjro}xqEtErzXH(;}riQ~ibaOxL)e|}#>30o1fIF4o^H3C2f7n_e!ST&!CfnNp3jIdRWi*Dbr|BfRI6xosV*Orm5#{)t*g>?zXo<#^CDqsBa1zbMK_vV-$eJW5Q|Zc)uS6LN>FwkGQ6&9_Jr>M9=^Hv6;O2vWyqb+bzZ7ZA;(#tV(40_}{7o!KuvepNAIbm2%RKey6OW*$wbzE}*d8>K-E@fOVZ9Y$UZn?#1xJ5tRnOy?MFl$V*-qO1al(y;j_RtLj6ANQ_TxD5J|`Tt$_Q)$sdo^Pf|+=e>@|lS5I%^EQbkPS8cpAtbn()ofFdM4&fEG$fv6PS^cuDl`im!q>GdHnTiPak^^Z?J{?s)L(b8Y`VYhR5**xIT91J>#$&fw@iHhGMe>Yg5-Gsa1wZ2eGm*_h>FB9ci5|fQw(mw1pyL}PB$AVH&cEi56o1vlj!IiqNbhFi(bSE7OY<1LpVEJzzqKRiPl-zaqy91bnq5c_-!mzo<hO!oq;J}_Y&`#YvSj2FT@uY_rG$LW90&OV&3RyyxMslGTYW5hC9~rX>X~Yj0xN#r0hW%mcDA3;un!(xzV<H^JjrCe^WsJ}!D0`qP3r!uPhz*)5$FOk^fY|*ZSuLP2NvT2FVxkPT&l5DC(R~P8pbKCjG+nWZK#*(<+u+GCg+^sFSgNVx$(xn38K5;M9gzO}83^gaEbTyJpeJ-Qv=-}KB&?d~a#4|A{KO-Ro9P%jkB>{*yY)~Ac14=Q&@I_TqguYmU(zYN`80G54c0s95ZI1p%Gw4@%rTpotoR1IbfDGjq47}AoE!QSVln$5tZ`4K%{IOlnnk}wBxa5J8$qX37kL1*W|;d_T*3X~a9ayc-RrlSn@l|oH?rP%BM#2@`BHE`&gl=ht~|K%Hy7-<r7s`^n1jPZ=FsBCNriK2!K26&DPvsFqBheZ^JH$317jR;XD$sNr;xPyGrb@ug!r?;PW7Oov_a?dq2T@ws1qRdQAiow?%2my=`@NEtILm?MqLKc=cgb$72Yw?P1`uTs<m9NFRxT+8dij&cC9Gr(snP@twyiOA>$q8pz!VwK;e46hH~fT&_qE>Ksv^BLL_o&%YaEgZ2o}|oC3f6T+7V(slqwuaS52?N-RFw=}$nADRLka88w1W5BsnuZ3mynQ!Rm113+$WnG?W)c*8;LjaU&cBu6ov;Bh>f6?_JTjBF$bAOFln*eqt1np)ZL!!@u6z@NVu1$`(#+{Yv$pZ@$svGmk}FYct~nH-Rdd+Cs`5Oj)c5gQF)iEr&vGIyj}li=j<V#WsBNJ6AMN!*<J3f%$zXgmasP)n>zhjb2uZrbb(Kz*jxJ4`tD#nyriBQ8+HT`-!|TrGG^2;Md<7pO-PUg+yygR=IiLD+`o>rZodknPp_!|kO(duyqFVfn$@%7yv^57yu@!m)=fkW8o|mfe_9o%1CUWaC18qh?FGW(%~-4e0O&^ds01nsNbuZ`@cIWu3{5g$)Ov1Aq5>iygA_2D$At5O9M~{(|P9Ss-L2CL@c!)FvQ#hs3@&cS}{6gr9imlTQ2qBDu`uaZDEC%EoKVcVWP_UE1-cn6OApMjTmPa0|o5Fd>f>phf%s0>qYU%XKGSsQI*;!!{itl8PCU1(=ay4cZCjwP|(|rCoN!L_7<wWX2YZ7#yNQQ^k4M9TAuW9qHxhuB4004>|FL`dJ5%B+9T=7=Z^2$LyJAE^IXf@WeOi<Ip-DxGLf_RIqI>1DSxr0vQN-J{L564hy#nya&~HjI(`-hq+7fH^MwZV~IQxjRiZvjQU)!b7<MbI_Vy0rsz}J;2Vr=lyHWnjes_}##x%|gGQ{gY*}~7*5>c8>?rImaDl~$Ba~u1_z2rmJhzHO18mwF5A+9V0?<J7t-t~3gs&|-d0?|SB1d`FZwqF9vj4<okN=pO#f>+zX}Ixd*3#Q`C^T%>4$B29!_9p2AfV`4KF$Rf4wQ^+702$dKViFBJW4<a1rEm{5*16P2Ov&W1_N>cm3Ps`gc)LBOg20M3OcrriYn~&VCiPypf)>E!Nd%)q}j6AM|BR(aD&T)46rMb0{2rc|K$c7TIkXSm+IbwP9$?Cn~QtRVTPOy<41HuEJv^xyTo*D426JC7=h2Zf&#JNFDB&06W<ey$HV5ZCkq0p5HEc8AcLhk&2IAoMfS5jp4K9uBF0`Ww}h_Lj^u8}q6zHWVCTEx<aIB3{#FQlk`0nmM!*!UbTHPjt1)VFrey7r5<;xTx5O4#Ug>$-?)FDx35{BfL7$QRfZ;e`%R4(tqBH0Ya4+VS;mg|yi}NM!AUN-7QPNbbj<9K1<l9GH+euncFY!$S$3k6!<TgJkJVet|Wf(?%wF{0ob8gFMPlhb^!G*wpZ<*1I<?3y&JfD!I@(Ch4llRZZOfVUdpU|e#XRDiPv2-=a(k?c|9wC_Y6#WpCMpeu+RagqFd`;IUSPW$36VcPCWYaCO$dK`9OW3X_+uImC!ZmLvk<p?8^c{_-q%N-7xtgYsxE4r<8Ck(nCeb(h6V$g3*%lF_z6bpdTSr1Xv@f+%A?G}K54!`Y+?EpgLfzlxQK!FuH@i$$htu!!inGU{n+(UwEJ8m6yEl1#)~-)CgODnS16xfG0N3zSK87w8a-cIlJ^_)9@MUN>+vksRdnI$+FiG4TDYhX`yJQJ!XM3448GBwshf=WaqBxT0BPKXj_$#9KKE;F0wp2K%ov(IJ>2%g8ysw0UqWA~(4OB%{Id>tu1KY*Bk1e)34Z65<9Cb2^``FKVZRqUD_f(|q1{`d_!RGEpH4eaRdIsz8Q=WE$@To6ejKpi?L=voJCLL^J!!(yLAXQDeb+?ryXM7Ha%p*uDKoqd${rDn7SQ_&pCJ8qC<8i+$s)|}H;UQTx`U84P?sX@zb5*MiG6pSWl}sznz6c1kOO%_$oc5S~s=1kUK+Z5*35T(<u`xIt?@0CP;fEg%EXT0s)#QFw%cCi3E3q!je91f^qw!%UbvnDSlD@}nXpX_C1U}mpFr<a!W$_b_l(fidHIzPuT2ZUji!In_jC9Xl)$TAiP)ksY6JGi@`70{L6K4lE7RaG2eV@DOAszxuvh7}fm?q6}p5hOawP1EwJxkldx)Hr6Lc9v{73JQr2J?2sZnsebCuLmZ-f58gCNA>CG%nQRVmFDSglUA=OXT+6hA`m9<%ZS<8B*gSqh|w6nz6^k*jPyywXRC@pOu-<VKcL@y2wpe4Q`l<Jr<^hdB2{$eAwWP@8(ci;<ZuNXk$sOhJm0zFChyN(KlN#P%?ak!>#M1c5{~~Z9yaKFhAg=lL8c)$d58(<yHKoLtz1T32Y8*ikv^cyN~A>+&Oa15J~fH`FzggKTR8uI(O9gJTsc#nnu<e>lBnVt*ke$D&|acymM7^x!a6)p04AZ%zv7Nd^6MY-lm&5U#&N06>S2m57t#^ax%+bR(c-CtaiM!P{`+b{?l1VOn3Fcg{LvK?N??cqe5|ETQXR0%qq69u-^Eulas&9B2GEtdTHN9r%!ZeurHT4u&VN3W&u_3U=~f~(Q&YlOC5HUkUtn<P<b-9g3xZ9YCMtKs@V)qLFR921Yg=+Ww{$EUr?TjQ>p)~S)WbcDItjCj|1e&;rLKNJuhj;LUYd*CI;1UkBVWAVCU8mmq}<4MV@5|3DZ<6Su9sAM^_$stk#iNsg~T4!Rs%}QT(531S%WSN1CJb8b&Fz2+A+Qp79Axk)%Xb%04)g+KN*^kKu-ZA7Ese+-0rmH~MOc0S3)H3uefYLdEQ*dOF}Uad0&b9>st@D#C~(+ivr8Qu%k~p4`vm@bLVlaV%UtSE#hke)D5}D`uqN6GS7q2MhHe(V`;}OzV#MG|k!v3Z-Cm6(lNN^DQ8ega5eCluHehYnW^iqD75Q<zf4<c(M~?xFT^_>cSMc0PfBSp!+VEt?(ei8*U4`((Bb&%=`E-EXL6+={+=Xo`kd%*Us54yck8ZTcmoUvY`pJ+BDYy<!RL&!0hhuh6``N(5^7l&;DszJ7pi^H(8Kn6Peeon0K3_U0z7LA3Szm4{a(}t1JRus~QJRK(5PrxERw83Esp2^ry{hm!sNVmhOjmEfbX@#g$aZKx5#@eHDn_#BSv#OntSqF;-Y_Zf}mwRiz<aXXPkSkv?km{JR^0RzJUJiRBjY^Gh-Mu@fa%w0-e~&pu$W2jv(&HY~Y%LMw`)@_7>z>(eNvX;gH3lkAiAj*hCcllSmry*5k-DXe-clsL-49~4Ljomh!xX-!EuGsS%Jv$^B#ygO9h;&h$zWt23gljCiT%$>l9X}FB|@|1>(&{gyjXB#Dhb7J@vXf<;i92jk&WZFmV&L5F~RG*CMEL1UxjkSm!IV3#9EorWc0D8_D@1St(CX3munHJOK+979-aw;p_WJk3|jkWyTR_*d*k6pQR<?^-b$;Io};h!s?ySDE26zOCvI)oJ#WLwd>vmYASQ)^PXh0E?MfYe(D<^V?Lv=TrCaJ-x`E6~V!r-a4P;zB9)RSw)Ep!A_sFM6=PJLZ;4sTBJn<cvvIot#my=#)&3O{rf~(DrsNXh84oWSpYR1|fl2#pKSf(xPL7SNLmc;FeIc!E1asbp!eU>W9(FZ$@H`mQE3y5LK&aohz-xBH@zln@#VHKbxI?YnNtL!_Z?&^3G#kV!F~KUt6@9LQGg+CmwB^2l%}Nw_mO;lM1ma+e1adse+b4vX(nbRZuekl)<M8eNCxX3PjE<#&$XPCeV2omuDmBHUin(!TjPae;N1vFzQCHQcdHsMOW$SmZ-uD8f86gv+1bmzYwxAh>DkOxt&?F?R$+|t4?qCTi(zq?1AUe=*MX%?44F0`~C0~yCzFzD;WUC#;5{_DtjLn7=4d#UI$^`b7g#2Tmy>NCu<)R`W7S*1>1^XU68Dv>#!8RjRTrxi@nM3{6Wn#UqJBgQOC`sriRc>huijWYQ0r`0KB#8)8Escj{>LHqGG^tt%K_>Hm(ZETCvvcK9$p<3@^28b!dLv3gA__(=KPVlFgBu#VV1yTtt*2uA!;jcbst6N^Z(spnh>A=)|FR%dc`TD!*nf4rte+F`WW=w<=`M{lO~uGF{i5uEkt#ZZ=#Cj!DIaHD+d!y@q3m>^qWj8aLF3T(D<7%!67`aaXOlTUHT67g%9v5pT~_1R~w06Bbq3r7EVx^wcW~XNYFl_}smX4y>APJ-OtnrGmAyVbAgsSOtq*4$YY9RW@sJU|(!o;BtPvfBnqsJZG=7fg#?KoouJQxnnv%z^-L$UheR6s#JtsP{Rs#B1xcE;2CT^$pqy-H8>v_^vkoYTLDTlHDhKanWDkHyWL>Fq`?BR*$7cH6sCF>CULr95_1oUPykKxyWVsY+SPt<+iP?K?x|X<f*HynoVs5!KM6-e**uRbZt`brvt$lNmF)wk*uIw9_4dnOzw?`aJ^JStZ-4I>$1l8m^upJV-h0l^yF3`wc)RA~ne_rlOwf2BS$t$&=gTm?^TG4C-hS!m<ri<i^3|KWd&L~(N3%H82P;I?K-+lhpoNGH9n^5|e(Y_yAN}2F(6<5mk>8yWdDl+ekN(~?=(|AsJFqun2P&8m=VJ)BZHx1_d%H}JCuzk{;`0dc)pCbvAva7_zJ@8gT03F+04AOh%tfr%g<Uv&iF5ocsLD49vjpKP63gdK@Yg-b7*<Dbx(ezjR%x<G(5kzLa4KP4CW4bHGT7~)Ii2CCirO^A3x>i`zi82L4Mde?=d{B*lh{3pXLMva!Og*y1xn8AKmYK@U%U18H;;ex#?en-yYs=1Z@vB7TkpSg`~BCCzWmCqcfNk>oqxOa&OgsN95it=%&7&0y4yPo2|YnbXmZ*yp~+cL=vh$cSy1R%Q0S=!g(elELX(q(g(jtOp~+unU}$2@lXJM_iM5DMPl|=8O3Y|-7BqTK1daO5o`tO??&ho=tVN#^u;sfYuP?XScB!CE|GuQ}v;{fZF5Q@Ty|c+#EbU)MEUn0V)VQ&LnGqIZmk}=2xv1gk1ZLfHmN8vp*dX2=f8)k|l#b!P)k?=JP9>=KBMR)j>%qO1$y+75cjgG+To2RPU>kELmau`yrr&VCcPOi%6deEJ*LU9d)twK1a{RMj2gl!k>(<-91%AH$%2)B)rr!AYM{nGE`#E@dv*H0CSHrf4%P26DZ?DAAJ_j78@%5D&3GNa;R+5A}AN=^k=U+Vf+8>Ty|2d52omaoi9BYB<Sqnb@*)O;|Lo%xxgRMEl+`5lM>6Ab%A-nGGY~Cktmt2-&4QIC~bx<Oz-hhpNvER_9kYjYc7FTC?c4zF44c_0G-4l1l<r8vtXK%*txEk<xXz#=wIu#G&-SINsJx`-@VQoH}KOcNHIC}q|VF5aN@dv>d0+`w#{`5JJU^5e1E`hWHUZ_VWmJnsmIj!29<V98Ymkpze3@JVbQ%tzZlBn2)R)uKONu_Fzs#1cTow_<Zb#->?YSvR%lrnT}+JjK*Cp~&K-)3hye+6sV%>X^T?|=8`8!v%w`sepv40iW$bM?n>u{UQ&w9bxb-HS)G&IN;E)*FXgHx@p>8hpv??Rhp1J`3Ni0gzw6kbl*LH2Z#IAv!x3c6KgoO8e)uXUAr;t~vY$PNlZg(@{mMbyV}KwUzxGrF!jO^+`G_HG2pW^l?g<eClWDtb;o0?bi+7hWgTRNDBX={z3&a5v}u#o(Z7a8OW<9u0Fx*SeVgmmn?6x>v)}wYNZov1R6FsN{Gqa+BiFz^f5Y_6a|Y9YhPB)={?HYlV-<0QsVEf5R?Y%T6s}TmFFp_l;5aR4qwhd&-PrDZy3*3zWaO-8fpxjH*I}|s0jbyH2j}R)_SG?a#-$n42wq8$0nW5VQ{J6>!rrU-X|GapXhLnaS138gVWWj+IM|rvdf%B5d1}F<3hpaNw{As<MbgeV}TNRSLm@QmwI}2ORK6X40SLKH*j-TCpoyGY>bQG7sxwrfkJ>HZo~C5F4k-t#&cY3NobfmbiV;yW9WwOgLM~+o1NXSr@R0qs=%YR+_czy=9Oc%2~)O;_#KOdV3ieRdI-}|Nnc}1rmAZL^mzaJG<zU(byB@|V$)&WhUMKPX=Q2a#xAT;O6$o@*6qCxbfDS|S+nf9{MY7K2f0diXiD$qxjTbWW*U)?W-n)e_T@IY2e*rr+L%*+&bFp!Ni*C@GfrU}oD^oAa3-ComPR7D^XivxfA2LJawUT`3UDb8886wbPZc<BHS3QW_Xw`AasrE)%?Qa!?Fh-o!jMojjT*iW9sm~g8@>-d<x~jM(i7O#IJbJ?g45TPYzxWh8dhc$7PC)xRFYlE!m3(!s>y?Ua_CPauM~}MD*DYP*iLE<OwLSVXC|?`d#L7A>~5;#m?2;FP{wqYIk&5N0X@~#T1;d-2ubdwlkKME&H2F<`o1f5Yle_mvvI#KyhMAedZ-{e$qAd}%%XZ`QJuR*RSHY|EB2qKajL`Chq&OFk5N~BciJZ2f*><Z)#QO<^EVHlP_EBzK%S2gd5UG?o<7l48Q)4&cO42U4lXb2(NvnX<CIWoO!_?5^U7VEW{FqWaO#tA)t7N~Q&V`>YF>-<2?W?2T`!l<!p}Z74vw>6wKI>v$K4}v78rLH80Q{i(`?sE_vdR6W?pk&z};-l9kwR;6o}~zoDKp^W&LFl02QD)ExV4M6^>7_%@tZZGrOLdUH<~iu4mznXU4g+aL126-0_5FvU%IG?pYJbUG#i7G@iTW?YI|WSh?whadFblT6h-pOTxpxV4dy*X0ng`#d>+eA|tC9@2y(b&02%zaGW(eq3e6B#)3BNXjVFr`>fkw)J{3fCx+>D+*hY<dRcBUs&c2FZS!c-l7SYS+$@AyQ#n7itqacjt>&g-M#|rrr6;2lSva7>xR3YYFU+~ktFduqRB~94Xgy<ijM%+qm&FPfS`Ej6%PM1bF|q=eS?d7aX0|m4n(Gcc={v`eS@+OzVJZ!4Dj0|Pj53<bN2JoQ+1nN+S?Va(HP&9V?oa8$5@F2GIwp)!d^DZ+5S~^@Fwc?$%pjrC8<Q6rIfH9!?iFBvn5W3E`)RhlGaga;7fQhsY-OOD2ZLe1m5xRL%V`nB2^5|g@;*94Ue+$(8aj-1#?~vhzZLA@9M-$aV9LyS(_Ro$ZaJCeIycW<OXq^tFa>Qn5Y!IxZ3J{C*i3`bPIH*TjA?=91@CrJU>TB?Ok9n$^C6gQCzCdteDazmo7gqVT2}E$hfAx?NXe3~G8=0#>$yMsjebxo6gV4%0flbZOlSj&#lG}xb$JM@c{&_>Ia5t2W$sC~yo`voTq&rEm^bGS3kfpx0XUIFY1vX4RRYx5mZ~5@{p@kSCyxDfEP%L{w$%k4*HdM&E}3yaH?4L-zpL80r?7QTZSS623^)Ece_kE<VltP^14sO^ZxdJI0G=Y~c=kZzsbutB4kMo}koe?5!G6<nGKD%);+?p6ZPQgWk29n6$HXXow^be6%;90N(^Y%CR^ro9eq3sU403dc+tOrRwnk-fO@T&b*H&4nq+&%CXfENJCkWe|uM$-59jU4t&Eg=b>?AoAN6CqtB{MioW^kI!#c^_C=SjurW?P|*Poz<nJ1I&X71d*Rt9UC;;gH6RE{YSL&NzvmVxGq`PUNg`-*GoHGuRFckD@V02k{gFByu?r9_tS;(Nw(pc(EHH8E1eDH;G~kDB0I;EOgiv1vU2N#=^R;r_ayxfHSKj13t;`;VC(t%Yk=*+2gN5$D2c^0ovh0*vy)v#$!$3qPt{9CJMpKA!=$=#KpVU?9%!>Y<A_+jqhMDkMGhkEaGA0kdZeQ_-wqffEBThOK|mv**0RHU;(5T{DT+icAB{TT1X_27$E9-d~<dML~5{NG-1m;IAm;5qa>c>LkXibt(A2uYfxc39kbOzS($B$s2fV+9nC?cLmc@a@~dHWhIK;(f>2o-SAJ1kRb)9s+8Daw!~vS-@|QNQ3WsYY?H0JE#^LfOPv>*R&hD`ar+84#+$3jil9Rbf*sPFISj1I#C&<`A(1zpPJYtjYfixL>B5-n_S>w*Ead*`kH^nr+%Yl~4w70*Ug;xHBo_j8mjQ7RCeZ-)km@I8Jx6{tjDBJF21JG%K&ufFj`&NVd&IOl11s>w&AP8GgP+zV;5M19$gU?L1K?B}?4Ag4aN3?3eI0dB{;5uqN7(8+~PTNh~q+T7jYq-l;g1tb78;-55tdr5*8dUkICm#K~#j9Bh#y?%W(#BP1D@%u~!Nmc}uf=+88Ou@G>pR&9_9>k-*lCV}%{1-d5McRsuz5(0K3djj(C-n7i^~r#)|YD%_UE{U&*1L^{9)QjK?<}35H;yg!6y#(K^%}}eQVO`1RaKbfFnb&ntRQxgExo6vKs7+$Ai)8($Y56nQX!iqr0@jXt@QnT%tR*VX5lEK4@vP)8Aa`f?l?C^~$Bo*Pgmu)B7%aOGsuGj)>9-_dy(`8smwqJLnI`Y#o8etzo~*9s>TQ$M@jui-J+D*&1hiO%xm>QTXDe>-ER3iPD0lYK)n<({VDSIthCzYCc6zU&u!A=@OL-fX<2i+6P=bJK@UU#F)P@tOf#Tep8F6@4mZ5S|+4%!bUW7ATPHBL)>t}|JV}^V8$FW8j%79eB2L4Fes^`Kz}P}25TE?Q`Do4^;&Rcj6f&I0{|iG!L}8MfF7bIZ<tjJ_NZO}>kRpVnyprUh&wK<Mtn0!Ta3A2(=h1*De>~1bc=Q}RD$iWn!q>NIBkJ=Nuhb!AeItM>gdba0??Y14$Q~>45rOtmUf`AV3c)XwhWC*gcVF5=sQ+5u;*`C*Lb6A+x`A_C*@LYaTDgn;-EQh?W{bsSg${@9*UW$Og~`PrQ&Ll-r-!xUebqX^Jypn1Kit5hZ(RGa@yJku;7@nAS=GHP8OF|vxlBKF>28B@vz@fh|I-yEqD_5$nak1wfz<i4Qtfj2wGmc$V{O8!c3;(3hozSTd3|S)pOP12JGH8VEfqTD&6@wA2>kv6AyjTGS@ts0a7C>osAF0=!n7Mv8C8?d_Cy*JK}9kRrz}36ogJ2nYS7$7nVO!uZYWy1r_{_g&@poX)O-6(zLzEcM4G;Jxo~TdM*b=S5$PZVAfJ}S8QajO}d+mof#-WtLiA^J2iIe@`;d;zybyg*(il&Px&w{iCR0&UN7yGBX*$xFA0RVOJ*^U-k`;dG#0`))p!S|w%H)+Ijr(99rZgCtQcYYL<0g|(>4+yC91Wy^3arx5$u{hMPWUL#zy1eMgU{8A#PGDXnF&^KAlN2IHV;%0dc)^V`0OIPp{uw?2xrN$QS-VG!lgJ7qksHfDnzukjQlfucf3J4vWJUFBJ<cpkY(hg)c+?1%Hp&Z-OIG$AhBq3LbAxves_0Ey{p^8%+kdhNtbC6&5eTv4LQ8il%6M(#2)>5JCN{13e~L4N4~VmCXSJkH++7J})$8%x7kM*dKsIuYkvLZP|&1fIzDaO-sZhpOu4Qf3vv>I_@}QGlIjod_a~WW!t*3u>qX0Baak5{P4qprS|z1wm7h~kQ_F`QoIbL5xA5-QV5EdU0|HUj-dxBt?0T!!=;%;Q{YAY2tvXO$j}CIa0f}mi38L=5yHYcZM7O|H7ukrwzxY+Q4XpSg}DP-LZjxRFO^dMib^Hp!vP&|S>Ss@rf((89KPUjpB32C6pf+*8be&P&Npx!kBgSp#)|dbSoa(y#G=u-5IdKM28>$l6z7eVhhnGbZ8&9V?3FwXkSh<xMKYlQGhQtKZ&S<UB`o>XE)X!phdL6f@6a@NSfgUSF{{|aWW8}!F-Okv&Q(ngr}|*^A|KuSr)^x31D*ej-DJZ-&wrYQjM3H~{U&xX_SPG-ipw8lpDm)9i~ktgRpX3N{n1d~z*?TbshxFWx}0-|{j(njDDoOB%kZaq+NP~xnqMm0WT(NFIrKXGi*lzSuM~agfrwq<9!9|u7ShDVUc#ZR)Ry6Mi1+}Bs13vVwLQ$*$&RC8>QUq>$$sF;ThUUDuq!CMwi=UtdmSClDPX<AY30IZ--FG5<SNO2RKO-LyvD97S(VuE(OWUFY4u0=8Wt7m3NcY4oFxDPrb9aoSr21Lb|a({`2iv9mq6IpAy~|tZf7G9Gs)&;YYXG>76?1^*3*xuyaM$DE6|UqS_}K4h5d>x?28um4R@;DZ#I9h_e$SajX5#m!?sqd)i@Ic=Vfs|lwv?tZ!jGp<Tob67>5KbW-udmTk_%x&LR>n$Yd)HhZ%qY#b6Xd4g3im3-d@E>=s4>gV^6`cDBL;iT%L}G=9KvUJVXdI-Rw}mGxk0DX0rP59)a-vrOI_j)~Hh#RHHC2P=6mG>j-pCE*du0~k7tQob1nyRi&omUx~EWHfTno}|LWMF9tGVIPpTDiNQYM=(p0<<ZHGk2hJ^0)`;pguw$4crSgZlI~^%A9!doc|h_iyPhh$Miovc-dHR^6qqJ2=dl=5<my`u3QU>R((^GuGQP=u78>YZ=^7(tFE5Md@9ZPsaxbo&02l8Zl81eiaVIR&3VOX)Z-<s@lq(;~NoJo80_%8QUhlZ=5F0`%rivKdu-yN!k#3ik;>W~6Pu+u^j^QD79<YXv5V&-d!0?g;lZk8c5rC;{v|Ip41|~XSpFdxB*5uwvS6w~81|!|Yk!w(SRy}tM7(vqJ(R;h$*x*q^%tMoV=@15%I3yYO6J{qd(p`4bagzt%8G%9g#YC2YNSv(5gZ>=hY7Q1eiDAqF(2MG3pLlQ}Z7-l#ds!=O+_-Tcq6Yssf|`qBkUm1>3o!knOd_M>#X~`Bz=z$*n3OMQ;R3FX^6GaND=E!aN8J;w;-;9u3}Xrb`7f4NKst!x&c4b53M@1nxC)ocMK>1i7`dW*_e2y@b^szzgsiaT|1s40MM_J%3Rh-4abkItR>j~fr6(f~2}1|6ku8YSW0o#yq@7dJn!VpAslviOmrY7<gH9Dt3%N8@iO0|<DqX7e(D1Pq=@UgEl9m!tDOj<Jr3p2tf?%oE1jB69h^;HVv0V)JOsw20nnYrA-=sI-h;O1@NNFkBdQ^f{Udpl#m9YO9l+8|Z6GLv|0FQWR6s(~XGZQTEFeNG=X3WiGFzi1aYLz1+%1p1ZTT+9?^5}K;JazoYsgJe2EZq;a4$GCF8Jc)rY4|M`eJB3;!%09I9aJA@0qal8;gUr+VKaYo-yjfLFz`POHrvd_u7Y;C)%CzsYN4UseT`^^%~-`;jW{~9i`EPGu8_XQnx(@G{J*)+v2;VAud!z7h@8=Ray0Xe>>4BWy1I`w!{A)i7~$59?XPz4EQ<?(YVMGgumD9zPPy2NScSqG_m&sW<!^Gckv|j^+bt-Ti)nzCKNJ+(Ehxt0{77#v*f8lMKwOG@JrYpLH>po*vl+(IVTVzwIaguKAwR_)e!lzl+y$I$p&5h)ey!<ScPO-O-9X%0sd{qBaxz3Vu(}JF?VWJg6bchw6(|nQCDf^K2u6S|`xiE64UM`})&-jQYjD=mzB`%ga&NX<9p+#un$8rY-r?ZZPCbY2a)*pyNPI!*+|1pblFChY<3#QYA@ccI+IIa!ZJ%kwrP^?j;_FCG>zEmZvrXZmDd(v-`?6NLp*{Im#7gU7(K3fa)?iAJY3smoYS7=VTKf&A2svu)H<)|GZSpXwqGVhx7=?ndsyI;fDF+I3<5sRwwG-2c;d{$gUoju3V<F+<kb;zw*;9a4ly@v412=N~we9}PcKRmcYQ!?mY4iBKN=a5sxLRF(9>ogj<G?SQivOsROOsWabW-I}G~;AsGfIhTy=SInsmkQkv4kzq;~%tjmu#!Zy+Q-%amnFM-Q{^x>jV1Y*tgvW1%M+a$Yq`9u`hO@i);t22eCvu|K;+9i4fL?A((wxr!gUddcT&Y<DHIrS71Z;o+hnoJ)fVvoPPVHqz~|%XI6ULaJ-8HcRhFK8--u}7Jw=##fsbk)lzYhoXb<P4e*o-yjd$mdfl0<Q;qQ<Nc3g~NF-q`oc3qLY2U)B3{r8%r4AnDR2fV16`#VEM_*Y88{P%0;a<B`6+5sZ_B@$Kqblg?A|<rSIkx#$nSy1F1sppM$H`@iv2VgEy8^34*f7-P=}CCzl^Y9=dFhQEKH0kI%#+R>21?+vfxo({-_=Cx{@gGqD5Zko%SobkNs*MbnLM(88H-5;O3k{E%x@hzot=DSp5><e7X0`&Key2z<)?m!Y5x1^QC~R^yxr+<%0s_5+g)+$1AoiIvV9n;e!nxSjmKBBO;P0f^~d0YDD=dYtK!2IR0m`B&!fZUKL5!}OR4nOgl76qb9k}W!{E1EM{r09*IuCyq|e5Ojg|T*-32glNz&?f`ol(8fAGOLScd<i7-u<t`tVmjz4MLV2Ty+b5pQL(;$wi!=;ow}WV|n0#p>{*l?_{X-^YEy@e4l))FNu6wtD>6b?j2<$0|eWx$hkR=-Wr{yc&ediwx<b*IzvP+H35?pWk~CUa~j0-hPR_j4TEc{a#f%3RR)2M_>EL+wXoAejmO1l|Z3NaO>@NZol&@DCYu3bYQ@a-v8C@Z~g%Q%uEC3HesCx`OJS=HdfgQJjWs*oW%$|QH&mRNjPW@M`<!lw~{8~H4OLSoW-+O;mLh%!O2Tv+{+l&W3>A~*FJRv{`|vc{{R=;;h3Ht4)LG%{ZU->qZZ|LQ@n){YwE=6%J{iqT7%W5-RyKi8x<XG_t5~QtC$jR-Pr9@OSsGDO`2SJuo4G{@PCZRJ*<=I0OQG_EGF5HmB$tRv93Q}uyP|Rh}B?O1xkdW-WpEl`*5;(x+Yv_IUsS*krJ*)Ns@tSx-k0L2<0(~Au;MAn_Jf1j#WUBdD{h6LM~M*P>wil<1R|E1EM@q5C;A}m~j`eBTCu*tUca|gB?;XWF2FttY(-2SfCI>Ns^;@<ADM>#xrXQK%hNFw(<Wo0=7XI3n(a<Yxq~l!Fwo;oRR@Zb9`DLx9Q+#3j6eQh}#&Gh}w|1#uK0&1&<?Zz>o8c5W!M_=Sd=Wg+o!{5G!z!71lTqtO`G>H6bb-)?H0_JF?i&qWpBE{T~6X`~5ZLN3TRr@PuO0><o4=_Afkd$jVrgu8|#=jG=r|tTOqZ|5a4ibHjite;eAm=t|XyELclBppTEzaLr=?lr)ZWipC{n>v`F_qv(2!`BFL!Cmmx!mxe~4m~l^|?o6ob?z~+8yqzDWt#Pxr-T8QRU-!&RI%6q~2n0!4RD07fW7=P_RN?ZQfilD;mzGjbyeP%WlPPAnN$0L6qJqnhgD%H&*CQ!elUiB)#Q+@t;Q6Ef`r%BfXgrN3s^hcS@3eWaOOC|xkKQ<X`PHLu{>$y(|Ik$vD}l~4OnckF7b{egl|Fj!9nx4=7B58JHlb#JH%%B<EvxN*=R0@)^v9#u-v=0FwK*Z>^0sP%VU#RVKaXDc+0k#lb@bv7j^F<M@h`r6^yAm6>gV&&%_Xy~27vRe_rH7eZ@)Qu^E;=h!RH*fs<E32Zr)12#%ZhN45sncs>W8(D)7~x-+N)@_@~d^dH-E_z5u^IeEy|BzxNGSlPaUFmh+g!UaM0wRJHw#9C!@f6TfHN^yDLo;e+y_`rgQfh65mg;cv5$Hb4aKfX=#Rz3@V;4__2HTTq8Zw4}Z(s-*Q*edp6!fxew=G^SkOh90v}Tf*;4pKWG;m-U|N_qH$A#^Wvhpp>WNBPAuTe`fLeXy>rE_{1_V8I%`YnWE@J)yvi$Mb`^XkxMI0f`F_&Bb8xBpMnBJdE6W%EzFf?dPvk5^x@&EB^!rq=wO#iTT?lXE_Di&us&HpcWUVYlo^GxZnWyrmmH01mcFb*eG}RZcD6PiPEtH~or8~!866dXbdgr<dPQ?Z^~Ym@5u?4*$+A|}!4ge!a7A?oL);Ak=N!NA^3gYc0iyWM`|q>0YBl(hAUU4AN2|Yr&$I5_>dfucN(bZMGW~<8Nt&Y|U9EzI+WgVK{qD|NzYe}6n|u}!N*&GGdlh+9#~2Eyp3buI*qqSmV?a9l8oIh#G}tOsZ(Y}rVYr7gXFZYV#=@QddiCg^z6p!gPk(#-jXxc|@aC;|-@Wzz_Zi7xCA;-5F5I`?|KRrbUi<LfAKiKV-`FRcY_8)(I!8SB?$OtN$A;wAJ6{jJ)Jc1xtJBf5ppE=iMZwJ*nq##Bi{xy<7(%FWj59pkFxz;}I+m5jQN251#N=hxqvCucqX6K>7UM9fQb1m*iMp^l!na@f>hV8*?da?O$DKRgI7jJ_jje8TASDmAwT22HyoG<(9nyO=*&I{WQ@kA;iq+_8<P+b+b|?)=qG}E6`9y;obL9)|X=diC<9~hb_!qwh;@$qwS5dS55@Po(sk%S>!_o8aeE7W|dw9%fUiHFdy*7~!(?rlv3&*EpRPmjRyfW6(;u`?8qHFkbz1X6sv$n=@hGFlSY~Y;avCZp|>lzPslH6ePVg@hLDOl&!iA+!&wK;_f%K?5F8gIOkELaRy3WegeKItix(Q}x}B)o`avo(e%&P7fExYg`*Hk++oA)H(&tSzrkOBxNVpEAB;D!WXL;7o{l5K`t*v;FjBG)|adX)wSPz?=PpRJd_`)@K-B99s2r@TrD9q+XYTK@V~*G&`+H2fYhfe-Z)~E4TwXu+wTDtR!SNPsnmkzu|8n1J+qvePG=#U#IeQDo?idq|VCM9p%mHNhjSJ^UWD+8zGu}cbJU(177M-U}%*(NQT@&fqOiLJ>q0T+w70W{Vp$fD7slMSIR^apAwjZXp-PQ#qeD9bMG?c1utp>4+xFY7Egv3iuM+HdzfY=lnpMj220Rx4%6le0mG48q#<T-*fSC|a@48wI(1g3F6z`-9gA9%9z&Y7cRz=*sP(BBQ%)s~2umC;5TE^cI9MjNFIe{M3@Nc@qWcorOLb1s5za=Z+1+e6gM(G*>DtOV@C&*M)t<3QOxbU6TNm2ki~LY|C^=1O4$EG1u<+A);S9(JJXu_a8KDl63~A0wt!3+;QctU;o?b83Y$ZGPCm8f<3Xdd<8O*36m@R??Bj?kSfWI=D)EQ0LjOJs;mRaf`wd$|@G*y0@DnCt?pXQaH=9QneT~xZea$ty}j6~7XQAs63<S_h8YiZ()^}Mu>DYO`oywyse38R+Yk423bB_O0f*uZd6?dEjYp^B{`_$i~cq8&I?rh$XWgCl|2`04-v-#yLVJx%XcOTzfO8Rfdt*y4VjP!Yvwm%vdbC5cjn$GezkO=a<@Dy*jo7OU^!_##-o4fCn?|7p>GIZZ|XE8*;*rRcv?TzE}7C4EQ!Ib{5EsPfMt<DbJB`3L9v{l*+QDgUre6BTCSpM!i9PMv>F-hZ2a7}9zFr{$l6d=?h-k0XSKLvqDy!TUjJ%+~+U-n%Wikz@&CzY<NFQ~)b7GJ)hZr%~OkBw596-kQwnno%+r2mpc1s00uo0?12Ob6(a=(t}>d_EB0hJ<6mtlgxB^(3-8jWmog4Ptdve-8}+8W|CDs&9<3hBErM{`0?Y%kDqI7Bz(0F^`M(gx7y9-i@%re;d3poub4+VTy<;t>-7y<Yf_)qPiRfNqHMGi>qm7cwcB2{>X3=Tlh|nLaUCwPJ)_ZZSXqV2fV*JOM<(bEBa}<c%8kOA7^;*D=HTr0cSV;(gS)Ykh_xf+hC}(5v{Ep(c~{J4Ff_Qh&WQn)EgZW-*6YC44)(pETKrkB+yA%!!+-v#{}xro|L(v3Pyhb^`cMDM|FI`LfPSWbK1z5xKl8s88ZD{Gkua~iqEobmG>jwd88jta8e#A4B(T1%ol0*Cu}{tJR_h=1*Hzdt`){?Q`W{zU4iOm%n-8_+5WzS}OW^Wz-HO~_u_gLK@%7w9r{}*gnW0ze3J1|UZHdo2ZFCZKT>orWQPpqQXH*Wk0>ypULu9lU9g#^NA2u4ogNn%=C)WwtQpnsd?^gIB95ZksqQ5+)C^c5cJ+;|Yty@PcslgZwwIWoj$mp`-zFoq7KBH(h{uD{|(LFdMba$0s;jBdO_GkRC>N~+l#mKSjY)$onl(@PkZR5Us$mh3ye205(Hp5DYKu1;EsB+wg>%qMK&hkgO(iU5-RtqTBuWWYu=YRii{_{WmZ>Bq#Vb6@CbxgMvT-P{ta07M)QNo#%jYe3lOs4v$I2{?e6h|ysvvCGJN0oE2;jg29(K}1_z5o#tvrmft!Y5`vvbqaxFqB(|7h03I6&#;G|2k?l!z>*7xV&B;0$c#)H{w)<9)D9q?8@C25!q;5K+i)$3jBN7H1XTfNbXqp3}^S#K6S2>;%HQ6zEe((=eFa~)d`!qjDFGIYo~cs=?(35*j}qc4p?XfIu2isLmg~cJ@LO=++~oYVnGt3uS#$2jh0Vu6ld{tSD_tN-Nj!^L<@8vw-f$(bCE5<j(<H7E}jqTg%VaS*v=DZp|Eqwrk|CYRd*{zdZU|_Vto|-^Z)+8{_}tMZ&<wl=l}cv{D1%HzXJ&$j+Tqo$N1+<{Le)j|7uqV^jMlQpDtG%i~mw0R0bK^%eoh=*CIuCrFu=YMFK`81S&Nv+s@NrzAT04rj|4|*0S=`wSs08+*@L(Sc9h?t@`$B|6BR!uO+C)-o{69v?|Nc%}v4G>>!3&UK0UNAYwAEzQYt@RhEpP&Va8^L28Qi_$%P7@QD?dx0Uxc3)~R@*ys|s`9+RJ?a&NroP9M+u``^SuYut8=2;C${fL^sbEYDLy9XFT=+cYpM7&Bf7C@ICG00iyHL|Wt^`#hf%LK5wE)bI8w|90b<4=M|gMJ+wJvIH}CrUQ1?#66Y*IUSEv*LnO!E-=jVBCvgF4eBE2qmtfZ=czOBPHUhnb%anxK6a*W>I=p7K#t&*|fbaBl1K#irdjXh?Z;*Z|}74QZ3UzLd)nNODCjd!6)ByLGEYiAd8+)2MHi4og`HD{@t4=f4}vDtJ!ZoA7VnVQJ&5BqQ^6IE!%378<5Ikaf#Ynm89Yy-aPH@fB$s<56@qHS5@GCxyZ+=-l<&g*P=S>lMFS^*>rIEs#rWeOY<>0smf*TzDWOk$*Ei9qrB`cV7;cmKIUaNb-$q3SUA#pqw8FC84jd@CDj|p5DIx0xMkJ%>=MG?-`a}~-`_%%wnciLTPo|)`&$B*iVqGXq~e>841~csh3|aDtQyvJd7kEFwgPi>$q?$1`j)ciSz4m>VO|G4Bprq(+8w3k0wzX5K+b%h@vGYDTC4OcnyK7yaygHjC?_%3XT^M<vYNkmX-6!H_guL*^u|dHrP~}SGDQz`b8ks_X?G<7GL`$wq*L<~fBCF0G@&XM<s9#u5b8!tXz26STSj9kLwby=`4YK?Cr#A4&ctg-_E#`xn{Zu@EC1&bZyb70gc*F}5N{xvzq7>z80Dpry%&BUw7}NO!!Bhv>vnOQ8T`H>pU0F%DIJ%MCTz}b7cLa=6d9_&d*g_E?Uu7_(CN}Y-EPH68rVgj|4`*<SHf(t_Oww-+CN$C8Cd5mEYpIQNihTedqpsqB+&kzo`_6!ROn8Y!d`K{j|;+yq6GX+2|vY)1T(eqL$$(9C!wCf!y4{9)SjDLal!{Dd5Lj46d7O@>E<pM1;}{?oc4Z$Vicv(VNa8N?<k3$Q`8Stq=!%QX$kk=1r|bxq+n<+SvnXLbE@Sf%3vbh;erpNbbh>?WE9OY%;1J<*BMWNp@5u7v<ri(VYq&Js0uk1P{$5bP5X1WU`LmEHip5XGM@nBm^)J>sL};Y)dYU)!M9ueYyp$UpWD{6qsGZ%F)R1Bw@;YJM?mC2Z`*#;W}^4?zKz-bPblI8`cATWzF;nO*mIY(cSJi1<2(A9!a1tN=Ec}R7Q5(_N%R)e&cbu4yDbKUVji@55c1MN8px+ey3{;Tq!-Fga8?Y`KHJh7y?nap#nGKY@asFVUIGBOb+7ek_r7-`{0Vzl!iro*)C~~ZLhuqnJW2Ef=^|s=oh~PR*e=j)m}6~Rm{vqbS;;FXkTgfntrB%e!>E60;6nbdC-QIR!?H&Q2Q)lC0R}cm$GGDif)AIKUpeW!JCFP+Katg6<z%@>4YP3fsH<1Ma^!Y<r?tk=FJHW=8`n#fzH;1MKM3WOS~4+F(N#Vr5R8<VGK;qhf|G{%dO)boBJPYbkHeW`yif_q*Mtv?7I7`2E5m2<XHDa(V*x9XczIGRftMMq0FIV%hC4j^ZeA>B@bHgVbtzKSNq}{Z+zk)_yc0k+Oha=RVx=%@^b)3$N8cl#QG-N{m)}1|C%m<WYW1r7b)bii|M<8W70`ZujsoO&e|#)YpCtMbcCA-0f>Z^3NKCNHEy~0Vg2mu;R^-Gp;1%|xpT$fD$B072JTYC3eFN{%KTA~E(ABwKMw2X^&WeJV2=Ee`r8ISkWyCXzs0&t;hD)MnlJ^eb#6WR@-~x^w9YWjE%LqEl=K2Uq=NI`|Qp}IHXT#BUdw1tv(%Nm`y_dkxCa!&y({W8f=<?)^8S8>B5h1<Zb}#WcT{-|6xd#*Oq3x%KMZR!VYUsO%1<S|F_qTchItThLrdwRmGtw1^BH3@$P=8@S?_v&!Y|70z%)AYVDN6ksmg{W72lpQa;e#IY$#N2v*@q?U_zWX=j!CM3^}^K<R68O0Hv>{rn-N#mX52EF@4QRqe*lZhlhpvWlkBvEaG~%sOy@(Yo2z6qL<<n(bdaLX50f^dNmY-_21fZ?QYEJrMi1FJ2;mmfpHE7a9)1mQTiMpm{UE&XeLe;t2{U{OW9JdWDDIx(G%x{x_9A;nqt}O>R7YtTiS1Jv@C0NQNI*#Qn$d_oEDOKGb4*^1yM3E;@9I+k=8oS=;1OxmM+rOX^>lpcIlzpwvlN5jM^R4}CMdgtV5w%XwYe5~);$NI*e2Ok)1~<C>iMdPqScC1j1w0V72|L`vp%clXaLR7q5-_!KyM)Yu0a7i@V{gyNCRjDihx<=#ZVIK+v3er0}5|gboxRvZS<bx1AE+nKtr4Y#YdEfQ+<Dsz1V_2?u8Hx(W}=7Pxo-`E{mnewLQ(w2_RJIW}Z**kXmLJz)np919SkEPa`DkVv0ArUIl}KiJhaMIw&S^baQv`HRt4KHbW%oc{7Wv)c7)SUOgpXfLsw3Sw(V@9xp9SsVBI4rN<HRoMnIOES={_*|2_uH-xf;>)1Za+QpQ_cFfEmsDhj);6Kdd<*(r-+-CEE{CSzqr>Y^K3gN=vl_6$djmPOE-NnogF<<kn5djpr_UgDLaFukVE;Jra;PM8S-(HZ`E79|VWcb7z6s43>ab-c5Fg68I@^~bK5UGh<u}789tAZRS1#m6d8;u%+Jfr3T%W=WSdp4zX7#4w^0QJH%dAya_TpV#1(VeO=L5Wj&I>M!4OWYvE*Fn@OFQ2My5ICsUK)Fv(1`W~nGzo^WZ%#q^2Acekc>QoWw-f8zZY_LkMnz3%wpw_;CQ0cHq9&u~S4B*)8Ihh)ty1vmrr@dayeKAcFr@O15G|*Md<a6L$>v!URt!s@>GlST0gYlKRt&1hbVN!S(jSe)=~XU|k28wUrs0A}Mx_cVt2e|hp4wVWGf<pFZ_;@>$uL<4D#9}6=>1}e^429^5wWJmtQhk-(kP%WC6!7(7fu<IPG(BDU5Vt~cJ-9YadG}Rr%YOhPe0{WaK<2;%ooNk!rE_#H1ngYPJOc-gjhis_%yu$SR22sBIuP&4rIocOJLd9KV=?3p3C?AQmZUWtYRWKaatO>p^9-y6Ew`va$_@2D-|6^!;qI(57F}rBOE{Y8^ZT_cQnsFT<xd`U|XMd?`%9<7(S11P(1lBiT#84hRGBS;XDNF30xOHKeRaNCg9@Du^i_}1Ksm{I)uqy|4v2HZX~*mM7L_OP3GkdeFg3CyNYq5#=9RMH|1lbL}IL~R600D&sCRjC+n(o+hHw<CVbHksYei104P8!=J6u~tdx08<^&?Sg!oi`b$*fyVHLy7@#$y@6AB_*i&{hAF{mk9OSKZ5i|m1JiRzN5aX*_E{j{HtfzPn6MD)??^=6lg6D3}K^UXJrXaBQI>TDBsybz49jrvclzGO>Cxwsr>L3<Z|WZ!W+mVL0fqJO>>FwoNQvi0EG6(h>pR(ehitEo5Y<AJDHWn4wpoerCUtZI~ic+>LBS0ZRd7h5U>CVW%G%hT)<-4nX`@wAv{-4u&qk@aYhv3Jf4YnUk=b*Wr)WbA{=^IlWTN?XG^Z0(A}W}^;n%DB$F(;@RsTxW^t?6%{2GfAL@tAw{(Xnk)-Dsbb~iq;M(QsX+MX9rc9afOPp)5^B2sVem!Db3<~ota1?A5>j+gkmbL&@gq}a<~5R;dk%6cbC#*ubsAq8;@&s8~`y-AvvCSv!^QFUHh^H3pK(z^v2q^?Nz&T=kz?CA1mK`yQm#iEYn8ZjzPnIGVidSy;DUPZU?5}v=iBdW37h1wX@@I5%(kCRN3)2P3R-nxY&v6E5Fi*0YRe^ARqTL?-S<%rbY96rsz<k$h@^A_F3NX3DB)Ch^HH-IK<L?BvAn&(b0D4_;dvfn=;X>bA*PPR-rMp1mSO;rxoOc)~Sxcx_n}CEifIQLAf7UCXkNBu0oU`Q;n;?z5cX1y<;$_U^96~>R7zWZR$%u9iO*&xRAUBaUlP88->^mdY%OyyQK9e_Jfj6L~{&_nZmJ;LgJhXBnQ=QGpUIvt=_)fCxtu;;Tck1iq{6##ec=BBlKoF%TS*HvK>5Cs%^Yh1cFyB{0e(0(le{rZqb*wRQg-9^<Gu!XW0Nm2t7J^#D<!Px9)l5xsh>YPG`5c1xVc}hrr<&couurV(s%m`Y8td0d4omU&<Bm&E)~QnOZJFrYtC=?Fq%!hkWu;u@Nh`(l|O;0!_!!4^u_avIr%P<NRc2zC?Qss4)mA(R4bk@<`MMZ*?ztynoPs_A2bZhzPL#4)^g<96e#>_rBmZ{-?*;G!qiE`0_jT<Yj)5PwO2Xg)f=wBl~SyW+d*vruRR>W&RCF&XRbf%dG#lxv78U+t+WNcAvj`e$ajP!>j#+=dWMAtM9_~?zB9}*ABZ#heJ-Y)g2ZCc0N#BBlHe06@2p$)ozVA3niI>P!<ftFd7_FZ1ucb%)8kK%pBD2wv=2bTtns0c{UrvKI*a++W}M|dVME&<xyV=v-ae!SpZiMweah`73jL6A1wR*zj^)q70}1)x7{bN_vzMWd0!Go=<dIM_3Zg~UMm%=qWUCGhKi?(IhG0Otn`HW=L`Jr+n4WNL0i^t(UYxjXcXILS2Sy=ddtn4T4IKT97pn^WQp%yB_MMUJm7$ofZL4ONisUP_W~tCd|6vg$USJhaysWSbP(pD%m@wUk8qrp4Fhz)7!RA1fqtcbNGD>sDo{Gw#spPIx3g<gN$a7w^;_+dp<}nV-g>PUg_r>8kan%$S_!fD(X2Rcw3`XKd{GLB(=3|}^9fzun(Gc7-4RF2y!<d<G}K^UhtbunC_H^G6Zv@aGLYm{1?NKX=m5$RN1*)o=>4sq7kEN8j8PH_k8XZsy8Jl^=f|7cS|m6=l>*GlEoL`gwbg3WIk=FnhM-CZ(%!jiUGvfHh1CyMkJ%ZBIbNqiJ`|euPAj0Xz_o+BzVs)sZuqi*y_9GU4fckDs5xAHN&BWHe;>Sl@;Yk#RHQ|752%4eZ4~T#`KTB5vvF}AM}^Ys<F)eJ{osv;1x&XT6IPob7^`lT%={DhcuEop&{ilI_WiA1XyJS03O<GFbDy=ym-?OoY>CV===U8`Ny3A9u{>rYk;NO0LAhy91u!Jh<MEiCRCQ1TE9A)`9q~8B%f;E<<nDd1*JkAnL!&PUh;i-M)5H=7+cJe=L!0KptziRTQ}`$Rp7K0F1IlL=Y*D2dy0ureliYqjR-#4leB;jNHx%XTyKn;X#eBGyx=MX-*%?JgYxNMy9mF?UYr&%Pv{Y^>K%w%6=w}iU9)(pm*vi@^CaMB`pp@0HK#FUcaf{}<Wt%(z3<kFLY6)nrzi2Bva(S2F1@g-vYa_-v!tMk2l|vf=PBN%`dPWA-*j4Cg1&}vR!vHK6$4LxVc$ZmtGS@sihFdS(FZxT2nt`>H!{dr;P~bo)vEg*Vns2WoowP}OBf(Vt*=7^rq}5eM+%7vf3dv5bL_RF#gz80H(INZF39YB{>ScE(JfZ|NJ?XD9D#mbNAzER;C11eJaj<m8@tj-qv&&*SWF;1pq3a3WNU=A`L`@`NP9^$qIj7hp_sd+xy(n)q&Z*c)&x(8qhaR0sGnh+GfjT7aloNC@R1SO1`f8w(;#sQ1)QTpWo0w(JRgUo|-&E~Gs_cd6@sdeiiHfUug-h51h>rL-MCogtqE{t;B?X}ngh$q8)Q5ZeDK82VTRu($-gVX7OgsqW*5eiUZ4C=URa#Zg-&B7`Yli_|i_LKp-tzTqVif`^z|<)DJ=5*9=pN70p%O~{YvF~>KislQaM}nDQAekX%s8mft>#uCKGnHM1jBOm;QJWJLky}vE(WIorr*sjW-0klI_&e1r1fYorepnxZ=)cib9)O8zJwj_9AT<KOgm_X!W-?(vCQX(Vh|WjjblyY8LR_vPQDLEfb=Yq5l6?c0hWE(uoE->b=$nzZO=gH-fll?KfKrU?m$u^kSI(X3C?b1{vtIM#~Z_az<5K6YMQi5%=W)Qv12j=nUn2k?{5ueON(K$I~dVix?B{4v|O0|cC0bjpIX!|_Y42P3ZkUm@YY!(IQo5fZRGoALO6sr877zX)09p6in9;kg9Z&5^T^GZF0ET$%Je~jP{FsbVx?bcGM|dK@K?j5=EuV`#Pv`(|C5V*BTJ5x2z^PL!i7D>c`deP^Flg$*v@{jql}VoLmtWuF8=pU#j#?ZcLY8Dx2CbL*MQzpH$qww+{vUJiLoF5@pbhWX!}}Z-IH{Tp^=n=T|a4DC^F)+{B6N%A$)1Z2gA}Tzc*6pB-YtadG;?3lO%x+4I7KO!&d8v5@<=A6J^a8t=u-i2b6}H<D)d0ZDBsUi<U)U*TM-=O_!~LD3GHpE|3#?)qqpy6z?V_t&z?7S)+nZ#=y0+KWANpvE`ukF}W3%5$g$}H_{NPHA^B2;{!#XA|JsZP@tu<n43;tZV>#3sL^9g9Z=g|bB|13+z)?mS|d*@+UwC0XD@`?4^ULhP^+<!WFd^K>ROXH`y|}aM%|crcl)ak)xt1y3Q{E8E~gZ}YkJ?9%eOk8K+uuz{rj)1ZYX|2C}WhLBzyV}RV;i;ZpY^*`QU`SJ!J=o;X`x2l!w7*o=%&xkddOWpduf^3<Zz*80E(z9pVwg4z3m;w7Wkr<#9xCc9BlVI0^TBSHZ#Q4nhzHU^+>K@{aRb9`>;^iZCrG=^EWE5NSu`qB+G#ZU)C0`Lwn>a@Gj-N`d4khb<^F+u5xkU7u#9e{QWcJ%v>cb}BS%HcsDh_K)$Uo|~ABhuqbuDuZ)#=B2_p-V#f-7)Ouw>gE<|_UKwM&*=me+cC(H#1S&o(KgkbEwLD(_@NHjL_yrJV3%ToI$&2d{J{DYlz%?JVGLd4U9(}Hbsw;n26QDqIn|$>R(x{md?L;!F<30S*x)>xz}01*j`fKoAH3v$F)0QKh2nS6&H#<pGvf{Hv^?us*lb@pp}N}8WljbmukxHb#uvb#eqw{(3ySP%xM0K;1I1B4Ps<8z(Q@{QIvy71(~7PG9r;7nbPdufYxLMJ<IqMvyggdiR2ed-+0ZwP5gYbmG_0PQQn(cub>3cEK+R@Zy6B#sN%2(vw#F&<B9m9asf4D(J@n0NMqA%lFZV($$4qi!+y;=%u>q8ar4<}!z?B;pEN{Y1h~JeR8dC%j2*FsGQI89rb%QvzG$aH(35kzeXHyf;4DnsaZ!z#iXnsZq{*cz^o+X(M8zToK&eE%iAZW<?QoOFUFyl>MU-s#~FZ2m@5EBe8;0_nICo#`x8a2HF8CQ1eWjIqYBLPNh;r*pb@q)soirOG*00~ehl6hW|k?tRgW%Me;Gvg~xrIJL@J?A4^5$>}Wy#Qg6_mp%CBIiNrT?OA@i|Cz-uBM~p9hDR`lL?jYZ+YkPs#9do^xTPA{S6Y(g<<@K%_EJz8=g5U>>X+_^wE)jJB7|wlk1suNAGXhsovPbpxY(l=ynt75cH9di{7u&l-pA<^rU^vj9A9CB9m-|RTox2o6k|a;xKSwJD9VxbUGfB;E38xVSwWtL~npwl^Kb!fap1ipunav{tv_Q4VWs%<<w*71IB|w&otL0XeYj|v_O9G!qZx`Uc=|z;9cIR+uKgh>$!OwgRk)RwwqAWBvm|5PsapfMTgbl3~O5)eb;W&bQ<*VHa)O)l9;d~cF4tBd=v8Ywzp!#Ua~t5rkbc1byqQ*t@Tfc!>70@Dz>Ug2NMx=i?Od|j+e3iWu(`qaSZ#hXME7x6MNfu_A$6eu#^by-k$1BeXE{)Y$mmS71a8}+9Iij`;NT>g#74wc~P6wjk@=b$F&DK`&<JR6$90vXO+Mm5T9!tvFc>dU}zOq(=d)i^8?o*%2jE3pW$I%^Qjd!XnahL*`Y5)9bKJfYA?b7WoYSznyGM->L%<UE+?=l@IH4ldLXy>LtEgtFsZskBZ2U9f|C}H!RauCrkC2=NqC>(Eg{rzc&Xw#V2+NCOn%ib{~Sj`In(u}YpkW=oSdXp(E_u<8|+JuxgT()?JrJOe8GFp{^qRWn`+?9pUGLCoj1(++Pg8fH)EYa$(oM-Q8Pl|Cbi~R4kFB-H#(ku42Dkl>$)2rSfSx&GE<cqFhvm%L8HOR2!?kNc2CjjIxE`~hd4?<u&{J0+!A{x$Ld9m_7kBM)Tu4t+LB#l%3j1Na<Ot;Aryw$g&IwG%?vAD9m8pxXiw4rtdWe(6l|zn6uMf>Kq>fu3%T@;tQc52P13m^1VVP8SWdFiu~M3lv}X$0nb5{$5~br?u)?>`=(JvDG<R}rZ@{qGtD?O&g$0P-I+W_%1o`+d987OPhvXiLCN?bhLB5(a6#TLd8+u_q`G^O>J2tWmkkW|4S?}bU5R8d9oWX`Z5e^ttAH>e(G`8mC*|J%S6rFCLi6kwgYdJj=g3NCt|L$Y0kyebE09D>lY8asIDKMR?zHDh{+GTJ2((v_KOqSzHqA}QIJC1gn&1jqA<Pr)zuvx23QnqYmkrbkJT3|m5k;98!-6*{wRw6=O$^fG@J>tV#4|mvcMi3e+{67<ZVXtl<xkb1fY*;_PHb-&^VmU*zk-^V_Y)7-*b~UC!3YMv@(s5E2qXl}{nT@249R4LY0z0k*-3zSqT=^KRHK4fgIRnl*;VWYt$#iwxK>qTE6B$AsLb3-j3DxI1MmIv<!aF7(>#l@i9`lOgu6X_I83Xr>l%{Yn(UtV)4M6w^7nLQZO0K+u{_Icl<0>7EjbO6_M|?@B)aCKASlUiH)g35}t1CR25m<H57&p9z@y1_X-0%L~rG6WVoCUrH+ZZX197?v>e@HF-3?m7ZPPg5#b;V%~F$W7%45G2uxfV)nI0th(w?)F13;rRiHIyvV_3Yt$QYp+<4v7mPhKU|v3tg7ca=Iv%=oTrmQ(`C$`W@zTOyPP$xeCq|HBT*7q*3bToa5sc(KO43C0B-^$a|FRWCHTpvJXr+di>@&g5u(Vedf|Ji;uHZy2GH%C~Xp08KKOy^N|$xP|Uj&^P42mWB8Ui6&l$wD5y%4PUi#rj$!I2n!d2yb8;wUr&j7XJJ*GsINH^rUnk_up)jR0uU60Bp%8i%Okb~gcN@EB61p7kMeh_RlVUJlNg{&H$c>LEep~EQxMzPknXw6@omg2FZ$~<=_qXsa64Bx43RBFs#@Pstz;S+zxm<;Q5)aE*B;-0rR}(z*fx@7LOvu2?w8gRi{5ksB#O#`ap)Heis0y(Vq#BHGItU|gEciXo`{=lD)RYlfYUx%bYkdqdEN&wH+zWZ5K1qwM$IoA+{cRd*OOe|)Cg=tVF1Huy@%9-GzGI{4+w7d~Z3H}S1RG~Eujz*_+XC@)JC%PoM9yECI*j&;YKeBdCK?2WOEd_j(fJ%rL?rn%<N`DkFB69mj15@9So2vIn@X8Ul-*htM&9_Ma{Jp=xqVTReNmEqQIdU8l6_H<eNmGAZ79ifm26UhbWzTQzIKuwd5^VJAsiMIR-d?-fi3#x>9YeA1((DX&Pmi;h)Ob8H4O8_5vP-JQzR0~vCu=Aa_zy?(z2M^8P6!y*+^Dl<ZG7W<Cei?vdqzKj;?p)N3I7;qVG{MrmNL7v#L|wR)9_@{h#`(b)U6#oQ#{iSFohf<$_#Xu$W1Sx7kb-7fC9!ulPzWd&gO=rJ_jyzUN20^Lr3)Wl&aE#9lRmS*C#@V8$Q`41bMJ)aS>uCBOCppl(Os%gcrk-T6|j^Fjc70C5^<<Ii5nFStF9;IdNUCB}}=et!#On5-*4<TOk}5ktSm6pX%-s*ctR#1D1MQk=}^B73{Czt%wo!IoJzDJ5-H(rql{0=h}kjdz@KghZtxj<g||{1NNsP&@*wax<gC5!QdwyV8(C)OPtX<I|D$25k$y^N`6`X#q`)eq8CKwMtmU+(Sl(&B}AYmOxj0RwqmbKDBtoTi{L<{vb3ma`%g%2;^ywB4SOsd~-SbYSQv5nYR&&{}r39Hjk_H_ypXAcD(c5E%v%?n({@UGsnU58+-GIQo1Ti^8%T@6thN8m9q+Q4zDL)oonlot8%<GYPLdLE%VfnbL{J#4LyYWmhVe$2}Lu0&<gK%8p=G)rZ^GTqN-#6bQ|jo&$dgT+YZUl()e`N&<X|0jo*+JVLju-UyD8Ak3T+r)$Tq%IC%OBbG3B$Up#*I?&&)t%hLN>Sqg_AO7OOdKUNI_skirnHVXXYW5|~Agv$d`UcODjiK-pJF&N{<V~s*-yWC{fRH9l)E1?v^6o?<f*p+3DsuB~F`H*ErGUt(+R01PXmGyqEf%RkW@2YOw%c(jFVb@SFDy0R<s5bBtwG${Kuk=F2|IpA_L?KLzhcyr$H3sg|haAHv<C!Yb@Zyd5rs)Td;Vz_(NKWe5$trz{;Ix0z+1W^r?RAYg0gUCosWJZLE3-omkL#(nRTrqhK3bvVUfuYwOWhr;c;I@SK11o>b3LtW;XQI(PrVFGZh`<mK1;^U8=Yb>KHocp$juiQ^kN{9yC<WnR17lVlr+4(T5%{injNZdTD<WK;X4s31Z99h@l9R!_G$LSompCsGv-z$sd<4x=N$Uqzl?U=dT$;Phl(}@Z_I9;-@>fyYxE~$rm;$wSx-pXkfV25W)yEMd(m<_%tGbuxPQEwiY`ZX;qD|L#8uWZ4$kTMh^z6*;>K~*bBfdSC=Q(UmV%%V$qF<&i8=3pG^Uob3`50dTtGWq-;xsicw;paulBg5IBoO}j3{=4#F{RRi5yTfQE=HPua44U#wEY4pvEu{QuM5OPtV*Z9%ZgZIg%$}ahT1pp0F@IsPW94D>|VYTMJ}1H~BcJx@K5qo;Tn%tD$ETgH!%kBnJ`J1wEN`Y+6%X?oyQz0&o}@nCvJ$*`{X{;ME!!on-^Y%3vLxGI=h(gqpq48?LPV%&fMb!&M&+0`Vbhnz+o<FxIJQpS^qXhD%`+!<YcyD2AeDSw2PTD<c%e17m<hU6_@%ea3HHNXmS;r1&m3Ot&T<bVv-VHGSR&jx`<ywJ+9Ze2?ooS0c;?H#IkL#}P^!b*ERy9-;Q^>D1rM*S>_?Os8Z2j8~l}9q&n8#il#zueg>8b?l$<>TTayk?poFU$~;Guj{DlcJz$$9K)?}N>`Ni=(3h5s|5IJV}7t3xQML(#d@=q6)HNB-E-{g8dmBVWM5x*IB?S5Fn)(xg0AVv+h%S@LRJyqII4+v2an$#V8w!W2XCLh`VP_J=TZ>H3Ckhzu;&!N=p7}|KGrJ{1(s*ooHY=`VzAWrbLqN7u3`DWV3Cq3j`KdTT6Gm==?5h$9G!t_l*<8CQ^h0~aE?hme3_nM2@6`>Kg)B1k5pqn0B>5-8D<(7v&A4WP-o5@B%g6vL}?_e!iYmvsZx#_(n)M~DGC(KkGHX8#x`dQ+J>eli^=%5^I;P()fX|?Jnj!Axml*@z%|c~m*aG<hUIVQfSG3_t{RFHl<$31Bi4k^ytZ}-zq~OJSgNX0VW<7<1n|e1!Ng-#c)_A(5yam(>=_E8-jR2BzVs_Fa>Ti=XiqYb8pLp6&BlpcMUp>}V~@~fa9Cud@%EHE>CkeFg%OKEPSaOz856FN$sVC6P7z8;%NP&sml!|n{i8$u32ut6tlORqdq;w@xf-qjhsj9*m8e=nYES+?RQsi&6oFLH6lHMsA7<q8z9w;Uo}LR<wk5cn%v*rYv!(O~K%ys4-(r^N9RERnAWOX%x<qH<4d=Ha0a0y=zw9`*U}W(EQPc+THp(w3bvH{tlKlW1!MZLb<!H^3&FPXzoc0$lFQRdGT6D|VIA3(x0lMUxo1O+#m&WRrr}?Z)RXaucO)7aqY1|MPuUx14PQqmtvT>kMZTJkAhzK@OR$o=*E!YfKH6OTbOf4>HUj=~)Rqn)A_7YkbL&s!OSF6s>g(|(-RhYK7>4i7DwE3osG{Lhx9d{d8!h}juA_yJ$|E6z|7oTG&_p03UMpGUKz%OjwGuh3<;H$)CpJ)a`M<FwntT7Q|z(c>2BV&5z8UFm$n;#C|)g0okriiZqW@g2TJFatEv+uhe*5v)R7}Z~5srV<JYnKTeFS6VUJCBJg6)JF~L$`>vK?A%<cH&}uk@$vi35?KC6#WJ9DB*E;*n7NO6bA?vl03>TsGBi`8uc<kmHRO3h0-#HKj?NbBBI-cPw)(8l}#A{^6*Ez7_#aL5%&B^`MaY=l58K3i?Rd+-KOVTj@Mg-;%(CftiATW#Gc{o;(YNS%wbrAl}Q(m5G#m-^>103@W;$Ek;||l16IFx2>YgW)Z6n>u@$vgHe^y+NW|Q^gZl>OnGNsUiM_9OD!!6m)MDM%N4mIIZ@2Jm^RJY~o{s{WJrMu_($7qW(P%+vjH<KVkWb96VirA-kCRm&M6amJbA}6R_I#LdJD;bVq*+8ENn(;xXiSY%)ddh*4tu(C;A}zeN+3FBT?wnbF?vS<ABYKN;Gz&9=zPx6OuR6|JOwFZ0(1zJddOla(7hJz)A>byMuL6XFSlEdT02Q=_wM~xQ)y?tss*@dQn^_u3#(*m3pDT_Unb$5AT`+>-bDf%EZOS4Q1zX;8){SRUgaRD<s-(SAO|d!lCJic2-I-Jh0@%FatIUUAYbvkfH~(=SC6`dtf!VAs3CnuEMK|2*dcd7Bn61{@G>ehuU2TTS6~iy{g!eEx5fH~rPwyK!{d<-MdGRe?D{Pl4CXMKtxzs>63D52Q>6T@(CqVKp{%Hs{35&x%byN}6Z;hFHYm6}p5TVyN|QqW(zqt!nxGO9Si!PHjr}osxXV#7Y7EYto?XYC@oson8Lt9Me{fzzyhqu%K?PT`;hwj(QoQpSby5>v<)cUnmbfnfzNKr;IXpdMonCaj1kpuv6QiI32;r<NunHSw)9uqUhQIlD9ze?DzgI<Gw1GuEn4a1WY>EQPqkAyzLh1mSJ2Kz4Va0OYj{?lV3U#sgg0viByDsoiY6encqVjW+PUUo3oKIyO>o_}32bUp@s92@Y66$+3%8<fEh_*vI$*ix19=?o5Xuc{%dV4&LrbXb8B#r==HasVN9RR~>_pi&aF+s0;9OFn!IS&3cr~PLf&de#DBicBc8*tArMO5JsclKAVPDCct&ga@TW*PU6^RsNq8YQXuYF3>At!9rXpEL<<?KGfqwUuk0Bc9K9$vW(?A$!ektpcYjIPUA2jwv=~r;<Jz_(sVDmuPTO5Z((AA4D`XHV39FjxF-w5jt9-Swi~vob_<kN6Rr@AC+iC24A$e%?lvxGoXr-;+#XTr|`?B03m#{9m&vlon=@TgJk1rfQ*H;=_?)cWSG^)MR5u|A4q$iUnC*lj<xK~pU2j`zBGD0kAP3?4WK6N=89kK2_WX<I)PksO17uHn2j|DYe07zCg2|J1*ahkPUnNg^`NIin~!Uwnz%T$CF_c2;}qo-JOq<UkEEOR2DnL20JOuQZ93<~26j|m>=hiyC1FaHR5}{5L`{`SIgx+#d|}4ObKIA%RHwvA&H$kS$88Gj;ojqUwQzeh=`1q0T-{%(CY<%(A*FaYJBX2$ITANkDJqLH40nu7wPI88M!Tan+*6d;*l7zzs7P2~O#OE6sJe<wz_0gO*`e#5fGa#fO~t5k1%`)RB5`=^B_TzPq2oHNp9MTpS@aY;rei648{YSE6!HsoA#DaTRg2!)F!ush()DRL^c}E$pITz-Mq2AB-)kFH<j_2-u;{bve#kGnQ|C49r9r-7|6Co{t9o_nXLBzdO)y^hh$;gRxsp2JlhT%<?SH_SI0?*%iec*dVL?}7Bwsh31x7KkxTW<GmF3JmBDDJ|tNIcFwf3$G5VR3xK(!1*+!9&~^O|&C4x0;_o4GkFkyE-j>k~2Hn)}OpJDXqQ20aWo+6FqYE+8HN=GTFb>s+6&;#*zujpK2?@-suQTBG%qU9!m&`iChrW^d;>`Zg^yLjwB|&f+()NS1%IFN}8r&q8iCrg6;(vlfiWYraZg?s!pq!Dbo~=lL4IRQX0VgIBDM@we#}R6mqz7V-vS;ksk}%18FkRh=7+{kTVMViYMpn$f=&Pt~p77-5a9e7CTi1|@!H@$C}JmeGY9A(S>p8WM$DbENmTnt@Wd)rX3Tm6QSXfw=KX->eOSi$qEfSA$8#n^9JVvl(;Slcr9B8d;#?1&k$LqZc|6dr6W6gNvGhXC|%5^VB+0Pi5ArZ}+3d6vPRs<qnvZ2=h&!<XB<mjT6EttpKN;QigkHpg1B3e8LG1W_VLnCg-TkMiWx=ccLd;#FwCGtXr$Tz8iKTGC_A|-MlWb{YV#XzVMN<5S2|*(MfdVD<|=qu}}$S@Q+d}aff!j#UROv;v-5q%9&(DOw9vT*F<Wy$S}t%2NZ{8a?&YEZ}Va{N1c`;KE8|947t9tU258IGxoRW#}=z~o0a=0ka@!7ZXsf_8+Cl?PP0yb1V<-PF$Aq(nZW_0<V9yuDPx6jzlZyGqgk;V>;qt@K3+m*!$|q=Qbqj+V6hMC*EWJgD&E$Ss)lX9=7%*=kY?Vgyn0ovHXUbXMnS@zU_#o8oW-;uC?Cw!m4i3PQM@YKk@cw`y6nm}_Q7Z+ZlR=pA+xJYtZ|Z}Yfeg5L}ATlErypbAYAgs&GF>K5V%R6Ea)nSLN5qUfNNHYXGF<Pegy17`D1(E8-ix&<7sxWP!0(4yz$q8oiw<WP~S{VJwial2uk^ZZRKO&>s8+BHz5Sk*+V)D?IX|nhj5QwOfOw^GFhlm&Mk38>)Ozez|N>@QWpzSab#CDTN(S|$y(KL*M)r4?67zZX|GB!aTn5BkH&SD!HLw_a8){+FHz^JE7sY34LcjIX6MR_+u2~jYlDJ2_F5nA90gfJ?foXm^XzWhH5{}R9&%;;On_G-r?UqJS)sW`7~C4MZ|QL87{55lv$VfkM^Z2psFrnKfo&7sCB6fTVBMiYhn2qLo3{)XGJhEQkt_=PauQCrVrrR!movUYG|uQuJR@&AM-tFu#y*;2hX#&t;4@2`;45S+G*B*Q!*#VrtB*o-I8DcZyEe4ZpHo$MBe&I)4J6)i($*LAePXdMUItB5aU@HK7SoscMXod1cNaNaLWIVgf+9G6V$2>s!a1FTC$3kAy232PUOXMsxK`0}r9H?t)bIqX(sChdM!+x>tS9&DvNCX_)J@tN{2qOagM?Kux^;mSOW{Kp?gu%+x$)44)mU4rr9Xn;l%ikpt77t60P<&S_EWzoH*BBz{XC=SB&B%Oc5#6{Dgz^Tzs=}kD;}%U-GBY++4JwZ&tAX9fxei4ZrlVkPWx~GZwU6^w%th9Q6Cq_`GU5`X(kH4V`AOdO1^`B+1)?}9?R|&-V2u5X}(ro9bU~(=OvH-d%<I%P-G`3XXkx@VS2noCukW15g`r|<xC_3C9<5Hp(sX!18$Ne3EdRD<L-!@r?*-=J0eMo;a#Yh4f^l$nLuU<>2)am`|xY_XBClLp$srQp(cB~;v`UbBi~l#?=Hi{7sgm(5zJ|YCJ}JD^{EP$63A-xqU@*R!4l)9BmQ0i#vZtS$2|cuxfS<zdYA+|&;>|!p|nS)M=zPVQc4<|0oGq5erQw<wi2Boszt#R?Dtp+cd>Aq<iQ0kO&HBQiF6T4;!B;QuVsl2!;@@M%rB`z%iu)h#VXM&f3diXN?2p`qe5R*@?gPoBc-ZHPd+X4V>957@;x~V^p?0w>BNLRy-3kT#d+H6^=6lg6U$@en{U4HtM6d;fMULgtPrf&6OrxSu_sFiHCnm>-i<ep55DgnyvCFCPfy<-Jblvr@#*vLzCU=^bb4IK;h|RxoHe>Uc>RZ`ub%%`Pv16uNI|-s%s5W@`wW%{DS_OQ;Vc+n%*5}2dZ+Ygk&Y>tZ<77aVs+RJpK_$V7jQMW=UDQ6;lSB>e`~hzOsUw#ZF#52oENhGk5H?#U@0F8YjC1yK8`R2E%VFH?oK5h6Xvrlh&9YHfbC*Hx7ayCcqOYUY{PMuPP@xlSM<aAt;-omR8queU5GXxZGu+#-}L$W_g_uu7OEO^rMKs_p=5LT)HclmRfm%NeDR2|^_QQ)bwS%1q$*U!GQm0DfS27-mcqxVhG4xDflL<)F$w+~Z|luq9vOW>XoPHor1jK$;upw!RIbvXT7t1{k-I|1qbulPLqyXFJ&eQN0l4rT<L@gcVOpVSv~;YR#$Fqu4jkU7BfztfPjo)FhXs+as2fTrH<k>=I|oub*0jTepID74E21=I-a=SN=vbF!TgQksP`3hNcO2~<InyIitrO5GO3AD`$$|Kk!9dYn)*5ZdS#uw;tKf-kS$|`dqe>Yq2Wm!Nt95DZ9fdWN4II*FN|O|Vs?A#XeK6xv`1a8VV5URE`5B4<-i`o!#f}v-N>=RL(2gmSbybnFeRyNy(tH(1U&-sq_6HQ9wyooq9woEOulAy^sC42bm-TD!wC_?RABtVkL6%Nv7vPic*>&R?oYQdC86?pH-l7P3J9_u#$=`3ifcwX^%(k8n5vEa|&G(|mGjiW(Cp$t#N~i0A0SWt#WGsx6te2Qau#g^qDhuORQR=^!(;UwU%+vu7N&F;=hC8v6J;(E<^1x#O^>T*$RJ@k=;+SJOt%2v+adv^8H*@jaFE44~m4aN9%!^THG}$-`qz(x7W_5rbMXa>s|6>m$H1Y5Yav$M`jc#|EPO@%SY|KBs-hcdU_u2CoPv3Q4J$~t=I_GVTnVT?$x%izBmT&2N=K#=gRvOLtL9)15a5J6b{N>}n@4kHf<mn66nswiStjt5+LQ*3V+){!g_O{!8aQ~s`;LY2oQo!!Mefr(gzt@>l8o$f;Fe>gZ7)go$nP>RlGy3lY|1%!re}~P#9R6z*{^leB;(um8zxc@izWR9dJNe}6zsP5QiGIhQlwRwv4|P{Zcj$}qj`@sxDZg&~?l<_yyQcW<M$ahHBmMH~BBjT(Mcbw*deXmKQ0D)xAb02sRfO*2MmXL@T<oGT$WJIfEf=G$hx9RP%0b+p%{Y6A+)t6vXi~`(8Ky%4u#|_SG(UkWDS(0U`?iRYl$oba(s??dY+_XAb(T@iO$-<sp`IZzlf0on+8Hz@PfOXfq=G~k8RWsmY2n~1QPLFvuih{dzUSViS&A<VDDo-$v=O<>O1E|j3{JE1!>d2w)G@9dUJ}(|7<^e^E-}*h<frf?YVCY0F6zjDQ1^m;WmAfWM3BgVV_q!b$-O@qk#``Ok`E~clcvX+5sPhI<}r64#)fdjJtxYb{6}y+r*J3QZU2FA#rropknp2UfvQ5$v^aWHj7(YZ%BY_$&NF~p(*m1KTs(*9sqfxIkgyu;ZC0Sh`qppYdKyRk_dmklaDKO%_+PK(fBbhw)BpHC)wlhhjkf>Ke^70u+EJ1ng4YLmO3D=I5+{A;@R;fAz@GM<I}a^=6OconJzl5KhbzSN;mn=QV0{~XuY{g@g?$2GQ@6zw?aD0^$iu<OyfNjdqbb!~E+x!DbU{g~@#lQj=wo%evZ+mtSLP*nX3Wx@e7B@6g?=|+uMO)s2q_m)&3ZZM7vqRNBA+_krDlx%kMI+TSp{5kn7{H}<qpcIA^+f1!e{9`N9x=X$r1z&HiJaSqoX9dP+q^7q@s)a9iAgdTHYsRF1`S8=?zGndzPI7dP~ybu+dPR#PmheN-|;$fdANwC9&$+g4L1V2#5h;jPHX`x0o*nn0td`E4A8%jOY|EiTf6?b(W2zZ}HDJ0KwMNsYnN+7vvpgU?AP?cW(~bh8~XNP?cirB*nGB>;nuWWg$R!#EA7X%=c#LeBo5lj3_H7VwQ3*NJhbBGEgTfOFCnmz`2MM&PhX%a*~LC1`^Fw#9wk?Ezk|drZ!~AD8bMa;l+kR(p5&*DkAL#(Ve&&Lc5)(BII<2+kbTAnr5iC25PXz;{x$B9zvtT_qUW{K_@{xeDq-FK^(OnK6rFg0T=WXt<_b#^{8$1C6!x7M%CP|@h4vWgHH5B*Iu|-Srj89wqsO5r}$c|mReuZNt*)x95X;BPCnO#(P33KTO(taP<yKy^pk=m8Mey6o)+0_6U^E=FLkae&dn;1J{;Ldh^r#_&HPXx@uzdnrAC=NWY5*1yjqx!8a>#wenP%CDW*CtoTPh{In|7GH{|VgdgA6@FByjEl%fOEt`f_nSL8?B!&0(j$9{h+G!D9s9<#Eez+^m}<wxasV2J_V{&4VY>mlVIa070sNy2(Ooo)vWSP5@eO5wG;E%FR+mkTORS8{${^Xq1CsjNI0(^J+buQTu8{soTdVzkIE7JvCBG<%drp-0MyL%k-^V<G3s%r!zpa8?wAHhj@nF%--~Q%q#V`Ja@9uFtjZ-ROLgzmGPVIl+EWR!%Sozr|1O+yzPW3iA!jW@C^tQQT%Tl({V}f~pz%YPdxK*@wzHj)ylqB=@H31BiI?n0{nz;$jLXn?`Qi&qfA&W7(v%GE~%9G(w3poEdPK*vf1gD!r0mZZs-fGw7`3v4}X&u`EkU32$gZl!<p?r{G3hMZ@J>JPi?{WZ%Mf%A*;XpJ<_li85D<DZiF^p~;7np)@haS%Pgd(T!qE+HR$k!l=U-3LM!$LB*sGj8uU#9by#NwOnVI`m~3%E~?BtMp4fqSqe0y^lWOU^~fYYje2@R%D);O!8fCh%X;C)Y7cRgM6VGy=XsgM1VRY)9g!7igRynFaZFnk^=N4Le^+}6k5Ek$;~&_A{Cl)ePcPJ;s^vHE|3^K6L%7;gQg!8|Aks0*5^p7(Qaw$N+C^Ye;Acriehqzmm!v|&ONFOB6K&Z|j0G;))|SAqD5w)RROzT$<JqNwio*gz?49@><3Nu?M`OoZy*PBhbX->;MV0j(D4t#4Y_<6}Ucn&zR$d9PJ<H5Px@^Lz=0`%5R9U^a>%S9>Q?LtCDv6jY1|fJN`(|0K`E96XaWuvY?M#3l&8=Uee_<nMB@gBy*G{jFlfDs=YYYK>h9XTguoWiVY`w{3pMEmg>d9m)CZmPH+R3P9pKdb75LJ^=O)4g%MWNcssAivTGR6>9lTl5oO|zthC}se%!Q)s>07o(K4$?^Csg{<44W3Z7_PtOEXI3xZeI1P#2^M&0m>+o_>(Z^D6a9R)7oE}>aT;^<gV|$}FS1G5Xnyn(!0`2|Bkx(jUqmuM9#KBer=j9&)CFE`x*Bp{iy!=8CFEweA@KXNtSjz}aB3X|*Tg(wsuo4G39kx5l>-f85i$TQo5GPzPDa*OK^H&zrbGWEM6``ilszT2h&r8Uhb~m2HEN`A79gSACM!ZQ-2-r?vWS}PRYitjq2uW?R0~X_Oe%LdWQIq@v#HT+`pX4h&C}^A#qTKvuIefly5}*T4(e&s?&x;(>h;0XJtNl}dM+K$@ggqNVSO?yAQDIfJI>HtNj4g`f+$bG`I0Y!LaNQ|aN|Z+&JvcD;JBvSj`DC7X-t(ah>mGxJ3w_GP)Tx(MOpwOz(D>EIvJcy*;Zr?Lze*I#t4_F^pqU35Uc4N>%w66ki`1JRy>`E)U=dL7G?r>kV%Yc$8olxw3&SAMjs(V*i&mFqo{uV7|3ZgOWBbrK_3nv{f1|d^xYj;;y`UW&>V&paP1V9WKa%dj(YWS$3JrnF9!;Ay9pb3KWY(1h`&+K9leLn8+_%<N!Nhfo)9=KqKwiEhD#RL#R|FXo@Gw{#})RqEU85k_j}bOt7@VJN`3OUUQJ52)hDfMm#LFC>t1?A-$SBk=%M?eE1f#n{ypZ?$mWYn<BFi&ZaBOJ9qRMxA`s<*OvF65jE=8tA+k0e*^5krsk9l<v&!fj;(I1w=s!fiNx~MW>V7h)fw3C!!Y>hTMBlj<KdE&ySVcv|q;7CXA6cg;T0c6GOaqSNxE3aO?U>twZdCNXDrQrNSLVPcs4>P(?%6AIs-)0~COL|T&})ndemmoI(jTTY&Yp8-5l6!pF?+c+@OnGKvos(bivEQ%)b1#B-1gysI&~!mfNFsp3%EDpkNfl@Vb8k)zlW}?SZ<hiZItBEZ)t;AhD}RxGrz2o3Vsqc#3Y{%vx|m+G7pI)*~I|UGU+5)Rej1?ByiB8FhQ2coZb6E?L+9QOCC@&#!r=lAo_mtLp8(-^XEq>a1jbvi*4eS>ST}<LQT;mPtAEjSAr?zD(?Ji@l`dfUw5MRRWTxl5>u)jqg73RyualGw-?b!zV^3_#v9<MweBXV&SO=@RUClc--7>3ek$^**TUmzg7ST)E}q^jr<P~0$*<Kj$+dx)n|={?*$H~hkgq`E1(6{;Qn9lZ>rmy2$QvX}MzGOAYHdh~FfgjN&~yj_#%Ewzy!+T;|46%@u(^ZZH1yrd703|7FVk5`dA`y0N7>vQC!LtReo)@C)mhR+`t6&i7{-)xTKb67dimFVjVsAK)Ds3?{W6%nIO)V~h{&uNi2K)l-#h^cTbsyQ3>RsYxR_`u@VZhMVEWnTFDTy{TcJ?lL9Q+^cUf0T+5UX0v$D^M8#$h<Qhnk+rw2-eZ8kq0j&c;eEFF!7BjdHe5gmWEIGS=B%qqFa;Bv!=vM{$1!i1uO5W}P2I65qnAl(aZB?Bc@O-#}$>uE>XH;a+nkPQ*M$tXgsjI&DKMOHeK(}+R+Cde7m`8>UB9C`|)!vV9N0kIxvf7BEx*1RW{_c=qw71+cJveB4?BRi_LWXM$F*7YY49kv&t)lcT5T-Ra{2!jsT5U^!eXf4(r)-x*LB@C9zwk`B(d+%9xOS*>RQi!D?HaaHdp|n}qd2hJbhW03+vbaF}C=HySWb=%p+HzH!QJN}YFq)6{sy98J+)XF0`u<v0u3yC(b^^~n;eEUY@txYY8{W`wi0NP)Z&y2um}xQWH-g2`Q?6!Hz&!<NFD7pva`b0~k3s+NS_bkHsZuZlJ>Bzfm<2|tEKZWVgaI1%NL4y3Vhatp3jRquykI9$<7c~rk4?kSo<{1qFXJ!;hJC3z3doR@Wsev~4F@rBm3V(^3&atau=|Qk706Z5f7mni9@^m%RT=(CYx|2sYK3SVZEAISf9srhe^}2y@^t&j`8;1_4KlS1my=oPXn63no-R6V)XUB({7DA=!lnbx)lJJmo(o$O&6|TYytL3HPuRD5$40T%3<zBBHqKAQTnu|bw3D<AcwDQ7mSAJa6bVdZ;^c+Ua1<%R1Y>DjPSJF8cGx-!sv{_OE%y1FjyXit#n#p(S0x7&>g0D^R?cu%gG1tHjmwXFbWNZKDw~9kZJffze7VTavZON2FUvqGi7=IICj)}vUX!VEH(0Y~IRmu)rn1Az8@eD514V(B&N8d(uU<!w1j7Rfb%1|xnp;UD(iV_pWmKRb>rgW;wy`>YETRyioe|)cD0?l;=hd@)q<p%g`k*zqV;!rxRzP5Pl$MLXf~dfCgZQo%theTege>`Wz-###yxt_lj~XWaqSQIA{|aLQxtZGTMoaBaraQf<2*QeZ3tWRn#oDZnyLf*Kml#-w6Mzvl`sv`5D|H$4Pgck~T9&^P3D4$cFk(Xt_$6-RDsdZPbFCG-fo=a*;x{%Au0ZTuHNZlJKXI7FXA7{XFV}q4V2ibTtq8oZdEO?S9%`nc10{}O_bmJ9l3-R%*w#&(5s$-(chK+Sgi1sZdyPngi*ju<lwnFOG{)Rsa!jkpUIp-=8<j1!{mYNFSn1#XB^u+`AC2+L_;%ARe_Wh*H3wk+*Qty)+}sb`bIUdunD@?RMc(UQnM?Qz_w64u<%pf)ilf*~wVjP~0r$Jf(Q(t$Efm+m>J5Fq@lEQeUTF+jVIx;Lg@=C^6(sa<)nmI*ht|k0PUgk(_=GreX}hs;_#D!(CBr?VWUy3vovA-EoP`aKZk(Cv@chLC{fh_s7Z3F7c%ZLxZ8UgpI1I<-z7=6P0Y+^9yn;95XY;K5L?#i|FnNsd*I^qW=B<k7Xz7d7W#W(xSDBB)b)0cShEwtxqW(Q)T$vw_^Vznf#ya)eO(@#}y0e3Sc=NQo|NYbbKRkc+U6tm{(pt$aBN6;0L(eLZ&n`iheSDVYW0Yhi?Q~zHf4=0(%<@rQcBP&G$Y_vtQ}?4BX?H$23BEbY@n&Pc(y39p3}u%UIdDE(dM<ft3)>=LAJs{RmMQ-oJr{+!Eonu*I{-<h?){$#Mz_V0){*I|1QrzH^Edpqn1hUIXvz$aMPbUTMrFfx%%w`6^(#gZ7s8#fzU7`u!$R_v>7VAI|4`}vY@8plge}mil`uzYC4;kd7ynl@&3uv`BOCA*)|%>bY-O%*&19WD@GF|DWMIx5{f$5sM8bQ-sfkj((Q01vKC5Z)te6wV_wg=3nYF0Yy55qC+yLzsFO<_~)qIF?xD1h$wg&5nz2};3`0q@X5j$C}7zU5DBmgecP$#wH;=Cc#*oevuU+DEH&7iiBlWV{t5}I`2dR;l`dfxFXm~90Iv1jZskk!+!<s!d=Sv$NWw5#+k3Ge4P*p^Ez1z)-Kn1YQI?+L#)cu%+>lsR27sq)frfrTG9c}$UXD6MtKM-7olS@GiNcwF@1mefG2L6@!&-7bm;K7WoQ?v)&$I$ipQ?SLWKGsM|GwH(T-I!2>NY+`{C$O21wUjy<hl^Us%TkW}7QtTUL4iZw@r%cGIER^ms&~U*}o6|0dBCS3Y5o%6KlwV#Ha0^0Xi{GFaMQL={b4VJKf;=Y$J1Jt}d%kEdsJ3$|O_01o<c%O>meawY5GI^Lr&$@{eNZOpC`mC)^-7J@H7T&69IIXfiDJPNIUJ?Ql$jAN(8>6@&{<eY4tBlFv+<BTXeYpK=FUe1RjLeF7VP2~PfMEHj*H@WoQdtQg_^^y8ET$e_qW>ZyGK>!tLr~&+SApWe&1f4{!h@Nl(ss<RA*5|?@0QCqJmPcnDX~mD3-aQqqi9wapYaP7|_D-pw)v^W!kE>%~1t7zmkh>9N2B|?Cf|a&!6m?B=-97(((N}dV=(n@dlVKCw*Q_Z3_?<ZCZJOCNlwL|I*dXoXFBTC0BaEV@hUC!T@HZO1W#~jtL~0UuW(Q?mzTq{v@BUg)SZEi%U6iNvB+mnARL0&x&F!pC*>8+#&-;HN-u;<}&O&Xs-ziT0eC6?{7toJjs$c8fDpVOVlT8MyiT3esZ9Br<_XSl$8>A6BAqoZp#Pum0EmJ6;yP!J_y>1gL(L`354n_;?Bs31&5vSViOc$?Uhj3?o}|}zZ*2QL3jCNInm}lkz7@H=3#Cs;u{k>;?W$e_zUvhBszezmQz2IHI)sMW-4S%HsB2W)3RJBOG^VY1jNz3RvTSFS|~)ozwMp751WDQlnxfc%!sXevOGb((eXGNFT1l#7IWcDIjJ%{PK)W*m?}@gA-2fR1UD3eL=E*9jBSK!m>Uy!B=1HlEyb;32?Y%7ii(tJk>B9oCI3zRNYJO*IX;4)AC~#xw0kVu03TLat`e)N9hN*n?RhtwA#G5sZXE?@TRZo|=HFSFj|*Xq?O%$bT1c-_IiG48wrpE^7Y*md3_X2T!?Ba>1fh^U(4nX@cj+C386VPq4%{iX4Qxt-4Lp4HJ`RqZ4<*ABhG-oZX!{%lT>#b@_wJ*+71-BfD;I&SVzgx&<Le>GHihKD3P|8Kw=N8#AHPW0M!sF1X2iXH+3r+E%~ui}z3*9Ste4JU<6e#(LD?9?7AQNu)ZSbhCYk|z!(rx^As|5%s%`2**4F}7)=G26Y*_6*fFBQpj=_)~*F^`5WCM9^swketE)|&6mBQF~a$S_^xZtYUb~BF%lyo{IP{v`2tOF8PTqh@WP!5XgM3YX-d+*2%PK%0@yL10;96aYb!GkNVWb++h;`ie^2HXMm8e856bZcZ8S$F~V_Fp`IqjM{h*Q$&g!!wB*VW35t3v#qNe3xa>Di<|li*L`C<MDR;?t=%rxAU@GV&Ou(hHSO&@3z|yAKcr0Si#IYt+;~Sc8HzE6)ce6K3Lno#IpulFql7L8wm49Y(tP)uatLxAoGw_K)bqfwt3a)m}KrY^sfUQpY+|2e4)JK?}pGvt{B}3Z4$4rbA^1{soFnYscBpG={UQj0(U7DtJ6Q+X1*}f@%NXJwTF~0?7M0fZFby@2Utl<{}y$oNikH+yBIUJCo1KV1-mE{^a*{DpJqn};h@R-{Vh*jb!*Rxk}Z=i*4LI%F``gcBS?Ir2sPDQ6Hy{q(qK&KU%JNIMlJA_#C@C6ktyX1cNobUfud>}x|!nfLmbC4xLP`F0~(QTOP?#0{Z^kXHcFg{suK0|3yqR^5Q#I)OwuaGwa1A#S*Yp^S>;Ju0*TCdm0hq%TVsu9zCXekFT}TgSf47aB1;%1>_=R3gncTNK%<CG-JdN0-hX94@cyf#kDkuY`8*hzg{jGGz(4BpHAY@C;oFJHD=g`OJUH1&L}|2GoyT$KD^rsRp9+M5wi}VDMbHjWI;vOG%J#XHuZF>3BlRoWt^QgSunzQJRIoRyg8ib0{Gx~a1bRq>Z*Y=M(T2HtF>ByborrO{1OXYkElm!azSp81+}a@>27f%y%3{2vJ1>q;IH1mLy%E-~UTd`0{S61BzQ5=;zvwovsN1|rt(xUi<hw1brNuJ)_++{@ODt<{0N3Z%uNkBMN;Pc4%jwEGwqIC}W}ZBq-8gtebYRuuAUp50?*CHiw#^hxb$X$HNeUlJkmy*e8%V@059N-rq~4^pq|h;F;;Jg0PorJ>qA1!-Q6w&jzfM(=;HY1eMSp9`qRrGqQq7~T@;9I|3M-4GPpP#amN(@UbVS<QijXaKq$#&38QD=CeFCM?7xmE1sD}(KH{4;6=IhfHJjv;nhz@zh$R*4fZ%VDPlyTCQd#MD~yii#j^m71W-Y0BO#4*-bd+gKfl2YWTeRI>7H2ZqC{>AJQU&%JQjD7BF*k`_gS^fUV<#;H1-yl6_S!mV)?-b+`p{w<ETxFbo*7HmuNX+k7Ag3#G(gmGioTlRNSm$;xc!EuJd|VUrvRFsiFkp%8(V81xlq>JXIys*hl`~-2OiVlrChuGKvWLY-4E_w~J9g^joHv??5=H^p503+oczZ@`DlOW>E_;bh2ITzgact%!(i`Lq3tFIv<pXVV^9r!-Mymn0+FPLK6`)%@em&sn)@CErBj-DJj0Rq*1b12i0>5*|Y7_!NvyFKdq!o|^p{tgRj#+hc4DJiOSQvos1(=2P$zD)J7^LU9mz?~$C3$*+Zv&-yfOnpsWkZ&D#W7I2p8XEzMV?feR!`Hz%5R_8xP43VzE^z4g>Nr<JdI>t36{490h!Os?p}4dXqddh_&Zq?eC24m%)|O+HZwUzMX(knrkCDyMfYU%d4jb>YI~%b7th~thSyn<Pi1~N+?WN7o`Xn*p%>`NQz$>tAKt#$3we&QNU()*8Yqo#_Z2(eUd*#>8`IAz5UhAGz|t(8mzjPtD8}QABsUc*hT4a5-d8bS@YCvKx}3}|F<J0*rXSAH^|f4-`U$BFWuL|+qYKlJeV>f*S9e&PPf=eXTk{Aif=s`7^ZbQ;OaYA*A#8P`q8$lp36QZH6C4NPWv>wLhGA}~J_1tc;|gDPiGMdY4u><eL_oxvnv84>w?!5wO=&87LWKy)sYbUcOX2nu?wjQt$KQS;E?4C?Q}6b9)*t8HY})O&?swtC?)}{d?Yj>iwv*}X&t@vq;6V58UvZ#&_q+FZ?(OXC-rregpxuX8oDqE3z5npugNG0A-Cbv(d#(1>27(W9pocpTAMM%$-4=ByYgUgMTo0>bjn#YY2lrYPleI^>5?aR+t-JSk@7{ZO&z<cA#Eu*}^%B^hp7bh0+R9rGryD|ppJ;lz-)cSB-M#C=Qx<o3b|!V`=N1B(yH~>s_d$rc_wd2}dv{HG`xx5kmiGSVbvCqN>#5z3@XuP?3I17YH{8>9sd|zP4IHGnl_Q%35$ZL(WTIQ03|SUw6hZ<^84bM-O1KN&z03=m<)PdL&b70{`fi32HQl;f&cQ`~`6}q$SnMh~IMgqWR1y-y64biSLkc8?M{V6T(^HwTq^6S3nzk*<uP`eGKdCoZN#V|r2df>vb?TS*w~!><Sym=gpf5NIujZ2gEVhvm1M}+Jhir{^cOKoZ|0-l`Ts2bI*4PS+WG}}$Gji5^?K#=stvf6KtD1R<wd6g1y~S+ZYhPh8cX#eyVKE<kmc<OtXpO}T&S{Osd~of>v@@M>iWOZXwDJBHt4u93n5D0f_e(>oCwTh+le=f#Ip>|jLMuWorVUg2m09`lb~KtS_JA{vOm1mA=$ZhiPGt4W2D0IKIw`mF$?>*|(%3%D=2ObR6e(*dYfyC{hv1g#LBjfBqhR0$<Ms2j+-x-dV*Rn;{;I!TF_agzV$V(Ha4P0ePUR-40Q`Ei=u}tsfgV{<8(y8;Vm<rgVsP61Isdr*0(*Rgt_(Zd<scnXq4JSCFX0O+oZ{d+!zrTWFWL>mB^P!>u$<MsQJ`8^L{|F~Bdc{aWVJs5vR;m7-_nnr?I);Cf`2tfSOdDrc$SO?pN6^uux)@h3E5}BS?ek|YkwY`wKm7uC!sUEWPcB5?UQ2hW?l@JgT-TY1|!W~ZiYkD#p5YHO-?cE+E1SbnX7@`0J$E>pB>9&Te&Kp&wepHGuT(c^w}?j=?}|ner1aO@QY!Z!M+lvKm0<NE|!a{;u${u#W0OvUkT3&Xam#hO2z?zNwCmjk^>3I#|su`vDp>m{>l+6t^v+~SD53=$8QkpfMN{K2u%uae2I9u{!DbgL18>J#&r-E9<Bmw&S;xnnDkh+(qL%NO7ZV{gVxQnLaw{cfUUJ=*}mCXw$_+s`#Q7Sgl~pGP&$$tMtQUF)FC7Qjn5r!TR9`w<Qj{M#r9LY{{s|`-ax0kF|`KlfPYnW2o4ueQG|!OmV%;I0l0yBf)FH__s;st33Em_uA0~YCgYf0AEb@tn6&|FT?t9;Pl}}0Rgl!aaU`u$M$|)g4OPVEaIKB6E2|+kg{ieUuG*giSFKHP^$F-|L*+v~lsBPzxIVyZBlY@fhwA|wP&%v&@YyefUr*t%A;6y*!vST(x&VLpg)r<X8a4#@Gh^6OGOP>kbyN)32l)#4y{dX42&|`EFrXNrw`o^P6IYO;E5_5eh`Ql6xme*~cnPsCOh)H)LD?LB>M{r|br`zdjI(r3b;7gBESslPBnus{sqH$$@=)XAIA3(d!mI~y-~2KJ8r!EG6S&c(We3;K8$%4{EXKn)nr7!RFht!>fE7ob2-?6^Xx4ycP0Qs5;FA=~eizUO=1GAYWa!~(=-~(ZAqVx!E2w&GCC?|I;J?W{D^Jo{*4S#rE1e;lp9O5BqRoM>@JMMo{!LYm0JX!Bn=)Kvv&L>5wc}_fj#^FAa|XTyrZg-u4g@fxHj<c;e+PyEMQv%shM2c~DYu%0;W}_97aGSJ<5QBvI^%!BwRP4&Z17cM7=6GNJ%moM2oCw&a~v4?&^O?trnCLiA!ac6&^_&*kqb_vT%cnNhju$W;5N0#Y3I{newGi>QJ1q(;TGV)fyl~!0gQ#VN0*7oser1-BCc{Zg_U`0Fc0-B5deL7;FJzilfc!$`j$5Zwpd+fA;lsuvrmmV`$q6-aA!>9Cb4JU3_|!Lx%N}z&zAQ;4gQRY+$8=?xxX1^B)5SVVT5Qpq#*l=N`rvobAAzk2a^#@kH=UiV=>P!qOYSfs_iQ4b17gC>y^pCgpAeVVu&99GIJcwFtm68y-(r)8;9P4$1d-Q{Y`ZQv%AR4&Q8<8`o>^0Ca(cBb`aZ~CU!SZ5M2I03aO?X5og&vA6-Vpat_?Oq}&k{6vz(QlnG$D!0>AdXTU>XHqU^BvMvyiE^N1)k15a>1J3rgw?($WK>_l8x=71xkreY|+=lI))`RUG{@?DS2lpQ(Kh2I+q$3a7;rOfsv!XX-Y#1e&fnlf?b(T)3qBYQBdfv2U*IdVT35Z8yT7m8q`pJeOeGct{IKrt+7bwwoFyIY(&t~w~wsDH)Ahj!}PD68l3K+-Jjk`*ynO*7+4p!`$eAVSJK=LpI{Pz^kZ{s<6)t`EyzeYXov)GKdOVa)zdAVF*^}m-O_hW)VHx83vN?>2&r67ccvsePjGl&0g%9&W@;vMf=-MPGRsBX@h&VtZD)L0YX@0sqWMfZ3PQbbew+1kODI+n(WrGPaP8O4UMVY(x@Tx2rq5mR%QZabx@c|Kc~Clt21?2Dr3dn(Gq2(rac&h1$WsM%heXXz=sf>XMFkyimZ8txMjkEx3PC2n_$BT}{(m6b&Wat4C_Ra{7I6p7h4=5g|(o~)?8s(?Qy{KM*zLL{g%w<+R=Vo&_4WcU>^fi>cp%-}69H@<BZNKJGyy%70rK41o?P7Z`nKbF3*7a+N+;97l#NvRo_8pT}#?P|pO#{skn?K=G2TFiMiDb83Wz*lT4_!Nozx9?wG?T>Eor*b~PghN~!%4iz4AX<{V`cw(^CbS<;&-nZvWTl-67n7M{&Hz02fU#sl&VmFeM4r9C6;bn9=PKbXi5XojHmJ@w5zJN6OfoojRLTrKma(s{9?uxs8j{f*&5Y8o{DR|k)9GRl5OFF>_F#-J-ChOX)4>Xpv^+(@XJ7<u7swcjtc20N;t1#*a||s)fm|!XrzF|Q5Wia|LMqxbx<M__c|Kg6*pEM2<Efb{RN4{G4a)dvN}+n@9XH;&p?7IV{}~G&LKi{eBxMIndDG8)h+>iK+lNL2;=tQ+e#aWHj{3`5TPgU!`Lvu&E=7W5PP`2VW~l|6ZcB+a7)w7GP7|)zLh<y($zV$9cx#$1=WsA@V_iewJa96OGyd;ZJ}ZX4q(eQ|$U5CtK-x*fAqk@V+wZ>B7YS_NzCNV=aXLK>fZARGYI_4v3Bg>V9n<>YxU8m3qghU8DzO#y$~dODfFw#(HYorR36#bX;3H1YLHQv-(&Js(g*^!$l_2BINUJJpkil=Phr-qRtf(MGDvCs-^f*PK%-YPQJ}}L_+remBs{%9Z0pjpR0@+!90XssZP(iRvHJj7hz5AQ=h7rpvdfa`u5qubWyrRc@t@cJeLVLT%m62RTh|KV+!?|!=6bms^sGTaJf!Q*!a?tx*EHi8`fR?xT=75G50dYtE5mSB*%4Y=Mh%1ezXGsUvwAAH}MwGxbZw@$xOhuRplk|G9YI3dwG#~>C?QB>MM0PS?h&`HUuofT{l#2!>1og*Pul0ztIm-O+Z@tV?O3$b^8X;ID8`{IU22r0(;5&VRaeG6}tk;nMx}qDc<R})|9^44v)W%$rures`P%5M#kRS?S;&nRv&+)|P^Zq0SaWA{TJho<QF%0K(v~Nfzo<&MSDdRU3DmyOf<H(#)Cx|U;esx3NzTV&g40}s1lQr@Ut~-O99TEQF-fkT2?H*Yl6n#y*uoWgnKCJAZNnVC0S;+}*skVlh2i}4L)2LGShdW2`e|t&mzxc?>GFd^fgUPaB9e<{y1zLa<^_AXYflgX0J82o6=zV8hX!mT*fDw^zB-#ySbZ{>2*#!vYco`|jAi^sY^2r#=WqWwI)lMGWZ^PLNf3)sDz_aymYbR+xXu-O`zmHlwM@MlWH>_45tP3%vc(?(Gcav7@?tOr`l{|X*=&peH!EOs}S;_s~hYxNDVonNiBRGDz^XMVt_~E0LfcS3vE<t>^wbQ;Kh&kWJjX?b1;iE?car^%LhYI3H_u$!1vh!f~-qk@4rT(}9m|MxchwTRh-@S)B63dTT_lS~PyZ3i*3gjCkjMlyOE`i(ze9O)E=-wj)`N6}7cdrg|FvH14FcVJOcki|c!29>_N&p}1+}|MpcOL+NSLFsWKg$i^5te!9{#{yTAWZ?~PWus|@!sxE1sZ(`hsR3C<G~mW<f~4t*nu{VQjmGD;K5xJD^gB*Q52_TRGj|CJq@Z9pEA3*n-vnXuQjSZY9^D@VLoqU7a)9fi&MV0dEme?_AWg`A%}G=KkH?q9OXY(_5abfak1j11i8n|!(Rzp4IEl6u)*-^qm4i?P_MPfd)};!mk6g2+f7VW0kiIxB8ML4Oy#_8yk)vwOxM@#>g_=p`cPpav-Ks9MHb~Z^hPPdZbxrf=QLykMz+vqc<v;Dn{ij>U~X8s8Mn4Hgmz62q3iC{h$Cimx}{)=1vC4{Nr9ThPL=*TA$@1lsydR5(Q5Mwd5OqE#MF9OEJ3OoM&q2cbke|(XegTN&$)_H@cJO<%}Hw2v?OG>6QxCzxkdEm@&J1ifWl5BL+GVWvA+RimY8T`I3;Tlt_Nnn?~c#{Vz&?={&<oBbhN32!h^e0+72t<WR6+5Rr>TR;!_hpqR@eK0qh7LJKz<pV+APjbD7N_7ZoU(aVhZp|NZy><G+nC?WKiL@u#S)A)usxx>S~<sVy=UrYLNdnD!`oP;Hc=*VYQjPxSn&vX5G7O?WQA`8)|oS<9l?Ef-Qi(*he+64T<vsVy(j3@OuDJ|R{9iyX}euSM?nn9HhE<laoY`C9ogzXe|6rr`V*S<ADGd@5sMKKnE4@@qy=k5u0cXWe04=Ewb5nJY+#u9V)gqz;TDDkyfjH~=<j)(mXTM~?aY<>T+3zIuAleRuHq)sx3>pLAb7ef%n}eEaU;iF~=waHv9#>LVQOIo`PClQ+*_(8IS`i7E84(wx$%j(}L}W<;$@_B<W3Y$G3|{i8c35n{w|N&`*g(*PT5o?gb<#U~#+E$JDWV9>dM*S4#xW>lioLgoQUeh0vLNk<09Sr?r(rv<vr98ZZrMD3jzCnKyW=~<eO(S{xe)32_dUzKe2UawkvA~WJXe-W&F;uu<dQm?3ivR_Qu5?ykxd9*O+_QtKLRxt!B&qP(meo>6;Y9pe2=$Aq?jt$Wh6`f=!jIYY5aU$Gl$oSNZ6nT|?p_Cqe-%Fx>uIW+0Mx1l0&;F%b2~k1blT{Uans|B<=kYenP>5L~Q{tjdz{`(%)zh`?XqXbyG@!xnk3KHm8Cei{4bl<NAtMF}C^$1Q?KP({=k1GA@pp9H^66H|x6k)~fdBU=pxcNogkLVp3?*PW2|c^2=|~XQof0o4@|GK-Lh4E{f~Z8P>5LonL`#tmzE?FJ4IWO&trf7=iW$R*@~2#pbVY^BUTeF3?_Pk;2!7F@wL<V>-6b-o9$F}8%;2)gvYt1xaJsXar9+iG=d54Af7AI8206=F2wj(NsDaicP(He8Xjzt&)(=ISL;8JGH}k`t<Q{1l=s!n3638&2*HOaftfZ{tl}0oqLV)KkG8}rqwXfKMDY?*8mIZc;uen8nYlwvqq*2h*_r;CSS1yL127B*3O86wWbVi+47hGNsmw=DWQAsEMr|1~Ag_L|j?7fNYEV?r`Gdz6z-M3Zla-LfVI|!KOBFhtC-Ri1JP}qxbPS7j)1v-+!UStaf(2TC~fs=PHAWB1X|Dil*<KS6W369O1WgSWYDrVY5*jmwc1V=+$!Hz5Y*4ubbUC9o40P35sYhqU(g|MJ#?5Z8ul%^ncn|Z}uvDl`o;H<6tV+np8{s%Y$|Em?I?0CCU+6x~C9rK5Bgy3K95aC;1kWY4uzqNY>{#e~9Tn+>EZ?vG_(L2IzWMImDrNqvT+6UnwNHpmfbROu*6o)zvy$5>Lf<eck@^GD6Z(S<N+1h9h3~TbkjzhLa*`O77O{GwhzVpn3>o^wQjrRg~_%J_nx6aM*v%tP}Iai=Zi*xbW8tlqvRbfW+i?S-)Wz6e^kgL@WVQ!Z$QlUsOzIU}Y_@(<SdG`G6y92uzIG&B+9u0`R&acRYiQ@ukB<%J_|J@6fXj0}TMcqaK$vo<#h7JWK5fx%{Cs!k(HvkABdI$KThSGA8E?5nnL{G`qX2cr#FtD48<)?!wm{iH*9qbd%*CcvA3Ji<&07%a{t?sX%EzUEL&35Rvf{vWo!n{jzEJDu}?lw`#*o-}lp&Ip!J9Jn<vTFS5p;u}{oNo-x2HucY4Xm^b!NE7xWYC2`)RM)+RluErm>-1{ElKoxI=&1He3BCNXQNTZCABS0RY_!^<md3}uc--f4&8Ou-&TwS7Qj|itkh2g4)n_9aC8<yHH}7>P`9kc*Cr|mPoP+dvw%(_2{EgyD+e6OrH;={3^l=s>|hL>(0(96hRM%Vc&xza_3rAnd{sA>vhvb??U@C}4A0d*-eJPP^?17aP_M6<jQ>NM8x=Z5mLa+#`o_++8i068==&|W-~r%!qLv~YTT&Czo6ZtM(mcKNVCOFlK$C-~uim|W`|QQ*A9)VwX9|bcS3RR=bqo@~y~QZu4%X=Y%|9Hxe!Ks@>U*|W(Korstlj;qA6~xs2h}+i2dwLSBgFix$A7On{wckv?-?2-WVEUrz>vRyWL#m3M?ylX*(PC8uUAQ4Lc^qOCy^K*_ln_C;Aoy|NUrz(mQ-7HNBO*jYcE4y0h$OILW!q=IVt7GXr8ysI}BK|a?o*FGv<rO&XCq1ab!YJKcv`NKa|YCD-kKhl&MCe%NSkZd+o))`m&nffFv3%aW--NOi-_h_S|_k3=FFZhgw~KAsBN=xW#9cBfWb4>S^^zRef))T@C0UR>~@?6t2NLTTm~Ar>pV9)j8Ssj!iXf>ek?YXnAimIBJm;*<BUz^5@L(wn8V3^20Kni`!H)WGB@MD<S)fzk_fEbXyf_EsZ1oRf8l)3${W5rY(8;NfoSNy;+UQOh9e?u7W$HWxJx{(-Cp6sqwUB%xV*ctFjYQx1Ll@KB(z4G!vbW+lvT9^sBey7^c^YZ0PQ1KhVwJ@9%FtpHg3Pi=%`Sy<CFC*w2(0<7jQ-$P#H#w;k--l-b~zk<uY@LXwO4^ThkQY3G}&cI9bQ-gi6t&$#wl+OdDeH8;YJ`6I5pwsrKMan)_8qyCDk6gD08R~%5RbV4sAx9P|~%myY>Z%2>EW9|Dy*P>E5mCKUaxHM%FqSZs&5vw5rqA2^gJjb~4kw?3Zo-nMagCT6Uxs8T!9LvP+INUknOmc?a*%_T(v6Az2K1El-0&}E<w+xonMxT0e1HLSl90NALBqfm6MWka)OK>TF52eNuWiKT(RqP#!3ySQ!iY5>NNzaOWX!~(Y*#9=X=^R9QA}TOA*JQ@z^kash46rFVA}t+{E%?dj0-dR=#LU&wac6~Fp|Q*%q7x<#XBbP^5|YNDw%i@X@`*6r9aYk6U=4XlhKu(*ch_-byg0CLeFSv8C(V#N?r#vM)lmg@LK~uXdZx_aOp>V3;j5o{M%o^<@toAB_N-ji9@%XC^Uben0ySR54KW7;774KNNdVv#;5nd)G&VoGUKZkhQ-1VpVV1i!<+17De`2Be;;3_5uU6&SE`NVkmv$L+!9KSle8==e5nj3z4C_$8#0gVMCuE#5`R-=Q`&c~+yTU)7UYb7e2TNiahAX^iC(d$^OK18aiDk}D_I*yQ3{X5S%};hWwH_zasdA;?fBIv-IC(+w2rr5fF6R4l_~+?#F^2`<CokVU#XsCW5_GnZ9d&_e43=Y*_(tgzQ~JOuM}HQ}MU|UBhI*U{DMn&w|DgTsRj{?96ijtZ$#iTt`mRh#cX!yelkU`aCnMR7=mN?d^qQW>?gPZv8%}T(vle?F7o6ygThtF{!Te^P9dH2JOOeKCg?qb{Y)lAcEX$CR8G-aQEC%Ibj-*=m4uv%Kwt|s>wugp`{|KgZ!ia7gzVX2Z$CMsv-i3XieZWLs?QUyFHWfbeI)Gv}hV9!Gyuk;Qqi4{|$#nNmtYm3|RY;WZCf=tOvCKY{T`cS;e<F|keY>bLi=E-gYM1?*#Z;dPYO@K_eJk*MmB@nI(RbN2o2MXyNu0sc?*8jn&z^tRefIio_sQ#hO08m|gV<Sj|NE!=e|Yoy`Ktq~;c-zMkF)K{06i;tBF14&2jYD0HWuP?sE*V)0`A{8azyMe7twziQ6@b`pyav5Xp~bfunE4KM@zuXm@-~{MZS}1G{Pj=vei5rXDM>nt?2EKs`r=VxfmhPLrmTza~;jH0$6HN4w6y1OtRr}`yYSmFXyL#^~E;uhf;>BmD@C-6oYYjQps#MGJJIKVRtYbH7=|RIvz|D*mMApvR(1_(O@}22AH2=jyPPR{*nw@BN)fX@n@W(jaIS~eNCrKlFebLXq$hOALfe&oLS9g6FvOo;U)8rlYZs5fT^vh9o<3nH}P`L&%cTK(5HNE!Y<D*x+=L>r1!U^)@D*rW(@ez#x8jE4sm&pD0wwA{=_1P#MdRfzlCSEg^6M$FGG{`5)~EsbWqGO;xVyf5k>mE!Ra_V%f_9ydm&bI@F(sL?{UYpKQR2Se;F;wF(ZxCA_JM_xO;7YU;%aQu@ktQ@ppX5gNtmLMf*>lDMh?kXtRq%eg7R?=a#b>P&D4}#>F}O1@vR5bV27DMOh78K*TyOGA@=(;bk@I`;F9?Kk%f+m+0Lv7Sb1cp$J6t;(@HOJc@aW#K`cV);7}H(P4XxNr&6Zt=3VZ(sN;Ni9lSpfh08sV6Ywhxx_zhXiAMXoVj9|hB4R1KiqXO-K1J2)s?fiY=~g1z%bp0HMpu($$mfuE`;Jv_V=(&yA7o5JMM0?+u@-}d&(LJPW#CW2Z2UQk*n`3<@GH_rb=#hZv*@e`pc&s*h`c2qSLq2s#_asF;d>l{QX{JtL4}uPP*3ZBs(-w{iKCYQvDl6VA#U?0dp1d?lOC}v!Xr&vMLQr(Sc84!+Cl>Wjaf}l??3|lI73z#${7JuuULiqciPmAEKN+T)Dm8d+|T~1rq4buRb>LC+5=m88P%Rj?`~J&&Pk=ZuZRVN&*ZJ0n2=t$zGY+^?*$?JbXj}K$iqCeyJjW5I6l|IUSZfG4`oP7lt8`y<kg{tpH&RVRB;J^J0D~SwkOFbe`i56;Py+oZUG45ZTkRUiEv1Gu^A#2Ty@kPB^O&nf_t4d7e&DHH>PVGNN=9WGv$K1d=!h?((l5|2+kCBNUt*NrFc6lV<>;05kGo91g*&T&ya1B%_)x;!LHT7SSFFrqdzgg|beQ_r|hcVphTF0_Z^~;3(4PBp(UxPJ#V80%*u_)YajQatFZ|(=Bz$1}JU!1T&!n?JhNr45RVqW@U#T?#W!|u2aie9i(DJI9?~q31-{|DTqnMXw?_3(u=BAmUVAc60r)w4$FKH2SKUML#1M&O^-<mX2v83lD^Pk>YiE6M4h7;-FbhDFN|c+GZ2G`@0j`P9NgS8j>NqZxLEDIQ;ccU>A(>=XNDZq3WLq2us1tm7&Eh>{T*IuD?jA>ZpD}B*>U*G29F7no9KfCKTXmSpQ!C5CG|FcY6*tcrlT=kA=e)KHX4hOT6yCwhCr}Ls;*?Kr6^z-jjc8a02WQS2bWrE*y{R(l~cf(?IaH$Ji6b$cmIBCXZLP<_tArgR#6#$CL#D&(Y8UblD;cIS~0kgf$Y(VE0PURcDD->h3fKZa4KnQi@J|_l;*lQF-Mp}mkoo~EPghBS;=H@vyQXDa_X=EqlQz}PF}Tred4o+mF9c6CSP#b9E#t$IRN}&e{U-2NXdw&Sj7du1eq<L>B~O-+JEu<4eJsN6%4O1zS{bR@83>XzxPGM_&c159@S1pt_{v8V5Fm$YAa#6DarF=fjL@)=2sUYmdB6G%PoGQFHOWHML1b`-5JU%bLta)B$=9@p)4s*PhuQc5G3j~$eT{&n`SssQQb7d?8O~e$`G8gg}j#XMNWGBA!D_N|Ly8hL7i~O;em2@G3VW1ka7)$1{7H?rIg}U9+F825JMcr12CJ`EyR^9!OVH!H^Bi5G(f@}N*YIIA(ucXt7wkhamy`UpD10Jj(+Ak@-)n>M<)0t<g|RKd2V<UXgbbF_8RrXQd*tya^T8gI0R~A=AyMoP+t9!zlI1Exx5BY+d%gW$J0@|=c^Tu@fg&3pQN*}e9u^lqpzKT!(C_T;Do<uUTtBW7#IBC_ttV+NvgcH)#FFA@sh7$a&hE7G3F-I&zTaXR6SmM&2>-fF1Y)}6<2$WMWE{FS&Erv-Su#FO7`pE3O&|1lNCQWU<J{yYaEotXaO>K%r`yqyCc0IHRBKQ;&KL;Keocr%^(|(wp1UURU4*I6&7u>UnAbj!<-EAlKLi>ZAmqP@rnI~wNH(aXtn(D;wVXyBWF~-^jGNMCJ$;f18j0kJMA|<r165j-VvQ%>oBIGfWPjum{9TMDE@L3e>sZ39L4{lj$(uPemRNPJ&D&mfY&{D-2-u4l&ep|R=i?^39kg~jCNH;cAI@T#OJV_Xz8VamABOvoqA<H$;atDUtC(pmXj5FCAci|nY)W9Pd{jePhO!j9~M+#7i)OJqhfBjp!Q5@vPW9<o<Y?;5eJQ)+hj2>bubCd&~t2-^A!VCOq-CiYSzviWf`BFn6)`Yp0tIB@T~EpEm=%*?O9)6K>L(Z)zCQE4ttL1+dE2bR_*gibv2=>7|J$#GPEk}@+7`$^s2qP!f@N-&I;_AMPP4w$lxs9vkKYMCWn;6qgI|#&hB2>-a0#1w6(9i>vU}5d9TKA&;B&9Nb7xUcnd9xF3WT%R*+4d5821l%cDKt@rT=b647{cczT3l^QkPbbjW4nE#SCDS^#P+B~0;4#~H{Bi#Y)kN2kq1f#<q1Qvq8Xh+L>Sqq^^F6I5nDRa>_)ohE>*<yhBLwX(7?7%Y<R1>EtPwYj{SZcZjuBy}L2`K;G$NC3X0D<&E@fr>0pR)NL;pS^eQbK|%UM88Teb+c-an4SUnkc0?ID2lXZLyElQ$c}e7+tUCVGc5vW2;G<&nCAXuC${{SbDf*Su@l>|on&+U$mZuHwk6x&Wuu{_zx);Mc~n(bRafINB=vCUpUgpb*Ynh=Q|FvIk5WoSReG!lTv57s=ZLq(9}B^tjpkD@>w{*`FZoU=-_*8f+VGgQ<)p$0JZqa1w~KPF@sRzG%xRR3t|m!uVW}cn9$ht=wp%p^7;qxg99h8oHssgWZSdB-)2HeBv)oLYB`gxOOKq2+FUr~rpitK@H(R%`s8$YKwwIvcV%VAX`!S}WQ{7PB%DPMoLN-8-9%?~+G>$M?GNwvt)M|APP(H&(M|t7Fs_mp?1pSRuy5)vuLXewMHT=4Eh}uR%?6K1FC~6?Nt<djEqaUYlFmXk~tR)%@(nIo1*2W}GY2Xdnl-ZM0_2fV^1PU7bp<%6z@KK1{Xut^r<)A?6z_yE!?pOLiFlZ+#)S_NA78lbZ{hyfFD+$__MP0H39N|^A(9;#rvO`dbrckue0kWp7OW~};L;{#avmbTiPMY39RxyJ9)}h68y)#HV>%I7yI6m9!U+DJFJ_F^RIp2&r=g&WV?%Y#RztN8xXD^&Td!ZSv3lgqLQNeij#MQ=A=hm(^o;g?7%oA4|$3c@v5;vMpL7&Y|^vt<uF7!K5um4mOcQ3@9==_EF>E6>%H+zlv?0H#<MY!s5J+pST*=V?MMbCC_>~#&=#3gPwcI>kDhA!{gfQ5L`!iMF(@&sVQVKmDOlspusj$Pr~Cyez-MRmkkA_el14Hg}Qz1b)lCS76bMxAA47hZ*4L6U`{2cLZujj6Y-*<k_RkNAx@qcajs6%>5)JrEm<)H3-b0)cfF!*e>yUY*AAr(%D@YKo7~(&DW(EZAeZcR49JyD~MmDoD3I{_8BR6Wv!Qd1xbwaue{4FQ-p$1m|mWL1Mi@S2&y+(&BPY?swLyu(%H(@55;hP_BDt>tsK>aOL2Xl6K?KAkZ+|FP+6kRj1RjUo2C-Sb>^dR+v+ClI|4#U$)4GvwjcrP6IbF$@rj}bMUVOmL@}{791q;A)kvSIE&q3wPK!>!oxxwjJi9ULxQMR9<1UNo_9=!b^OovL^bk;5@MAgB4p{dlR@;2w3uWGO|o^{+di$rnCt<M#{cTHR1>A!MpuEzJ?}R`5I~wf#$;s<EKI67jZ)e*y)*6MjNq!P7#96{z;vfI(!yo&iXARXEiGIs8d8GJ1Kl+iGSO03jSnV56+TYO3s4J%Bb(c$tm)PKvrI`(dEHs@Q_VRqX2(Tyn1Z`pIJ4R|SO9&YJw)ySoc{$MI_%VtWyI^(WN(^I$zFdL57Wtv4rav@ut9S?5!apznSa~oWv(nRe|k&E1#W7q6TW%zgk?hBl0xWeI@&Y!L3%xXN-I|7T)!(^mKZ^2t4MDazbbBxD7=8oAFWuFA_xK^Ln*q8F(_H9S$0P^P^uz?f>Flix8U7&tM0Xm@$R^THORqK^<oxkJ9lQ0fZ0BW%a~Tp8TwTO)2M8Y=Zvcha}3qwRws$-y3aukS<<*R9puRx-p=8Lp1J@eQhodOld1u4Gow%`pG>FTOEGTaC*eqh-DC?aC#LqQD@A7)k?jvs%cZVmi_-Y2@JywTb}f7<y0H3`JiBVwL~674X?qO+#cuB;5s><~PZi@Cioi#`1Z7v$F?43dU9{)#0Rh-m#dllYU2_2U_&P|{*ljt(yFj0!rYd=_Ha`v$hdC2r!wk5bK8>k|O9AI6>6kqm#5xsgBtAI~={sbS!-whBWE4lN`qeaoi<DXi58T4558MK3G*8gRYKNe8scxvtT-F*CKcIASRiAhziwmtTM*G7$hUBDO7MQv_cb7Y5Ro!Nxu4r*!4)4g@N`^7~5rGR`dK&$*j?O0#Xi$jP0Ym}%eCt4YbsA4*(tfF}ufA_o)))9rY}dH6pazLRR-J>2I@N>IA6d3^OxxVib#t4#WG3gViid~akp&F>@!@r*Pb@qL1=nKEOy`!gy)p`=eU41PJ&f|<bl~gouj>sJ-J|GVtI1PBMoBae6uk?9lX!|)_Fv|F1sJrc)J}NL6l{f@WwxJA2R#vLkFGH8t~!=`b+3b=SF`Gq#xd)AOT_;&X;)QJaA)Bv*fy2|=dvIzftAL&2&RqsU?gyrr*h8;Ui%u<LSf_aj8_UKV@RIUaTkp~3vM(|z5e)YvwDY^x;<gN=Iez>_thpH7==>eJw=||XHkoj8XGI1LbYW(6nuo$;kd2yA9auJa!?h<!iIP%bs}sNtZi>Lt&^*odv6OX-_^FZNzqQYxp%excTc*p&5<m-wT($H+}S3$^#&|b9u!&*zqV8ZQU_jjid3mx%oYtM&ey)xoWh`1pl%?nQ#QPUp>!0*X!@4K=(ZBH3V9TdN871Gk}>7QB0Yd?yB|+5-#JNyyH*ps7?i6pE7i@0u`=NJOv*yjQJzk_`*Dxs0;HI{U=qJNO(r02;?4aYw4$($jZy^zOXj@73qQ%dQH)|X*|k!z6lkc6p&IGhb=nyuUGL(J%_6&_JxvOq8Ak(=o`x^)J<=bL9;_RQ+gQH%Dtr`1GcmtmrCUlCcB5$)4J2E;ifBC;M8l|%t@-gP{%fuF5;FE_PUx%>zs*k%XBD<d`quRquD85fiWW@bVR}G;hPtswWAikmN+;gU^wJQ|pw4tS=F8}2&<Kn+iy%|a!Z#JI2G@1GHi-K<U#}o;FhKFFsRI%u1H8@TD4vK3F<v81C}eXd&GU4)#$ocfI>y(nv%|?@gmHFyvGlX0kru;vu8m`4+(&C6bepACr4==xXpb_pz`I_8>Q1!R4!oPO_}Ow@NWT!JYZo`KMxAx)sxYx$myw*-^Js7V06X6>LG*ll7)?YdQbJV|W_1yr=T$uXku!y3;*R|7$4A|GT*S;LoK_{<ZLd)icA@CH;T=v=Xo>g^UoJo`uq9GGG8jMwBc_c*EXtf!o2|4K2SYO>)#!Ex54bOifMV~v2s~f$>kxxQYOx0qe(P=<b}1SgvT<nyWeWAXD+NCxs8n$Vh?7T;<wILsM)|d_Dxc5*;~nW5_ajd!_KqZ&;K8VisR-geh7miAy;o2-hlC3|yc5S`cAf#Em7w#C-ScAs{$U94@(csc1J8C@lw$V)A|Rn3#S~&M#<)O$4fe5SAj}||p;B^tuD_KwR5{9>07FbOmm^UCFhgYjf`<R-<3Opq`zGp{reYindsr|d`h?a6>$;9W43<Qr)$!qE3p;m(grF576CiiCRQ^g8x|F*3-fBY+1IUot9C<e`U)jQw5zZRm3lHh!%jKfQ9CnSkJRv>u2`L+fk#>>+`Y^z`l>wEk*vAFcFkDp8(l`S-X~5B%uxkvLt~v|67ZFA3IIdA_T&F&v208><)4e18vILLIf-e&f#s{+BiVt8NG5Xe?u-l4c{QJ0PG84?x6K9kS(!*YQI4Y+7UpXo{Id*0#^4R3~0sIT_Pd&y{Jlhp4uOcj8LQoo=;7mc(w+snIJZ74Jg?jYT8Lpj6OF3G=&!Mp4@4$3K(>%rTh4BctiX)^9*~J5c8yAD&+W1CQ-y2W)bICaT3GYS80t$vo1qLg8gn`Kcg7K+Rnjwm$vma&35MA78q3t?G>J3Fjj_As8Ikn6bw6=n(hU}q~?MbQ{<|AAh$jR1DmT{oZAuwJ0CX3@?CK0=G`_!5FW-KN-WFVq2RcaPUiUj&f=?&<3XZ;k^(%e%g8gbyCnXzk;Xk$Q0!hhPGTC%87wOT`m=)Hxx3Kr7o)YluVmCV{Ji1qwxb18PYYNfiPD!Q*fXNL94Db(kTvR>9!YMi<KmAMByJqe#IZT-|KWjT?uJgfZFqg@UpAyW`)FCi?=F%*0lda31fUdw8zgr#p;h?%c8Hr;|wtm?Ye8%KpH#0rY8l4eVWvy}ncb*(d}TIr<(40M+)<~c_pCi**xTFj{sV?7wh0d?uXa$(_L#HzjZ!i|fl?mCf1$`HM+IY?oeLsG6JmK!VfK?>y`l6z^`KMVJclu$N@fPV)<%_L4#*8ar7Uf}I={EWgFViaPCQqpsAXibdI2hzb51HJ9CijLY!C=NT9G?~Mdk_IuBT$Ah$?<mJcWu5=-s(WqM>!mT@&`JN5syGh;E)aPj@`;>^IG{LnZR?xhsY>#7`$-&b2Ap%!SDv<_EepWJ2@v-ol-IO-wzM7MVa#VNdT>4fDFqe*(Or7%$VC-twWTt(2(>Nqf_b}+^=;oi6-Rq$xwXs2nZTp;)PO~Bl_%#|;OLGOLJ?eXQkV8&+jy!wikthfJ^y5O?!$KJO?^Js^w@6WvD&9v?L)_-h_{T^&1;&L2dD-}w`2!B1tFVc4h}-orn`V<4QLpUMG@?2l#bR6gK3IoMW4DLmSencnTVxsjA8x-!FWbps=rB=P2+66(P*B3>Z!myMNHW_K1jN8AO+>yr@GT#1X2f{*UQ^H%tN#aCUPFi%&9VO@vUkG_&z~d6Uc)Di9D_-MH90%bRflDR*TC%zCekSh>yQr#R%<^J_-9JBAbjr8jO0#qG7&=5u7Nfbs6;%5zYvc+3bz5tgY)H<kCDXJE@9miy?OswA_J#N<TXGpriHaPvH#Nn?&P%5jF`9hl|eML83pP;2Tyebm(v|=)tDriN^8aL8;Vd#RnV$TQfc9HCUVGxC?L@G%SCTzb|s%Qs;~m@=TMCYOP|mAn23<-#@#Axg_$@8zmTFkZx^$?CR!?@Z#3i<(IZLufG&tx_a@}t;@G4aheoG1gFxnWsABhVJi)qMQQ7Duc(s&pIH<$xlSl_@4<bSu3f#c&|wM6tm38gaLwI|OkrUkS$<9vwryIzL!lkJV!zuk?1{^^@^uyV0iD){aW6ryH3~vdZmj8QtW>e?Fv+jYb5xuo{mu1+F=Gj~)s(B*GiMD0x+~TNm{ivgyUO_P)OlP+2f=tkYW9=(!ks9P$Z9?m_z!8fpmY^6kO&z%1eqOyE+V)`X*8sUrAX-#W96Mg$p}6ks}LbYtq|Up8XXSCI9#=X7xvwJpuUx&Wpi(Y*GAoRJbPdCAXViaT<xJdsm%?OVTYu1#`-LU&vM>)Y|8uj;u_D6vQ?LN^14&xZZKC<y!-$x)7*0BZe!eCaX}<a%wk4=R_`Ui<8@_zF1*HrnN9ZaUT}|-rU|d<a-18>j>F%@m9rxG2^%NU818Z=W#`O3u^;2Drbh{>a=wq|kW6|=$xNhUAVnsc+7;>*zAXv7TX9VK3Q=L#n~s2hIr9mN7=yA&ynv+~YrQy+5+gQj38^p{ac@ejt%HcpCCnN<S(2pJZH2Z;_xyRtJzrI_lA2^i70J94N`ddpJn+o{@{$m4u6SqySzf-za+8p;rrVy`a=m=Fu~0WF+d2b{8K&HBrdbqs9_&_vQ{Q^7i0%oDcqhZyD!B;SQ<x)4>{PL-BW2?nVNAv`U0DD_hLLeNtV?=0l1f8P&Bwn%(i8!UM7M%RuDDp>Loej<v<<#Prpb!3SOjE)w)@Eu>83fqrkb0Ooij=UgQ9y_B=2R9c$8s_ZUMi-Y&=LP1|0B2b_!{!p*G_Qn87<_8J){=m#{;-owOZt)J3J0I!Q$IycgkJQAM)rp@NE@i?x7-Sq9~y5}b%dBu-CYh~N{4CMT)8&B@^A`ps2|0#l0Egt8=cW1m~AR|LW<oj5Wqh+mboT*gn}h)Z*sQ1^fg%L;=Rc2p&SU+=fNC%d}L`YDqH3Dfz?*g_)Do44+b#+)39!+e-=KHR|7Lqe25T9q_N42XMQrkARVReQ!q(Ln-aUL|G{9T|oqwbkn0d4ReH4^UU?K^D~hR6I#Dd!n5r%9g8eXVnS^t5#6+u&c~pYY>1|U1gA^PX=lbix*V1b`ti4(-9jdUc0b##paxf3m`I&Os@(nSdmT~V6^ZRVO1C&BqLhaP?}%OY0JcjauW!WQUeR1VQ?fy0Bg!U789jWs$uvbLijUhkJ+4SwQ@`?9G}(*4JyEVy`gWnJ-4=+5zXC->oa#^SJH`*sJ<J`rFas-94CkvP*-KVm*n1czyUfgb;xlf1nB{+lxp&Lo`h^LT%U79Rn6a5%DQ?Hd9_wvV<Q1x<xx^3k;Lz~9`kORs<CxhvZX6xN@LdAs=^Abn6{$??jg3?+U;9g7jJH1g7{loH#cAU7z!T-LnQ5j&=$h*liS|482jCwz`KNuExQ+>E{-|pUfP`&wgu8%huoo(u88iLX}M<yNoNvGW`$7#3(zQ(WA8i|Kbv;<(FRH9^`TS-1ZT8YClTsXMP_f*uIOQ<#*-;si0GkA;=CJZ;*LIZ8G2NuwBJ^DMXa~n@l`&0zT^zj$=*85JX=SD>p{E@MfdaJ;0dc@4YS`8oTL;&Ikj_~S_HG$n+~Fh>Xz$}Of$hNSE(_Q-^-vzs11=;PBdFm;fpr#rdbi?9vh`}!AUyN%$sjSlL*!*o)|KtP~Q78G&ysiyF0a($A&XpU^?NEFe1=>gK}e3?N;W^Va1pr3d%?*lAZU_Pfpp=APyJ?Y0zWZO=!lPjZ=0}XG64n-{ydgF-v5|do&Kh{y!~*4{#rZg-p4-BSX~!gw=3F$Wh_bk8stbOtA^KF8}dXxcSnJm$z=s*@~t7j4I%j)bbS`TFWLwa->$+M6DgC#}Q#sWkRIFe-tN15#SIzX8|AytN*0oP(sRJ()&qQMB!(RM#^aZmGo@rC%GdZj2w)xBvKgz*pz5{_u@29w{Tc^Y9IwdC7n*1)W?7#62`DdW{h3@pSwbcnT7{p2vta_K)SokG05#tP`0m$Bs7~ZWPh;Z2f_Lt2IB_+Jwfl+tN>dTs@hc-P}ho<7V{dmtXyh<BqJL}YLUXIU}QQ7v)UG;FH>caw;T>?a=F0O-`&Ps(L1}XGAP!(22*%Om0SYm^l99qIM29u`gFZiYonr;tT7krk)gY`VXwZf;n!M4n7aqsLvxov+fPM2TbW>L?hey~8Oo@<fZ)YKi1hM%F)fLcnpK4!#Fg6FoeNggeHowSCmW@gQr(vz^oTX|N+0jczdTnv?@FKc;8QXkL&R*1IT#(<)6fTAV*Q-c5Lu1@5Az2beT2v0G)FS1rOE&Tshb#i<3W^&j29IL-p4m@;T`JAN1h%lcct7kMqULg4sH1eyn)@w+o_WsZg`v-?YNJy(m{j=8cA+9CMi%2oRd1$&{V_xHs_3byd^)>Te2!%^2zTy<@{F8)bOFE6py*$$6WDauJ|!m{88qLA9F1$b1hn4&Ido-yH@tj$L!H#_UNO|9xY}29Sw5R{9C@bILL8Z3abv*kBWB{w&0>@Wt?`YG%rTM5QM+5{V=5gDd_tuQ8oEHG14$1?b2nCV~<=eGI66KUi2h2&$?v_XT_{w3|;2K6^1{L#`t)F!Mg~`K*qQX;;R5Pf7@X$Eo^#;4up8*8ObI}kge^M#!_jDRC1s`v0bl`GT?plXs)PJm07#0nzqpwlqU1xmq$>5Bb3$cM51Ok(4~xs&1?&+a15moO}amV+dfv#ZZNCs)CZmM5hZq&K($Mp0x|^M{S-q>0Xgd>B0MGe?nja-i+e><Wxh>ubi5^ewx?O#pAPVv727|IMme7fbZn;sFFk;rzn>m*Jj@Y1<;JK4XO3h&)o*<cV!T7nTw!2<rz$RI$b*f}1oTELVx|nz^v-mQw`Y^&Xh}Qkhsq;=5TeS-#g!9_EDuL)IXN@;;WrY0Aja<fV(wfKg62<YRc`9xb5xa9_^w!z<##U2>T>oW-NvO<d=cwjwj^~HD@J0yUt%y-7VxzS#<y)eLC?nHybdvrkXG0v1%h4FCSI$Y7PYJ;^yXZJ19QEM`q~%0@*E?4;}F)$AyUh+3a)gdb8wCbZcZk|Wvy$2z<P1UW$ADAHIKbR6A!~7*fj|DaVVS%IMkE7#SlHEs{;NTKvFV)y`uEOz#EI>U{!A3`Vt8NszS!u;4o(8e*lwdFFn1vu4!#^W~Cgv%X*GDx1RL^@Yd|Cu~Y4&GIy7pXor;JeCh1HJ8%pg5&i+~_5<AE+m_>$*>T;PGG(MY(5a!rN!Ah+Y-Ac>tVX?8%j3#&$0`k|z!-CEWygL;qDoDWf4?$SE)bGe^W8PF*E#cVRyf8=twq~Xd{h+gV@Tt$@C&n@X;<{4H8Qaop67Z)l?k;l3|}cqVY2{O=nSXTc1I~FM)ijvULamg@(@iLK^_7ChIuMnpQJKio;8~s>K}q!IEcChAE4Uv#r~)CC)U>ZM3V0dJN3HZmenqy+r!F`S*?aRvHoIiTFu`zwTHH33p5}%sng67Mjx}}A3RHLXH|9CyqA;iocg2;M2mrgINn^*&h1k*SL|e&^jb^qDg&C8Z}#<K6fp+#CTZ<pO4!;G4%5XUQRGr>t6z0v!?x`_sYa+I0B}ra9}qy&C*?qqgzc{~QJ!Krb~D+>bTmlr#AbYa)WVmIp`Y+LS8zEu)l!MTi1(+MgVpCmIbfq5{vGvJJGk?=tvp@%ULLCO1lgo3A9m)e&DtK?PmiNX9G>fi=11S+kF9uju5i7Q1+s(}@!FrKW&Pm1*{cXh+x4<utwL#Qzvl#RDshz>6K%)>^C~!5AHP(xv`sE%*_5=Gw@+nx(!DdgeagFh{TAyBg=2S}dX%b6^t?F4Sh7$oFwW8x<8rB}Po=&w4>OX5*3O(-iq-OsFDDfm$`GiVUDiNAv4P!a?KpkxtIyNLf<k#5r7YGkDmjL>F~<7bpbysQgXOkOxn6Z-2ma5=a{9)79flOv$gGU;i_{Y@G|_PbQ%Dde4N@HX2*zEhG0)R+z|F}1-40RswdRnIIJSW^;(?NQobdw+R&|B@#%0B+3Lyod#JsvO3|+=6*L=d${N<HX4IV(_@nGiB?nj`aO^0a{A~TKW?5AzT!NOSUe8y_WFIFLzQpuXF@{X|Pq8;CMYIL*5tg1(msL+Jz(w)c%kB{Y~$8ypKCnsG#icp-lU+?bjj%Ppw^>@!b_gvXl1l=q#vZrA)TDK{k**G>-CMQm(I1mT+(sXYSuX9wCH4K@uMxjm{7uK51b1V<&E$4C+81+vd)d}0SntwvfI2ys)4Wm0E&kV{caV~FC+ZbW|NL<rm#96+~B-Wm;q6^PH{A2``ksXQfx$R;?{`T!tkvHB?p)<OU>OzyJFkrXwwwLbRKDAS?7)xDZpSX8}&*R)V(}E4Aq57v!^N?(SyEr^~PM5w}b#0DH)-xCkJWRNt=PP!v=9R=-Kw3vUtVb^9x{nv9xL`?$4<aU13YQ-ZdBWm}I55rwYK^&(_ginggzDH1`}2^L$HXI%0QDfMW1~q#bfG(zk&$s%$h1sFMkW!PL6!#s!n?){-Jm`j(vZ=UEeq-D>m_22Xs|gE7gga(_GtMi15sDwq;zBO4?XuX&lGp5xSEeO#J}4b;zepz1KTT6sa^Hq{nGFKXHnUi2hKu;oyo-M>audp(z)O?(s^)I+T<gkN}GJBRB1*?_qj)Lm2tnSrG*I=^@5dc=T~F$N-PtG9Ai#NMWzukxq$cVjWF>*!x@de#3~iWn_B9JGJxg<kVjMdg3A}_Tli|EpG>lx(m1(eZ@V}?S5FTk^7m3v=#90|azUFRas$3#GsulxlVX7a;JV@{_pMu^#+*wz6`5Yaomw)-XpOjJv{Vt-cR`m0L9}UMMJ+8puQRx>J7}(bJ0S<ZytbQ}^7mce%{%tQ8gG5W%ecExSEyu1u!kvXuFfhvB7#!d5o+%UamQ3@m1SCI10PaFxjrRbi%t45m#X}+-1QNWyRgRlE^VDe&wOL^s#GtN2e7Rm<xiMpFwXL96-~ABC9aog)FNHAR`elgtH*LALR(#UIX@I_^>i{I>WJY5TkGoyf)SI60WL&2G>}uEK|t%~M&qgV4gTNRXP&z7Oz_Hh&sbLmIn;*+X*q&)9FKene#{Z>Lw6Web`nQJ=3RiRI&=(6>9*DMeI8ny{(Q7`)O}!GrjrkwW?HA`w%DQ*w{{h!`w$Ac^XOcqQeYK>0XLIlGdL$^ZC1X>KQWgg1ZOQ|tiT_-urR~HS6B-a)_PlT^a^7=R|4E}%ZS49(U=_wIfp$4?Zmu{c9)bCH_7DpIho_(t78(i#aZMlVPV7@t}kK_SA~bs`a+24-JS?0Mt4g<MPDv;)1kf8lav}WX?w07Z65au%Cq$%2?rZ3lKl<xy1Hm0BE7=V0GM5wr`R5P)rE3M=O_=#*Ab;Wq3UzqXvBnUDW7VL$tml|emkd_O*=9#hmYPNm}Hf*h-0csm{}(oB>8N<4UT4CQv6&XlNkKG{@@U2v?$|qbtPq`ND+8^am7b013g4*8YxZ@u;uao*xuz)*}IhD6)Y`qn9**u#abx}f|zA58Sd6vp0X3vE_PKsqt47$v?9XFz~Pz>g<U;GylufK3eFKmA)}EfvSoxKd4zU4_;}5t9-3@LFsl}bRPD!RJ0GCgj@~<mDraqUN?y0V!L$TH;Mg2OdJ4)k&LvN|gqbHO?z~VZ7GW`DsCA62pAKUn<naJQlHg?zd3Gdmuco`$11e|43W?M5zw*|@vk|DoC|?kK>+yx{BbF!$JFNTD$2LzZUdY-C1s$l{4fh?SQRfjUU6s}tbudfuIl8W(#Cqt7P(LdC#|bs34~Jth;r@Uu3#;nh2cAWflO77!^1f(&KRkTTV+HU7DS(${c&xppb37KgTbbpN8o<XK&+j6~BUB|8&+}NDUad{9VslJLK2k>K0k4&LQ}e~A^CI~nJ3$rUv7P#z2Vk-ujZ$g;c(vfm%d>R9dl%?B68&ynnLlQpK3L`n83fMit!t2)s8`BLilrthmf-cyJcZa{%n?|3rShpzJ?(l~LQ(Wxx-Xd;FUc13cR1=c4ZZG}yxxO4sx8X{&NCLU?FH~2BMZmzB$TVB%mu7ZWt8k>$|Mx9ZI~uK->y^x_ya^SqQ^u=5l=f2UH&9-R%#Q7k>9_4>Mpguy|DxTwSva)-QyUkJj4y;a)1fFOF)Ja_u#?YGiMvYGr#AmQ^q5x`dH=nm;rvw0H2%zeyl8g2+C5H&P6P9w-8~#Ta}!vRmG4?n7cXol-pp->eYB$Hy$bZwLfk*>s~JzwlQYRoj4xjZ`*p`F+SC#_OwKc#!R}fwy>+SWS9)1Ns`a--ii+#-KnQIfr!4A2^gPP!zdq42fj9*x;fw6uLXIk7YKUsLDG$DHJOoHPhK7$jY}z}&)CKS71Ygf3R}|%a@);=a=x@3jzhXwQi@b{u@{YVnF~YaVuqgfryU@Y!*ymJ*Dxe~1W;P<fI!HoxZWA0o%M}I-0M8uh|WFLxX^#*Y;^AFPJHfc2Yy8T#<{rpOt%LQPxbYR%tKAGbyT@Bb4ha&ENU>GJ#n@9^iy^S(^;&z3;6BZ0HL^N8#T;of{W6&zl*i9^QbgnA4Yo(XnIW~pdQ6ph|$ID9?)~D2`deqmlNMZ1>GG6<NLChg^MkjA=TMwdpYf4jRu2aMA>xca~E=B)Jg-OwTD{4Fpft4%7K`auA;=S)$O6yy~-_8H)v1YqFJ>Hk2N1hUq|Rl4>cGs&8T#C+sYBG8C~l19pGtGFXe?ZCfhE}zPbXj^)_r=rWnGJQiq!J{zH^A0P1Rw6+gmn&H%VoT4z;cd2>ll979U_!Y~so7}!`(ixZ6mr%mc3;C{OtHI?0L$g#%rBX2-aTE&fvcrEgpORHkCfwF_`ive&(bP#4m>V%pX2YUfA;UsLn6z734@YTI|uA~!c>XI>!1y-*Q)lo+%-B8hDN0MB@RR?|U22lsi4BNUN9>25C)=MCai!@J$_mtB@(Wr^mYdPAPP_}r(7*Cl1E^7g}nth_v0QBKl2LeFE@?%aFGq3{-6vgK<gHc2yb`7!8Ly}j9$YJTN_Z*0@L=&O$%^t2pJ~Q}@51pmq1L~%YO&Cv{O6u^^^_Q+}ek{Cl{U-MF0Ip6eXpDJ5KI#UD52X3cS@Y{$q6I2ofzHb(%v_NC=PI{{^aaVvG?(}LuG46J67NkA;qlTRGaRb?nj*&|CV&)qe}W*WNQ$ZLO3s7nZJPPoB-w3yja@JDtWJ^5;h|!jr)$8!t=)Qcis~`^3JmRL-|BbQMGmNG>#!|1e`x}*)-Gd)sPU|3SuH&JNNHw5%lh0ScD|DR!EXRcc9<*$j;7Hd%;LD`pRFO;1abs>4Ytpj&5ylw{pRKH=H**2U)}m>Xv}4V<wMrD4$_DsFh<_4c7MMs)(mqRO*?SBO$@5Q?PP2XL+qQ(jfzI?>2NU9yc>F$Z&j+z$CDT-4{0IIIvn)^vub4`x^nJCo%F!SE~Eev+SeUcUepJ_Pcb<RH$Fa`s}*%YV9jh<GmsNG0g{wbQstM{n4ut64Y=%4EP3(#FlE}}P|;KAxW`9unDma6wjziod&B6+2Z1$6@_6XiocjISr*6nbxRVxMf-p-aC|%NP5~166bZ2B~v|IA7UVQ%Y)v&QZr~g>J{=Tc%PpDV_Po_#&nGbC*+)@k=kh~7!2_73LIF}^jxowKG4^>CjN7gts%mBwMG87_DNpmR|wJ2h@yR7n3us0f-m35fI6cbc)LCSSG!Q`4k4~TE#QPdg8TikIpi3WpskYvMB5F;TS^AA+^3OeT29m>c}0^9M5au5$mTB*wVpjWdv!&XHitI95r-6&W|r*kCCvfLT4iX`~?Vwyc}O=7_y#e~BX&;1opxOSqEqEI=Ac?_Y+rdAByomXsOM?;K6RNc=CYuL6S?96ipm>c$qOkTTn`6@h>wEibpbg>^~;}Q=u*h4X<_o&O|QEzH1!qDA=Y`2`{Z6ToMF7N`#kyIjtl1dO(z(gf7vEW<4NR+6ZU{dptX~CSHzjPU6hg_vmt~e=u86HTFuc_6u?PhD|k*eUG;{S@ec%f0B9|3OF$Zv}yzCBs@o|M_%1v|f9D8oW^G4*82dy?|U&nBor^O_sYbL4gK;&g8hN?wWJ#MvBm11|^BETC-jw{E=f$7@&N{1}0Fz^Xa~Ps_VFj=KBtS~J)Xs&~ni5+K*nCsH%?Vx2VVHCXFzXSQb<ZZ<~wP6)f02)(Y<K6O^s=%qC&JrI|{A@n+lNN5WwEv#(pd1NN8a3N;{EYKKrXd;^<MwoI94Z=n>CD~m|juQ81^~@(+IYjYGwZN8Dx1dfQ@6GBc7f8p-9u27OA$;Y}$#lfqTGnuOY7COKR2Txm88J}J!8d24PES)6b_tL&pG;BgibM<A+8^#Gz{nj=qOtawbp%BrQs&B!g;y!3_x6}P7sWNzk925J1Kr&Lxv!6^8%$}8oMdsINoHEQ8;mK7LscAy^6Qnu*xlir$k3ST^?zF3s$8e4ky43Xj8{zXU8P7)Mz}gES-NQ4&JJ&@5&WvrmzbNvrKqTl*>y;hluRfwIN*@3w<uumJK*5#L0hisw(XFS7@0chOj&H)zJ1D?BwIwPmJ3TrtDK6>a{H7ECJIGWOQi~1MIUpowXBZoMU(R*yUeL)_s0k{Ps@)trY&i)T#x_<fk5(E6xk6kN(FbEs&vxkv@{CnvNdNaB}aowcGBwAZTeLT5=|3PM9a}YnWDKyS(u%+kWN%zAP7B_c89<{a$kxP5-5tt;KnDnuHU@$qH$xOpChe#VX3cS%JE^G@23<gMkG9=onP4Gl8$Nui;S}GK#8`QpVf$l&^o|F0NV_<mS|2~24&RHE@H&4?mGbWcJXF%FWdFt-zmmkA<@!CceMLdRH|^?*XS7VQQW@dq;?U)yDXI=Ez=iN1YCfz7SmN%7d~>ddF3P+;fnD>4_edCjcttHQmI-~kgXe&2)rAE7!Tb^+%NnkQD2@-%qv8pMI;R`=@NMg*&;ITBIR-qd;Tt9-)`O8rELknchrM=v$N}zR*~dNRPpF08WkmY-z4!z#3tf3>IL3qVAYB$-DregBabgCu?aL~$jCobTwXf5^-*~&X@@9RquZif8&F27dZ2a(33N?1ZDz}6JAf6lbm|SK8L817-R>q?n2q7&4v`lK4^u!Ix;5DKe3&LkK^(Cs8g`PsDcaGo1&0GHwZLV%g4cI9Hm?SJTS5QgQjEsIs25Ep(F~vTTT!)8R;^+Mi@L=-<6gCTVfnPyS5*v>msD&lxu)V;P%bFnB9RLIrRng-%%d@2Yd2<F*40yijX0vfdp-rIb@!vu2qdL73S>;QM7^Q}eT?V4$Z*mjFA0OOHJ(iW0j@~X5dw(oI1JdZ#D)ge5wW|YUyGs@Tt5Qgm1ez5XNMbNx8eaR%LtXJA)>WWk1&F%KjNywO+V^(0fFdNH7y7+5feG;3=o%QE?&6wA){V=1SEQx!ujX;zUJ*RhQgGAHb044sW`)kB)cs<9cAa@eX%v+%)s}&_CC}erD|NLHAkZq$qhSIBSlMnwFygTFNO%i(L6Wwu|;^cCd28_<Bx?8fOH*+3sSwSd!3|;l8P3bql8t6$2qAriAR*(n5SdXXP*bvlc3*nUKP_~bdjL%r6JS_$7uBOB#;b&S+y203k9MkK^)X+9|;|VBp)>00)JpT_7W5>U^vd1@kGj$rhpOb3Ls?j%>bBzFjhbuLJFsesv?%kRsb{PbYw?aEKaS`a0>3L*rW@_a}h+S1G=WsDx<<0qukLV7GNU0#-#*+8t8%BS$huLR}_L@6KAABHwyO5BpJ>jZ&DV(I^W(3HIHnYg7*L^&6DnQ5KX*3Mm&SV<xT)IRoKb5Hc0JSW$kORjq*NkA2vY{Jmi%@dI$#x>E|$FA#hP?2ZJ<Z)%$Ur8x0AnZ5=<0n;DIbX80`7Pqf8Nyb>K5ZTz>3f9&nPQqjcXAQ{`cI_JE*)ABBs-W9Z^pAG<ch|27T<N*GfC+>^UH{$eZZ*7eP9NH(GM7Af-%ziA18ePhYAtEi?hjK0kH*j=jx^0*}CerSP^Iqe?S?R&ISDHNzwbB#OBWD`1^Kmex)z}p~!g3)|hYOI%)3;Aq$)Qi3#@V3d567t|bGE{q6WMH(EiB%?+&6)u=7NQOrOA-Efp6q?SqkiKrrae`nL1?Rp2Y=KAtXI$rt%n24=ENlj&NlQ7{L?)0|X<4DUG0mpq$IR7*y3YpS&wo{zNLun3E$zJ9W#3!h9^eFU8-wMwQ#e&$@Q-wDrS&L8R3Jrd2a4MdqM+xy?crOlC1St#w`m<!#$sc}H2S$N>jN0q4DfY)wx^G+1J-=T*<DO6!!+N29AYUXY+oj4^B!q6uhhRayy0R41{vQ7ErYqdoxsoMz2HZLaRZscb-HYC@d<VwFN+SYD_Vr1G;QcUw9HWkAa;k1-Obv|Tw3Bg3OL4&Mfd2|Kpf{E$IVA5;NHW5T8;<B`JA08uqG|B%h?a~Zvg?w=Crvt?XRa2VmBOP5u}<qADty1IFz(C?IvG-e&J(Askh(Ym*FUjEh=wp5dIYqVSnaH9j8aF@qWt+01BWEGj)oLB&4`F$h{WXpFao@Ks!tY_v8pjLHbXk1FYpLd2^TF0e}Lcu-g%z(JUO0(iOUFwP7jP=DcVB|=sv<T<2eH10+MOYVR)=vhp`wnvJlQ%AhmtMSl>5tIA#=AJ0)x8(cXsUEON#R3`@ntY?6o&xY0YnjUhr_DtOArwbV3ULyu)U#nox&~gA6(8n?M(YiUMUv$ktI@cl1s^py`E@6-5A1QXaGp=qLtfL*A$B{E(a;0U0vQXVMASLyqBqAVG;6V5oIY{7{ByG+5}-JDAdQVU%L2wcxChI<y+xP7q4Bu1-0(pK6Q|GqfSU@ajJ#ZA+(0PM{l3Hr+T_|WApN*%i+r}y>R*F)y<bK)2-6~`HRm;Oto%R75sh2npoM_?3-1`<!C_JUdSJ9`<6hq8#vG%ztR_W-M~uSgr1Hp{o)n)VLYv9J+^lpBSwjhq#VFr3{nUuOe!QW(*LLn5MZQV&hSE3$U)Z0Plpp-n-!biKA_O;AQFl@iaa4(uD&%b7Eve?N;-m*Le1kw_u>&=+YDxyXcyM!bO1a2v<J8hgb&7C6JdyLuM371Ij7+eQT;IUMlq6V`~~MXr#Q$HbYhFN1=o4WG-E=cZW1c>r?*c%yPiFF`_!)U@-`mzbR{|WruQ@*-C3b0C|%hVd>f^QBhiI-QAp|>*n}Bsp#aqYQ%GNeND8rX<ZZsdazRE0_$+i<L2cA|E<=|D<^|@%0U<Xs#l48Qt3)z<h}r*yg<j&BNXP8dC`#?2K1ghf#}tyR3V(%C!8fjs=BUChswO<qgw#vWkcAq;iDlVXc*8MY2Yn*{F~7S{2ww>dO%y9t_pY(m^s73V_i<P3>40FmeS_R{$ZQ1NHT&tr>gQyecRioP@%k_VY?!RG3s6m7pddKvtkw5Y)(7Bfl^vNW{J2vowaeDJVTP*?b$46dWwzhVrX9SyLPCa!Ee^x7c4c?eiL;!%#h~&n(GD5@Qc4D<kJC}I2QSd&3uM@Y5%2jMm)BkZb~|OOi;`-=XdF+vje6tWE@{H$cyTuYxasJ1DA>&w&~2;5RpYF1!_i8>`HXvLQA5vq8_*Eo%XG*T9teW=nY3Ab#1qy8lzK$TYJWDyHOru|KZx?&Ve(AWEF-ld%VcXn$N{AAW)0e1lP<k$3L4R1FNITZe;7CyI~t7lBV46nni8rM%cQT8f^MCL-{U3HVn@x83;?f#H-7;Ah<!84DDTk{_lhM}*9){s=Uy_H6z#$}CndA4eS9Ftqv^1N-tB$mq|WCv-O!N3jy{lH3Y^v<TG=3b=eS$22rFqa%Oqq!rtL=;U_{!fQEHP+3h2IpNU9)Pl$6$OXjJe4nG(TDe4q4uFA0`xn<N#@B2Js7Z{4Y|8=7OsQw*x^I@H_pz^S_q;<h?x>#pO>S-J8YY1>9K7R}eJW>E5BqZtr{oQ6X-9=0!>t5;zMX|eUMQhQ(++}Ikn65B7-Gq+DE@yOQC7Uz7~KdF#vd{qo&3zQ+pJV0-tKNRtY+HBG!kJM7wAgM_$@>OXaK$B&A?kp~bTH)DL@M3ePqKZALsc0QkO+^){m`cIG&Yg-X_Nb<!bx<`GRb-y0)1V*N+!CCy_sDm`dM|G;*yYUp=xElYscmx?l_EqGQcqi>GIK^NHBu-gg8-vx?`tP(+%|x;L8^#c0eY)QLFo!aE8CceBE`0{ZRjgWUga0N32SlMu4T6ClxU#+!meOaMdQgXYLiF{6J;t0a~Z2OoVfJkVc=%X{nMvi)76oJw65geZJ`I!<4Ff7G3vegtqQ0@_x)gkIp}Ln7>+wly0|g4U+%Kfix}$!<QIMe%LiA0MZ84{L0|S-idH3w$Apl<Xz3;O?G}9U7gP-}qDXH#9L~_ux}O|L!&ldc1H1Ay<o|HIzV)S64L)w^7O=2Xek&Dot+`}i67H_1QICKk>+}dvPUuLJ2~BNyVy%9D-c$m@^Va#|Zi7(=xqS_li<e~_)-NUG*olOZl*g5{o$<f}#HMLH)>fl;<>Aq5JOHGJ7O8iEgTw$cOl$n4*76eKrW4eAz?k-8eIwx<YWM6$gXy>z<)UDO2c>8)6cMZ(D<}v4Fo9tbPUeJdDpV>_Vu=~K2rwHhHc<z`Wk-MC&^S<4Krx>nq9FyDh5J!v^=Jlu(V0rs%-l=7n=x#r_&_tZMVk_WnNZ5H6&8uBr~(|Nwt|J$$F$zYfF|g|E9@XjHijh?jSK-(Q0-k|1&T3>81t3xWC`oz0^m}GgphFCS4sny%@;~CytY6_I}5F{<YEF?G3TbT4~9NC{;-7hI7%j^ZmXAj32AeVC8;fPDzzRB5B*xWV_^ZCOH)W|xeN=&<xHbA*r8R_K-m=K@g4=BVU;M62M=TUG<I5EgL;b0?(5!$d|j!{EnzwEr*pQD+gd@gcuVYZ2imo@-^NDQJ~ZZSY~t$Y+px$;u^w_!EskkzE|q0*qG5BDC6RD9-QL*25SYO9X!EF{`>7e%cB%64ud_fyYw%%F?GV`$<hheR+r(mARAt9XnhSRc-y+o-p<YBzUr1>?LA%KJd(j@=3IV*Q8HV5!TZ*Wpm*jy9ISshfC~_PH>65Rv*8qxbA;Tz%0>0SmC+~p+{NcNkF%zLyn@5fhY5QHX{d=f+_4-q^%JJZA&q!lOH(8x{KRQTIM;6Us^lVYxn~n$QdmZ<j@d9q2BJc<z(uqm;NN0yL0ki`x6+D)+H&VqB%Sxbom{`}PSd5}+v|1wkrf5X!7DPkAWr{y^E@E}wKtz~CL;37Zp@>HBpyF<)xB(HD3rNcg;=G0joE`KmhxUMBI1f@`%vWNV_Ec8^aoBxvbw<%TD0VFdzUt$xY2nKQ;7>{0Ag#uKH`#*~kJzg^bmkt7xFy^3x5vuIhjk9X_Jfu&1UHAoS*A4ntM_NM@UZr6XPOLptP0OUWOb}VShtp2Q1nBvU!TVz8n+gbk=H%xB0?(TI@~}x$6n3w{uB@uY5ORmc#Uv8S&DbO@imway!=+J?njwz^-esdb-}nMz2IfVmbd%I9%}k`jZ*d$;JDtsNaeieRq;ne1(4PW%lCbo;Zi~L#KHDaqZiwJ>jdrY_(8C~hhadX$?OSwy++JQ;T$za1<8aAIINYQhow5$1xHiGlIgtUNokSR#saQEZ*%E^HEIjHHA&gS<aI-kO(?WV7#de|#p|-thCZnIQHYu_4A4T*RE1-)ZL|s#!In?$l%s!1NeRn7)Ols@d1dCjK=3uErcVj)5NvU&I;3?ugbxrbjNYJb-s~%kQ>Besc|Ee=UgANN)23<=^`5qt+R~g)VvNNWjxk!ul;fm}(~5#F;O!6|IPlk@h~%^uxS#{l&gGK;88>?4Q|xw+|7G*kK<lMu=qa;wIzifgf^-&VJ5R^wqt1n>cdq};h4^f<bGG?(^FsfbxY2pK-)lbg)H4nE($y=*=mHVtsUx8UJP*ft7?1XnQ5^HAXHQ&x=A6#1N27EE=OJp6sg?#ZAl;k;MF(jZA4PzJ3<bJ-c39F2vTU^8lkQ4Bwl<Ykfw*M7+G<|+ZdYP9mS<+&=-580=1$3BOIGEZDE(N8c^|RxNKHw{e0aVSkMqF0q8XBLE&Pm0!!TJGAFMbkU_EAD+U-s!6J=c{OSwGyJ7`6Lm9v~(Bq<mn*1%>_%!a?z1v}!7$C&OQL**kN`CwLCq*<B^D(4Ed=hq2dNfW2=PFvftgVDvaCd~3CY+p$ERxqM~Kewye9KqXpM)L5h24W{l5ydBQkWD*21)sTnJ3GUK!oxomHF*4Z`)y}R(MXX=*`(^&pl7#F6=v7a8%4u6)okJ}U3qmVb{lOD=ZnXD;gJ<VCdd<TpnGU8!3vSLgwQprIh2g9&oD?tApX*K0neJ#J;hcsk%p;JXqYfkDQcHWooXc-JBU_%A2S#%(4eY$$;z#riW|4=5ni{cYjx8h5OKx+RC)C_kk`KIjO7a`Jp1^AYN}SVLmfFVa#`Ohc{mh?(J~r|bqahUhQ8J!;kV8)A$IL6+Fjv5k3@KyiBKlG;|7Jj1u`z=muPSp%`)nc$gLix#=;S6I2sXe3D;a3a+oX9Zsp#K@eKE8DpE<{mH<v<mxr^L0Kwkf1`_=u{J)O$I+j0RY)|g&>PZ`^;Q#rH&%9`ru|~RukiekpJXao^w~M1wOL~u}%qjLIW@r$6Re{~WdoiAn@n$qC%%D+1+SSdN3Y&2bU1Z}Q6gg-31@BspST+zvgPt31+vi$4yYnfJM#szS4P)vOT#b{KxfY#Ul*4QbBP4@4Zv@_DUIal+&v?SXAU<k&#)Oyd*4&|jh?d7u45EoH+{7SQRoO!X5b0UOD)>Kq62G34ct%J^$W}$|w#`=M>bG`E^ttW1i;_8&O*$W3;*yv5LM5Gotg2Ho$7lV%#6ion!%?^&kZ6IrVGLWKUHS5b|6GSmwXAIS!2NU2j{*ouPZ=<{fS`SrQS&HN&O9KDxC|N_(~T!X!T`-eh)7~$f@HI|m6F)#^C&7%&peup^(1qROxO<<i<-?})k38xSsj`O@|5BRKq&IWsb$ZG`F9{E(@v;1@%D}xeqPEOZO^I=l;I#u)Saor28k}Ig;$K{W>cP^YX@MyD2N;I{<dMmBqmh=`~OOuHBZi=m26zsPp!Z)X^IDHYE{vpHRzmaGxz5xX_hiEN4PbY7osLLR*nYCfOp(eFSow);$N3*2Pl}0v~9n5WHw5i3>f7NRkTwUQ688IWf90!g=?6$X{|3}uhQFIg7Ngz$*je(B~kH1KA?PpYS#=X)d)o%i8|!cTatAIDyCU5j`IB=$)E$@?5QO1v-4>XQvecRqds1^w_zimf7fi_UQG`f<Ei&%5|sk%)<PO)x^kc{!P)G!?)esQ0NHP!T7wM)leRz6X^W%>w?xUGV%v10-I<dqxUYs7bu#>0Qqy_=rClmx&XdDs7Hfj<O3k!Nu9)olT{d2|8n+{WnCrh(UxlTt*)|HNhssbkaqjmNi7Fb}&<StYY^yz*DRWMt4<&P1XVCV35*$vFJoa4;yEO{-pk)XF;fz7SEQ0(f$6x^%+iie8OmVy22pSvY>yMXU-Mrmy2Iu%WNy#Ax-kb{>^3^`ju~|HeJ8&rAr{F@+5MT0q+-{!@8fSyE{3c8GV9#O~qDAx)MKNy;xmT#Y5j-V6%#yU-KBtC~%|_j|JQ>C?sdoE(u)&?=`2_nsFWT!Qc?LMundRKrIUY*~pn8^ig=!bXi*7oJ^>@z$><0-=pW5mIUy$_SMRNn%f10My-H#?&oUawOzBs>g{B4kQ@3h-b2Tke|*D0M&#xrd3Y|zA4v+e+EK!;pL<mt+AqC9V;+LIm@xxmjLF49EkDNaV|m?ri#)sK%b<{8kzy-7M9b4OUNA7yzoPN=5_gP`9Z;!rlk>fkyKQXDTW54{>h+5Q?pbcBltGsbr?i9AgQG{W=GaO0EoQ}GCuLfGNc{4T>oia>QP;6M?3N#5&VHJmLxe=Nz_Ih48#&Yj0Hlc<Nwd;tNnp<ZC{5I{~wyav5a(*qiPg8;eTXwU=0XurQE8`=;R;MGx_GoDcez(l#;8^JT;%@I~;GAQ>sFh(DfLhesH_~q;cs+;si5si66hRNT&jzc^96fOIIOGd9mO`a4!zcP%8)^S(Z^ebs+kaY0*Je9bU941*hI6zdu0V1FW(Who>m@19&1A#M)CU<)2;b<+%*Kp}zX=%FG;Qe%bE9%EMiiRkgj|Ygp9W2MFH0(_$g92v5Azum58`x&}YCIGJWwfEp6TIawKqC$LN;oCZ?u5!X-cg)m*u5j{j&XT}aG>3T8whV3M$;i<Ki~vCqXs-$>rOj>Y4E23P;1~4Gp?!6J^TmIS})RQK4ldd9Zd4>{34U-2r0(ALZgxR`V8)Rn4bQ%-_K>;2F{XJPkqI#a`H1E3(TT%&S;$1xM%^eoKUYQFh*)p3=dJLp$}m=pp}PD=bt(soNp9gM&0f}RBQ%KMgsZi;FSb0VRWZ*-lK#3$oVl($Bd?kFnI2D57x$F(-MUWo}w?4S2D4W8?^T0sCy^E?L(w^gWdpfqtk$i%Q!(R34#>w^zk6dk&gsEhKS2q!T}62oX;qVBQ?((9QCMOj6qXuaOqcqn0i(WBEyg)?e<e*pNwY_aBk3N^Q_p2<5>^mGjwU+G-wORS0Pge7%r1)c+W%t=EGHEr2RQfX@kakW41>}D7E&Cs5Lqm4hX7p47kJu7~Cgk=@r6r4ZQ()yv+dUDkZz!C_eMPgAQ>FFbeVMQ{psgQvaX4c<m|zV1u3~Ach}_??9T)(JRj1NE74kCrKA~{8{LfN&u3J0}1Dpon;1JoKn1Y8IT%9E*Wh2N`{4_H9`E#y2%}&Rh-qFLDwu^58`MtLhO1<zGdRjV4|4CLt2P6VU|fyWgr^zDUacqXT<x-Y%S~Eq4_alWc%>|owoO*A)_+RGD}d6DUr6x*pMYh)IU?pY&eX@%vjLyMoGVqCrU#gFs2+H0CF-8ia8ucK+I{0n%ob(7(^XDW(l4%K>`IR^LR4srh|0C=LL40@1waY=1=S91ANUm53ENY4{5QW=lS?i`cndXlj&@p4v}wZ3POyJVpM|hqBx{{s5j(eG^GO*-))>hT5wK&;VWipp8=J>+8g2)y}(C;E&}4~bIcZi+?dP5ot}ZAn`oyrhiF<$M#w4Tq6c0x{uHVZEYxg6eESr|)hp0`I{H*H=ED`6g)<ILy|ZE__~Iaey-cm2<u`}O5ej%=lZV~Iq=)AoE?WZubQpIsV1|ejYe?P_V_4(@L$J`%AUh%!e8QU<QU2&?ADE_M)n?I{w$r(@;&DREZj&F8`R0JQ9HDDLNGv?Nv?mb>Ts+d@isA=GjU82Gqow!AjR1^_JT-K4M4wQYlJ9$QJjQ1gajV$l0d-ykbBcPQ9(@Z}D8PTr5LQ6*!(_PEoa}XENg2H?h^BceJfDnaqqtP`)x&tyl!g9pd?-c4q5&wpG*ba>$=9WZJMeYD&4;51PCr@ir6@UwFX4eRFUJu_6tglNk(2%MaB*GWmd2Tk`URA(OmT-?g0tu%%T6gd_R~-Ja1-OyP725ra-9lGhOLDN|BzD*hCul041Jh!_s%#?K)i%K6ZVpf(mB>C*=`sQ$N4N2dEDL5l$}VG#q;>`5@kB8^Fy7NP`;ywE9vA?G|i&H)oU_6^T&w@$x`~eYeGMX((K2tUJJKg+<fVe!dsubwRQPgc;n{vYd5w~yNae0_#cLi6<QgNj*k&P?qGoJX)a=OM{q7s6cA=}rR&Zx!Q@7x9@}{cvk&@0ZV-jlAdr0w>jmT4a<@m9af37y-bc{>vown5yl2<Nk37Oij2O^deJeMF#^Oa4?w4Mw*|HwsW0zlgV6XsxHd+i74yKiBDMJ(-$_<<h45fBVC~z(X2M|;gw@o~Cf%`a!H%An)2?YCAb;2M*=(=|h3Ehg8`G?;sI%7SCd+5srWJE79iVwrW52j?>XLtmc{Pv?lZ=`<Pg><iQ-Q~B`k{Xm~E!XIxSy-O|7S^BVwu}(4TgP{HB?Bl>!FaX{_le+z6C!M=11}Rmzv4#0dCLH@N|MjojSCy~6{?}bNXVR<-`GHRzVq-;^ZfbR%B^Jk(Kv2n$k<n>5uxp}O_b~;QGHfGrhE$ca<<LpO0z|^4>>Zo&o$qFDB+d1Yz3<0fA1%h)et)ApjQfav*>?V3GY%kh(u%MPN6F7gpfJZjgwZlptnLRdRdJ~5{ZPZ9FqE37>w3ho>p4>O7S?&K4Agw9N?WlY;fRxw>Ng`<q~KwHSha4%JT3|JW~<Cp=80V`2|Up*;NOFVJLoLmR6Cxa}doyb}?}3URiyv%}2B8;+zh!?NH^YH1%YHMIQdLO4+PdYp>R_V;vdu0`O^tx1AH$?`8yG%hd{q8wMN*<#J&O{2g;mMPM^_6spU<>Z%nD=+eQRO^`F2EaO#Vc;^7Cw`DV~BG_`9ny-6y{1|Szy^Vw3saM6zXq!i4sTt$IX_wEWdPVf3@;Yg=7t!fL(o}^48wd-mYG^WN1GZ-EIcq~%i;4EO@~Nu0T23@q<=<6B9oPkxN^eJwF+QBVB9hG&2^)>wLVT8p83oJ;M~AX@5c+I_+yMpABMeG?;Ja&??PXr5!r)5=9Zv{Q;YaJ+$@V5PXPlG4udq^Butk^IrbU*v_BQ?9akWwi?Cmly>1=bAee3eNUVZ<<n0%8NuY1dE=jugH@;Qr~<le(s<s|>muCgg0eK=M*S&m4%SM@&`oklkBF|B?~t8LB4V|x9VUO%SSA1uAb^2|(^f;W^B-H?W)UI7^6eYd|CvUZ^x4PrFusas!@;hy-|2+r3{a<6IwvWwO^da>tdWMLUp<^tf_{zh=F?ll5aHH1YbF<NJZ!(@b4=M`VcS>pDoy+PWE24>7<?I-FC@D?u1<8jtTU6yXWE<OMtlYZXjD?Y;S)<;pauRem%HF}gk0%>00A&BUbA~90#Jea`lxfk;Z_7MUka3W`Eog8?(^m>I*xw?rtYd62*GV2)1x{DSR3ofy!SZu+<QjwD_Zpmp1h=5~>h%`7?$RrvkS)g(Y%dJnj-qOp1ll%y-&Xf$j;+`0yIR$wYSU!IoWUy_Y-u4tKwq?8OgR21t(zGky+tp3NZ#-2FmCrx(v1(XFS@q;BDo;E8!BtQ@^UTLNh3N>io=X~4^Ncee%VDld%y0Fw5+04yEaB2X)hh`ZZY})nsDztsQ75lX<H;-}U$S`lUW{6EyD6}UcI4t7tP0D_K~-6K9;KR9-SMtia#^78306hxVu?j5_p#KPU?)p0iPrv-E252m$ptCTVc~k%jy7c9ASf;SbO{P;QDeL@(>Z&Gn^z*N!b<684Fx<mXSSttv@h1ZPpHhi^jZ73U-u5-<BQ(<x_&r_GY!qzISmlDv7m?v2X*A|k6;p0*M6f;ebyaKk?j(lajSpr?nk4M8CPA6fW@yk8D-O<fBV#E|8V?jckQ`RcYl2NDmt%o$=Y(Guk0Hp=AB+J@WTZqEM!9Vq9@$yNAiBMP?ZElx+=d`3#g=Bn@q-*t4>Ad03lC(F4Hbru6xvLTaC0<wHJzRg{NZybw?G>X${cY2+n(F@ZVa{9RQx;9eJY$n<p5@li@U{HEH6dITUW5KksOZ%nB)+Yc!ceGoRbWSSLQC#uSd~r=k7v)UO3(sOi_-E?U-(Tp-zNSIUN{UL>h#Vc-iG;(0W_=%t*{ELI`aSudN%LrPaQoxqp=AdRpFO46;$+k9yo>g$)-A7&|cWkDD1yQTZMZ>sF-4y&cw%4{($-h*+)&aCi`yd=EpZS||f)fEWP)_&JZiP?K<!VZ}mL9WMFb)A~?S;Cs>0jsS|De<>`*tO~#JI?C-Lqv+g=oX(&h9S)4(-_o(_PP0q)?Ivb1p?>xER({QN6CwuHw$w@-(_PcuRZiU)|f0LOuG}p2AKpaog{k+X0_qn+)XDFwvZ2%tw?3qll%Yg&yK(Tg|~nHgZn@E*Xx&V-v5i&?|=J8zkTZ~_rLp#`(ORZ+rRnA{U3jyU*7-L&+h-lKOcYXJIDX<Wq#vUlenU=86#|YdLzGm>kIGv=tpn={52TBZ{PZ2$ywK!OUu(oB&z=ME2XN7V}0_xkH7x4cmDcMVOr%0*?@v`DI!zsAs&t*gck+m^D%w<yTAG7@mt?Le&Yx4{O_+Gf9u!BfAYrh>%V;W^?&-0*S_R$|BwH8?aMo7YVa>W_V^2LzVqfA$G`a5+i!mV?VtaWzJ8XzewM<f*vn^LCz!*qXC=R4ik;i1=8WmC*-k+IpMIlu&pAeS=_Z!`tQk7l5xhVflLb<9W-$5tU-+~8|MI>2-~Qt98-MoB*Z%pvZ@>2L-+u;I{Lg;@%XIv0Sf*cKSh{3XLJW7zQPuYu69sihvP--1**(TidW4)2#)ZP==g2p=Z+{x!6BVGR=Y0IrJ5#IA`?38$o`oeZ>c<#77;Bp$`_!nQOsqm@y1$0$gv~g&D{dVrfg>R=rkw!>1Oy25M)9?YS;VB|(0&BW%?$p31RTx`{*OLwXntIvMON}Y@1`&^Je7!z8s+{`qlO*<xj&OXj+*jKvsSC39W={r)f!%fVE1#2Gf-rt3{|({Vuu64V3}Sb^2E_&qt36J4tRtFk&s|Qznc^+m+O^FH>q@!N)rsjrYzmG;f8K&lSYX7q<riMuo(NOdFx(PS%UG5%g*MOZK`&g+-`GTS=nxr+jW+eQ3L#9;A=%^zzzdHjEOIbu@;|a{F(k`@8yc+$uKDxEv?TBjPAyzWf9RDiW05}3u3&b2rSkk?hYPiX#=v&U=GMe=C<&8!}^MQjusd9Trr#ga1b_%YgFbAlepo!t)(%^Rirg^t%_DuEFvvoRZ(rWywz}<`LeINYO3Z;_ELc2434S^D|}Rn2W_YJIoQJmRUp(`S*H+S+Ct;X7LQE2&u+K~P8L$w$u>ow^h=;G6RE*;$Pszuyp6Tf+T3wg4bcxX+fN5QXv3V7jnWPv5M$SK#|}E&$B-^u!i6!at}E%By!_JUx8ux@Of6(_?hGE{iYI~z%#bb)8VH{l#8i?~e9=>_DohjR;lvu)WowRMISOXUXd1gxS{ImeM3Kp-L*GE$9gN>e>&I(1AVP?^M87NAV?ZLPB-+d?jm^J80BAY^u$`P~w01D^GkRgeLv7w{0V=O70~HH0RYm4}BB(W&f?899TC+BP^ie1C#e%GfK5E{xrOmNuZp9h|<j&h$*tbMq%!=o~Qd_G1>L6WnfTwLE$xsxdt?E~eELn126&qFSue=ACIN?XJR#>c6*<OI>rVpgAJ&1;#UgULKp0)OkoeG**uA2^8-r8dH(Xgks;1Iwr2#`o+T|{c>`5G;0I0WSvj6ud%)ElrYQIyb8Vg;N6%sb!x+5NA6=lI8e`Oc62{=L8Z)BFGW<@f&loA3Vgrzpc9e^QoXWc2N~-~8d*KmRfaJ@cAaECy+xGyYo=To?8foc;q8O-4TasQG`;q6hlt%%}ganoDG@V;3)6-@179sv`+=c}%T}<7Kh1Qf^C0vX#`<7GY?cEg{A{Byq-pOQuXC$7Axqx=|4naLR*oaPaM4ef6DR{e%T-)Z+Zz-`@Y;=T}C<%2pO*A`W^<6m<6Tu(41Z&&knGYh-o2`v7oGVUAOZHYBY<OlI4H9NT>19Cgj{7ry)6H-CUywUrPIO6&EPLn<*$7hbvKy*d4DllJZQ?QG|?j%IBg&FIhe#@aK#ee2KGe*4xpcYgcU7bWgF4_*r$og%2f&;=H*W&Aa#?!q}yLS4Y<<rTJS;X0S(`4{gBt>dLPL-;=L1Ovf|KU<_5QuYZ8nE=KyKrlcvre(NAKJ&&q|L~?8t<j4CzP$6D|L^{fzt1Y^ci#NQ{lES8@!x+IT3T)|sPV0^8-R|^8SJO422|Vr?OT7{g8%bo$xbh#%#V0?0NCx>yYNnx5nIBp<JuG_-ULQxmD>ObGHwO@I}?A81W&T+8NuQLsw>3K;Ks#X;2yLGwM)3KTCG~n?!;NHu05=TSB7krjFjROZy@ky81KoFQ6I0zR+F|6%;*YFQ>yb%ga!cJRPhD?{9%PMO^-G!8))-^$Dw0o2N-Ks0dSFws?E{T7Hb1mtu2yBk(kQ{pEY3ETEgw#g1P*;L``E=)xcL<B-F+g%^*aL@jUA_Ha6zX!bMcKixw3-Y>liel7YbHWQncok7B(`Xv8-KB>{>zpmhhKn%P-+P8&VCrrWAi*zs0&f~ax;4v=iC)+%mHDkF|t?G&As+p*xR?*XxCwc%Io*rc!PTaC__8J`-PTY{>n7O+c@uXy9yJe)E3hSrC1xr1>!_En2@&(2D+!irN<8cDO8(p`is{f%}_V>N1kx44jPtNU0NDpB45y^Hzxk`=cmVl%t(#@06#Lpwv4>CYDTyq|vV?os1j{q79^IcnlB{0E!A_pH3m{q%EHcilqI)Yq?@7u#<o%0eVeq=`t$NlWKN$%YiQ*fMcD#h2TrX!vY_M*r>@Ha8R$OJo2uER=vYB2mW-bRnCxI7@};XfTsk5p~NPY~D8xBIfvrhDtOlVJ9?d!<k}uWfcZXrOG0ZN@1<E5uB}8JEomo!ZFQhUcW7!$xL9D9K7oDe8V2H8=7Wcn_)DiCXNvPHT>NsaepA#@NPOC<wd}uBJEoh-dJAQj1}_MZF{HE7LW|IO7Vbdu4P%?fKaa0;u_^4(^X<QEhz`+qYa7C4!cp73wu)ub_^1Y_PX&10}hK=Pr?9|D|UzpZd`Dkb%fOwX-`}mu8NV$W`IIN5i|$pQ5DqCQN8LQa5Y;I6wp!IvE0tsMtTQkcq7-@R9fI=8@6gL(5=wDCecoewTlOjPD$OXm1LN0jU<4gD&>5wluAy3;wJ#HKAI(3T>-{=%MJiLajhCs8eOS`nBu|N`ArLgsx?(LGi^oNj=+OxPfxkt<O{!|`T6bh$co}8UOaZEs8~G?k%4+u7nPVJ$LlA8G0p%dzujuBHFh2xT~{PClPJkz@8f8%d|4*Sd3lsg$7AC6c$64FH*R@%@sGY7_sXepN`8vUQ_|_%?t+a|8CB)mKF>?thFWDs%X8h@Q1yA;IlTtw_2Y6KtSs;JA9R(T@95`@`mnr<<jP)(C-77(nH?7FjJ)j!cvX}_4g$^Ddy}Z2i!cpPy1W}6j1EEnwQ{(S2d5>(H#_#O^cm525K(`l1((Bs<sx8m;dR>)!Qk&}rJQt*rH~`3m0UGGl#J$hl28DIC*S`0kKg;tpWXlQ4}bgCU!z|CjDoRrD>%ja^Ha6$wT8n(L7|$6))|i*0^ejq3#yRGN_7=AQlZ9wvYKV4CWqsY5S`kylNCBjayZ|`rRD6JqOgk0?4a|Hlu>`w8#q!ycU9mTBVuttIbPtQA8JL74d7@l?&&cC%m%xYnCZh1nohD*o{BnyRoBP0faqyBAh)M6irPaUgNqs@uAMGqq7h8R7HjPYVoJtgr!6>l(C`%g1o)5hishjYT8d`o9?HY=EC?2R6%?wKyt_}Vpt#101_NK+U{#P<Q*$7Xhtl?R-j<0DPhufa#XV}HAUz#=Wv#5JlOf+rYa(S|_6B<Qr~iKZ(_b9_>Kn&j{OWt(`&VVKa{v3U-T&7w^9|{{zxmd?-}}lt|MKg%fAbTvDHt99`@g*N<~NUj^4D+w=6m-)^HU=HqekHAb&tRGYj}VB?SDkisBiuH?Vo??_>J!#f9bpE^+Mg<|Htng|LvFAEQ>3?`;EW3|Gm$17fsn3_VTN*9{<hP?|=98<NxO?qC7N0R$b7={hz+^_Rs(6__c4o_u3cv($_QK1xo-c01E){SG9G^Fb!<nH~jbZuW+Dmzxn3zm%hi-Tl0=T|JD0H`rg}b{`os^e*X5UcfO5NynX7oZ+!*8gpOA<D+>hw@!DsWo5ojQ8OlIE{@g#l_cwp1Ko6_+-s@j|=d-^!{>fMFf8pzV(S7{yKYaUF-xh1|+qb^>?mv9?_^02o#sNi--}q8_DC8LR`TKwS_4|MI)8nuI{X5_O&ix;MpNqk#cfb6@_x|Q@by!jPZwf=EFMWyIXQMi{Kw;Yxv4$rZ4Im8pbpcy({NulZ$NPWvwfledYiURIc?R?GFaAx~>3r_rob9lc6)ekpuYLCT2mf&V&u{Sfu!~>(+q>WU;qm7_$BXdpPk(s)gFm%!f+ujcY67r=$6xqg@BH~cUcYqn-9P!&@t3}J{Oi9w{=z>4mZRyxU%Y<&`Oh8y<QIq=fBJ97U;o<i8$ZQ4f9?yS%inwxQ1+cKzJC8F|H}SrZ~yXZ$KUv;_g?>ss={7@VrOc$i_U?bI{~Dz%y)ihm`M;09KZhgcfR)%gwx-D77)>CM*&|M2iTrMFQc{){I}!|R*s3)^y*Aor;GE1$JLcjODdiozEW>VZQh4QY^k_4XA_}!-hAWT-@ML9h1j)(-y^$+;wdVpGba!BT&{mUZjfmJaEN!RPz>b*JZIne(U0yXXBzjA%+vPMC5Gy1cjfT!%_~1&91tgc<p@sFphFf~x^5vZss5cDQ#KtG-u;#<x3(%K?^W3h{{UWjJc#Hn@Ja7KUi-2zxO$R*E(==|Kd{~Ein<)mWow3``U%K|obN({fjb}N4NerZSgp5|YHu0!-UF%lPOj!VnX2z3>b})gekW7=oj~<>BK6-1RDdou;9^aNrX7^1SzDQQ0(W%(FaHgW_q&=Dx>ks%_bSvejw7#f`<_tKP;Bvv1|PbvVd*7UMdk2F)D9=nIX(m?>_j?92Sb8|3mNzJlPKW6@!5A@`x9V>n;=sF^8sS!yI=el%!mZ6F~|=nX@Cs;%Ri^M8LtCds<?N2|E*s)upYCrU-{+ncfP~C>)XHj1`BC`?#Dm)Dljwr1CSoL9blo~{>|TTvG>0Jwfo<F>+PR^O|*r5zWEm*!yW(dS6l*$-v7#H-v0S(@4o&s?Dp$lfmYuB`Bzb@_|^YD{>E3!Rd5hAOkk<+fA`;y-+F`Dsl~cDEkl;rA4sO`Ju>%5ts7UDfefo<TP8(abgb5PlHT<L>+DW~r6<JD6JzIs?TKS8Z&3quC%he@j06i6cUnRMcQDjdE>=$n+ZFh!B}A|l#4o+Y#rOT%am0Q?$N&6!jwnRhElD80^=lC89a1-H!;qZl#R^fL$cj+;7$gG;0*cl|Sx~G3%JSwH-u?AAj$i+!RYnj8YPn1Y9+Yc6;T`m2X7w?%x-zp`eTUW;2ScPJyV8}k5u=PMijPKmQ&@&U7%t8T&{D~aw{=57;WT2#zNcVbs(6-9oW)L`z7tLMGTkH*vnI1K;>FQyr&VpuhmI#=?k&~wlPuzkH<rO4+9y$n#w4T9oTiq-Me&Q0&y~C^y)lTRELJ&wquj$6xrdF=Y0&cGB;SwG>qPXCPN?a?A*4H3#cA4W-=gX>Km#Gypo`jt<brlrW;1fy%eP`5FqJDKm=A|JbJ1vBE`4nYgoXbQQ7Hp81x@tnUv$RWXQkZ`%>jW$*c(sVw@*F2@v-Fj+o$wPK`EM@+L}aYxgTLD%)3Q1!q7;Re!{zY^_s_dAw3QNnT%w}`66W`%vH-(O#wf{SCDtZ+~<Cp1=w=%N}7y}0pC8QWu3(G(08rWc<}0-aKs6AwNMt?_dJQiL-bCHhn={G7GjwdBStGs87&cb(?Lx4T{Q4(>DCJrK5=ptzw2t}C%lg*TU8*ylrIVd<$T;LK%!{LhJ^M7gKSz;)4rDSc8Zk5m7mwVhB42JmkG7B=@qGg@TZ!oNI>roQlOYLh4!4OMRAZt{Wy<DSvtwInJKD*J#+qMY&q_k;{Owz54~y))Tp!+>N*KONU|x65?F~+idyk$Ix1(cEl{Ri{4_{*0or!)vu;HJmIc($y43=1T3i{9M2uTE1H#&V%Cv(|N29o#r;`u_08G^j6P<?R8Ci6Mc{JL$vi6q38@HRbtAQv&9~;f5l{LJ*HX2kZw`~#)po+CbWSBx?Rt${uEIf>pz5U$sd>TZfy(uz%GF7*o(%vs1U2|FLTonxnTq%#n0IcXl-<8*hW)`dE>;<>`q7G!^1RUR-SiR`J?#=0kk$S0~0@Wh?BCQiY6a){`_#VU)n%F@TqtU|YGK!;#HrqOR9vO0-8Vo@O?YO61ya5uMth^JqR1)<Y);84BtJA`&qD%#$#T<{ExTleanqAq-1JC~QQLh(f<G7ok8xhAHRuzoV5gP`lgiOh9kI~NVPUTf`;W~1@(X4xo3w2Kilq-9?RlB)y?&+%9R-$;T<}*)Y?r(BiIaezm)v!9z5v<)Xoy<I-5Z=dbyo_hIdEDPKmr9t04Wlv1>A-k|XfhTS?|mD~Yzo0Ww(lk_o3*g5kl3;eZO=q3b2diuYqfiZBTyQW?VmV)Lkba$X;p5YqRoJYYgGsGj+;OsbY9>{>o6sNN-;kRAXzkw)$FN%3cqCVaZj2%InZ{QKs<}iBA*R1lHfezN%^y+#A(Kb$|V4`nn3s*RR_Z+B}lYCsN=xVSYq}izG{KKk>#ygE_4}hv<$z+NtPF(><i$6e4qOckWy|1HbZP}m^SN(Gk3_pv0TmJG<1i(P-Tbiz#sDK_8pX}^wBQq2x>P`3|$nqE1m=*A1D+=4S8Y_>(<?myLZOGspTO^e(1(doWfj+P}G^n>+y~;**(LBQ4O|T{4@$UOjYY9{Gif)n%>C_o~>q0)yluHFzte3b&+XS2@~?v*IUyv7br7Y&jPTZ83)n8uQ+9y$0eCNi*wkG9+sYxI-9C%EP(hUqdYTHp`Df%hpwfcbmSei5>y%|Z6RGI%t#0$Ps39w$-;4hlhnPNIJ2$p*i=v&uFxtuYf?H0T$^#gnIX!OIgNq>7&gzJ2T?_<SQeA*L=&=htzs98YFf+1{4*5Q`7Y47@2K+^#&mTZ!P!MP-wp6HvJ+8HCc>DnuoAj{-)fX6b@k#)m;U?pTZ&Uc9b+^YcqL;DzBc7exxK78eSKNW^!l>a_VUE3xglX-m;Oz|8dSwd8WSI(JEE^ZZ%7nSXn3cWOtjCAUIYBfCON9yz4g0t&6GdFy6{D~=a6kc@E)zuL}9CPDtKkQ$Ba#oALa1<o-|xJ#D2Ue^ZLdC6C6&GJoe>6*Pgt0@Al{u*KhvO<(uKH>o4EDbQ!?-BwNRQOytbyc>vW8P>`Y5nR)qsj4+HlKqUk3r4-e5uK>dUXo|jO8Ihh*3IrfOOec2&Wod^S$P2^%6lH)g6y^2XHYg-srkL$x2LGVCqE0I3I;O<!#+|5pr%sxsLDGRjY4=VHyJtVj@C9}QXCOtkg0l_D;)_?p7hirpynf}%)y<bKOYm&ny!g_sE7xycyL|JO{S9ioaB=J6t;<`bPcLj<x$^QYvre%svmhQFB$ITsEn~vL=LR!M+Q(v0FKphr`25w&;nwA=m#<yky7|e%O~YAj>-ryEerfZ+WB)fU-n@AA>gB7Ox2_eH`mDQovY<D;cr**eQuy%!Wt1X?V5`^>^cuM{sLAjQ1QUqHP`I2LMaIh8jh(VvE>k_bRj~-AX6ssKdl<5!2gg`H)xlY~HQogN5io!q8=;7ReM{n)7j?UFmLVqYC!K`-9fb-PzZ2_>n+{M4qm)I{9&9_nUtb4Lt$<pTr^BR6x;0;r<jyN<VEBd-2onwlH{h?YlLy+gDBzBAK|%OI@Dv_VoiM#)2+R3!EZPi^CWP6v-%pNkwgG(74sCeiP0Y7vIL4qi9gg8R>AZrzIT>!RDC;JP@JPd#AnkypirSrN-hbL^PSO)O33*nb-Av|-{5G%&@fs-wS}=&lI{qL<-SD1m%WvMZRJOd1ntLjY;AZ^lG?~P%s@nLVeI5XoDm5!&&u{D~MoZ*9qPWv6KZ5uxT3Dk7yfa>dX9F+fbhPfc4J5P}&dR9TG8AxFOsegTVl|VoQVFI7T(Jm<d=YTQ6UpI>Co!`DZYVPmy>#PcMD^)#Iv|3Hk;i&5_K#aFI*PO3N6I^e)rR`83&zP2Fee^xerROtbtmaK3yhpBg5DC_DP9qs77<iNvOpgi<y$5^Q?6*@9>VeWyt7Tbijk{`&x@9S>*5Q^t(#PTUDg{;v)t=|v<>W0GDaaf?L$)<I*7z};^@Wu-R-qzYe&t7`G701{dy9#hycUr2>*yhKZGq&dbwYNx5(|{Cczz%bm8pNx*kK{<2jvY(_yC4(gFXRW|+h5Y+Y28&7cvGqg=x-C$yYPl<WZV^;9{`S{EJyVx)&YBhxsZB(VLvf*f$dHM~RhH-e3ghTV)$Kek&Osc6SL&0n_p@~^d16Tj%IsEbGS5suXtL}Oec7<EZc#)tMNIf5*XhQRps`$p_)JkaxWj3j+#NU``~5clbjIV#Dw*`!XeMykr2ixC1R)<Q97+`?ARh_PTpEozP5l)`vx?fDrd#L6QOYHI<<=M!n>xJboEN5I8=3RsTKaehZ6I09*SR76eal9LKj<7Ux@QPi2Bqk8j+bX^BYikhU1w*s0bpe>N&N)@A66?iFrEep_cTO2+A#0>x?9492i`TD?~^EPV0*^g2A5M}tu0eD$Ch(<Ga&_h5PfXg)FuzudUK6z~mHR82~Gq8Sv%<_z2WbjqPYk*U=03-g_w&i$sf?*tu09PEKM;rNUBKKn{DIJiNWg&?z5iYyi(xC|@^&;Z9_YGj9!((kBy~k5eVjQsRrDICNH<bg2XzIGew3?V-3y4eW5O`3ZiD`9#yQD)MP(H#IMbV#Sm{Agc9c4A*LIo8P_gTY;O&#!XD3%bz@%AG)l`x1DR}+IrDJoPGXo=j8q%5WDKWYP4JU*oFbhr@XgBsL(HS6LX#T!bAz-kv%Vdv0$I%adkR`s#1R8pcWsSGigf>1;;%S|ra26rV?OT_&%PZA!T!=dVwmRJJ%7$`dfIfzT?-p65f=PrqRAXWmVdpPBDra6Q)G~zmkz`KyFDg==IXu_AY70Jk`c`;Dp6Y$Q!w*ggV5{+T`=AeCpm*=)n-s(h|(14+gRjYdy<qGxX{Bmsir<oFn;z9vfu_}mxc{XvaNji-8BD{c;i*X%a;H2!5tU26#;aV_-4Uy5mvJwk-d4i^Sbe+amdS<u=z%12xfWXw}FUmi;W5sv~G}UMpY@ycS1|P=kR8F=7{8bnm0UYUSd7pMm(FLg%)9!>#f#Kh1Fy4<ku~rx)(yE07S+qwt8atRx1OC#u*0)}`woc$!?*K;c52MK)(xU)6-ZN4K<fL43LdjRtteI5;&61`PRjewoJ|SSnFwCGFU1Q<ptxvD;BA>121d?NO^nzhRh42n`BWTu1Ws_!FP!el9KyPkrn~swGWP&<Mpe`h4xDbz^Ch@13XuXt^Q`h>#$piy~|L_k`iUj_0CSq(rbX>isSEeAG*vh#K=FTpgH(C7@H<pr+vh@@6s>2pltC=GP9%aOf#;Ex6N0HEFv8LP=Q-QOaUhib_<RI>;tK5p{3v6A8N|ZdGgrUf0iXqr5gGDh4h$_COwHJZgD|-<(jNFBguTDo~W_CEIJK<_qn48;GtAghtJhu^IJJJ@BY}@cXpXey&3waD30{B2@^{mK=E=vhNKfyf%Dgh5d)8`L$fjU&B#+W0G8|T2#=_Vw;l@`#im553yg)QM^d#K&Rv~%qyrhQ;tzO*G?@ntWe=Rsa`T+vu1h2s_O#F}+`W0Xu-U6){+#-&c%%&m^(D@?_d(a0TDa_J<LdxHMj<qz^<Xc!YnoE*}6Sl?+NW{Lg2{r&E}TFuoBPB|OG;N88b&(w$=qqjsfOB}8`0R%TUnT~8`ZiI?zf%)?oy1GZn%2ClW8tY1#8=F^U!A)_kWY-Nwqrk1JIT3%@OQO9|n&nB?hzSno_#`skEXhblN=-p0C?{pdd%8@!sgogv;5~^<XPY2Eqelm2TReZhrgFqLNsrTDjl_=C&lR?g9g}S91EnErG(j!)Nnij;;G9^JKqeVHr>TKDNp}kWC*VYbWRK!WkK#$@jP-@fE>#UAOw=k^SU?0WfH%Gt@29I8>K6PB#bKl=4K;>9s84r{mqNs`IS#FL8Na*Ox7ZLH@NF+@UcTI+A%)X1#{m$vE$~Ws7b{@L=VA>D<_IS-+ovT~Yu>1Pxa_rnkq4zY!ih;bg{=m{Xhz`;0ePmd{!i;|Gumz!_{o=x0BLcBV&C)%g-fzGOcQ8(8%6@7TC!sY{qXG$V7929r*Rppy^<)<jYbEw9l3d+(xhMi*z<(AK3k17pXSr?a6d8Bnrs3(o+-s9Z#C;{knc~2ol%qwe7!yH@McU44gyW^{N|;X;s3^pg8WQp9o58awb&lv18nGZTumQZlOGr__)D!cNe~t?N)9=m)kVuDoCT|40~(Kk7XWx$w{D6k6+Sn}Kw(%pTM~Q&>dt+idn7v~@v~9<+}QEXXp_^5$oFDyF3gZzh{T1vEG}w?D6ulz68Jkp2{BTJwgsz0eHHvr(F_l$C@28PypND1TAo|HK-7sQBeHj7JqGOae1hr?;UJYj57R8p!cXm^p(y+byQAMnxoI4CiM{A0d@hB^Wn@zS@o<C?{qEnjj!1OrW7}baQYX`gws77h4_xJ6$1!GvryP0KiGV)j5odYa$@KFwh`R`;t^&8q=TgLQ4e00=OD`GldNBElIPM$uQuub;g{%>5)Jvh*<tr@!`-TX>?h3nZIdQr1#1VA89BTdclOoo-x#`Gs#kS+%6`W()Qt<x{f+Q_m+~>epDI+p*1l+yTl3N}bq&v03sLr<yT$C%6vScaVNRt&qkx<#OJbq#6lMxBD3(<nK4?Pqb+I$~VyudJI`7~?md$gCr#vG+E)aiIUn4u(>bjZ3REGk;dA$HazUS<juxHTd>sxnl!k(^=3GsFQlMGp~inWj;?F$VOJ3UIgGI=8bU=od8?bLJuPQWh#&EOw3y#RqB%Ev4W=8HY_+@M09^o=(xgkEIkzR_-}-QK3#-Mi4AV5!50H8nFYHcnJQWW;QCq3#t%<bshg~bgGLOfwkI>ZTDH2yvX>ubyvK6&6Y#Qa}fL&2M`yFFN|Bn#>A)Uu@o^TcyAa{KT4g5bl3-GDrj?@b#HseogdPi-NAAJ(Z!s|{0dsX7v4=;8_nK5fw>2cvzq9Tc~lETOUSwpHWWKMh;ysB6d)l&=&J}#N9mA=nsCU+BjWcxg|*?{U^d>*!YR_jq@S>bH|aPk1Z`?}t|+dK+=U`6^*KYYAUUl`FY|?YJ*%-WN7W?hOz9xwSEo^n1VR#eQ=kKp*mEF;CEZz@&WcGQY=E2v(cz2aZ`1hUA{y;@JQ!(f_WK?sLrF&a@r2BNwy-7d#w_1YG1Uz=2s?WQD@T;92NTMj1`wzZya2LEH#?|{cVu6teg-nJ4$9QS9(+l4(R>V|ezb=N4nPf^Xk^15=SO$hJla$IOE$|we8$TI43Gi83*M+2qB1k6Z$egZYi@fg^I~pJ+tMC`8^(KR*=S*x%&v|FocNcpI@htK&?%6|?XVZ<*yEQ_NtTBtpAuJ~v^LZxt8PL8?l3IrZ7`cUS49&xu?{}J0&#NMfhpoGy8MCzkgF%ZFR(>JqH&Qik_WY}uw<BIbfmEVtAxQl?=BDEo_COWcl9psX<$9vk8wq!Jv(bGnLH%q4(;)P;c)M<B^cEI#EX|NUnTt|=fcFFY(9#WERQ*~tb(@zn`NNU8C}yxdqkK@^s*R}(iKQN^|77p^Q|5I21}a*!rw<c)!HDQQ>2Xaaf%ZE@$S2}Jlu7(p=;%2MB!cXqyvaa9eNj^-)u3k4FtPq+&FhRb66RuH!okhe)9$M(iA7dRhAg(B{Fdp<Wy2HW8B=u!ZD_Y%CrsW6pb>T_(>Yv%Fz#P^SaMts_E;11`cSR=0kO|8aYc?1W&~6K$}Z}`&u+Wp$%ONPCNmz5zt%Nwx8rrK}a9Nu!g*SIyfkTNQ=V=E0J}O*4B<8L&~itHIaNpZU~QhThchgIhsqZ0oCSp_y1G(F3XK%XPV%vxN@u-@DK=I;*ybwMD{Qvn3*(1kWnNfB^8tA1_DkJ86W_`0Wv5GW@FY*Q0rcm)vRXIJvNImQ!RR-vU(4_$^6%I^WzSXk*caul@x^g@pC`_`LEwkx}l)C`RuNg$D3G%8oH@fwNc{;Jt3<YsWP&~H5)vAiKd@VQ7epf(4sxQ#kCq-IfdL_-TWy_<NA8RIT`lzlKU5V+`?{kTkUuOg+OWH1mJfw-rl;oDXp#BF*@KFTR!LA(?m^U>hup+62Ax#A@B%kJ9Lh=k;vk~)~8U<>GoMk=UYp@=qyl3!tEHQ;{47e@Uv%%krVv+?~N`e!17#OBO5xUJba4rZF-`C13xvobUALj6`;dBh0}`XJZTM0t!bDSfE<%4P5~v+a}6$F;WEa>pTiWB)J#czN4@LwL9Zj_iE>duZ1O~ssU_!fUB%dR?yFuqRI{gG6iycg!)#o8F&kl&Sa$y68M!{^O{roqyv&O^$7j)VSa(iSEVC5gcbKYSy>*_;7#_lXdMp$NYgG^<x)Irpyz5a#x+(QV{kGNWHZ$@O$g(yO9(O0FTwfKJ?E;geHBDE2A34)*aA7rD`KzwOpXw=!Ph+)1W7Uab?mCy_!;yu>;yOXqIdB|$o?h8=b8oi&vU{&jmo2;aUIBdyHHb`eUp>^N$~YpK@XLxXBFEgJiArVf$qZm`S(5l7au|8Ni0mOHl+|RdO6WGDYG70ysdy^=lpZp7ktu7df`(36hKG#4Bw;!_t0noh`WrLR`VB@MX~&NFkdZ(qoER#3l=T||;k(2AP0OESEF^jv&2tp5WoIA<dO5>#o_M|V?$t<Sua?yXG4mIAFW&PCXuTctp{q+5ft{ClX=QH^+Rt+}99I~N*Ai2wquv}DH}6w*w(`uh$*Du3S}rrwQq$7{R;17e)BM87n#Og&d0*scn*hFKm|OdD=tLMipAM%vi7LkC;8?uM9i$%&T46Spdc>J-3XsVNO)p};pcibVFq?Qdo=i>O*=<b>M*c%|p`*VFN|VF)%girxV#_FaoG3&q%Sn{B8F$<QQl5f~5gecl<l=GNT3~?Gh;&NkyUBQdGMS7_5i+J=^UG7}lFNV56*g3zM$swhh|BDxE5WaFZZN0;zEi*iMOL)&@1zlz0Am1Vmmczmr?RW##|{?P@9&cH*>C{&#6=IS>LpRI$YT7dP(VGr?koR_E4f)z-x21#sP@Jc_UpB6xF)B~=&8VZT1hiJ#ri|uz2ARp&dG2zn%wfMC(Y9)+!ABf$&Za*`2i##J*)ExVgIVrv(#gkDJ~Cnu8&eY)@U8QUo~pSb=FK|A>W4U@0d26c>+_$@tpeCz*+f2vz<!cIhVwl-JW!qn1D}7)5+KPUE$sDz4M`YK19PHs-20DwEI1{7E#&4i)d~LD6K!*`bRVM_Ebkss-t*w{f6|lyd)S9Z{Kw=DnIsd*1Lg&h<U}FY6wMSSlD;c8=*r~QvQ}b%s1_XmT-zb8$$kQCkcfLmy13LA69;sc{g|Ptd*=f>YXImjP?7hlRRKg>U{@WBFjJJ{l(nOC+-6KNX=>&!%yZd|D-qM92>lLo;f~WYx$A|8b*EW2Xf*JTPvu!^w1eMV`fOMkANpiD{DbFUS}IY7u8h|=|vO6IQqS5k9ox|Y1QJzZ7KK&zXB$cJh6H+%)=En(qnD|#Z2_#ocBiA&8&A_(U&R0?#<>yxazhL0R)S)r}1NAGz^6fgELMnb>4aNj{Tm4@D7K5>(}3puHCYaa?-F2iissPPcn2>bRFNwgTeR+LXQSx3UYfvPG4xC;c4sIeN>Tv(CvobUQ@Hw`2V_JI7?OBjr&)#33}pMExd2Vk$4`YuV&_s>@y7?%_+aI8pHYK?RYnDi81TwgSpG86(21kEB@>cKMv;``izg>&5piCrA_~4gUQ0iRo)u_n`!F0>s^mbKknq$lFoC$0I1}^sdo@?RZUN0Wr0-K@@*EkSE$*L;k(;|6(jnlJ)KOOf%-*x9H}^z#L)X`I9}u-kPymdgMr6_R_I|XY*FEtqF#k!?^(NWFqwacZhui#YouE^i|7SKfJ<y)p#4#x_T#jyGtB=0y>j+Y6F-h9S)gDWqgDtKSBxz7SzDlga#9+Qxmwqn1l_2zEz*0DddaLwlO;cz*i<-IPR|Hd4=&T7bLKJPLCkx@anUr43*0cXoM3<%Eg?<U)&PC+Mnd6%{YC*0uGhI(B#cyyOw|E&nB<(qEW$nfwQMDq(7}6Sjw|KF){SbC_V|P6+lSm>Oh>Z*XhL`lvM<G{prEAOO?epT4Y(ew3b=+DvX(ozOZy))1boOO4Qqhe8tihVxfr~71<No(V5eeuIqrEkfb+V|^@e9r!zK7zA|A*4-+wqh+KxJMxE0RwOIU<lzirER=u-`x%oS>bi)coG=wKb3UOAV|!(-Y6EpJPBeVX<P4L;ylHp0!o(4fM%TM$IZ?DFPx`z&?rfH4FO=Yqx<FyVnrsF~M3aagEBhusd>eUqKyg3vUx576Bu6POVxhF!UHAdtdk_WICMU4zC%4T}jwMldfGiLQ(gN0w?CxJ46Z5Pl0DeQaiX=gamM{2%+meU-A);ADYOvfJTW%38TAgi3%sMnM^lU?P`^MSPLeZK!dh@6t7a_-tM7NpUY4IQdd&O?rdYc7m2cFJCgrUoLvH0Z{jWSceG|sh8L}i|BS;pc!Y`*{29qvAGQz=Jl_YbEGT2&ym2}WHdfs=#e2W<Iw3P&YGYUkQJ9b>BP^;6XD83Id{eGtlE>0oE;^gB%a})tH-ypg9KPBsXHP)RZq#q82!@SuxxX_fT)lcyhm+h+Bhyzy(lG0n8Vn@skC)H7-Gm~4Zz;^W~il-szS)2Dfu~;wyP3H#x@P6#R+R}TZPlc21<?_x>d|h@rN9SW73Ca0tjrXX}X=tdq>(VZ$A?u%6(g_Y+QUzL*NOP4z7D+j_=2dys}DsO%*=dDH+KeDFb#lMl$EUk0Sh$3vbbkT!?~OND0oaCqQaIHtClN(oSq+5ZaV(wJ<cgeED!(bkUy;r}JWS#nQqOVBN_6bjb}1nH2KDs@!Y_Ws0|4&1A$xu3wrYXkY{mu0V&)0n1zAv?53c&$%c(VTDkd&(81W;%({l+=dAwECd;Al{624?xMXbB++ZorWM-rUusK^rzCJsuG^2SV6Dx`VXq$6MdfW`X~e1v#?=^+HW3*9*|1-*M#poU_$4<G4dVgZr_9fa1vY#vTgC}&?HKn*3L~uuj=f)|&ZnxPteu;dz(i+g$Lh17Qr%u8OxEmR;qdn8A84)fsA)I$7&glxv&B=Z=#OnwQuTl)ed^2EnlWRhNtLMh{lAMUbRB)K!&uneOG~wuZ75x>B7S5GqBUMF(<&H8*WDCHUd-VRaF_Mx$+<5}Vd=Dl<{Gj+@Y$&e)DO=xKYbWmvli)zh&Xj&)$7p~iq^>E3GK*pienJSbxMX!7zTXjq6^imAB|-YC_8wq=@**OK*c!um!^bNmpI6P)(54Nze}1s$&)9D>bH?>mlzkCH92gsraF1!#b{)XV36TyatYH^WU!<Luvqw4;5&p*<02^I{aTB}luP0lGBfB2@8wgbRBl_X#NtJCL#&8YxPI^MsZ(vU(b5@ZGrHi~Kam5huLCi-aj-FN2<d(#78pZq95jLQb?E{N)Zdd<)>(VhQB-)_xl@P!>eP#tODCdTjC<4KYEqsr9X5j~3$57X`<6m}rsA%FCKp=NQGSz;x)J39%o^t0^*{-NkfatPaZ+p%&2oDP+-cEVCzc35TrZx8iUtG1uqD{kBw5jj($zyM4^88)!pdTSS`~_r@G6e=kTOJ37ilPu>6cXTAVON!w)Q?6OHd7W9ps$wt;Kv~x5^z+htYWTTHsN&yYnlA73!P9X%ysy63a9BXnm$WOBBIp_N9&@ETzw-Bmo{-bffhc#?RnQ|NUEZ$d&j)n&0BUh}3{y*<HPi7=WUz?c>(mXLBF0?yERr>bGL67jyf=LZpZOg9HZWg0CxDyD)8f!M`VtQH!Bue?iC}To|pe#{PLsE+=SeDW*NjgCQ^eIMk2c?(H5O?7hj}9KQbWZtvhEJJ~({&9TyN^Yq8vqy623lVb-~i|}=vEhDFPG`t?p4O<pEP4)?2XeF-K_DjL_@`HVxy8tR;^mFxGh6io9n2m#jRIn=U&bDph?0SgRyYc<|4+)JWDHi=J<PpP8q2UUH@COttKNn<C=*MYi<LOx{UiHMmIYtFM^xg--WRSND&#08nE^)u0q@PABWjmjA<?C*;g;Sy}eQnFHnu_<6Vo1*0jN_S{F9xVAMXTXA-R%ShpV-cs1{@qslh~2AfXf3=L^14`PwOO<@|_p*T>LkJdm=|TtvM##oHVIwms0S;brZyfHFMLV+Tb8d*8|-+_YZimt&py8Z5zC3!EyobY_BS_QajOwGVy*`-Pg%8w@f@><sI!z>s^nV@}4@n>>#8E>WceA<lkv%KOXJAOL`Y`wDp!&Dv(%skxr?yGYV+C9?pFmC>JS(BD;s)$u$Q5oesyB$Wtva!NHT3?u|^5#92P;cvx7m;7Y?GN}X*6Z!bkomU%;1p>nuAFD%!(!mS*53QF;Vs@f>yC6tKDMO>;nRnar?_2g_px0A(akQC^Ff~kV;sq*D3n7+ZHpC?5hbArOO^sn&bV93M>R(+lr+(kL6MMNmo!Q<AIPHQ|8^%X6wfxCw$65vn2=Jky4Pltf`Y1$B41znit2!wB#wZz%KL0@Gvkjsd7oQ1l%+lWXKMuN#lDRgor()}?5QaY2SMndCsl2ck0cCPC6XOp5Z82H%p2sX0pPkTf}cw-8>;x6O}_#>gCz2x?40u7-fy-pQ@u0xkDK-%4)Fdq)|#AK@M^1jItRDLwQp}%Wi&o)C=`F19hLf{mK1OGzC!jW9k3eUg|m4d;k36gWsZ$AtP?d3j|4|3OWHrv?YhogT!nT!lK5WYmZkPv0b)nSS|xN6(5awD;mVA%xdKUUc@OE#KZ4*PIX;nL9_E@jY=S-}k(xsF}xK&NRy*$)PyVWDL=-ZI~GpSpXcNQz1F?e6z`Z=nd<f#8Acc-hL|ba%pC?2A?2l?nq+5lajBhZj}%RGg+cou%UrH7%2Io)54KP#N+ObX%S0u)lN7M@&gGa87}T9`+N4;&4F_F|8WkSy1CyfKjfy4Z9s@Pgl9H`lk{$!4UmdF@X-ShR#I4N_v-<v-}cYqEBjHVw?yAqvw#9=`1HTet>kr1h!I|T^%>^`oo)D!*v1WpSu*+o3NW$6z`+hFeyf{ATN#k=Y?K+4oOEmzO-E|uA3<-cA9H=be7mRa~ZIY7pINDLS6db7P%zWxfOGb>iA1S_iE2y{>bzGWtOpAL4^AxIJyVBGV`|<DnQR{_hA^zc5|6Szt$C53g6^3@xWXbrR&n6JUbrci6PlbtV&du*$N$~IqX$h?38#^n2dF<Riswy_(rq7=%H651-K7EeqLa9-={kdXlm!B&tT!<;I<53i_4?6KUgR*Yvu_AmP&czT=6QgxdYH6i8KR9z>Gf5Mrk@Ln~&BSFfOEQIShZNm=$vbNhAGr^5ygT3ne6Y$8bB0auUepZqeHWYJ%J3RkDDu%tr0%)}_cT<=@^u3-@YiL|$mrIs@5x_UM5~{9Ga_V01uGRn35a^LzWOI&VbviV*t9fPn7}=^>{dy5L{ZObu)r+SOPj_L~3pyX_a7-=S~+?i@idS>O~(v3m`)HQ$)bHpclaXSMd3Xz^zw0xxY#(%7QqOh64SWW^m#tZ4L6b1CF0(>gPa9?Hl(OP$b^ep)NutJ#nH$NQKI@Xg+j`>*$oxk@2BdfT&go`ED?gX7xE5utP*Wos$BxhCbS48D=Cm~kz-;_&R%m9`w-wkv<G@P+O}VPLK=b1%)XNinQ{7#0h`<-K%=%n==96;9IOMDhvaKF1_$!XNIMs9CsP3-pS@<u6b$x^7$S=M#{VKo&#%4(AIeO))3soh1|aaCVDZ8sx13)suMnAk;-!bG3443-$_6Zf`UhUlzjw3yWeDhDJG-IC%}PbCakoVSvrTO`>&$0r8mLm0{+_`4Btgc6uygpTb5ez!!W-6Ppj(QsH;hKi+lrrY`2FNw}C`KnsuWM%dW_4q<~>)q#=K+uo`K(ZZ-sc2my7>f;ETIzm+Pn3jtmoJ=zgU?uJE<ckx;XV%Z#8F{>52;%*dC(cuuIDGNsiQB}ol{m$a+ru)>Bgqj|1dlZaAS>ew!`qLy_A^YdZNGEhl*L>x^8juz2CrgVac2~#>Zjx!nBmwuTbct$HDJl;*JQk)ti1A(yDo4l3DU)B;({_yIT^V<NI!2J>wIXVAMotUaCtxA4>qurJK8aQTLzSczqJJ8zJ8iIZqv)O8Pb=Ms#2&YkTnu$gG7$l1|1LR+9)B<D?H`BWIi0<2{md7r-(sgJ<`uD@Bk)92j3?&$cj{s!|S{{OO#1cuDQ#hbAIOXFkOIo>8CVzWIX<}E&ANyg}iGM?cy6rd}%f8+pcE4*ADqp*jT{HHCnnpX0!2ScxR{Ucw4AXjhj$W((Per;frS`6_Zq}v4zeY;BQZv-J4VPT)fNL0yb32H_ocEfs_*HA^tK@fl5h>Y2YNAT&W24RUm{*kJv^)7Jz<XP*Wi&IAx$Cs~H#;7l<ch`p4PbM50zDS5Q=X?U#LEcR?IRoii#%#~WO)rgOyzKsq9-*eI(*9X_s$8g$93pbN<U&;BBM&OxWV0WhyIn_{~R>tAP#eabm41!);0jKrWzYI6!vC6`LMZ6|ZmrW!HTBA|px;8tcrs!0PE+@8)OMoY)UdvdwE(X19A#Nw$#2QyTQIi?oP?q+EuW?57!>M4OA!Ad$;in0xA+est%eXFcN)e5E{GE->X3agK6)6g$GU9i#&()bdkL(_+|8|w`qo-wdeQ(UXJmE$C0HyT3_kDo3YWFU>4je~|WS4w$Mwk+s+&~9i}an<-Aex&@rHEwrxJH*1YTQJa!W<_vgGRl1gbR8LwzrW4LJMShZZ-~6i-uCYDS=sYjb1d_IiEi6c$AjHw1c<r;nkM+&Y3P#@>(&Z+rS345Wf+Ioqx(C;-0IPGa?kmEZoGiQ@{)gQWC|H;1K(KB_ewIrcw_WqN%`hsl8bJ<ZOh~O@sGlc!mj%Tv$h#W3coGPHB-(-g{c6@8Y?*f1ee@?;w7I_XTqgtaUuff;Ov#9l+&FvOao*!+Va&hnvf*Y(^IjOIo_N}w6;pv4xy)-K8$l4b*kHTQ)!jiT{EnbEH46!d)2K><QZIb;*qo>dwKab)9t8f3M=;b_!r4`>cqJGCrvntRZJJ|L3p&~RtdNR(2e->4x9-4=LLkmumL25fq-f-nWAqoD%BB9<w7D?f*5fxH5a*js;(J*?q5%eG3H|rui`fxD{E?Zg4;DmM3;<|rM^3Yuc?@V>x8-8Y9TR#5?7PSM?Y#F>*Bx`+e0Jm@K56`vgJ@0{0`+3phB@K++sf$8T0zHbxOOtuiz9s{<C!|S(`2^?-tNuFLhtUIvWe=c%o4qJ}DH8DA%r8=N=s9s(+OI^C8Ad<Fo=@kMWFa1e`|elyO%ZMX4U+lN~Wy#yq#xQtJCE*L`c;vekAOUNF>6>uXB7G)B(RN9>tyTv!Qvf|*@keFFc_Sy=rLla9|fD)g=-L?vJ3!4Uc}Ckw90tMn93Z4QK?JDm;DLCq07OKLH!s?9fuO*d&?nA+jB-~!8oG%TvEvkMme85YUeS24H#cjgtrH+Ss}wU6GylV+)v0u#-<<@Rn6uO?EiM-`r)g>PIHYqS7Sy6Bq%9W2L0<+j3iaN>Y%rqfz_N4c}2Z$PvkUiA7EJy|QvOiRue1%p@27GuF3EzLg)ZY&@`QQQjUMbw=_%V?9|`o%Q;4A3*pw|W{VX-yCge&Fpm(z2OFugF<Rho1e+QL_O=d4qYr4UPKQxY?FieNu@y)6in&DSnrH!vP||x-{*+(0mo=`TRD|$2=p<Avjai4d<-IzNYB&;B5yHoig15U2j8ht~bt~Ca?JKb#y<@E?@<a*)Z&O;cp^e#}q7RD+yQ4sUm!pS~%)3p--2VMtLtKAIx;sqzU3oxk99_^8w{GCv78KVlxC89|PqI^PCbkF{VZ-u7v1ea*G9KAuEdw^&5*&in}tURS{x7(R&2eg~DzG)N?a&Vh$_fmDa$?m|st8F%NDL(VP*4C`cA#yo)A|9Qr*bt5C1#4~Lqx4==Xyyt|WfAQk2WW51Oc{!k>!4YmNZJ-1B8&Q{bXxB~6xA%G1ATL%dL$7rrF3YmEin`>C0@_DZc=|>ZlNZ-5NWc#bCaz^!4)q6a%#VF6G&8_y<*7HEkt(&zI+h%wT;;NpC8cnyyS;8@?Ypfugr;E{|xH5auJ2p^2v$MlOKj5hNm))<s#u?Sc+lMN}+9H&$B6!ok9V3-h1rNGcr+nrN?{)^V5RVRXS<Ki~uEqZ9-dK%w+e(by7h&>kF;Af7niy;Sz9PM{+Y%+!=o%cspgBS=LFy^lqwbeMM(6+CaFA^rUSXRK(mpGlZ9YLwU4;U1^*a09lJeU;gOx>VwuQR?a%chV4NOF;P?7ARGm{KWjwYM*o9I>8#2X1N%mo5uN{$E=bEN>e1MXCY!bWd0Gb+66w4Ny0*Lx;P*W(1ts19|ZN()jqL*hE<PEkweOn@co$J*~rVF%{CVw%S>jrOx_!w(4Ox4j~-j5l0wE2-CC$ZAG>=cB{%+0$z7t;#$RUQoCoqr1nC#W@-wSm=X~MePnNVq{6rcv>gG=<W_amEFjsaS)mxblWiX^f!;ML>RH=KgP^>d>ke_rRny$8m>UNs7k(RytqE6X7Hp33@u!P+d)iOKwRXYgd|1tTVvp4r;16F0d8_K5bj53>*-)gwYKL}a~urK;j~OD?RuVxyfAA?qe5P4h3KKx<8Zb|41q2A(U5&{1|rY^u8FjKB5S=r_`cNj@(o?O{MKM8%(}X{Ok_f5`H1tG&L>TOzop^+y~?4Y^Sl?6Ew41emC+WoZ+k<La#SyocGjW)p5g_Qi_ApxlCH?M=BBV9-*B=~DqN86Sk8jszw)ku*gHvN+e%j48xq1!mN+ewR$l*xmtskdiAW3!g~OJjvY%@bD)Ur+Bvs5hRW*6+<Kho#<;6$?qHrjioT!(#nXirwOx#)HL#rRw2n3%6Us+DZ(3~M__2r7G5hb@%g?_kJ2ktNc(S9qnNeAvUfkwre$mh<on2-v7RlRRWQ)bylEPT4LeP#f7RU<;i14+u3Ji)wdeBMw%@#KfJ#F<^OpI~%aM%Nj>t{WVEzGBCfhk{={bPyz{M2E1Fxnx1-N?anXYV(Bt5(3HU97FnA$)J~CPsZI96|(lI8Atu2f@Ro=T2_$i273636wyz7fAiu_9L_)WIbx_Z8Oo^n#PyS-)keWmuwYsibE2%XCfe@CQ-{FOR2!}gO+<`lxbs9zrs4_jgf2U{_Jm4i_DWb6K7>7UJ-P8wB=PGy=8R4Jhyo7cT#3JnWmKon;u|@}Mn*))I;ner_po0{Fu^4GSMc<d3sY`#GhgP~kE({0lWCT?Os%M#oi&&dVIhV^R?S6(X$tK(l-{uNK9O>Z@n|^y*eol-K9fb35eTO|Bp9)HXQasNno88OSN##tiDvphg^y!HFmN$u!O6jld<+q3g@9$k6In&$gxDw+_7IOeyR!=??BKa}j1eAD=;24>r}5F=@yX%Q?#ce)LH7Fa&0coA|NX)4$%mso=xk%>`Bs1H{L7awUq0J@esTWgi;JC?F9$mpUtPTT^2_a)y{}$;*?)1~>pgw3ebLLeUOs>NV(aDBi+r%N^X%zY`9<T4))(Z0k!2T)IXTX&^93bR?lH2N8tnR$>74^z(?X!a7U}93j=ZJ7H<+>=-d>>}B>cNL7oB*^AlbB%#bP+9!M@=)sQeTE0n@R99L~qD-|ikC@4w!Ci|41IN4?QXe;Oa`eK?ZmCfp0|NlkEvf0pBRfa|Y9!Wr*=I62Jre)szAhc|m~+?P#Tr~CQw;fJHwdl_dx)Jn|T-|p>y|HDbu+oL@UPdYr%7snsozdt-W*?WV(-67Z^Cpt`GmaluoN9P$2Ju+UX`~A`3k9!BZ2d{BA-9LGPGfSD_KIV5t)66hV9eVR(@FGGvWyXy{X3p(RCus=kB}?&K1l2-*?Yv0v<v3r=dLw@+Vngagi^j(WWmOp_Y0W@5o-O)1@T}h(qooNd!!3pedXj3<YsCGncuUHcn4B|cq5TOV{_|-zoMd9cRl_Zs)u1aVvk+3f7#zp!65+MboJ}V4mZA48dJ9C*I^}zfbS!CMPr}4+%1y&SISsTi!lw4yN}akE8j&3IJY<dEYc)X@XaBR)1{P_YIStGvx6~g-yA~y-d_yi)O}`5Evtmylni9f?(|!sXoG*z6T;2taredArvtHr`CD}!%+E_)2CdgDpA?fs!M7UokWU^|crTflWeVhw;Tx}aN*KdTGba@k*o1m_6VHUt^HO^2~$4e3k61qUbu=jpfuAT38_us;OlO3Pzp6n4-k}DF=9vPg*ml^xP;S)uu9F8?Z>?p_QHd@4fEj%ZEmJ8Lusn1~?cudPKMEEV<QxuiMnJLJJXbLw^-nET1bWX3~+7WnHlga}4fZKegGwOSmOO?$N?iwdMNy%z%x=SulQm;NAB*B|bX=cs$HSIcH=F}zy^%ib}l;4On1PJPzOfC|3*}+NVxhu#kk5&i%)%7QDAABee)J5HW#L?sUWIRbYW_*-i%=zvtK2lHxN_mm7mDjIkl;6Py@bWsvY7DlP=YJI6Wt4y(Hv&>5LYk&bv=R$d-hsNoh`^5VIvb(0SD*YwwM*)Yo_UljA4zf+E_~X!>CJ{ckXq0;J)Zl<v1v*RskXqMN(LIIs-OoTqK|-<391B?P5r#F<S@!HJWf5Cc#$^FEC_2X^Vi<*PL6gnbLj_kgt#ywq)rH%ZPcD6xsK^5=tw%QYd=3a)jv3xt*M`-=u;iL(k=K!j-hn?F*}OPn}$iEL&M5+r&NmJUHH^I>C^xMH9m|#jzI?894u(QDLP7)#2KgIcr=2fX0%Q&BKw_Fm*v&mis;5IGeYaMC)%9y$K^(yimTejepNSGp1^ldn*|84m82d#4sFu@Y%<L#yaH||ByF&6r5Cu&_l+r#C?rL2aV@6!M}IO}TmwH>+n4<@ou9<1&p9>-q)W7vNTr~)4?h9HQ2NQZz0O0S;+l*AAF+RZ@H{wXI4DBz{hGiBSKzEQ?xF0`z)(#Ewb7YDMWT-o`Lx8gHF4lPujS%rraZ>yBLrPVIqG`-(RsJ~yNo2(@AtBAe~<iv4uSeYNLo4=P79yU8kS<BQcI##n$g>fb)C`TT2x^g^^+YRrP25jn;D6J7@w7jvdP!pnwIB9=LF>Wy`vw)x#?jf<`{uoBGVFPyEY47oy!3CB<9I=wu@hj9OkBE!nWmW2l1P|ql3M-84Tt14=UrnKYY9Y`uAoeTpv!C+dfTQO?Np_pPP)q98m8@V_!pipV`<!>xTm6AKR8eHDcK`s8z4$uoKQ66ywJ|&<J2hI?40x9VGZS`E1mi!tbX$Bub+PI*O?1bWQ9RFii0K+0HWk{DIiGBMv&^zO<6<)WF6X@JCxn``Cg54x2gh^%^A0LKHE{g>Qj@(@ijEAc4pfo;vr;nLVIA5Lc*q%#!eQ2YXJ>w6FZQcE<|x4%V_MpXt41P=gBOp`Y&fr1i9MPN5^{M*|H3`><Wx&>^C2*iX9Ks0u^WK!eF4_`_(@ob}+y`FG(qE=PQBRONGq)F+<`9p2fhCpd55s}mSHePWL1cDoH9hC?0X#mD@X?_YTUrxzx;z6JVflgkpg5scTuN;yNn%qJN2affbxDDb|*&>Tv>Q$_j(%my|dx-7-ZV8ExcxVMxakf?o(!pDln>3g&!;v?2h#q0MUR4bVM_V5Vkq)Xe^z3-+V>}aY26f!7V0iC_bkm7|)gEB|G?jwBvRqX)ZAAFy^K798ct}oP0W^ea?+<R+>IM$(YK;IT3!hTgX%-*0kCAAzba)CG-3nCM(gq7#fk8Icyo)0FP=yfz7dW<xUGAMiI+;;XGr~!BCXat9ij$E<g^4t5}-s=yf5P5g_X3t)Drb?XdQk7gIj+A#(CU{hoX_$JeTV1z#ag7KhUDa;=r@WV{&Z<A4Qy$9Sk>nMO_3SJyd7UEb&$pf}hg;K#1orU*E+dS_-+xn}dWHa<;CHtF=GaxtRV+`~7q7!+O#+eKI1rRZ()I5UyVm43=K<ziTzaaBNZ)h@;F&_JO$r4&i^(_de}D4B;lcaelOJ$p`PUDJ@AmAk55SpzyL-Inodc+m8~`b&i^F{5bHN|ruMZCK`{VtSy>4S0DwR4<zI(~PfA}u@;lsC?WS8Y<Cr7&n$KM?uy@L<^H*fa8`;O}KA44T?c2A&>C*>Dd`<wmaT?BSJ*?YV9Ztvvi_tKaB_u0W=wtw(u?|1fz7RJAS#IB=>?T5YBzmd)z{$}rB{~xdqs0w}icJJ-}@w;y00!?}F9*F$zarW)*Ybf@n+aL~2H<hd-Op>>b=ab1C806lx$r<f)$(*FhEcl^w&u~Ngs5e6#6IO*K@85hkTwEiuo#4GPL-2Pe?piFc2segFo2VEwkP2U<bduEgjZNsoC2&R*C_}Cb!vZs5I+}i`XI9}KTASmwS)+3;T|N46aI*hy50>M|5LMLC6O=f1L=SFIzY{?2XV)-!xQF9Gh6&KU0QUF$ZzZ#1E9u|t$Zd18)9&BQUr$ERBYMdwxXov+<QWGaam+HmTfoe|OzHdf>%+H)N7>Q$-|l>WwEKH|6PecGUcZ|TE@0huSTJYb0W%|g7$fmXh0Rt@6YA~}m`ybIICX72Y2RL<GS*F_K(<u;3SDUCy-Q;>E$)g8e^(Y}#2OR%KP&R=_A1A~F1RsJ&F9!!uwJIhT5Z$qhBu(oDUqQmQA+!Q*n6nvzeL+=PB|Od>C>~bNNj@E61@aUs%5wf-pTQ?s$xA!)Gq9}(d0I{cPD)L3WH>N{VPIr(Y(`IB6-8cB=-_A&K#wKlWZo8HUK-c5q_{voRxJoFqU-8H`_c7^D8zsu8FZ3l6q`ogL9Fyt!2oxgkOv&8>DtMzXm8+&=2mGCIaSoB>rJ%9zESsL_$Wxo%VTr)2+O%(%SUwPD`l|L7}_dBdhlfOrkKk1=a*kb34OC+r-XhXt=-=3UBzN`tL9^VE+b@>fqq|uehBCN4uM5+ptacKfHsR<GVw&NpqG<$xSJrFFg{^S$^H+mk3EEf~0Ln8=4h)f#YI1)%8(;ha$pNo47f6s<9+Vom=sQN^GlaSYcDN=fM-*vOJUq{Atg>>*LfC?E@a|AK?xJGRQii<5T<~W^I^;25Ig=)FeH9K+^4yMojHhhQz0h)_ZnXH;(G$rrsdK01qReCR5kVol7`w?((&_Epg|lqmsG$yv-3&<~02a|22mv5Yd5w<mc#n$DcI}d1eCwIR~Hxke?!bJHd*{0K#bd38LM4SO5CgCH{pz_VyZ!aX2_;+B{gK%^h=Ze$qT*O@xSR%dkbxb&8+~8ci<YJ{gXh@CRXwAXaww<<^$bI{Sux@{geQ;e(Vje>P_44bJvS2uLBv&9F!y*?dIGvy)c8hUSGh;iGwuczel)_9{)|UZq@`oL5tYALQL40oFygAzqF;mzB;2YV2$>NyXF?ZW@5pH()Yg@pW%JyZ|!6^g#Ga$s^$JwI2y;B~hog1aIF2L&Y>8;|eZ1f#;<>S+c2`c)|!>)<WZ?-Y!Gu3Z*Kh8>u?CasLqh28#iqVFNEk1~)wVrDdaz605KKd;K0Rb4jue%p+nZpdyZ2g;CE6bB{}$v}3e2U8p7@N^|i(B^R3e{Ir48t#Q^#*j`UG5-UG#;5a}WT*_=nR|!L49P=X8wX36|m2N$T99DUnNc(O~$~4^BsIeT57rwKJ>7wXXWb~`$@?D&E`HGif2%iD-wRF@t(ycXC+blLOC%RB#oCEr2O7NW2m8P*9jB!ky0jzUrJ$|~YMltm(nUX&$(2qj!D1OzIo}M<;;DlI;(kW~z!}ALDxJ8Sr-p=zEo)))uQRp}qz!k%kU85@7TNVYUkVdJ!d)dyNU~~J|o^a0@;}MDTLygq?1_;xQ1;%dRA;pu$-zziT3R24m#;;+J1gqP-{XBk0JvUfYhuuIe4bUTwHgc-kWM0h@r=|%#Ge}yS7*RHmF$&^EHlCm!bBfY%l!S#08A%By>kAMCTfIi__(4LV^rOwo1QgYFtW(Wro93^K2WyH@|6pg$HqBpk@2Wq?JAK$2>3|WO&lEzx0uEuCZH5)N?|7jc5cjq}9tX6mRFCrtIp+m2^uVSQi?yoan_eDPh>zB&sF6D4TJzHz(5YtA8z6ZS)O7_!i>5mil>;6ynlz+X<sFKgJa-lT;LKIOEamccFL2;<iq5^Td-$H4U&0@~*?igH+3OJkF6k4Ti`Qk^)AY0<fVS6Y_p=#Q%;k$%9+;!48|8&m%rJ&V#R^2c@=-s}w!eJ7wbh<}MDh5EHx5<%q9U~$Qa^D`>zMXYR8z9mNj|(kJ~`UkeP?~TPgT-8Lury}g|P^B?8g~VyzCE)9=dul9m#IC8>)zxWYQIq+kgnD5~Bha9=i{3YKfFVUdxBmxy>E#0<kn?w^?{gE@QS0@s3hObTjpPgsko{Sz6v*ao=}_K4OEk+-jsxFW-RNpVIEN(ux$QH*C`M>3u_u0~a)#EvEB)K)fVp^Bc~Eoxr_BBkx~LCI#C3E}+bnUU616t24)n1l_G{h`BR2lB<Zqo#JT|JimT%BouFPmJ(o7K&8K>AHE`jTMeqXcaShe{}H*~JbkX+yF8onnd}8yZmAutxA@yqQ9r6P#-5styA9#uWdj?>0#7D1`DS=JXj7?=hoT}_0g}sL+K9aA2m#hnTn5GJhnCse?D?ogaWAtg5D*18YA_*WFwxE0sotsRc|)C`+}zmpWm=Z8&8~IjJ@v?B5d4;P0TndY@!AfaYOkP-+}6`mtXlE<9fiXC{kN(`*r$MNuVwVbg*vZtj`VMeN!mlkg#yDVgP)?oyZxLDf6vr{y(7*lwmcd#v&-2KRhuuV@Y9`oJHu<t5KiGJaB^W1esIz(+SU^g`Jb^7&oq&o?zEonoVgwwhW~PdaCngL$N8|ofd6Knb{Y|OPq)Gvv+GWwsq|T+Tqjr3=6s4x`E@%S%&(FsXu;qIR}5ATPhi{8O0|#jq6hLn%7EYQzy1LKO_3yA=RhgY_N&dRy__9?;l(|$nLRQlKU<3T7=qry?^6!`o1*HOa7HJ2K_P%BmjI>9)Ug{jHrzlWFNDK|3w1KYSOwRMQo_(~cEp5(YM$}bjd_6{Qcmi|74&4My+ynx3=rL?Z-MC{t?`#++wLJ#WuEP9V~CPW|1y><S86|ns>x&Gxf=ZOS=b2rChd^hkXy=c)GtHl;we^KU8nOhtO{Pec)HRq?w^8#x9{v3*!ueF>+W;5xD5sgS6sLjj#2L%eGNQY*|gm&db1f^UYv;|G>X05{noGHw9`u9|F_Phs*7GRKN!8sf%$zyOAb4u>6T-&bBmLyHJvpeDCUZpYE{isT0S~_5Rj1ZK)eA$PuwBFaCTr=X>dr!qw5SxW~0(E;ckQn<hq9ls~9hx-6ABboP<|+{4o{I&9GQdoEtJ0h8493rL?Xbs_fnlEX4kyZ*<pug-BF$$1}6U4^$0l&p(0Khw!sR17Ib0;-qRvLNA{AMAIQ<6!`;n6tm0TSRQzRi)9CPu5#Na8NcdU(}`M*@!nSTiA;l`iT()Glwm2P|GxHSxoF&D7aYGw;m)00F!xS(mW>#>bNk*qFPet=#{O>VaHM}5i8q{0zx!$OO|sSAda;&&pz+Pa1ledun2Zw*BO+{0Qmsr7;(csY)||9_V1)!pk=AU$>0n+FFCys{LvWTaoWtKqb-fz8!gbVH+v}*3bhPSo2N()CN}x02dC|Ot58V~dTFE)w@f7cewO}-8fobKaB+#f<vh(~ow{Y^q(cbY7*|+-#yGOsz_78qce(gR#JURYQCPA3Hga&|4;JE8v+s<&yl=pPQcmlHK3;5y-PA}DTdf&sbI^PG}B;4B=I>*tpu*g2GX?O?9M!0YQ6@jx4QGmo?h)jiEjy!YE3)CKvi0C;rMj#BKSX@I>luW9ZXT7=D$Dk%A{J5rYEPAlA<OE4i(H?W|KS|ryaF?)<Vy7YF8H3t%>fDz!jqr~Cbzfy57ZjP0ZY=<ufM_0{z%ly@15!qJ(<@-@LBzr7KskP_b48Kb{aM!Q_ZJ}L(_3VQpqGef>4Z5F*<N6b9|zcWymgDk%{qm>O7OjgxC(d4k$sl#b4Vn!h5p&qd|{N65Q<FN-=bi%e`w+*IZYM<XmKe6poMOt4c>j|Vao2MU2dA$_G%Yb-vT3MH28*8J=!#gQ}9wi2HGrKada?{zE6YUB`SN&eisWxH@Rp`&VPp8SZv<IH>aJQvnBntM2Ij^d+?UVs>?0#i02cQ@X|<QwSn4?Fd(?*>nb%rbgol|`CTi-**a37nX2OyvJxx&fj`mzy+dX}nFf498UUIwQ{2v<zoA87&gf<|eWGVA6q0pImQ8QbDWeZl))eV-nA?<%5Ga3>$3E)3Wd2Tt>YXuN``B0!GNwk+O*MtUT}OJLMxZ1hjRj4Nbdg)5Cx+W36CldgY6j%PK#vLrLyd*9V2qwO%lJ8cRI%=*&QRer1hb2NOBU<$SB}i;FY97lXUM6+Cg8Z(c*t$pd`#Z85`xDT8V3-^lwR9MWIfkMs`8igaXe&(klY*bfS(g19Cv7Ropy2BY1nE(DD;=7A`An1CAN~H4Ml@uCKDf~4%p6`E_9qr2VGN`#ij7XPI9kRE*m}><2CGt?OMzT)SFgKbEN!g$=01?m|yi8rO6fRa9m+nlWAZaM^UJNvEYkN4pCJMWve!h(Di>>8z@8lC`P~tWgrKS)ta^Cdu0j`2A?$;12N>{$tby!)Q`P;YMDPt{^bS7cJ1)InAtN_qR_ih?+&33{1H*~5AM7w%Y_sv8qu#F<O)VacZ`w^CHKRr8C(=hP|Ue1MG{DBZkQ$dU}ZcUCD!<y>ZXvaYaI6dGH{z{y2IT}{PF}3B8IyP<a+r$t@1hx!+`AqE5d&xEjd5+=BynPs*kRa6!ld8_H0>uZD>`hk!N*-R2xn*Z&6kAcf+j++$kkk;;=jXlNC5X-uM~)iO)zy%g<O>EPK=W-u$FG>=JaHzgwDDL31v6q8{N#n0t|Y%4oEwX4c$IE!P>GIt7Ke@q7n_9AK?<^ea{`{&B_@fzm?87`;Zk2A3X0%oQ|e^^Ko}4ndSzr`b5xk6f>dPNQYReOA<aDqTj+R?1GUe^}(|Rf_TY#jB*qF~+KXWUdXGXmAO~C@fGN3g;aBRq=TBRGJ%FlRD#b-A3*hl^%z?2t<Du3xpl7y+^Lr6=zObxPS5l|FW1bS#LxZvJ8QN<QnRjSz@#xr!*bAQZaKF8~$CG&4$HCk-anGpR+-(s2c3V?6(AT5xaNV=nV!#EP*6Rs)T6)Dl@k+_uqu=J0vT5LP|>zIS^*DQg4-M#~(S~MOXwkyF;<fr_pp3fr}T7(tzd|?eLkkk<k&t9A38wxk5pU6@hPVYFfXO3W1Zj$Ut;gOBTXcsHlPE7|us`a?Us^El0cdB`GhhiCe!JPm;mpcFejSe_&=3P9M4#&N^{2BiKZ`{AMU})I}FVHEwz}zDkU%tbiE}3wCnDP-U2tn;f_U+99tnbXgjrG+Q5Y44YOlP=vpBawPNMZ6xhN51!8uSZOhCH+&i)s~~cZdC;Vz>@q<fTS_{b7@vgL80^}Tp<ft6DYo%{o;67~VYrJ?BdqW-PW6;4uMjvw<2lOuD<_qhqt(R2Y(rRy8)MwAQqjpR=LQ@5Nj{h*g11J+1b>H{AGioia5th_BDp`R^f6B3rv^xh5kTnzuqsk2A({gZ!pvy*{4%7<pAh<@6HPl^G*~3MHG!f~6u43gVafVciJX!9v84*vzFaSu>KrSaUkwYjG1!FXhp&$kRWjm{oy@Ot&9_}NmeJw+;pf(FMbj<75YyZsEh_8l7p=gWVo}H)aw{IE94m<VvO>1Q@y}<f9--d2OR7Jkd_m1LIu(TzhU<Gj<K{ngSDBaudTje{Hpv#}ZSX@Btbzs@1Jc(@o;C?txRsCvJ=GzzebqC$FvDz-q+G*#^?-)x>K8~~jI#lI8T3)nKFabN@@bjU?F!<_5EyP&$D;?x<ro+OiC>n-#b`3`HV~h-5yLFB2a+g2WF4+pdgifP*hKz~q7jm=%=o(M4{SG1)E;!;?OdANAv_=SFs-VV-mbS&zF(2S{fvSummn)cFli&bBHL%AAmSdJ71NMXrXlW|n(A$;(YVgl)dsTM%B9P~K&<1;IB@g=yP^v#QM%(R(&)JR6E#Exb-u=@mL=Lp7Zi5bi||EUzk@$g_a7&R77s(zyX3}ZwIr)6`n26j?%=<lcHrL~r#fK*&nKVOsRo#mYw{huCt(?FZM;PEiyTW)`D`B3FXXJZ<rTY&hXq;&x7X<ORb}Blb_z}h7o}48%qhZ(@Q#g1V5?qOf(!?dVm64uet^@|iOwCG?iA(CDPo_Uk-Gst1)}DZ7!KqWMG|Dt1@Xt@*Zy;A#;7oKfEL=`++!W_=|Zg*>}0Z^cM^-a%wdwK+9gL4YftSc${t7Fk;ewH9|nldu93ET>YEDAma8`s-C1Z<=@qn}nl#WmA7yRDNsY|0h6I&R{MJ0c<!%)phd{1TIu(u4!h%1W2H>hQ>V!4=PAQ3usBbm}g6FXdv)viqfSi(ln*o^y9z39ul5X7&m9+<QF9w7qpcmZJP^(o>wm_nF+X?t>=~#n%MklaG=<*F<V+T1OphhwLl<Bt5(6&D94QIDQblS(@^bFQdWX@M`%MOveAe&4~c*9e#==^4gNZyb&7G_-O*=+LZ&Z`ow_r?=qkYCPv171AOp(hft&5)XoQcTK38dss*-yYa@)?3Uch`Zi*+ZLx?l@%ImoPLcQx|r||V2PBD1229_qg|Er4i}Ba%;bZHsm@*k#ZO8N+oE{#)2fp#TRu%6d!pi1!@d!u-?Wv4MP!dbnRf)`gNIyt@sM%b&aca#Om0~Z#|MIRX0^M2xn8|iWLDmkh356XcDJsBOt9X^Rn5a7;emG}mi_sb>-Ugn9ClSTJoy89tSSkW1=gReaztz;@5oXv3XIkG$u$DBIf-eFBR|uOO4<4WjjXZnZJ^bCLt{q5qh;g5El)%KSmt@{M^qNQc40^L5e|Xe%{{yf4NKWL%6p^Ndn~7E@Kl0vzX&IGiP|aV_$4n`9AN|fU*M?agJl2ByUi2$@4Lx6V3<!ZBoFl})7fw`TZO}se7pPo-rLQ0Z{NNXNDgJ(1WBVjW`5yMk|?AirQs?%&#BB{(qG`%vN<QlV0Vk6De9S08WTr%Lw#DVMc}ksMoG3d#}oOo5)1wQBe1*d_RxYqFciD|wT5L=N*<!QQ{c?j+OaB>vL>vmewsH`mbH@8v-AO!guf%4w8q6OkD?XV{U;h*X#5{7hwsQjwL%cCv6Vjbuy>RisWXWmw}|Hho>V6sashOl)Mt(U%}yhA;dIua=xgh1B3cWh@*)x1(SW(?Op3p9yJ>Q`Kxa8k%evik+jv+n$@kD@IN3@;&ND9F#p>!L*K)NwDW*kXd8Jr9;N-5>Ox5;pJ(s_SRsQI$;`fkj5^)`GrAWoVEKNaXX*vEzt>icPT`6_NI{%phAF%!xBx2L#mXYJurF>Jg-c1%ohDV%ov-{VDVv{4lD;#{pMD$O-dw<l?hq9a2S`(t(`_DuMy*c|uDc|Rxw)Ib4l=!&wy;MuM`!M~??jYMQX`H=aSU!Up#uZs#2?NQO#b$o_xUT#nB;&J4hx?y%xOhH3{L{Y)^?5iI`Wy;0AJQdV?N|BB9}e*z>Q*~N+pM4ut5`za9vjOhace6g<sP`jOkxKU*cJw|akrm5frh`aPN{xUle5QMs8wNbYMFLDPQ2z`CB<|!oRfoBLZ0!)0fj4NZ=MG(4?EKbP6}&HA#1=M`kAqdvm~scpsMkAV?fJ6#ShWQxPAhUSSJnpbo(2?2WW(7ECPM)>0e*_91K!SrH^B8O7_+&=nE@LpDK#ynU2_fCywH0W*)GMp*7q$x!#+waSR0$yzfrwqowFzsxU5Gn`hTI&<}>a%kcy=Z<5=7S6vFlU&MLcNFTKZp*`8l9i&O6-9pMc5p_lV>Z7kATW8y?vFW0BFq=vfXOk{%rH>e^0s+aPmqy61Cl+ty7}p5TFKtn*Ca^Q7otLm?+b`0Vgj*ZeAX!>=?ILZ<W7dQ=qn!}DQ37C-$kaS$TB<|6AIx@fm@j!tSQr`RASm39+$VUEAGv7^_z~9<jbD7Ij6~N*%R^q|HIN{!7@F;aHu?jQZy$Pi+YHV`3<XW`XE3|r5>9EkhS#4_rD_we4UYnU3V-s8vLq{6l*S$f;ROr|yTeNA%k=Y_<=t{h|D5Sf>xbi-Slq|8^vH|W^^3-Sc@dP4ukLE?aNJ*FHe*iR7Y*n7+q}P8y1v~HZ2J#=U9x)gjK?u@zG-bmZbu&aFTs~<4lD|c%Ms$}d;+Hso^TX6K^;MJiJKns0b`4&k$iOjv4%S7P4+~&)w9R3KKQLc32{(59;=h7oPHb^&Nck)0E3`0%4URGyWbMOZ(P&dJr<xglt_?n<wudB>)v!KfW^&EJIO}!X*;#>Bt+l|5Ci_dd57=s-1kiNTjB|K_DCgj0I{ZU7VLMe{#d}33-p2Dxy~mRQbI<tB;X3V-#fCgSk-4td**M$@H$Rfj@IxGefa%{K1E?M7uI3)lxC>=fGI((NGo~42o;mr02XVGuou*)%=L5`Pp-+y1;l}yUXii0w<NX)$pY=rXa#i8CRX~Ouzv}8O`%f@sRuFzfoZsCbjbgNz(vyuNHwUe>kWYO5)7k3cAd{I5$Z!>jD}-$T$^XT%O0kk$gG#7E7po{2=&NJG&8iOQ8L9nP;!pLAc*}T<}^+lD@`9QpO?~dJxDu`HAvn7=<h5Rm;E4aur90o+U#mY*7@~itoH1hM)F{5%O3rMiV9#n;aTUw%K{S+3(F(){U+4-n2GVj2`RP-Rqo`dbEe7Vq3Va%?D;yAqw`}g{r7gz5;XM|HEB)lTgq}C{Q2N1{TzqvQb&JLvhNx=#G{8|d2$-E>rJK8@Gv6%pOW^3u1k(Lnc7~K4<~0<6Q}y0N33uRQT0)py!A}oC2u-2{xyvIkM@`|Hjz@~*P1itM$ef={MgZ2Kc2BejApGgiN@t}PM67_;5_(_=4-lZKHkkwSaZ?klp$;wn6)C|{5qAJ#azQ{aBbri7JtA#mVOurAMOx7jdGH1(bLlVv*jZicA^Ea^~N|~%m|>(a&eES)rm01yaLS3!ZnGhBawrHa@mk^TE&dszv>|@Ii~VbmUK;yoqX#*?|?Xc*F2L{QP&B=6yIL;1MZq!Am|UC4-!rO{7~UGMi-<n6*ZKBrGE+XT1QIp6jMatm=i+8GLhn8Ctta;tn61-nK&2MEKF$|=`^twK%@GL83h+#EanvJE864}^1OOKnT+<Rfyu0*3ge=sU+y8l5sAqpZUjb8?_;-&F>VD@#M4F-<J#3C`&!fRR(vgL8ta3rX<wr`yvJvyOt6tAukzl_-QTFmZ>Ys`n*@C+>2E7Jx(BZ>29RbZc)lV;5;e``PciM5X5Q{vaGQ!;-=v+M!sj(d2`^VsSzyU_u{cMr*olf&Lm=;~)Jw5qc2W)PTkWNQHS(-so$ll`rF-je(_BNFDQg{bC8E@~z-=0FYDN7#_H_ghe=O4d^?T0^t!hLBh4_Zme9hvyPJywl1a>+{D7O-lQ>tc^aV0JqrzShj8Ra*-M<Z>r@lC>h3=M>wUKCd$8iFsI5ISBtz}ktdc6(M8VZ7oPY|}Q!Gf=bD@wF^V*Ob-Gu17q5Of0e+D7PYc`^u1GLOY`*Y2!}P^a7jX@BkqpGn7$M<r2{(x6<yYcYQwSB~bC2RB@GXhwF-^$x8k6<h*ENul;Cj+aJ6zmMyb6WjLm_sD-~}riAM6>aV)9z1jez{l@5k?6!9Ua^`7s{LuNu@o@4HODCfl>;I<Rp!I!pH|(KUK{X#pG1sG(5p74s>OF$};eAxc1i9B3@evgRs>g>2Dpp<qnaR=c+;sKRaDF`R^*=^R!xB#apan@)902`LgyxJ>v)wmyC*V|rP18;?r11<}oJ=tvFRn=o?$eMo3V=ZX4#?0K0w)urQ&>ACc^;1AsY4xdh~?T)XgICag53@C(ZCIR!SBNPTw{^YVZceI4M&KS#sUjM%ptIp6ir@n!I<S~N@~<zH9!W9otpFJ4ZdjSh^mmu>NlrO!qqgj%53Ux1-nG>5FqA5G1>{uC(@or!e<?4Q=HE1hM1kY2nJ`hQ934d=R?tAI4c><nvSK4k-#k0aXq<dI|?~7DcTps-MDYFi&EzEi5HS$kgz$iOUzpT5IAhNSJ?~>MU^?-RjBF*g}`Mq{7T8TuAq^bi*yx__A({X0IHKbvT#$E>WV=ltL>RX;?!FwvQFahN5#%KLTgG%gM!d3lQrtT!C>a7b-I~fZ;(o7!&Sa8E>pn<!fg)>!k}j=mtaOC2opM?WP`MeVbB|a4=@6JClu&iv;*_n1z1qEIK3o*B6dY3?6?l1Y>t|$I_4x^GR3GJFU`9LT#@ym-b+l@ij@oRh1Z{jvv9KVk!91xHRv#CAw^eM?26i0v;IUd)-Wc3Aq!DCg}H?;+5E{)x7eQ#2fIJ+?!Vo|&=t4RcY7y49KJbThU@&81K`D_a5d?&21rnf1Xz(Ao`LpW%8yut2%++obG8=O*PMFd-flDr7B8X;ZZ{U@?;#asVYR6Uw@g@z9SLk$&m1Bk$*tsR>Y_Xf4rG|RfG|z0h8uUg^OKA+1aa1J)<c5zac?%5gLk*;-)1+c!Dx_U*74AL+_|f?D*=TW_0CBT6+pZFI>D5S2wdf2gN1wvHi-d(ejnTzMyeh&4bj@d$S|E1C9KXYMt}mfSv-SA8kz#Q?o+AfEQn>1Eq2-nx<abr;F*W@7`%()H@dh>W*U9q%%Gkad?sgC1-WS_t&q!Ye~O1;XqJcXxLZ>g`b}<ZyjtcQ&dDgZzQQ3NQH?*^n`5@Bc@5=|4uw_5g>>m4I<7G!d=?pobU-VPaG1PS-|?re!+G)#`|okNq@5b2$olqsY3=o70qij?6Kr10KwsTFR8-S6+rN+4;opR#cr;|Gm`X*$c7s<Tx+iLLdcdH=8(9=a_heuYAUgeg&`yr?JSnXsZMeGa<1#yfC034m#~Yv==<c!*%b=Q5_p%Z!od>foaavr(Hk7;I@i<`s=S2`{RRM>%2EO&hPhV(Da2v<k=G*wEFQSKvJF?Cxbr-*}UR{QSgmV$=X5=#iEG|?=13bL0#M(1km_Egm5$lwNHZ>nK)`3b!AlnC#sb84tX#!M!qp9asQ5lO06l86Hqh`$1uK!Ls0gqH+u91}<MNwSlI1?wU(Lp>nT@9_V(Mp^p?Id#bg@`~n633fGC(~vb?wDP~9xwy}&`B8PcP;L%l(?c(lhL9gn?=W}jpGGc$xiYl+1}cDh5?FC)5ox)V>D_O^L&ak`kOqTCUeTx%#)aCN&=gLxt)AB%khpWJ_;%{NuPKH!5xAhYd|6gBDU}uB3Z&|VDbe#<%f5BbPuujG<;K_3P+H=fUjxz8AK_?GKr~#k6M;x?SYPDVt{QEktS=BllZPPWxgS^t^C3#R!*mcER)s3Vgb={sT_Dv&d|c}MH!a;^$-8{KX#KJ4*&9h93&@y{qz5M0vqq@uYdlh>*R9y*MIr%!vueflfhzqnT-DJfByAf{&{@aPF~ZU|F8e}mw!$^P2kPH{v*BW(?|a!nZq~3hGwpaavswB_rt&b`Jac$^y)AF&-ik4e)X6C>)LNCxtjd-&;M^9IyLOc!#Qr6N%HOC;ct#N-|v3Ecl=7)bvM_){^MW&<-hS*?vgn)^S`F;Dzj9KR+iloDz=J~*l9iawBzRibp)qs;98GFe=hW_O`<H_o!SrqE+6QZX6wMwy2@A{>1jkU6N^pKnK-S3Ok@n!nc|I-!WN}RNp6?tcxyQ}1(xlD$yFV7WWKb^M$2@Twkg&czwwC<yINJNrWuQav8|2rmAsP_{E@)Z4VJ1tF+#9^sbX~|vPJfcO>~7LUzy+O!&txERE&Qw^N3NX@tk|?64c-B4F*jg+(KZAyLRy~>68bU57S?Q{^rMPKz}RdyXVjaEWLvO>(ANQ%1PLl472Kyl{j^$<(6GQ6l^&Ll){;~Zc8Y)Zu#KV%5HjC%NXWwvds3cdgDtU(A0Fc)=^ADUMd5FTu;1!&3^>&vX|I6Q14WyI4(o{|Dwr`!x^(JmC_&ET#c0d*v9GzB7_^BBadRXl=`KW_T6}<n<1us&t?A5QiMK)q32zEPHo^4C*gBFD}8Wd+MD=3#9=O#Tx(2+8b)OYgi%?A7(2BM-pUf7S}~$(S5WM;7c<36R2eeGuY-uGwT7caLZDD2u&5C+Jfa#LuR7HsuF6Om-oD=5bKxbL3{SOC|9eu~DhBX~B%Vq%Wb|@g8|t%q9&}X&Z`Ndjqzm^~eB<gBP@l)~Sa<=eODfQChtMSx5EUGxW<@J4n2ZW5!Mggh+BV+lHO6SCl6-}CByUJf3T9gBjqjRG=UOW0F2V&a>d{iu)XKQp$*7*RE=Ovj98w}G_~Zh`R+tmgUyo3~rdo$E|E!oOP*aDiL13A}(G9G?TAN?lPd7^zX(Q{I9Lk1+4KushqRP<<*DFGo>ej9P%t_upnLFhfM1X&s8WkHI`|bJ#XRqbW#kg2Zr({|4_!?8IojtKtuHc?+Z^BG&GXHVQQ$!f3o6(l-PizGw>Fb2t(hRrmrwa^Ao9*Emy!xEB+lO9#fXL$MG)tS^-$mR#pb4lCMRSwwVk>!qQ%i5e5rUGV=*0Z=1^(XwsRI`MeqIz8i;=OeGF2~<^E){#o5);mB2&GIw@ZsP3|Kp*ts?h>HrJ`yw`sC*E*3B~lR?h8dF`SM@}fT*PRR|oc4=@A`y=szZTxrYSc8*9DKU%D;<AZ<xrBmh>AGEO+bqA0gYRB9S|ruecWLRnS&kly&Bp21-*kTZV)G~Y&!7J56a44X*4767yM1wn7e@yFgCC~ii_=Kjjr6qh;@Mey<~+fp$wG|JbmnbXnh~XUZZ?|#b(l6^A0GVa{n5dnc0c^?PX`~~{^|AMyVLEBug?DT_WcR{kxGj!Ry7;g-d4$?B!l7HH9U8Lm~J0Zu1fNInp#O~T1vX%xYyYAHKMhMhnKU-Vj9Ye2C>=tOj4U}r};zI^k`Supyk3i+kH4u7~k1UDKJ+<m$Zy5s0JrQx<LR#V{I_PVuW3`-<32+40@6@Ha2jcJ+lMnE7xA7h}J`XTd@Cah2#}40nv>}c*Ut!KpME{gkWwW>WADO0-<CxHI;>Wy(Hlx8usUHywNcMfcOiDy0oNopy{?bBZXyh=2#A^Bvf-NbKaO6giE>KSX#h89S|s*p=PKn_*`W~=tt+|zZRoAcBE4+C;lobC(6u}rFMS4hS<*cRfzD;%T}`Wd)<>V!s;G>c>n(J=w$EBDn|99@nQUNJh>f<kjv{JwRMtv{=>_#J!N6o(a6|$>PZ<B>Kz5oaBN27ByTC1LWTuK@S}o(0Kf<Uizq5};cGZR@OLvPNv&~FD$5U`^w#P;-uvzx%<?}hhG@~>-Z&?Kb|QG3VPY~K=a-DuBv66_A4voCPQ$TTk0#F)@{>q?*w!*yT$gj1)vS>8iyFikgXO_CmcDgPDO1Drfg`D)c|J`SjSc?po@WL&U}x(oRO2M71%i`wg%XwZaKZiHrC$^bD+Hg`&9ixrUoSVN_t=Zh)^li@`?(~@sNiPg8XR)JuA1(m@y1ck^#;Vn@V3U0z;tQl%czDf(zIJF1ow2X+5&w6<F$Fi5@<{wHjuL0eAeXbBQyoLQE{tCu__pa$dEPe@cqH}FhKs^8=fNEX{4`06YFSAax3z!W5l6`ag*~$W)eK5A#o~B`ClY=@`T78|Lqex6<<o3;>ri|VB&GID(nd-JD<pzXhdZT;W!LzpyT9AYFd*%jAjI{xcdfYfM=;tK1@xPEPR0%XH#m{fr{S*`5p?b$I0Zs>UsZ`?_|;OI4s}8z!#6?!sMB7MT7w+s~rn)%sTSIQGNkb?uycRq9PkA)ERz7xoC|9vB|zFG)KT@Bk@4{qnbH!);)Q_T2g5KttEQ3pcC}k|Cy>ZU`DFBYMn<ZMqNxsHzs`5dAD{7S_QTyr--iC;wJFe>O0{TCfdgx4qZ1<`f$6LdPw}fu~f0B<FYmrAE$Q)>LF!^ZSs=O#~=v-b?7!0^NW`zxsX^&zk_u=n9RRJWA|QQA9Y$+E-!K^OQ9B>id8xrR)`J#E0(&f#}_U+;w!aY<{|k=Iys@r9ia~&-3x3WL9V^w@_`N-BUSHC4xHY?q(@c|#Wp?m6H+6JAwDHizr(Bx`~>#B#Hr!Spu(!#V)d<{<MfU*N%U2>ptrF+8WNyH;^YXkIW#t3UMZ!Lv{s#8f;|P^$10|x94d-m{I@fhrhic7{o1EqVvTX{dpCs5H>tJ&&*ChEE@q4_-*qr2I_0?km_B&vEK%@=B@dchx%NMO!CWl)F2M36*TaJJ3>`BXixBaJs|LXY;`5Nf@wS40UwVV#uiG@8WyL+mgE)2DG4qFIlZJev40JFJEk2aA@BCg>o05>;?NR?MXRQYW7i%!IyR0`Nh+Ezs1xy`xbLo<Zn$`V`xs-`~M6UGmDv+>JIpruA)bFmcD2jeLlNsOQ(wI%9+-Q^1i63hc<#uJ3`1$qJr9fq0p}5!aq-ayvQ?Qv^wr_FgU~vrU#8N{q=EG6DKbhVU+r4(p=6PG3bj!Or4$py0WxlOAGkybpte1O3we+ayO%;I)b_6Q|77zl5RvMRpD_feb-Uq8Ucm!S8?ocD)tlpwLx^Qc7HSq|z_`dNERum#`<xa7wD+1@W7$+8WT)S^LlO)cPE=x#J7A%7xxH~g8PoBt%ElHM)E}3NF<8q$R5iQ?h2hVgF*kzzmDNpI@e6LQoqrTNb0|3hMh;iOuM;1BeBzh!@u(LYIwWzfPQBPP1hWG=~M2;MK>GKD@i+Mg%c`;_V_~USjbNlZioY>e1V(R2)+U}i$P_mdCsRT`^{4UDwL6;l#94%j0(i#EnUiILwUV`724S3qv1Me4??rRkx;n(L6qWu0@B@2`+Q~Mu=Q@pa9rft}fpJ3ynMHgf{997qp-G859wAJ1l*l`$h10TP5#v?#iPzHgGxZKrT;%*!#4BP3}K0NW+4m#@$z~60dfAw;!h3<CS>Rb^`7(m$-ae-!tvx<2Rpp~ajw_m<YzWzGdei1?G0n$kN)gbP+3fm*N$efsIip2KtrTL`-ucE8x1eJV$^yiU@Xs0^B`ka#Eq45O9)ns*xDwaqxZL-fbxn2XcJ8#q`aNrRu3>Q^i1A3qw3DM-L!f@iGRwqz-HK+Zz;J8cdgOZ6VAKG>JhAN&RM&LC@h{?OQj>*l$i{8+tc@Jg@{>jj<E1>--uIINN)xYk28eT81yH8sZ0Emum-R4#sk3@U>IsWf6`k&{9V~=KnXz@R9X1&|TS^uvl1)0JAf8fVj>9;C_FrZuXGflqMX=YEF(4NAE2V-!cR!ia$*G1l^j_J_pf@!|`!*Uj5CMm8l5+ypb5xBoN6#I;h;4@W(ZzW%Moexl9^0oVW=L`#215j9gJ1GB@%cu3~q4&I#-JTjN&;>dZ!b$;8@@czux83@*)4JQi={enMZJ)V<A)&;wDN`pe!jt#&(I~rZ@xPeNp!swQu8OD6x14>zRwA6p>*2TwCk7`Ct9CCGNodjKDW$h?&FZHc3HtPHn%_^PMHkB<CtjX&Z;bO>PG{0en!z{FTW$+U#Q&Dle>^1;oKmd^Pi!0ir%Ba^kP9ufv|2P3>UbFCoL^uR%mj0oG>6IGCFLFsV}lEW?XWYX5^Z9QFgq`~)?X*j)7E-1)GbsLj=GI=pwu6$d@xrdPJWIvvH89|zgk?Mk71KGn>-J=%rDGl+DM*9Sq@qX;Qb1iZh@#TO~}*ImleQ4CfF?+2Oja<#6`GXj6l`~LYc*v<0-C3<ENj-jW(vzZZ^5j)1Ny(<J*_4bNiWW^ytprgd>ATW9QkH91bdvwwR|c$s|*;YJ{@VO*Gt5FEL#Ciw0)N1)8(Wru5e~{6&B4oP{P!0%Gx)=_>UMFufEh4dsb*W_G_>(u43C6~M$d;SKR&z)f@;APF`6>I~a}l{&A4`YKs2D1jD~@b4E(<T@4b0Cn`>8PcjX>CLm9tqOvn+amMZw6uetN8E?kB_Jz^2n}Oc?rwbdN;e-$NtpZF=#3~To6K*cmYzI}ru-6gh3)?6I&jM2kIwD+^BHAMQn&uopJ5Jn#4pS_iqF!s%6jnz=Sz%#v|Jx_+B4OkZo8jun@{E1tN(UKzx7<+fp3z2a9k9ZOl#_S>er*b@mkTgQ_<7Pil!(PKs{UsvTFEo6|ip)`B-W7>C0!Zk?4CJ!_${r=8@dW^{sMI;APMs5<woD7!gKm?Nkga%`GGCj1)52J+)LSEPDGWFM8Kggk)?V?Cm1@$;x00(pi<he!F|{`X3IDD~2OykS{Un(^{9<3jD&=$NTA)p6_)S0=Zr+xhF*l{e*sBBW2L$rsV*Mb?*viK2}~BD=v*SE)J0(rIhcrT+TsA;S`eLQmL{6@j<2PqewB=2pdY}OW=&`MS{;9p!ZU1PHsl6r<CT>D%>2)vAk^&(nPA#N31||IHq9SNlzWiuq2M8JuE;{N0oyVmiIa~5InZ(<&MhnutM(~)p^`RGMyAdQscv|cW!;oG-K$x^T=OaaUiY~3g?y(_{tNkgEVoXgjMa;F`Z2=QA3x}TjyJlaDvbgXR2?iSK(&8`M;4jH$3NbKM}l16UWlx&Zb<5#{RXAW<9R~of`gFriQHgGGLhwFN?57M{`3zjpYo83eWr|{1E-Y7Af=7D6Y+kV<>}i5?0Aw+A=)~9D)!-fpk0F7tb90gB{(UHkmDIt@&5Ulg5lzis|n<mz$V}4qRV?Gmpga@m@Op-grP!7^HBJA#8FqwHsF5lpGfA(d5!*w2;6r1siB^Zc~OmvC0b_BU-N1vX#vzIz^<>aLa_Llx!bK<%`snoyg2bv&+%syxDkyfR-A$G98jbjPd9;uGjj6(!;}Ex5@|cQ$)>!X|hvttqmOw0|!L2jupdQX&-bPK~0sVTD)FyqI=^g>FFDjZ|ug6`&{dlMecccXLSjML`{%3PlG)x305bW=p-R0#?Ynfn?5}!6K(64u;hmZY1P!1tjj!U#uV?*nyW>^jpSbIW@fCT8>1r71S$uCC{Daq8xD=xxY=gm1*izu+|p&}8mC7lG7R$9?@bW~d61EXPN|l_I>}TZn%?X>hh{WH3lXgHjCA0P!zNt~oO_%VHIRvNdf5=biUtSrjmNqI=?xh))Iwk}--*tr>VZ{MNpvMvn-jxkJjn2QNIFfTVe+cvZNu#JM0_idlQtfR%t_<R2&?8Wpz=UXbTPm-vN>93)3uS^)*YF(j8;yP8}G%s`CM+^v{TaHC}Nc<V?40KmL2*xWxm0qZ80>kGBfbT5)jj3k{7VY;rh|#C}ZhTgK;QbF-X<n=yD{l8Q;3jVDZkD>{N|1mWvbE$H;8dYl`lH<uJ%OCb+XYI@JAsl`+|fIj{vk!$^F)(nwrw6V?-hlmicU6p+XvCdQVw7Bxv%H0I*JN*t9m8JWhlYVk<oh!YkKnh`5k4K{ewqV*asiKYb|BZSt*+^RHULJp{9@&YZ_il~R!IL(z9Ns26~->q)9%qBNV+I!7{S?1i_8L`ZjaBrG$((-QA$n7Pb3wg#>3|EXOt(=qy%f+O)S7azaerl%w(Pn7Z-fZNYd^Q_TkoZ&Po64k;F4(5;KXgJ+ntxcpmc~f-#W)#Lp7m>t0Vl*Y7kYDY{&U`+7x2Y(k5HEu1;z{+*CDIl2+{W(#JWyZu;qL*C$LS2H&<kL_i{8l$Dd!lprR)N`8>T-nCA=jvj(xk?TGJ@BOK&6BmmQo;dqiNPK5ks4nkEvc0-m8U8Z)iPDf(1ytt|$Lf6|QG)KSgnvPh$Ril-$MBP#_124RY6>rhE%w#x`XFlTJy8H@pT<dc0l=M&f;03ah#T)JNP4hn00z-Div>&e47`@y59bIL+-|uDL{{Cd|*l3EVHN?NOsv<(Fwz?ubz1R_Na>CAdFEM(u9-KtFa<q56cl2Y}rJffs8T3XZ*cGQtJw$SnC|Nv07)7^C8(({yhlCk?*X8YjZHY`mDw?hcov7yCYH{@&NjNXTlG?ar^EljS-mg~DTg)fGXbt<_QblRq)+R=+$s4>yr=8`hjDT5u%J&+}Cg&3%Fk^(Al$fl`+2j@;8K_7Cog9;>i$D<gtOuO~{?O8E&vWuZB;*KmX=z7P(QB?DNL;MCBxd4Sk7PB{PW+KW<%~X{L?C%VwhR`*E!Ao#3zk)@T(On3v$$hWhHy8P*t;#prJ0`71_9fn6-ZkD#=K!|;8_eN_Tr~6N~osz7L!|bc)Q;F4J=_h4F*H@MCsnk&gC}KrS9wdpzWouCxIi;*uQpX=hvpQK1ym;r)7C{+uJT<)!lpiU+gIDEOwEGE}7<*=DgUB2JJ4L4af~UJ&EoomQ6^By56o7M*)vu@I307tcTupxewo{K(AU=4SVN?o3-G{(Tc<$of$@Y)+SDg17q;7_TIta@%z`UM1H;b`wxdFwGCZ^Y{Uj*Li45i)%Dn$64KH$K+D*Ch0KgTYes;1DMf-%8?vKK5If{%*rynH!oNlQ6Z7K_3%KU363)W<hC7UvqmJR}wg*c&olW|r>xR<P$rwd1QO?7Gm|O<w&zLn(y{r6mjUDpHGhTGbISXaJjPenQN~&zvg71cxW8_)X!lKn;$c+3&BER#-yH?yc{~^W_Gk0!Jg86W)C}tF2>-cJUh2OG%UOyrVWer`*w6w(F)uK)BHS>*`$A&dR%@_(c#BA2B%-h1oZ1Qi_NcEvbW|u8#XSv_Lkv5!{(8#c#hp!kq&H9_(USW7CqnvT8F^MV2b6;W|-IsXlk(?1#|Lmh8b+(v7hhm7BRL`h{f+;7cJ)Mk3&9o*_FlSe;9ETy%a9%&Iktjhn1JpE6?zIz4c@V^D`3l8|<Q@+367?r=QPHV~-x*rHoA8G#p^V8TH@*q@`s+=Z-^7v|{>lZGo|;LB2N>C~QAhn%GLWr^&Gu~%3tUVz>Pq%o&VwnNvXKVYg$e$lKxif=@l^J~cifKPt2oH><q|}5q>Og&n{F9f#lx|+1E|!EXkEYRn$uXL^l*-)d{|vJJjbx`(V(xU+_Ei)Sf#ZgV4g}DcauvMU{aoLRpuW77V5)z_{07Yh6wPU?6kKomc=Vhy4Jm?iBqOquw!Vq5}YQS3n16Gn1jzN%0_>E)xf|LC`y>F^U242?3Z?)b5YD+QJZ+yr+eO1Mz~{{96fa3cw{^zUWU7Dka3-^h2-A-V(F^wh>^85>)k*b)2y~$t8_=26*8~C39hv2F5R0h2>uVn_`z96Wzu4v8=?_{*#w;H=TT5}+B6J^+=~Uq4K<nNfGu*o|NV#KqwT0ZwmQ?U65b*o4>GPxs%%SzRmppi(iWpUpEkGJJ28)|@Nm87rru5_9~aZgLsogv9zS#C8!kfF-O2bk9F<Z<>pS8)qlth{c%zi=H|A^+d;zyaaId|EZE3a;V;Z<720XDb1B03sq!Fkr!Ar5T&Q0m9oG!1}syyqB%=gNRO8^agcU4Yz<hraX+GE!pAz?ML2RX9`8cfMh649}TET+#`{Pe{=_eK&r;4DiC?E;Wq`D^LTVMmtb3ttXmi<qmL=1)4Vu`<h$Y0a``k&iAYaWIW$Fhq}>yZIHl;II=G)RJR9I&qwGBo>Y{>Ec{8qN0nN<q4|x;%Hg(I=W~Fg~+t;JqgMOnSI@#j%T@XGsECy?Rf*HVGM<V%j{nX=rmb#6(u$DzUgLjq7E3d4YTNW>d;T9fhnrdmpY6mjaCv-*QP;1JS<(WVPhyF_n@FSXNuJ?pn$|u-c~vY*oie<+?IU<(>fsMUfu@0`S~G(hBCS=J!pE{i2W(OYle#jq>L-(mD0VYEp+4)KT{0VOviRH9$pP+nTvNz*G=OB4h(jK^2u<mrI^~WQKxeww0!uuQ7-u)ub8qCjzg=d8>hB!XDLQo&8qBp>l+-6rw<KE{rKB)JEr4YP{KHXPFF)5{3?cu!cEl~{;H5T#D=aOL5v3&f!kOGf?<4)DzGmeh=U<V0X^*~6%}nZOwP&Fu)(W{5vRIW)pfOFNo0dbBbMs3KV!I~)qhVSTARTEcYXUD6PHVhrg^`LFnFGV{wEc#M$V|yIzr*QT?VwTDS-ac3B1}}T4%!Tr+4Kl-G2S{@aQ*tN7?b=hojefb-gJwiqxJkKR7#!bLE8I6CFGyyCGe080oXI5U6G8gq>-AOfzO5p5asIO4qG-|NS04Mtz(^g6SDt>TcQNlQ)MSP8c0oX^1~Kd~nmfoGx?{;pcER8Dmb^rsukhWrid=A;xY?l#Jz%a47xTL01Slk|nz)EzT+lwO)*Rmqiz9W;WgQ87ArV(cbRKUUqPp{dWK0&Eanw5tsm~laYwJj=s}fj+T40PG$F>IqZbqSgrCsU(TNOEzlv8-!8TZ`lCsaH_JxhD9=%?L8+HVFqT1#f|ZWW-gF1Ym13WaNY;vWt}0lD6KffEus85hf>O^gDJcE1LRwxwD`_+{sUV$&eONOksmqp<<GS#q!!Zb~85<Kp1{s*0QsN+$aq%<u8aB<J^9v1RzjwdwY<=kg%X;jY=XGWjv`v><i|`Q=Z2KXwg>;?JsD$Vf4Yk8eJ@V6FhZ!lV?mKP)oD_Ubb7T6Tc@0%_9CEKlghoOp?l{gx0R4<Pfki&0Rc(5y-i)`sKjluG1>OnHb&$q5c8^S$xDdyen3%q>Am7txRcvA1rJ<gW6=W-7tEnMXj={!E2!-v6_dIP8@+0SA80BYR+Id*4iyn=z;<!zuyM7RiXG*eE-YER~A2TYDkI^G(VBng<lP-X_N(aet^g%@ZKb&;iK$S;R>HlB?eOv0qBS<vgt2hOLS;(mPh<Z2V_>>fUx{e+(dDzO8X;JjDUkfNLuE2{o?lnw=LJ#ZqRve(s;r&$K_Gvntc_6LP+G9Gsco)*B0>I9QoA>7?pN)EVO>vbkSWYhs?HZftJMQd4{p2Q)qX=`8OkpcYaZ1>l++O8Ca|q%LF|1LOVj!63lLcDEJ(nHq&TKfyomfEa3v=jRG@#Yq>38K|A-YiYb-Dtbyo4%jzbt1A)!W)uZt>9^k2+FJkgI_vnrlUGBk$<2NuK{f-+GRyv$%|-oWm$=O*~8XDydh9aPki6kpHoZ5oEm!-TgsI5f<y>*D1PI`>I%znOmK<8LL9Zv8HK}5&}a#F!Q}>Lp?aLvCc|st-+dSrx<pyC5<WvaJ(Xi+xcWNDmJA^1_Qnv<=N!og7R%l?~;v;!Q^&~L)s{=DOeJB`RL9MLXc=A?AWQ(;WHQXW;7)}G=|47xmIIH{QO<$P)6|;{E>Jd#g#Vvb?TO18=1GE3TQK)C}bEnu?!6KwR<sW8p<vxgCvnB5J8J8-JhOrA+&86m%B;P7Fm}Wk{?grW#7HsJ^mp(+WY$t`$v1(>)qEs>_tFuL_;EIFa(_>9U$y@!^gyNU8T{}$_6Qq{B~oVTSZ6=&qZ7JqfWyrS{<q#qbRP%vLq8%ZBz2c*Pc$M)sxy%$Iu6$7pyyXz7M3M@AAoH8=q!~$HrS9k0@@mCPoBiUXTjUT%#rD%1sQRyD+GmpL6Zl4rznZK%Iv40ZwWD3HrewH2I(MCGvSIY%=$*O{6ys{oRzDsS(;QH>ZidiQSo1_oYqhaj)MmUnRf<!U-aeU7@Btjv{&E2c4B~4%3U3R|N4In(e@RCK0UskAv06bA^Z<^WJ5lH*RrPWcWK;5O;tn2^AI~AW)lnk4fwV*2mElT3yrYqYEv^V90c1$X#~Y;6ffzrbpp@VFz4)Yv5v1*#~n!^i{e7_?-(q?pFCNjVpGOP_5xb!UhVM-v1Jvf9?uv5l)|;b=L6V;&hyFHZb1~&+_iK``SKL<#mGB#*Ieif>sM7g&9E6%b;=J!1YDDBK(8m5OK7qoVhF7lK7IOr2Jy|NgneLc#70yLCFo}jeW2*@O^bgNJjxo%vV#hCZqlZ@xeOVFN|Y5^^A;1s?RzfjyI*lBKV<mid%vlu12HktRPY#4-?o!MQi8%Smo*Dwo#q@!;~uL-KeO*JZAO)=Bq(l9+z3XSb27Q%NhBiIa<N#9m+}V{>5d(kwrZ1r4<ih1)Hb-Qlow1y1}NS8~k@;0}W>&L~cxnQ+@zn%=7DwVmQw?^hOYq26qH)7Ifr{z2T-?Z7T2%AaDOtM!}%VzCDHdYkD^<DA6_fNz0_li_7t3mT#De=BMMy1_q*jGQS4-6i4%fd-9NxSc{D9@0-0J4?eto8_6ZYT2P~#!aa(_paK^i_d^FaLkLN3=fal`9TA+3hKg^)_o8@R!2L6wXDVQNz7mG5Hd)%djOgpjnkB02au}`%XnR&G0DAns?NcYBg;$}FOSw7RIODKAH?Jp`KMwQTXRdV096AAtXgmGMB?2Kn!@!DBKhL(me7?2So_?e)rgdzxqakf86B2A*-t_X#093_$hZDO%N{tE$@(H2p+C}doN4WCItnj`8QE6NtCPhB0tk*5GM*Z!Iu9Hh7k<Yre+(PSDjgFQeYtHOnYZb~WyJ5;A%RKCj#FsO>?oG?33se*Vqdc>Jc{QJfPSEbLsSs^+|L9%rw1(Q(OlpyCjVC~}+fxr6!4EV4yhU7pSwh}zDjBz<LFzagwXPy&76jPGnFg}etCEY=Gy>(ZNtvKh<1o>{<ia?L&4%RL?ak&+pgFR_&V@g9Zu>L^ewR@BaOE3kCbeyvrUBq7`2#hG+)pRNv6{i`?W$v;jJuv$4Cg0PuMJA0+$(awh%9%0eVz}H{U}_6T(G0u+G88eL?bt4VH)*Ys_e}{9>SK4DmAYoqFSbcLcW35C&x!^n7lb{INYw2{tP;B<({9<oN2AL`jhRm=y*N#KxuG%hgFR&N`ZTM_+&J?4(ZX&a2`IYq+MmocQfht&do14_`qqk=bz^B`7CY1m;u)~f`#+PUT77LKjl-5rRfAmfW&^&8~$!MKAwy(ciZ#%1@_%O`c@yE{IGFST-}W~-fxv3ZATuxC_mewN9H6uq5Z`9&<-;wABNe|(WJEuMMd%*n(tG0^KvS(#+UuP|8lVP)%KT9x1Mi3{jz`FyXc>9!~b2p+}?Vz(;E!7Uu-`;fBs_o;`!jqd}pwoKmY2hi+l?TKF_!6Q<<FXz8M}LCA~p!IwyPD#RUj-tU;JuTnzDwLzt}`34+SIPEdP5LR^8wPFf&M(uOj=XIc)KzkD`y=#5E;8kwztuq}gRq%@nY#6dxD8P_RgtUVu&;Fa>Pm@J@`aLQx$s!-Dr1{eO(XJW-W+ypCdl2k<f5QmHhHeu(JV|Q@f#ig#KdTV-(r|}NCA+-PAePIm(dV^C6;tpjXMx<W&lbbqiP;yn1vXm15_`Q~i<af<-ce##or;Xi;ztoiI{n27j%C8<7Py@sgGh-u)0FrOe@(fRh^ATpK<N7!z_i9*3YiORK-Zf%0AaWaui0}4wF`W@xaW$C@{|M9&SWMJQqDFW)&KoB9bQ?oR-4FX$JxrHC7r<gXC)YMCv-7MIcVLK&^++=>nDIz=67EA{d&LPo2p4L*-QIbYnD&cT7_0+Fb3Q~{>TEJ+9WLS-O>QHqF?SS{NC-xZu*4U`oY^g)2*b&&%8*ero)1%TB0s@e-UD$Sk`pV2`4`C_uVDAwz&{)P@v8(_0FCkAKz=y60P-<X%oQN?Q0cI^^20b8y<4}WxZ%S?_bR0wjb2uQjlCh*+~j8}IM*8jx?P?eUT3NT{&>0m*oAX)a;jqmgg3n@fYMlDupS`oH@UUV<Qe12f0Jyrx1O)x@Bu77Vz;8YB<Z1?G~W={6jZQ3-`@GOz4MYnLU6;_Lqu8Do6_H8ocCtQv(9!JW||gq)GC8{WCk<KKq0&8L>U6GuX>Y@5p;2c*xVi+F_>y5feoYPxf*csOnJxoR})cZVSD60q?aaA1l|_pq2joweK^?tad-di?ze9V!v)b?PWBHEvUhtYKODX}jt1aZvO@fQW0P&Yo9v30`bj8D>eOn)t9WgeXRGBQ)WnK`k_b?41IuAT3KT4Yik+mkO05s}J{;}7?L^G4nvUn<(m<LQYb4EFMR}bQMq&Q5HK{KdK1_#{Ov-gtc!FS5Y$Xgh>e(4_*BrUkNh0S+P&PVrq6|T%IZVjhKy+Gm84p&^RAU*7Y%ZaymW;TeMA(hQGS~67C>gR#cRr^45SbLvirI=*XOb>=fgOJMOR1;{4f$s~7Kqa-aqhm(l9te_2a%l!RhJ>78BCYj8$6#Rs@An>Qr@YUYtqGKS~Is5yVi~m#08>SWM+1R{@N%eQi9T-n{q>_z<AfY9)<1&Rvs}~6`kNIM_Osw4!9*8;gK@b*w#HMH!VS)jnM&hLXkGEKaOqDDs?1|#d<8S8SZcE%8c{Nn9>kn;wa8|f`4FQJ?+k^WC}ea;|jc0(<QE$(>c;R1?|~@6BEeTTqj!5Zc_^_FC;`>lJ9*9eBVYj!Y)29<$#I1a8Oa`qlqGctJY$UuIr&8Uq^izYiOWe?2*VMXR-89`@*BKO6n}is5?9)uL?}`+ED8&{0J5f9nw|yENv1uAGS%^7b&HnN+w>)?YAcp|G0PX=1^mVj_KUoyB3H<q;%Il(lPv9g{cGzFxo+5>82roBS_)B=`sR6$`}J^&&L^afG-donxIg5aM^8dHYQzTC`OeK7w)OuCCt!Y|NPJ6tK|GI|1{BH!V}amt9RwctH1nHKRMcaeR%W+R)_ndhimzY(0YMO{OiB`-~D9%*FXQSWb|+Ua{+sJf-W0T!*PJ0>ZgHR&kMPFtGfO;3aL!*n&g8#h3@8E+Xb9;d!G~f)2DuE4TQ;5&HhEh%-%f|H}?E8U$<^VXLFs2s)q2!$yfopTNUCqys$j90Xreu#R8wENIomGn!Z2ed+lt3NMs1if1Y0;(&+m)-zii8w65{^PF6E*Kd^N2uskWE+I!FQBHkmNJN7_*3p*(IBGO+nM~-b{F{MBv^dC}wKs|?Ixsqdkvl7rD&Sf7CPWIpJ38`qa-KFVGwB#l4u7tN1tgN_4r+X_|SI|vwV#oo5Ya<_wARbbfYjOQt5tr+@c$QL+2U))>x`v&RwyeEdwPO$Wx2qj#;CS~Kr@OTr?mTBZvqML{vF8q`LQ&Jr(yHR`6JP>d{hz1C^xj}xy!TrI;-hLr1M#&O%{74mnmDg}oWQ{8-v)M=(-*Ls)JjWT%p)jT^b8Zp;R7r_hB6#8CTR&GFQ&s!z_;Uz^E;h`?mfdCvKmHjsqQHSy9$tC0V|4v3Cz>5poC4yVv1)#->~u<vuYlF|1Gq-C@=%nCd@rLfRM}n74Rvql<)urSkTnay=ysrtq%y{_!>w*W@C7jyng?I^&IdLI|ouTBb6aG-v(+pH@}A>`f*|5))<>Fz&5_5`x@7rQ{0B|k;2xxZcF9-yZ9mUcv#p+C1*E3Tr5_pGHYXm-Ey!ht*@gI3sU21aD;h`oV2|UJV$0p(Oa;})7ARAH96(ZN{l9MPvF;rZHI`VUBumL>YI>JhXd}6fwv&=+%}0C{EbF|5<~3RtF(iaItzy~Ng1XR5p;E;rznba&7kb2Z=AJZJz;zH;6+o2AEjiG1LGVab7p*P13nq#IvaE>Nv+kz4&nZoL9M9S?Yc|kizGHx7<a)fspG3EiwtQDj7<mY5eZ(t?WkKgrPGmNRDr^i&}K(lSzTOb;(@F=78l>j;_5}PTbesW9+OV{x>tOZh!&m&K8BjLntz$J)iE1Q7>ibIuPIMJpl*8PB=mc9@!>Xs6M?@Pk#v+3I8YNx8*-aG;g9fy$)600PBqbFG@ozpw36+emyuEO2F7R!tZ8x;KPIOQf9qH87%oheav}b^dM8G2r^@;(zFMb2-8H{*b2nB3{_)-LYs_bj^Ky&kwv=WF;n&+++xTE>=UD_@j-f!UsylSU;TiwN=D6#9Dvblk9<B!hFmruEAc%-$_=OY|d{l}8uFl(_(^k648)sh5DLg9O@~P>%M~72{^6}~VrbA^K`1IlFSpY*|8g-Cyg>LdcN*tYWb?Zm%q{31oB3H??MdmVtSDs*~79S-msqlPQ5bCVV&PW86Tx|S0_}=GKD6WUlxW@Ahs2T%;;qFF;NIrih`W3X|X?FR3Z=8Tg0ZfZza?t7E<m)htcX0b)a;Cn54;euVdlAE4pH84tcb1V2f;CA4A*~dzlB*$hk4=lPp1}U!sUB#UJBcNF123_JXF)P2s+By*|5SUg_AMW)n!Nx(gfM)CGTJEP5?_QaWwk?3lRqW*<$GeO1Nd1vEXPO9G?#2>Q>laG{d0I7Mp7Gy6$>VBg0bYp0<-GJRdpoXT}I2pIbz9Ln+PQ95v+~NVwD#jSmo^hYwpT>+sLi>R~h(^mB!F|B}=}PRRGrwHU_pyVY^!tf-o3aBUu}3WXMPD_4>P)?;!bjj7Cb*q5%TPGbD${Bl+I%U9;8l@y=iKOG1c%uSGR8IO0-F&E)dJsFw`CX;t&F<tB?0)I0PO$;5$e6{0il<2T(^YTsF7vrxeNScyjWo(AKaGKf(+W&yw^(u>d%0Vj`MBK|b;s$!%+2<Kf<bv==VZ5b1mNP|OXzeM)MC>Yjnrx)j^pRTgYzkfKp3|p+(KR5z7;`e|MzZ5U`O{WhuA=Sfh!yi7D`BOf>MXjv5=gNLkqk0{dl<~f#Oe;)ea3jNmn#yi`H}UP-#9^T2)7s-lHeC2gC}2n!(}{NYJfyt{h@$vZnr=Jeo+Kov6oW6qal4)}Mqy0|!J+ouY>wVPVdcQaZ!OU}ERjB^`K#e%Y6AS*=f8e12L^1ImbKPuB{{lJ<2Ez*7>h4@)OM3&h8H=c)WKxbiJi|@MFCzLa(6^<8SjF-y-n`t&x&`MuUK~uVI1B;hIa}%%Ly~p&Qq%oS1`obR`(wR=^#ztJ0~BRI6z#vy-88r!4KEMLF6t55LOzn1>3rKg6kEd4;%e%w!B`sBi5xFBG)m~C|FI9K5Y?E<s->#?+PO5jlTjj<wS#dxKTB3mbZu@gx2)f4+ly!NC6x<$RNEg)ZH+$edlkz`K{Oq<i45~hNJ4TVd^=o62NJuh6AiU?F&qeKrYS*3ctrM84K|Mit)S*Ojtk6{i*gF*=7yITtTd9RtXbim8<Rj@&`n8rmdcQKD$V!idG%30)JK2$e(O}gUFgcY10n5eowf&1NLVW93HAc&Sg}(^2`4O7jrI@l?!tj3y}aWdrN*m1`sDhLSP=^#s`s|n!rk9uGRt}TNZQF9>GHleq1ka9~UL>D{5(NIE|sMAsZp#$^Gn?d1+PTU$CcN=+x%!s1M=;dk;nZ>bC<G4I3K*ir~ixeFBF?b22%=A&``V97<~<2RiI;i+#v^Z#2!CU|4>%YT_7_L;Z<B9O|CV@AV;|PLTm<QKgW}7PV5q4J~WQ!gm|tSk1$rrfUpA;*OJ;qt(J=r+0U83@sW;LHN5Cw)IpwlQgK9IBQQih?!WL0xX4Z0@x1Qqhxm@Xm{8F^9oIF1E92)F|gtkNHDfEtpb*K{1&yq2WdT+xW;_jp|l#Ta5{@(s<qbE^^F_r>uCiRX^*h&?6B9(O3c>ON`9Gcys@@NHHKB2+Vr}d{8!ocMxg8FR)*koxNT-S&FMq7FdF9?5qn@rU9{IeNY_P5-em2xVRB}yD8*3G*iZE!6Gm!e+V~y7Z+{ot9qM{jFmGiTs}^(O_Yf591y1QEoNe$Qn5-&hQ5w9ix!Ns}@-4;8+E6mG_4%jIH0!3M8_A(3rWNc7h>i3)8HLys8N|Onq$;lCs)$fv%2ub!5=G)Ev)VvmwkPV3PJN1SnQ$pSYHOCPrgqqFC7qFD$xkvCU%Y_cYAv-RQ-~K0&{Cw70F@tQcWNi$&0AH;lu)&GWz4slxp!GmKT-%g(8hn{(76e_=Qd1OHB+_F%jtV2m<)9@S9iK=dJ}pZ0}G&*3)ynYZV7ua%qF6xeBJ}E-i%kxW#t^bQNlI)b~HIGzVmajQiY!(Jao*CJD|Vw5)l>$Ev0c1LgQ(5l?*>YJm%gL5vA2BF<RQ7Je(&w%#0csPCr_={wCJnq`(6UhJg|BITk8BmDB(%D1FRh(G=sUjbM?V{1JIkdSuv2W=IVx;GV@kkU+-F(1TcMvzxJ)`Kuy{P~)A3s0CUM4N)|(;2KCd!(J1=ujPrH9A(8bvI1;b%^boZc*AMYr;)N=)2l>IolC9xZ2;)=Z$(Tlm@+QlDz}3*O${51d6}6?D@yzFvp6nI5Z*MB$S0B-3CV!2H+>1d?(V`QNs~oRq2q@JjYmhb#I0L|<|Oq4uR*DS5vYJY440`ol-nMm$o;^JXq(4tcyOC9#*;jG?kCM3pJN0SowQa`S-8j}rJp16$+P8)pB;h^4h=@?21#@qxn*g?$8I8>IJQ%9>F`t{z3E&M?bKF?o!Oj@+3A;7?1m(SzA}_2%s))2+X(88v?5rX{#?mF{iDZgJw3(|OTNe&Z|TMdAhUSuK82B_=g*n5*%7SG^6?TJsLVd~Kg<gj7zfa#58|`#7UeY^QlSxMSgu-S53f-ys^-^>hdqj*Lk#>woGRfB;N#%RtZBd-nZ^u6so_!rrd=}ig>#Vt{C=SGYnz!Hm|v+uuxr6t_#sPIgRt;}rK*rGLK!sP&i4dp18HJlr3!=98u=msYz_5kW2k|9X$SAY?7&BPb5=DNzsE`PD};K#zj-J?R!4?}oc{&b3%;>)fK(|SGVJ=NVudnhGRYVCMiJ8W^`p8?jREl21Hvd<0lEkT{4}yizcj0=^$Yxp@2|t2G3W_!4Qo{&;^LJ+HK~alYNpMhmVE``*%65HG8<n%?K^*$kBhq(a8xMoSc#*T9K-2?{jq9f5B?Ad-4W}dH{k7DtQ4*M+DM%edk;5{i*X4l>zZID)Iwt+eFDr@IVSPE+sp6s)e53+Q#F2{0iD79tvbAC_Hh%c;eEb;aL~ecS067vTz<^{a{lS`;$PYMr{9u4<@3*1m;cbTLnlZ^W;u%bGj3iDp~PKuwp5c+g1^H*=MOiRP)bRYUs}nrX+jsEY{BaI>hwpEclM8AQ~d5zpU=*+%d69??DXQp=?F87rpXUUZ9yfp!uWA9l3l<{_EQ@=vjChO!E`7zgxPzav`y!LD@7ZAGL}*kuWooqeF+xyG_5CXF!oSy7R5x7aMLzQe5d`Jx`14v>V=zL505bjZh;DShn&(D>3!}Y@v$kQ>4i(Ej%U*qdzqQe0{_(HF=5Ic9?(FsAWhW=`vhO|L=pfs0PCgOq7)MhP0%PaTbLZ!<u`E}s+Z;Chx1=PUa4CRtm5QAhn*1whT?&-W4xL{uM=hlU8QgIs;JfYUVvg(9U;G?USDc)wNbg^v)B_9J>Y)QQNOLP&eyAZSsaO~d<FLn;prt-pwnWcp`sOBFdm8-#Fhl?Az-RQ-TxLE^Xbg>D2aATI~2pU5yd`guM1zVsEBpGcwAT@EQbH+^sB&u-;sI#THuG3bvpMavPPLfo;JjMdJM8yRMEp%hJH7BENUZFc1iy<W#ZqKJx)h1c)0<XQXg42N|t-C7|_-Lc)4m^-2*d>Uw}-`tXLVoLx_jkG7Zx%l@hPTWbWzztt&;f#?v7Z`_3qD9d8L)SMC6*eBKx=f>86{re)puQ#OXs)|yYR&q7PPk#_ds@lIW`G38Ghfv=>VR{j5Wt!u5$GZe|q5YbqzD5GH~ey^2swP|pn_dL+ZR+4m4?&Xt7Qv}^|-V=|yy;F?|>rUsfj#{iwT*v|l*Es5R^@aZ4IjFvJM!plofFY9fQGuTNIChgZqk2A4VD|AvWCScZkW0{du_6YfatV}+__q(D0z>4m3_-M$2&ijB4z7<iacjs2u(L75u-GXOxdE(?Ihkr>?`|?}q#Ig0yuSI{Q!VUjNqMoN41uSb*2O*5G@0Kmm$!~;)`bO-UeYjY5`_rc?DG|xakou%>5Xr56txs1+U|ms!E2|cLh#~^5h-bV*nLyP<Fy+o&vf<!FLVs{A!-q`jySV>`|uk9fVGmtp1<)i<BV!>UN%jmw9!+)Mb*d(vcpUYr_F=Te(DaTLLi9s(JkTjH9Sqty74*FG;~DjNnQce9??|60R%&=iuhB1(8P=woW78%pj?6y9KvW(+$P@ZYDQ+j{%n3CY5fd-AKgIo&L&+I{WXk{IRYhSN~m8r4H?j%di?y-U}t{_nTPLjcJRHGr?_aWU#RE6ZZ$iK%?&?@SSQMoKy6A4j*zM>B1=@#jSt+eQ(;&n5BWq=WN79R3;`r>zMb-d2aoy$#Aw|h`;7CrCwO>m8aKJ%W|RS5XGKPQR+Adk42#t0xxwK;CbrHd?+5Zy#;L$Pr&~r;S{0I4HESD;SJ|3t`nzAz;b!pBe@{)NQF`A;4~II`!^9jv@xdENzSc;X=G>z4hHAKnqyGv%-C2)M3+X8W<=2`NGp+$U9UQaBhsOnvi`gol<djytEHHKlm=Q+M<Z9>M2+kWg7DUiq<4REN&k`cI-yG<*9|kIwj?!xs9?0W}VQ(E~mA?!1w0y=co$J&r9Q<bU=#Nq2?7PO}j)4y9x)Wg)L=o;xcu2L3w4vYGMu5oXrEBX~6&})=?}`OS*r;NEy0g{a?1NjcPQ}Vp92uKMuJtSTHPpx~0|fglN-@KV<G9Hm!M$kBTvrWyXVlu!Zc97;*BuH!wbS339v)5e?zDT@KJ1RCM<?TxgQMcGb2L5a9E=Z+dz1X2*F8Dt?!2{jH~|Vt=c?|(q!>?oMYlIO?j0W;j(gq1>7?7suZwZs?Tx#gebwfE(K#vd<Nf@2vOg(0hsAYoderL_?Wt<>KcA+iBm'}


# Kiểm tra môi trường offline

Giữ ảnh gốc, TADiSR, DiffTSR và UniHIR; bốn OCR giữ nguyên.
Dùng nguyên Dataset models/packages đang có trên Kaggle, không cần cập nhật asset.
RESTORATION_GPU_CHECK='off' là mặc định; chỉ chọn build_and_probe hoặc unihir_smoke khi cần kiểm tra thêm trên Kaggle.


In [ ]:
report = collect_environment(
    run_probes=RUN_PROBES, input_root=INPUT_ROOT,
    configured_paths={'models':MODELS_DIR, 'packages':PACKAGES_DIR,
                      'pdfs':DATA_DIR, 'prepared_images':PREPARED_IMAGES},
    runtime_pythons=RUNTIME_PYTHONS, options=CHECK_OPTIONS,
    checkpoint_path=OUTPUT_DIR/'kaggle_environment_report.json')
print(json.dumps(report["hardware"], ensure_ascii=False, indent=2))
print()
print("GPU:", report["tools"].get("gpu", {}).get("stdout", "unavailable"))
print("Torch/CUDA:", report["probes"].get("torch_cuda", {}).get("stdout", "not available"))
print()
print("Các package liên quan:")
for name, version in report["packages"].items():
    print(f"{name:24s} {version or 'not installed'}")
print()
print("Pip dependency check:", report.get("pip_check", {}).get("status"))
print("Runtime VTM từ phiên trước:", report["prior_vtm_runtime_exists"])
print('Trạng thái báo cáo:', report['status'])
print('Input:', json.dumps(report['asset_roots'], ensure_ascii=False, indent=2))
print('Asset:')
for name, details in report['assets'].items():
    print(name, details.get('state', details.get('status')), details.get('required_missing', []))
print('Runtime riêng:', {name:value['status'] for name,value in report['runtimes'].items()})


# Kiểm tra môi trường offline

Giữ ảnh gốc, TADiSR, DiffTSR và UniHIR; bốn OCR giữ nguyên.
Dùng nguyên Dataset models/packages đang có trên Kaggle, không cần cập nhật asset.
RESTORATION_GPU_CHECK='off' là mặc định; chỉ chọn build_and_probe hoặc unihir_smoke khi cần kiểm tra thêm trên Kaggle.


In [ ]:
try:
    report['restoration_gpu_check'] = run_restoration_gpu_check(
        report, mode=RESTORATION_GPU_CHECK, options=RESTORATION_CHECK_OPTIONS,
        payload=GPU_CHECK_PAYLOAD, output_dir=OUTPUT_DIR)
except Exception as error:
    report['restoration_gpu_check'] = {'status':'supervisor_failed', 'error':str(error),
        'traceback':traceback.format_exc(), 'model_inference_verified':False}
report['model_inference_performed'] = bool(report['restoration_gpu_check'].get('model_inference_attempted'))
save_report(report, OUTPUT_DIR)
print(json.dumps(report['restoration_gpu_check'], ensure_ascii=False, indent=2))


# Kiểm tra môi trường offline

Giữ ảnh gốc, TADiSR, DiffTSR và UniHIR; bốn OCR giữ nguyên.
Dùng nguyên Dataset models/packages đang có trên Kaggle, không cần cập nhật asset.
RESTORATION_GPU_CHECK='off' là mặc định; chỉ chọn build_and_probe hoặc unihir_smoke khi cần kiểm tra thêm trên Kaggle.


In [ ]:
json_path, text_path = save_report(report, OUTPUT_DIR)
print("Báo cáo JSON:", json_path)
print("Báo cáo TXT:", text_path)
log_archive = package_gpu_check_logs(report, OUTPUT_DIR)
print('ZIP log gửi lại:', log_archive)
try:
    from IPython.display import display, FileLink
    display(FileLink(str(json_path)))
    display(FileLink(str(text_path)))
    display(FileLink(str(log_archive)))
except ImportError:
    pass
